# 14.9n - YOLOv8n controlled grouped baseline

Positive-only calculus localization on a conservative source-group split. Historical scores use a different split and are context only. One seed, no test evaluation. Every image and label is checked against the locally audited manifest.

## 1. Install the shared environment
Both arms use identical pinned Ultralytics and training settings.

In [ ]:
%pip install -q ultralytics==8.3.228

## 2. Verify GPU and frozen protocol
Use one T4 worker. Preserve the exact manifest and its checksum in outputs.

In [ ]:
from pathlib import Path
import hashlib, json, shutil, time
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import torch
from ultralytics import YOLO
payload = '{\n  "dhash_threshold": 4,\n  "policy": "keep historical validation; exclude connected training components",\n  "protocol": "calculus_conservative_groups_v1",\n  "records": [\n    {\n      "boxes": 2,\n      "group": 718,\n      "height": 640,\n      "image_sha256": "6f59039539fc5a46486fdc2ea742adce693f1c0fba6918141f2ca79e33cff41a",\n      "label_sha256": "003400c3f72780c02013c5794205dbc2ff5ce3e0e03688f01fdf840eb45d6d12",\n      "name": "calculus_-1-_jpg.rf.be58ef879db16db61cdf44111d954bf1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 718,\n      "height": 640,\n      "image_sha256": "ddde63a9092ea2f4664e44524da6e0cc8c154ae067fc71dba02d197f78eeda32",\n      "label_sha256": "4a450a549579032f1be1e9ac2cc0fedf9aebe2cc485a36cfc80561d5154eecba",\n      "name": "calculus_-1-_jpg.rf.dd05dc49c98e19ab5dd2c26a2014232a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 847,\n      "height": 640,\n      "image_sha256": "28bd357718a3e0b760fd57b3636cc59ff98583125fc01aeec75bb0618b916fee",\n      "label_sha256": "492d2cac155cf18529020f7a6b6c02de4565d1569c92eb723a3c23d0849f445b",\n      "name": "calculus_-10-_jpg.rf.16e22b81af79383b305ce7b3fae61ef7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-10-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 847,\n      "height": 640,\n      "image_sha256": "9d4122b389eb0af9bc67fba49284d634ad789b6b27bd7702e2dfe77e3dcc8fbe",\n      "label_sha256": "da4d18c96900981dd15ba31660ffffb7fbe0a3fefab6d0fbb846bfb08200a45f",\n      "name": "calculus_-10-_jpg.rf.adefeb50085a2ed555d51bfbfe4fecbb.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-10-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 932,\n      "height": 640,\n      "image_sha256": "9723638d2c4a811c09c61bfddbba47de9e0a700bf3f6948ca5859b5147226fa2",\n      "label_sha256": "02071fc323efe203b755e3fdb63c4400655456bb9b4d0f5b36561606a39a4bd6",\n      "name": "calculus_-100-_jpg.rf.1743bfd5c1a1f22c3e495ae1100acd40.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-100-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 932,\n      "height": 640,\n      "image_sha256": "9c17ff6938c87b063c122819157f09c26039e3bbccfeb4d0202aa60b4341faea",\n      "label_sha256": "5a6804f6f3130bc807fb1a714622c5bb44f0d5c082c34fd97da20b668c779cde",\n      "name": "calculus_-100-_jpg.rf.ae675ed095ee8526a1b629a3e03baffe.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-100-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 848,\n      "height": 640,\n      "image_sha256": "f8fe02d7c47ae71120b6e00792b6611ed9fce69959754a4f953233ab923d4594",\n      "label_sha256": "a79973596a26eeca823170fcef32a970933635e62ddfc7517ad37dd457acb5bf",\n      "name": "calculus_-1000-_jpg.rf.6a06bd10feb365f5ccd7a8e4210bc646.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1000-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 766,\n      "height": 640,\n      "image_sha256": "1d3d21047dc5f65023dfaf65b47b6494dff4c4647d37dd6b4a17f13bd4ceae52",\n      "label_sha256": "c2f27d03c850699e74009e6f92cf7d7a5935ed68ec419306c34233f6b7b673a2",\n      "name": "calculus_-1001-_jpg.rf.59dfd25b733c643dafd1edb9f354f29c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1001-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 766,\n      "height": 640,\n      "image_sha256": "2b9b3fa62c8ed47d723dd95fc8605fa13a0caa345f9403632fc82e3c3ce9366c",\n      "label_sha256": "38e8201e3cb0c1cd2b0c269deb18935b2214aea7653554fddce332dcdce4e6cb",\n      "name": "calculus_-1001-_jpg.rf.99399fa17a7c3145b9d556a6fcfba190.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1001-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 766,\n      "height": 640,\n      "image_sha256": "f74f732524d2d386ec0612276440e6d4a094b6398e39325e97f3fe0861d6d0ce",\n      "label_sha256": "eba33073c76b58c4af84d2fcdc8a7e4fa62b381fac03d7c1b2ccc47e77e56764",\n      "name": "calculus_-1001-_jpg.rf.ab9695bebee2c45197a0bad5bd2dcffd.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1001-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 849,\n      "height": 640,\n      "image_sha256": "66f7ede3b9923baa1e679e3271c62fcb58f97c39538780e1498667600d3cfc1d",\n      "label_sha256": "83c8db2573b1d0117b281df8ce6e3649ca5f13bbbb91aaf142d9445ffc19f6eb",\n      "name": "calculus_-1002-_jpg.rf.e5e5821319b445ebec9bd1737e893f63.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1002-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 8,\n      "group": 530,\n      "height": 640,\n      "image_sha256": "057607b577faa68d1c1e513748aa8624eb341a9f62d588a402810edae4bc74ce",\n      "label_sha256": "76b305daa673ad4795a4e621658ff2a42de7e43aca87c5adabc8d8486bc6b9a7",\n      "name": "calculus_-1004-_jpg.rf.63aafcfb6b203f818fd49307e741b1fc.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1004-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 8,\n      "group": 530,\n      "height": 640,\n      "image_sha256": "9cf0c64c1b3e77863fa81dd565f63094e7029062c4a9f30e275aca76f723c80a",\n      "label_sha256": "ced302f2da08c5187c6c315f3d82d1a1fdcbfc3d14abaf8be6a522f148d95c91",\n      "name": "calculus_-1004-_jpg.rf.6b038c0ab965651ce28a6972d8729ea4.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1004-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 15,\n      "height": 640,\n      "image_sha256": "e659ffd56827729bce1fe20c43945c694a7bc7cb9091e55a4234b18beed3f594",\n      "label_sha256": "f7377f628d6c2c58c3033dcc489a0c2a0dbb02b5bd2cd49df0b7d97efeb061a4",\n      "name": "calculus_-1006-_jpg.rf.01d788efe12972559f5aae9201075ad2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1006-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 15,\n      "height": 640,\n      "image_sha256": "796ca703ce751a276dc57a46cf0d12453f1b1d3df66a04a22234e2715693aeaa",\n      "label_sha256": "3fe5f719726d64f00f3104032d39a93add294d4ac9613eff96ac248b0b8b176b",\n      "name": "calculus_-1006-_jpg.rf.cc3cd3e34b2ee936790bdf8492ff02e1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1006-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 15,\n      "height": 640,\n      "image_sha256": "5fe49ed82bce71d08f83ec66d22adaed1eaf4076f8e1d9d0c949a899f8fdab67",\n      "label_sha256": "4ee104e769f52a7a945a00ee726201b28bf9a8e2b56a5ad01c79023934b72584",\n      "name": "calculus_-1006-_jpg.rf.d67ecf1b1a6352956abcaca12e7fa8fb.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1006-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 8,\n      "group": 532,\n      "height": 640,\n      "image_sha256": "cee3daaff5954442b35bd78e493e619ea291c332ce3067d0965e6cee40544907",\n      "label_sha256": "4e1f2c77e4e394b26e805267389941862705a1da7723356044f4e0a452dcf658",\n      "name": "calculus_-1007-_jpg.rf.e5874a12ce6f078ffbf54ea69afa32d2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1007-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 19,\n      "height": 640,\n      "image_sha256": "3f4e0c085e90e8444a068c37e657e06b0ee04621249332f95aac956d34fb0fb1",\n      "label_sha256": "61bbcf27fcde1b800b0548d1b3eb049be7905697e759b65d5dc2ca12fc68adac",\n      "name": "calculus_-1009-_jpg.rf.1989fd06383caf77c00eaae520c10534.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1009-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 19,\n      "height": 640,\n      "image_sha256": "08c20ed28b13436b0f8893efe7849c1011188f5b499bf87c1a7ac65eb7c447af",\n      "label_sha256": "804761c075bb1994bf111fe5ef6c0539064981dadd9d6c8311e46d673a3c184f",\n      "name": "calculus_-1009-_jpg.rf.3366cf8c18c7662a9db7087ba005466a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1009-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 19,\n      "height": 640,\n      "image_sha256": "9d961f3a818d1c42ad40de532a569349f08addcb58de48a2f8071f3748c81327",\n      "label_sha256": "be87d0b56b12f23243728317b6d2728ff934a71c788ca8d0f8db60a8c9bc1d79",\n      "name": "calculus_-1009-_jpg.rf.955c4bf0d6e37d01d9c42646ba37f4b3.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1009-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 913,\n      "height": 640,\n      "image_sha256": "c42f9c90e4c2f3604fdd123daf0cea09aa0070c3ef76444ee3991b9614d707aa",\n      "label_sha256": "c627bc746637414b1dc107c98cb66354e45a081d0f4f24617a9a7f6add3c9199",\n      "name": "calculus_-101-_jpg.rf.046bf0822db333b00efc6857faabe916.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-101-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 851,\n      "height": 640,\n      "image_sha256": "8b4bc6ab726104f0b3a9df1d1a5b01a2f9cfae5e2780addea7864e33db8cf236",\n      "label_sha256": "4f925f2272cb21e10c5355f9190af05c91b99a1be5b886a9efb1a7e9f1d39115",\n      "name": "calculus_-1010-_jpg.rf.8c7eb5acaa27115a64bf5a63c78a0d58.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1010-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 851,\n      "height": 640,\n      "image_sha256": "0c4e2a8d88c247cc75f53376c8087841fa8d9234c2f9b51894d9302769199c32",\n      "label_sha256": "a42afb607e54528b49e585cd263ac6fedf00511bcb1db9fbd93cba252dfce6d8",\n      "name": "calculus_-1010-_jpg.rf.9b8bb4607223e44a369ce45cd6643be3.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1010-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1009,\n      "height": 640,\n      "image_sha256": "b0d8dde9d40a5b4fc45bca533416a241c1d43c312a9836e3b8231962867b2194",\n      "label_sha256": "dd5e2fa5d2b6966fef71bdcdcedf19e7d4911ec1e82857fb086cfe7c39c07e11",\n      "name": "calculus_-1011-_jpg.rf.81bbdc8c59f15ed5eb3b3504c9a027b6.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1011-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1009,\n      "height": 640,\n      "image_sha256": "89cf46968d515eec0d94cd8806d13119eb08aea1d17bae8cd417675cdbdd36e1",\n      "label_sha256": "b73d6c0f9768b15fe09483ed00cbdc7209ba2a2ad50e5772ab59662a1e875450",\n      "name": "calculus_-1011-_jpg.rf.b5eff498dd0bebde469c329963f563d6.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1011-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 27,\n      "height": 640,\n      "image_sha256": "2f93704a800b1370f1e66010466296892accd865f26fb5b16ed5d32b330e18a7",\n      "label_sha256": "0128586b46c4c74b0b8a57da0852b219e1ef78592c46034777be41f7bbfba9d7",\n      "name": "calculus_-1012-_jpg.rf.111c0517359a7583c94318324dcb6343.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1012-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 27,\n      "height": 640,\n      "image_sha256": "a06508646a6580ade21b7130dca7e3068771113345cb67c1734338b26daafece",\n      "label_sha256": "e65fb5694d5e3f734b2f7b18adf3c5915bc71684c5714fd31879094e9ae776ea",\n      "name": "calculus_-1012-_jpg.rf.8f878cfded022df7bc6b16a78a32cb27.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1012-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 27,\n      "height": 640,\n      "image_sha256": "c029af001e35743f15ac65b9fd1ee8cf538e396802d3500fba00ae4329f97db6",\n      "label_sha256": "97c5a7440a1b1636d908c7b2e1c6713cc74329a0db9c4e9c225dee5f860bcb3c",\n      "name": "calculus_-1012-_jpg.rf.d54c63574928d670a64c9a121a23ac65.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1012-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 852,\n      "height": 640,\n      "image_sha256": "7b28f37b9b14c871fc1324abab8b7cddcdd51e2d987dfd517b335a0b7b001e96",\n      "label_sha256": "928d5d082ff7fa1ce010d8388943095df8d72742cb6f3a1fe149ee4f6dea1396",\n      "name": "calculus_-1015-_jpg.rf.21cbf1334e7688249ed8031a01819873.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1015-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 853,\n      "height": 640,\n      "image_sha256": "10e84ac0e252bc04a0007938551d5044b6d6fa94ffc4fb090582b67dca436689",\n      "label_sha256": "dfd7d2771d94e1c5e092ec43b8d085e9bdadf20a7cc63bb20e852c1d560dfeef",\n      "name": "calculus_-1018-_jpg.rf.f152936b9cdb4859c0bb726f6a398a2b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1018-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 854,\n      "height": 640,\n      "image_sha256": "4a391c7b086cce054515c9fcddc5b72e7509546ee1debf3d4286ccec802fcd07",\n      "label_sha256": "793a663ff39e66ce33032a01543f36115a1e09c7ba75a99f907195280968596c",\n      "name": "calculus_-1019-_jpg.rf.3aaf4e8ebd64ca2f1f0a336f6b8c1e54.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1019-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 747,\n      "height": 640,\n      "image_sha256": "37f0e74d8af53d1311e6441220006f573b345561227884c657dfaf964a369d95",\n      "label_sha256": "076f9916385d2e29343edb680997ad9a2ca5b9b2ec8ebfff549923546644211b",\n      "name": "calculus_-102-_jpg.rf.98c687ec21de69305307dddfd0296d73.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-102-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 747,\n      "height": 640,\n      "image_sha256": "0d8b7fb91b01561c619cdd1040e2155caf221600385e2abfba5363f496309a99",\n      "label_sha256": "b674c32de87d5d3c6be7972e85390c643bb101b20f14f5e272167a10a47905d1",\n      "name": "calculus_-102-_jpg.rf.adc20837d7178b8e31863ba5474c07d9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-102-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 747,\n      "height": 640,\n      "image_sha256": "6d0e1ddb2c5d2ada5815ecc41a4bac0f1b660c43bd406fc90931ab4c9d44d252",\n      "label_sha256": "b17280c3a2edc9ae0ca8b91ca96f7b33578a2ed5a04833e45d08e0f8a0568fda",\n      "name": "calculus_-102-_jpg.rf.f9f42831494d6502981a623b0fe8001b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-102-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 562,\n      "height": 640,\n      "image_sha256": "821ea3464b5d6c0d03fe30b0fb77c46adebff58e17202b332fd13b7359762973",\n      "label_sha256": "4368d62fdc873fc02576cc634446776532f5f67ebeb49505fe435355b60e7a25",\n      "name": "calculus_-1020-_jpg.rf.7302b2902da207a0854acf5d7c43b92f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1020-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 562,\n      "height": 640,\n      "image_sha256": "a6d0365601cce8e7eb5397dc94753a300dcfcddb8f9179d9bff3b60c1fa51c7f",\n      "label_sha256": "80f3c2f0b458385de6c19b0aa2b4a7b6190e4199c1a6150ae9069771b58a6844",\n      "name": "calculus_-1020-_jpg.rf.b9ab1e203a5d896841203d84426e4569.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1020-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 537,\n      "height": 640,\n      "image_sha256": "eb164a733b3098ce9bde3a42f055db88a270718a5d2585c47353bbe042447e83",\n      "label_sha256": "80c1816b36c1b479d7a1dae87a5449e901ab61b5c0f24c8bc6a6c1697489e022",\n      "name": "calculus_-1021-_jpg.rf.2e20978f70d9eae1451319ccef2a778f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1021-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 537,\n      "height": 640,\n      "image_sha256": "c2580b097537fdc6b973d5bf234dfe46dec4b96b11f83af65db95622d660c60e",\n      "label_sha256": "bca92a30a0b3aa1528c508603570321ad79e8429bf5a4eb1f86711f7743082e3",\n      "name": "calculus_-1021-_jpg.rf.cb53fe3406a62dbc73f18ffc8adf4e86.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1021-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 856,\n      "height": 640,\n      "image_sha256": "e76fa2b2873434ca390d89ae27bfe1535c2b2d3b22618d348976ac8ffb8ee9c2",\n      "label_sha256": "1fdf9b3448e7e662745750be6c2276391230ea61e433b7c60b9c6ae39b77a7ca",\n      "name": "calculus_-1022-_jpg.rf.354f11633db40424a7db0fb1ed735f4b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1022-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 856,\n      "height": 640,\n      "image_sha256": "a89e71b19fe06a1c66f2a8da274392d1e131402c16324e346f9d685b44e62dd5",\n      "label_sha256": "9dd14d0486c91ec0a96557ccffd7fed43ff6e2b26f237de09b67f59ea2a3c1ad",\n      "name": "calculus_-1022-_jpg.rf.5b80c67fe46aa11a33550d74554e3f43.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1022-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 858,\n      "height": 640,\n      "image_sha256": "47c6d170950e3e9997c3a1afe8b3baea36991b80fa3df1d2f60cd9c4ad9f1935",\n      "label_sha256": "c74e5ac4776a027ab18011cac4cb42d697a2d0d80c186744e4f62828e571c0e9",\n      "name": "calculus_-1023-_jpg.rf.2742df728240e3508795c68b8ca3e36c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1023-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 859,\n      "height": 640,\n      "image_sha256": "d0d46d6082dc7cf3b10fd1eac25d1d615c37b81c8f724ab78fba49932076dd61",\n      "label_sha256": "197f7c61bda8f8036709ee3ab204357c17e7166fd11f0bf58e86b91ff1005c27",\n      "name": "calculus_-1025-_jpg.rf.7b7a5974a78c24dbd400547b49078d01.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1025-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 859,\n      "height": 640,\n      "image_sha256": "bb0fb7ac134ed1a24cadefe70ba25b2c4238658a257dfe2432dcf2dcb2cb9a60",\n      "label_sha256": "4b6ac08c6e08ad651b6f45e16d9308a7be886aa19cd2e7453b4f0f8f57b7c8e3",\n      "name": "calculus_-1025-_jpg.rf.9eab335620d8721cce627cccf53bdac0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1025-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 1013,\n      "height": 640,\n      "image_sha256": "90b9420d4c5a4a711b42feb2feca4178464879329383bb7589541ae7d112dd62",\n      "label_sha256": "854baf2a60b37f8b6abac65038b7975ac0fd077e51b9f6311b3438e107d0918c",\n      "name": "calculus_-1026-_jpg.rf.2f8c9e2b25c83dc57f612d8b91dc44f1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1026-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 1013,\n      "height": 640,\n      "image_sha256": "ce25500a02d8f64a81bb1b59787d37e05ed65086bc90b1495f4a0fb3e217602a",\n      "label_sha256": "98a1332e86964678653747c01007ac1afac5f7290995cd70c53599f622594ca6",\n      "name": "calculus_-1026-_jpg.rf.73e16f1234a9b0f44024d15c9159a3ef.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1026-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 1013,\n      "height": 640,\n      "image_sha256": "0c7b81c602fefa131ceaafba364d1a6b664e0e0f6756672f78a0bf3a7e6ada74",\n      "label_sha256": "adfe18265673b77ca054b9203d4b60a8b899360afea3015b0deb2c2d2e1d49f2",\n      "name": "calculus_-1026-_jpg.rf.d3291c3727bffc3e0456501a1d4ce91c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1026-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 48,\n      "height": 640,\n      "image_sha256": "a18f1b9467f06d6a31881d95e238ffbc3c24f4d94adc4a7b9830f26c9e7e99c1",\n      "label_sha256": "eb34c9ae25282a382f5ad5c5b8d0415593197e2a2da8448c3def0dcb270d60bf",\n      "name": "calculus_-1029-_jpg.rf.163beb6ca7cd3f1d9793f2eeea3701f0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1029-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 48,\n      "height": 640,\n      "image_sha256": "efe36c0b5285f3cf640737855bac9d2033df68d05ea11d251ae0b7448b652fb4",\n      "label_sha256": "034288e93d547a3cf04ff2511825f6f1da4dd7b4afddc8c240a3e1762c504721",\n      "name": "calculus_-1029-_jpg.rf.5df5e1f3f9c917c7a217357db423134d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1029-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 48,\n      "height": 640,\n      "image_sha256": "1fdc66e3e60d74fcc1bf69996fa7b4e5d91d77269950428e4810d9aa17260c8c",\n      "label_sha256": "d0faeff3bce3e7af2099ab66d481981abe6d5fb4c5d59ccceb8959d033b2958f",\n      "name": "calculus_-1029-_jpg.rf.dc5ee3ac690a64f4e1be692453258303.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1029-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 860,\n      "height": 640,\n      "image_sha256": "8b2b03e07a838b2e97789664b96d5634e3bbcfc10fdd357132d6fa402e4bad64",\n      "label_sha256": "2c9f0b8022d13fba255eaa2b39de763ddf683fc3052b7567ee0f9788171533ea",\n      "name": "calculus_-1030-_jpg.rf.000aab7206ba3e12f91acf1d6c7b336a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1030-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 860,\n      "height": 640,\n      "image_sha256": "bc131885019d9cdb778e92cdc06f5ba64af68e35255d201ed1678c459aec3e91",\n      "label_sha256": "d6ca59fbb8a97d79326a9f8aba741d46e467a2545508e1067c1a8b40cbba6b6d",\n      "name": "calculus_-1030-_jpg.rf.316e4a3a31dd6f06c87b80f156d3b29e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1030-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 861,\n      "height": 640,\n      "image_sha256": "43e337690db9c2b9a969c32f32c695a97c447591be77a3c6a5f9ff3d610a117e",\n      "label_sha256": "e8827f3331d633582d135883f436a67a984a050c24acfcb59402b0d313b864fd",\n      "name": "calculus_-1031-_jpg.rf.c4eb04eaf334a5d16cde423a90b094fd.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1031-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 54,\n      "height": 640,\n      "image_sha256": "4ce2fa3b6e73c7f367298c0187b45e4af0ea8faf0ced6ac698ba10db30d0b5a1",\n      "label_sha256": "06e0b79b37cd8dd0a1381348d66239b157fd9063949c9d8c17528d8bba0323bf",\n      "name": "calculus_-1032-_jpg.rf.25d2007e176e01fb665b289551170210.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1032-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 54,\n      "height": 640,\n      "image_sha256": "0679ca14d75b6c98a8e907063f24287d7f5c9ce4b47221cea341bde79e442981",\n      "label_sha256": "ddc84b3aa0ad7c3fb7edb2457c5019a64a529c5ee425dab5a443c259df1042e7",\n      "name": "calculus_-1032-_jpg.rf.39c249f4f2f0ac2483f491c4a547c2be.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1032-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 54,\n      "height": 640,\n      "image_sha256": "c07459db05d08ee6b53e6357c976f9946689d5d0f12a7d5c84377c758f110316",\n      "label_sha256": "63327ede1f79fd0ba3e6b380a93c597c9f39563402a625fe5329626eb1f98d0b",\n      "name": "calculus_-1032-_jpg.rf.6bdb4621d44c235b153334cb01ea607f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1032-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 881,\n      "height": 640,\n      "image_sha256": "f85078adb5ae812f39e6bf6bcd5e31b53e0c3d0b6a87e7f9c51cf11ea4d39cc3",\n      "label_sha256": "626f6ea1e3525528d7eb74ae2e70135a45b452b628e4f19370a7e4e2d18c792e",\n      "name": "calculus_-1035-_jpg.rf.80b24f362207dcdbc03bb344425f8aa0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1035-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 881,\n      "height": 640,\n      "image_sha256": "62ea0f7fcdd545e87773d1facb5d296887194b985e9ef07a3b386bae14980c19",\n      "label_sha256": "080c875125812998a05ef6d8564c09d15353ed93a3af6e662a1458bb87fe2287",\n      "name": "calculus_-1035-_jpg.rf.9fa670d9bbaf815f0f0c1a3e726fe18a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1035-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 864,\n      "height": 640,\n      "image_sha256": "1491eff7ba0b974ca47914abcb69f5a3da7681d65cb5d05bf19a388174a30834",\n      "label_sha256": "8f7bda92d71d2f9160cfdccfb894b22352ca1ac9c57d2ba5071d2b815b816f0e",\n      "name": "calculus_-1037-_jpg.rf.3580e60e74c13499e1f89f41ba0dd114.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1037-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 864,\n      "height": 640,\n      "image_sha256": "fad1bd1b9ac331c1767dc0d69a6e21e034e8bdb1833316c06e1d8671304428e3",\n      "label_sha256": "674a9611c3f3ab75ee9ad5925b788754568f8193b340806d5f03f075eb34e6ed",\n      "name": "calculus_-1037-_jpg.rf.6063a712997977dddc836b7ad229bf39.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1037-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1011,\n      "height": 640,\n      "image_sha256": "6824fd79ffe95197f08aad7f15a549e51ad3bcddec9861e8b3ebe10a6a904807",\n      "label_sha256": "2aadaf286bd04b861ad0b4fffb172ed41bb2fee9706d4e191ba0f74bc402a335",\n      "name": "calculus_-1038-_jpg.rf.06217c027fe0486067d0b8c97abe5e14.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1038-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1011,\n      "height": 640,\n      "image_sha256": "06cb8c7641972dbedfdb0f8d88de3ec8d2364e132a8c305540cb1f8e084f0de2",\n      "label_sha256": "1e7cbca626048f68692ba6616e91d6a279da138dfb5363f69ff528509bc939de",\n      "name": "calculus_-1038-_jpg.rf.549bc3953dea7acef340b57d3aeed905.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1038-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 867,\n      "height": 640,\n      "image_sha256": "c9f5e534bc8c2a58f10dde65595681177f0207cbe6725d327befec4636c6cef0",\n      "label_sha256": "ff8c7c27e5aa2d3a0f54c352443e24bc9199ec01b5600946ba150beb710cdf2a",\n      "name": "calculus_-1040-_jpg.rf.4b44846511d41e0e1815606aeba75907.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1040-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 881,\n      "height": 640,\n      "image_sha256": "c54e02f07c4721ad462a1be72c2c406decec61b56d4a283f000cc5be1e352d31",\n      "label_sha256": "b29345ec934c8af7fe398586bc4e9e8673524cb07c8478ce5b40d237a029fa45",\n      "name": "calculus_-1041-_jpg.rf.10da0aa508fe10dcab7c528a7ec1daf1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1041-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 881,\n      "height": 640,\n      "image_sha256": "a24555c649faf3a1794fb86056605eea4ba844c49ecd103435de84e43d3f3708",\n      "label_sha256": "1f0c62847da9e2e2f29cea3403bb18002be740d94aab133469748e58aa3fe1f9",\n      "name": "calculus_-1041-_jpg.rf.27f13bc1daf969232b28baec3bb86698.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1041-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 881,\n      "height": 640,\n      "image_sha256": "a8f44aa96b74383a99b98e363268b315d9007827c58b55a04d6da4232f63a061",\n      "label_sha256": "d20cbc92292ed119803dfd09bdb7711f4c71e95bd1fbe42ccb7e5191df88b55d",\n      "name": "calculus_-1041-_jpg.rf.b847740e0840fa34ba4c2317ee938420.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1041-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 868,\n      "height": 640,\n      "image_sha256": "468cea0daf33ec89950f7b438505d5070b9d6a39b2cc1e6d54a1944fa645c158",\n      "label_sha256": "5ff2ef78b035eb68aedd6481d061381d7cecb229af9fc3d705d24e4e88c145ea",\n      "name": "calculus_-1042-_jpg.rf.a3219179efb5e02eb99f80e63a6a7fa6.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1042-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 868,\n      "height": 640,\n      "image_sha256": "693421a62901a41343eefce1d07141363c1f80b7c5474eb0730d00384c6aaed4",\n      "label_sha256": "18d910073722cbc976a711f6cc019795a8ddadcb9f86bf14598373d433854ee0",\n      "name": "calculus_-1042-_jpg.rf.e3b207552ea9d9d0490a8ef6d0d378c3.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1042-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 548,\n      "height": 640,\n      "image_sha256": "e0db59f4f44e0564b81518249e89a0708d751b74012b117be3da1f5345c1e470",\n      "label_sha256": "c5d0b5152a46c1eac4f312b70f07010cc5772361d9c52c5cd953fbc88692c3eb",\n      "name": "calculus_-1043-_jpg.rf.547de2db5b9105edacb126d97bd7e53f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1043-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 548,\n      "height": 640,\n      "image_sha256": "8d1f8806ea2c989d88b986010bfcce43ec4ed865c036bffc05be35ba496a6efb",\n      "label_sha256": "18a6e4c6228d1219f40787dce0804034575e19fe0c6fdf649590ec7fad135217",\n      "name": "calculus_-1043-_jpg.rf.56803c439ee80b351df8e56a6a850bd0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1043-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 548,\n      "height": 640,\n      "image_sha256": "bb0201e61a44667c4184d77fc2ff563ab9d7fe732569bb3f389a4dd930bbf1fb",\n      "label_sha256": "6e99bb1daa102113a6328417bac769f5674ea1d437165f0f270a58707f691523",\n      "name": "calculus_-1043-_jpg.rf.c9508b375ff5ff16b71c49741af1b725.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1043-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 570,\n      "height": 640,\n      "image_sha256": "ea5b350dff70dc3eab1134a491deb3a942894d33a1b1adcee00bc92d5770bbd8",\n      "label_sha256": "bba6bd717beaedb3d08ab8ed9801a28bf8d9e795610816692884aef2ef5e8f8a",\n      "name": "calculus_-1044-_jpg.rf.052d301a4974613971511df44063e54e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1044-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 570,\n      "height": 640,\n      "image_sha256": "c3c9e7e478c3bebec0ac883e9288c03a7978af687abcc1892e073f9ab78be686",\n      "label_sha256": "071f4caae408dc216081f6c4b3bf5525178248238aac3ba909342570de95460a",\n      "name": "calculus_-1044-_jpg.rf.2d7c870ec7c72664962dfe27bba4cae3.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1044-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 570,\n      "height": 640,\n      "image_sha256": "0c2509df9818c2a5a602b91ef2e802f3dc039abed0f4d336300a7be4e6fbd85b",\n      "label_sha256": "6c1e798028231615d77f17def2d080262dc8da376018095ba9e72a70943eb83a",\n      "name": "calculus_-1044-_jpg.rf.3aecbad562f1d5cac6c55f450354d64d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1044-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 568,\n      "height": 640,\n      "image_sha256": "b231ad4d42240984a6bbf6ffeab7b35b87565666e73fae430666ed785f039ec2",\n      "label_sha256": "4f0030caf908e0f8575cc1f12b40815c9885893d2049f2bd86d905f9756f54ae",\n      "name": "calculus_-1045-_jpg.rf.7537f25c534b56a5288f93c7b45ce3e4.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1045-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 568,\n      "height": 640,\n      "image_sha256": "7b28be91fdd61898250c6c2f60cdbbef8a65f4b585f8b3f22f24fda861a82587",\n      "label_sha256": "0573934bab69dc1a704d0659c5da26ed1d9d65a8b01066e2b9b25e4aee89e672",\n      "name": "calculus_-1045-_jpg.rf.c98df37d4406d048bf6e3bb59916e52c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1045-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 568,\n      "height": 640,\n      "image_sha256": "b093e9862f5b358a8d5a4fd6a93761ef2bf4f7abe5f0fd05ed2e618000cb38bc",\n      "label_sha256": "29b1c666f5a533bae4a6f1ed60c974eeb027a96d6ebdc55540b9dcecc91ecfa7",\n      "name": "calculus_-1045-_jpg.rf.e69f4d4f5d62f14a25c37b05ad257030.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1045-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 1018,\n      "height": 640,\n      "image_sha256": "412cb700a75cfe0687a9dec8c0b1dddd1b188ffbe5155ccac3d36c8daeb6750c",\n      "label_sha256": "e3fef60d778df29b5f9155f1c28dbb9bbb80a3fe038b0858bddc13f5453b9e04",\n      "name": "calculus_-105-_jpg.rf.9763ea5f4c4996144ce73967873a58d4.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-105-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 79,\n      "height": 640,\n      "image_sha256": "deafe28aa9e064d7ca06bae9ffe0181a412ca64d8baa1ac065a71a67da456cbe",\n      "label_sha256": "ac0e5e81aa0b38386298352e2345fce3f4e0bd9d099787286217498214ee19d7",\n      "name": "calculus_-1050-_jpg.rf.7a8808824c85f02da253b6e16029efb0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1050-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 79,\n      "height": 640,\n      "image_sha256": "b2595d324a9aca0b35567ed9743ab825a7e3f4408f5c5f51c1a0a9b4911a2cc5",\n      "label_sha256": "430af90f26bcbb24ad9b1589b807878a0b82f965b10441c72df21924d8cecd53",\n      "name": "calculus_-1050-_jpg.rf.9d60e64e311fefff8b44ea8409fd8272.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1050-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 79,\n      "height": 640,\n      "image_sha256": "cdd4df98947c23b0dd00b3ccb49db78213cd51efb617f2fb99be6cc0664fd9b6",\n      "label_sha256": "28705e95e4b03e358d8cb32b2750225610c01e2b2b61d40e6da0e214394e3de1",\n      "name": "calculus_-1050-_jpg.rf.bec5c302120f8a9b6eeafaea71e98170.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1050-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 82,\n      "height": 640,\n      "image_sha256": "947e45c3d2b8181d13978ab6887216d8d672a1e738d75a578e44671ec48d743f",\n      "label_sha256": "607b50c38f75f8584a6442436beb39fe37edf56ec827e04fec4fde6fcb62852e",\n      "name": "calculus_-1053-_jpg.rf.7b0d09a22eceec08df0ac0e39929d617.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1053-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 82,\n      "height": 640,\n      "image_sha256": "f7b950cd24491159936e93d1cd9e57063fe7ca3b0b2ac332c706414b6e5644a4",\n      "label_sha256": "6ab6cf55a4f0d3d08093008e38cc0b67fd92847b767f85ccabe1d2c270c5dd58",\n      "name": "calculus_-1053-_jpg.rf.f73de09d19cb249b2adca3ab7ca4b722.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1053-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 82,\n      "height": 640,\n      "image_sha256": "c6702837f3d3be0c370a915cc50fa20a57f7d50179e8fd81c1c07f64fdaeb7d8",\n      "label_sha256": "01dff92c6af87fd13f73f78dfa0a683555df7a2e5ea794c513edc2ca1b3ca5cd",\n      "name": "calculus_-1053-_jpg.rf.f9e37749ce34d24fd8e14ee71524da15.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1053-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 869,\n      "height": 640,\n      "image_sha256": "db76a7f08e03cada4195a306a09389d888e6ba130b3571d6b429611e849eb47d",\n      "label_sha256": "f79222c225bccff5359cfe036581c3ec66f82e4ed818f0fa9bbab638e165a4f6",\n      "name": "calculus_-1054-_jpg.rf.49cec32d3aab2fcf12030acfecb83969.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1054-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 869,\n      "height": 640,\n      "image_sha256": "b99325dae52446801f30aa5f716aafccdc5c2a64714bedb43c682a515db72d8b",\n      "label_sha256": "1fe9a8f11d0202ec9968f5fd6f0a19e9262adcdfd088730a407c19e250dd7e28",\n      "name": "calculus_-1054-_jpg.rf.f43327cc7835c3009f510a48a3283a96.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1054-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 874,\n      "height": 640,\n      "image_sha256": "385ec2cf59ceb58afc8356d2ef869f0f1b4e1938d7d0356d244c590582889a71",\n      "label_sha256": "821a1b6eb92587491147b17e115bd3be44c1e2286a69778aa6d348f4c6df971e",\n      "name": "calculus_-1074-_jpg.rf.68f36330167214a59c8762ab517272a1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1074-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 874,\n      "height": 640,\n      "image_sha256": "f5d1e5c96ae92b396f683f2f866090af16bde24e64ce53c710565dd844abc07a",\n      "label_sha256": "d5f6d4cf1d27de6d02712faff2546ec095217c5cf3886d1a5a5696bc4c6b4348",\n      "name": "calculus_-1074-_jpg.rf.f05375b5fe9dc97a1e73649a8bab865c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1074-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 558,\n      "height": 640,\n      "image_sha256": "67e40f2d1728e1f9a04b340d4bb2b745be2f4d1702ccc8aa0c0a0cf8f589500b",\n      "label_sha256": "ad14d5663350eb3f711be90a02ee19ddb0680679ab6443c8adcb15a70ae8193e",\n      "name": "calculus_-1075-_jpg.rf.049a769ed2921a624fdf88a4670d6fb2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1075-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 558,\n      "height": 640,\n      "image_sha256": "05e3b5d8a02078e61df3bcf02cb030211e25ebe311d611a92a7c782f1f4025d6",\n      "label_sha256": "91355ee18f6b09a8272016ae96f10737e18f81c986f4881722ef6b536324f418",\n      "name": "calculus_-1075-_jpg.rf.2f070e52b1feeb370d0b7b9b89553e46.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1075-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 558,\n      "height": 640,\n      "image_sha256": "eb3267e6c9b988740d543af0912543a8a577e2289786ba7dc392b52bca8e70d9",\n      "label_sha256": "dac2b6176253190be4a6906581a87f8f0d794fe8d6712c2933d83a00c2199a70",\n      "name": "calculus_-1075-_jpg.rf.c42b6d6b75ea9f1bde7e5257db5b9355.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1075-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 875,\n      "height": 640,\n      "image_sha256": "587022c3e4a54e19ea353d8a311662b05423f22d888d36ee9b4415052f6e8ee6",\n      "label_sha256": "b5d3a617782d9b21d748c95ba2408cf12a2f8b56c1fd252d3f06c0daf1d4701c",\n      "name": "calculus_-1076-_jpg.rf.269205f42bd1046e172a6f464a307eb7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1076-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 876,\n      "height": 640,\n      "image_sha256": "26b7dc41022f1e7aedb79e40ef73958e4f9660969575aadf0f1cc310c18f55d9",\n      "label_sha256": "a3c03d4aff1639fc24dda5b9d5b84a6d0558ed80ba3f465d78cfcbe927f468b1",\n      "name": "calculus_-1077-_jpg.rf.b932b79c53edb9961e0ffd7a573c5ef5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1077-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 93,\n      "height": 640,\n      "image_sha256": "775a6ce3b1022832c1b93a409a94caf72a22a72cec4ef784e7264cffb3df0278",\n      "label_sha256": "55b0d36827a41869bdea8221dcecd985689d834b9aca1171034d995cbe77a662",\n      "name": "calculus_-1078-_jpg.rf.140118d2c3b5c13b89c1d94b56b55845.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1078-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 93,\n      "height": 640,\n      "image_sha256": "89735c2e6e38eb027ba50fc54a31ffba2e6ebff03edefcd76d9b1339a6cc15bf",\n      "label_sha256": "254031376a8ff5fc0b8928fff0f38bac0669d1be9ccd33b017281606a2b75524",\n      "name": "calculus_-1078-_jpg.rf.261e3bb9819b1b8e6ab23d44ad2f8bb3.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1078-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 96,\n      "height": 640,\n      "image_sha256": "5e30d45fda4853796bcd7b887cb08b808d76b505f56a861ffaf81958ac9f7d92",\n      "label_sha256": "6770ade571cc7b6c24a9942a45fc022b68bef3175c2f5da122fce861d94ee75f",\n      "name": "calculus_-1079-_jpg.rf.96ab4147d5373769a150987680afe102.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1079-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 96,\n      "height": 640,\n      "image_sha256": "3daffa3e90a41d245f504795270acef8d7957805efbc198a9eedf30c05c55af0",\n      "label_sha256": "df946360c30dc85da67de1d239b847dae95cd64d0e7c63f824f1d6c4c272160a",\n      "name": "calculus_-1079-_jpg.rf.acfe1eabf51cc737a71e5d19c53d023d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1079-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 96,\n      "height": 640,\n      "image_sha256": "af524261d409eba4aba420bfe77ec25417d3458b7c044727f13f0cf50a67837d",\n      "label_sha256": "5f5876b4579c92b851682b48b7f19d979b53b3a9e8a5bd42487c8a23265b6eef",\n      "name": "calculus_-1079-_jpg.rf.f1841c7e353555e3720e8849239d07ec.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1079-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1042,\n      "height": 640,\n      "image_sha256": "0f0f8341edfc4d64c9c88f3e51989c3be6c39cca27d415fe1022353e593fd5ce",\n      "label_sha256": "49ebf7201a20548dfef325ec7058dc8716c2dd79c77f2035b1d19a9dd1905bc7",\n      "name": "calculus_-108-_jpg.rf.485a50791354acb4dd9b5d7e5f3c2c52.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-108-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1042,\n      "height": 640,\n      "image_sha256": "cf3400a7395bcb5d99124e07c57cfbd7cd1bca3d2d5975414a3d125fdaa9baa5",\n      "label_sha256": "a6c72b9b8b240fdab79cde6acbf618feb7a33425b1d4e2e5a30b4c28a231b822",\n      "name": "calculus_-108-_jpg.rf.55399d3a554497c123231a4f47732bfa.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-108-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 101,\n      "height": 640,\n      "image_sha256": "8f22c51cceda6a232ba3b7c376f0d5303888167dbfeb68fd13dab9f615996e90",\n      "label_sha256": "03745832959609057adec5dfa11241c1cfd17ff714773e382de7a699dc9467ed",\n      "name": "calculus_-1081-_jpg.rf.07e4fc37b1445fc4610d605fc09ced9d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1081-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 101,\n      "height": 640,\n      "image_sha256": "5c04985108f3e4dec7f1ccf6cffb73a838b7d232fca5d968b671de651d801cd6",\n      "label_sha256": "7396542560a496fe27bd247efc1684a71d14038e5c94f3fc991afb0866aefcb6",\n      "name": "calculus_-1081-_jpg.rf.bb9f410118b4ce9f16fd8b005ef160b9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1081-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 101,\n      "height": 640,\n      "image_sha256": "c3c4db11ad5ec9d30530ca399b9fb4ce272e5db329fb7d0e1fd96883b1394c14",\n      "label_sha256": "4282f2c01c8c31edab5eb4dc9a693f93e3f2c645f209ce1295186cf40f16d90b",\n      "name": "calculus_-1081-_jpg.rf.d93701025f364a7ecaeef0fcfb0b6703.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1081-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 102,\n      "height": 640,\n      "image_sha256": "89a316fe3ab3d6324efb3631b5c42beb9cb80de78cb07867f8e2568b7a55c0ac",\n      "label_sha256": "181ecdae000d72dd85920557375733134e1f711d3cc791ae83aecf53725ad549",\n      "name": "calculus_-109-_jpg.rf.e57cf5f6c22d0f5400863397be15b92c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-109-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 878,\n      "height": 640,\n      "image_sha256": "8d2c82016acfcd3d5a1e2cd6eebfeebe40f61363aa0b6c4d462c9cc371fb7e78",\n      "label_sha256": "d2484d5a80b6a28afa62c268631b9ed24139a9196aceeca9036e49c54f06e022",\n      "name": "calculus_-1092-_jpg.rf.bbf24014f3109f48420096a898887b0e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1092-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 566,\n      "height": 640,\n      "image_sha256": "3c55a6b3927cd5dc5f316e3aad2d2a0d4488ee3b180ba9218647cc0baa96872a",\n      "label_sha256": "3d2e32fd1566518dc56de33a4b365e4dff823b59fc147f54fca328a158c5cc65",\n      "name": "calculus_-1099-_jpg.rf.4976632cdaaa96f723e23cf62bf7a246.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1099-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 566,\n      "height": 640,\n      "image_sha256": "656fe3b30a078571b1ff3c4bac04c58057e56793a10eca7e015258154fc29332",\n      "label_sha256": "771f86bc3b2127ddf58927feed3313ca6c10aa14a4ffe2f36324e98ef5310003",\n      "name": "calculus_-1099-_jpg.rf.532d84598b8ab983722a1283908f83f1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1099-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 618,\n      "height": 640,\n      "image_sha256": "481859a418094ab510c91035c73147860ced998f4d9b31d3d966f450b794e57c",\n      "label_sha256": "a29c8d97fb93f18281bcbd7ff07422bd3ab6c68add98a8f09f2961ebf720d98b",\n      "name": "calculus_-11-_jpg.rf.2dc89fcc4b2fa22d36434cb999a94cfa.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-11-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 618,\n      "height": 640,\n      "image_sha256": "018718503fe050afbda84d1df6b67216878c4ad030d2b8051814f1d08079875a",\n      "label_sha256": "25218a200ea4b93d76abb0cda602a31ceb17a93426890502a1ea9d387ca35526",\n      "name": "calculus_-11-_jpg.rf.a668b3de140fbd6d57e62f6995011ea1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-11-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 618,\n      "height": 640,\n      "image_sha256": "5b3dbdc3420f58ce5aacaa95375b3ac7915b7aa82ea5c46ceeed0305b49c5785",\n      "label_sha256": "44d8677752a7fecf7318003f30e37ec50d1180e78dbecbb26517cc394c8af631",\n      "name": "calculus_-11-_jpg.rf.c01694bb4e61a647bb6787aad5c62649.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-11-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 881,\n      "height": 640,\n      "image_sha256": "142c976cd7043e138b7a215451a2f2dafb03e60bdfaa3893b82e43b0f63953af",\n      "label_sha256": "65bf3ba2837352816d769bd8d06e84ad4871f0499d550f053d28061499824f3d",\n      "name": "calculus_-1106-_jpg.rf.c20c0ba5ae14410d33bead0e71238261.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1106-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1015,\n      "height": 640,\n      "image_sha256": "4b08267435247f9aa94b8636198a9537adeafd5fae83f7d8288062b0aa2d6eae",\n      "label_sha256": "6dfb22f2e1b377db27ab32da806a2a53828a09010d45bd7989005d86ecd67c11",\n      "name": "calculus_-1109-_jpg.rf.0bbf51cd4604e96934ea8677abe3c1e8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1109-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 203,\n      "height": 640,\n      "image_sha256": "7da4184cc63120c4dcab4903b7fc1213db18183a8d0970eb19a750b8738f1e17",\n      "label_sha256": "2dd00f40e15f450cb601020f26e5238a5ef75a5013fba8906ac1b88261945ca8",\n      "name": "calculus_-112-_jpg.rf.24642d5f7a8d8a55cb4ebc841fee8e16.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-112-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 203,\n      "height": 640,\n      "image_sha256": "55be736244e6047d0fb846b7cfbdd52baabbbf684b810009321f1e53794bb620",\n      "label_sha256": "154e4461476c20c089f00ac0f95fd049275a829f7abd6c07a1d70c8a80cbe496",\n      "name": "calculus_-112-_jpg.rf.84e93e8185a77d799e8354bb87fe6158.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-112-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 203,\n      "height": 640,\n      "image_sha256": "d5f1dd0de675abab39ca28300c0c6a284f59b417644c0eb4047b5c4fa44ddc19",\n      "label_sha256": "01d5b827d2f8934b0dc3587e8fbe7b17937aeeeb4fbdcbe02651afc814cda67c",\n      "name": "calculus_-112-_jpg.rf.90696579c2532b8b2ee551a725be2085.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-112-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 884,\n      "height": 640,\n      "image_sha256": "8a5eb8dc5098167539c941a6670b1d817f88d6cce00299e815bad2ebb71262c8",\n      "label_sha256": "6969b87fe49504ea173d5a519c3261cd0ebba00d61ad40f3c93435698fb945c6",\n      "name": "calculus_-1120-_jpg.rf.1b7ab51f3a1dd44534d0c6cc03905ab9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1120-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 884,\n      "height": 640,\n      "image_sha256": "834846c4a2c62dc9136f5d1793d3f53e57c20e12787c0ee7bd333d831d968689",\n      "label_sha256": "9d314448ddf0306451f24714856612ec145a9625ce6045f0d8f47ab3fc630038",\n      "name": "calculus_-1120-_jpg.rf.fcf2071571db693991a87d45751c4f2b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1120-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 886,\n      "height": 640,\n      "image_sha256": "a35fd2c387cc37fe22c9e9ec16f74564087d23b6e0e55c26767cf5b111d14fc4",\n      "label_sha256": "2d1e45aa3e91c4042d7dbeab392e0a499e9d2c99a880d439ddf965b7f0043e84",\n      "name": "calculus_-1137-_jpg.rf.94054c60e48173a13fe3e566a25c1774.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1137-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 8,\n      "group": 579,\n      "height": 640,\n      "image_sha256": "0feceb65eea96577f361bf2ec83296cfa6426f590c86d57285fda12edda18a91",\n      "label_sha256": "cf0d8da4f481231254d201d8d8c7b5e4e4e210a3981c924c712e1d76daebe362",\n      "name": "calculus_-1148-_jpg.rf.391a3b3e187de7d5d1742370d861301b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1148-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 8,\n      "group": 579,\n      "height": 640,\n      "image_sha256": "ba78795b08cbd2e2c672a3b64c58f50a47364cc0aead40b7344bad09871b4bc7",\n      "label_sha256": "41f8bcfcad1d373a52cb153987a832837c34da19beb422d92b91e213bd1268ea",\n      "name": "calculus_-1148-_jpg.rf.a50787253a51229c795fc194993c5e2e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1148-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 917,\n      "height": 640,\n      "image_sha256": "f85aeccee6da6e5f75a15172c922a212f7113fcbe024a9b28bbe682abb9a2029",\n      "label_sha256": "32f9ab06ea9906c56254d32383074c45e9b2134ec72e76784d63cfee1232cac4",\n      "name": "calculus_-115-_jpg.rf.01c3350ecb43c24ab392fe303cb9002f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-115-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 917,\n      "height": 640,\n      "image_sha256": "8306a31bb7c86f470d3f643c477526338f2fdae8e8b98ee46f441f9227c2973e",\n      "label_sha256": "0ed9d04e6405704a77dea6aa3e1271ddb71aae4dec643a302196b24652114de7",\n      "name": "calculus_-115-_jpg.rf.3519a202e41b4b242a61fb9e580a91ed.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-115-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 917,\n      "height": 640,\n      "image_sha256": "3537fd2168814d295b13b4baf98ba950412f93f6ca9e9ae2a3a29de3a5a3650c",\n      "label_sha256": "24423b4cfca0ab80a417766b64f67d07ce4dbd7c0579d72f75cda7d379646051",\n      "name": "calculus_-115-_jpg.rf.5b0fc47e7faf8485648e9f89492bd6da.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-115-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 887,\n      "height": 640,\n      "image_sha256": "c0a76dbaa4c4bfd03e27114c836ef9b160a5c118a3755abc0e00069499651651",\n      "label_sha256": "a436cf351656311d0bc4093b7f47bb384e3d6b7385f309bfc5f2e0fb8e78d543",\n      "name": "calculus_-1150-_jpg.rf.2b45da5e8c782c7cf22ba4f288637665.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1150-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 887,\n      "height": 640,\n      "image_sha256": "340d7cc56743e6c5dfa44ac61e596d51cc26e4d11bb794149cac3b4b0ad9cb6e",\n      "label_sha256": "62d714670e091977a721bb1cb5caeb18c10c10e53af7947e4e2e6dd0ef0429c9",\n      "name": "calculus_-1150-_jpg.rf.65b1aa5c2f20434ed6cce24d7426b81f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1150-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 718,\n      "height": 640,\n      "image_sha256": "7e9be1394c7afcc057101d03130dbc87a6067498dd04da2e11e3523347f6d208",\n      "label_sha256": "4f7e7bf0756c1f355b7dcc22fc61c5a8d87f4dd4e3a5f5b08ff300e7a0f48393",\n      "name": "calculus_-1153-_jpg.rf.26f891247a77c777d566f9ca4ae5a0dc.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1153-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 718,\n      "height": 640,\n      "image_sha256": "a43f9f0969c1aecb57058738351782d8b34ad5bd621f2fd80fdda3d7ae679d88",\n      "label_sha256": "91ecfaa550b229d7d838421a2dbfd64c41814ab4459df208661f4aff33633ac1",\n      "name": "calculus_-1153-_jpg.rf.bcd0b13fc78680bc4a2dc39b18d347a9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1153-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 921,\n      "height": 640,\n      "image_sha256": "b760e2ea42474f02b645a3504dba7dc312beb86d80a1782df59f81301276a005",\n      "label_sha256": "80872e37bd714dfba9ea67b53497e1298aed2745b25028a536f48ddf9e5291de",\n      "name": "calculus_-1154-_jpg.rf.310382e21dbe0394ab3562bcedf9bcaf.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1154-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 921,\n      "height": 640,\n      "image_sha256": "e7645dca2afcbf3d750bb4aea35150776b9fa0f070a8401408c5334cd816c013",\n      "label_sha256": "4ee39ded8db363d2f96641deaba5fc729f90e06678994dbf08b028d009b6a71c",\n      "name": "calculus_-1154-_jpg.rf.bdbd081a2660318fd52da67ee5193fdb.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1154-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 922,\n      "height": 640,\n      "image_sha256": "e00ee9de21baead2ed3944f16223f3238713a2ec4afbc6338905965d5e65489a",\n      "label_sha256": "b944ec0348588bcd3d1225f1ed8f05ec06e950401dfaa2194d38ef2c5e858e52",\n      "name": "calculus_-1156-_jpg.rf.556a72b6c751b75009fd9e28ad4080cd.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1156-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 922,\n      "height": 640,\n      "image_sha256": "0941bdfe0d72c86edbe8f0eed3cc8167db1fcf8491cd4cd688e12c168c92364e",\n      "label_sha256": "e7c06bd772e597333ae1dad533ffbcf2dcb67a5755cddd657135e2e4438e1f4e",\n      "name": "calculus_-1156-_jpg.rf.cf9219c61f0eee4954201fe382e158ba.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1156-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1017,\n      "height": 640,\n      "image_sha256": "4b920bafb244388dfcfdb5d034e756eda758dfa57ea6bd9d12c3ce8a80616577",\n      "label_sha256": "53327de17d1f720bc3a47dc2810f091b7b6af35435bb482115cfe60dfa9e52fb",\n      "name": "calculus_-1157-_jpg.rf.e5344b97d106be09efd93fb03f6e3c67.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1157-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 9,\n      "group": 638,\n      "height": 640,\n      "image_sha256": "2c6eb8be7cec5e731ea926125b7fd9ca2cb117e01c9aaafbcc6fde020da8531b",\n      "label_sha256": "4cde52d3411678d5db9034dba940172b6259dd475143c9a50aad65abad198346",\n      "name": "calculus_-1159-_jpg.rf.292640d27c06857a663d1c63198c2bde.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1159-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 9,\n      "group": 638,\n      "height": 640,\n      "image_sha256": "0991ccc44e340ba02c99ca650701daac1f2947612778ccd777f4b31c0f3e1d5d",\n      "label_sha256": "6ae296ed3eccd3698fe72b6ad111c75ecbf3108b0ae23ee64fb62784fcb057c4",\n      "name": "calculus_-1159-_jpg.rf.bafbe976dc0d40e63391eb3b32d77db9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1159-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1031,\n      "height": 640,\n      "image_sha256": "3941e1e11e0cd89ec087a1a5bbfc00c27ab397e27dc800018f0a742f552be085",\n      "label_sha256": "9917c47ea5f2c366b00eb7c4822321022810c90e9d7bf67fbadc6ffaaa8d00e4",\n      "name": "calculus_-1161-_jpg.rf.55149cebcf1f53839600df56c6bfd789.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1161-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1031,\n      "height": 640,\n      "image_sha256": "bf59698bab56633e1a684152840b39394b7ef4e23661c43796417ceff15ddeb4",\n      "label_sha256": "a6d39dda9b528d99e0198b5d1c5c1bf4aa629248477be82ad26c64e59505b0d4",\n      "name": "calculus_-1161-_jpg.rf.fa9f37417112eae0ba4d0dc70d13b18a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1161-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 701,\n      "height": 640,\n      "image_sha256": "6480b241c119f5f39000a32e5e3da21c8166e81982b9b0b41083bc45e91152c4",\n      "label_sha256": "8e26d1abc89420c99d8c9c260c27735615afb3886017b225ace8ae193d422306",\n      "name": "calculus_-1166-_jpg.rf.16bca5e4e8f4fcc4fd130d38b38da73e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1166-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 701,\n      "height": 640,\n      "image_sha256": "2b84049d0fb7ecece6a39c064797e28559125c2a6c706e057e8c8d88928831ea",\n      "label_sha256": "6b9af45209fb8e801dc911496c6bf0076fdb075aef72d72e5d992b82cf59a901",\n      "name": "calculus_-1166-_jpg.rf.784e2091b60fb78365be4adb5bcc2adf.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1166-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 701,\n      "height": 640,\n      "image_sha256": "ecdd07a556984a052852f872c53be8209a27fcf942d6688b43bfb1740d72bfe4",\n      "label_sha256": "4c815b90110eca5f6060be880b8ce4658496544a7c4a69bd419dd57af4f9e812",\n      "name": "calculus_-1166-_jpg.rf.9a9bcb145ea8bca7d8a666eb3b15fcb1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1166-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 231,\n      "height": 640,\n      "image_sha256": "1ba2e03242c0aabd1358eef54dca1fdafb9268cb08be506576910f3a3d728fac",\n      "label_sha256": "7adf5da0a9c5a21dbfd72e3bab799b179581bfebdcb521f15f877c6d4f556730",\n      "name": "calculus_-1167-_jpg.rf.33c3de54a091373d25289c670572d145.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1167-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 231,\n      "height": 640,\n      "image_sha256": "9babb22cc8ba97f2b622224393703c6f8954d578a37c3df9a16411eeb6fd289b",\n      "label_sha256": "7a525a14f347ce5bc5d818b5b3abf03bebff83c45bc54eff0db4d5dc67fef4d1",\n      "name": "calculus_-1167-_jpg.rf.dbf97d3235d3c72a1d603d4644e40ea1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1167-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 766,\n      "height": 640,\n      "image_sha256": "2f6da8af782239fdfd85b543d29c98a33697a16f94cea472fa273f1f324d6796",\n      "label_sha256": "295e3955b669afe8e2dfae34ab3294b136a718989b5ce1e551ebc049c657f403",\n      "name": "calculus_-1169-_jpg.rf.6d0da0d65022978413cc47d4d911c7ec.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1169-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 766,\n      "height": 640,\n      "image_sha256": "0b5201361c95079422572183c14ec5841728f4a2192893234bf481ca2f9eee9f",\n      "label_sha256": "9b35cea6b738b9e09c4c515bc45210d005efcf31511e4d1808cbdec896bd6d03",\n      "name": "calculus_-1169-_jpg.rf.cd7908ee35f97074e8cce3b0ece2f040.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1169-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 766,\n      "height": 640,\n      "image_sha256": "567ee3eec9d50e06a2a32b16417f60af403f95e125c8d62d33986a987626d399",\n      "label_sha256": "0084605a37cb6572b870807396a9abe9f9ac7c298283ff66b6f8cd84c95564ac",\n      "name": "calculus_-1169-_jpg.rf.cf413d53a6eb007f663ea844232e2192.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1169-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 942,\n      "height": 640,\n      "image_sha256": "16ed5c23682d00f0a524bbebb2cce07166f91dd07b16ed54c1d6c869b6b5852d",\n      "label_sha256": "b0b858118b7143ad4cdad33a2201ddc224547a9c9b5e8768086578c0cbf06147",\n      "name": "calculus_-1170-_jpg.rf.06e44df0c8c3d7118e51722a75950db3.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1170-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 942,\n      "height": 640,\n      "image_sha256": "079ae664a60092094624e448628fe8b55c55408c0c4dbec2b741cba3d198dc19",\n      "label_sha256": "ed2e809f7c9cdaf75e70ca8abcdc4a04515848d9f6e1765743632bee4c63d707",\n      "name": "calculus_-1170-_jpg.rf.44f004611ad0304b93a5639dd01a20d2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1170-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 653,\n      "height": 640,\n      "image_sha256": "2ec5f88e6d35975bcb5804e3f9198e145e1f313149150ffa6274ee6d228d7944",\n      "label_sha256": "7874943ce5253959011aacbc4685637e9ec234a5e448200fb6e62b2aa1a06f4f",\n      "name": "calculus_-1175-_jpg.rf.6b583174a27c731a7deab5ab4859ef8f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1175-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 653,\n      "height": 640,\n      "image_sha256": "a0069438a87e9cf2ccccc28eda39924583212b56480662e414bee51519bed027",\n      "label_sha256": "61e54325061bdf3af59c6691bb62a1711a3fc912fb130a61022561ef0ea3bb87",\n      "name": "calculus_-1175-_jpg.rf.c7b06a1bf995104b86a4b40ca1b17bc2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1175-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 653,\n      "height": 640,\n      "image_sha256": "aaf60f6c4eab63932e5cab86575d495e1f4ae16d3e555bb96cd7d4594c54ad4a",\n      "label_sha256": "218f900836912f9d6eaec276f22de19ed9c52abe286ab9e2942a971721bb9a09",\n      "name": "calculus_-1175-_jpg.rf.da128e072a229263b48ce7f8243a2daf.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1175-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 1033,\n      "height": 640,\n      "image_sha256": "755f67e3fcd65f6a3d5c657b0998071a7bf4772db407d560294f7f30cb4575dd",\n      "label_sha256": "048d98b609e6d5bbffcca68b777ca92baa2e0c3c257a94e24c9687a5838a000b",\n      "name": "calculus_-1176-_jpg.rf.13d0f4ef8e94f6fa30b73c6fc0480f49.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1176-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 11,\n      "group": 723,\n      "height": 640,\n      "image_sha256": "fe64b20ec401c4600b21d74624694f328290b249cbd19fc7672d6150c4c9d387",\n      "label_sha256": "9689531186a64693132090abfd1933a989c022d90a6eb4452b87d99855f718fc",\n      "name": "calculus_-1179-_jpg.rf.15959c8865937a2b2cc12d076b6289d6.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1179-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 11,\n      "group": 723,\n      "height": 640,\n      "image_sha256": "2bcabcd632e3cc57a4c719561ac90ec8160cae931b326fae37e108af55d2e23f",\n      "label_sha256": "73a5b7338a4216b4903e3fcd799cd8de3781bb6014a716c34f37948ffedc6f68",\n      "name": "calculus_-1179-_jpg.rf.fb6888d8de7959f088ea51682a5cfb39.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1179-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 897,\n      "height": 640,\n      "image_sha256": "6e98385eaf6a4dd10bdce61a547ff5dac7555a852c7f1c06dbecf23f6b77a85c",\n      "label_sha256": "c690ae1265ae13c6c078470e7d00be255e4c6f39db9665274d5051831ebd7dc3",\n      "name": "calculus_-118-_jpg.rf.bcbf305301cd5d3e7d4357ff1fb5f3f8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-118-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 1045,\n      "height": 640,\n      "image_sha256": "0eaf587aee513b44cf8b0b1d1d01f5a22939119ca27efb2ac04b4c4de5b0ea38",\n      "label_sha256": "02c404b9b8750b2e224f5374d5c4238ba6b5750f67db97e9d4e34cf0febd44f0",\n      "name": "calculus_-1181-_jpg.rf.ddcbb5791114946c1d9f84708b6ea866.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1181-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 900,\n      "height": 640,\n      "image_sha256": "80202fb946f1348bbb412086c0dc644a00d8ccf0a2edfd1ba05491436be98ea1",\n      "label_sha256": "9ee1151775eee592a18ca747f8532f1e56ee74d65f2c6c83e9e3e3b3b3507bb9",\n      "name": "calculus_-1182-_jpg.rf.a203452550c5a1fb9a0aef9750167093.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1182-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 900,\n      "height": 640,\n      "image_sha256": "eaa1f4de46d205b32b25e3e21a618506c0324e37c1db6c49322e1386e11b6a3b",\n      "label_sha256": "0983d68a6226b2d709e13047281102ed69445938defd92b225708d99913df373",\n      "name": "calculus_-1182-_jpg.rf.a98997580e96e390129d6a6288a3bca7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1182-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 920,\n      "height": 640,\n      "image_sha256": "a74d0ffa82fecf3ae8a48c4420eb78fc76fb2f5d406c79b03e0113e697b89272",\n      "label_sha256": "17f49288f1a8315e18254e1cc9f0591e3e210e3e6a3eaada86843a47c4d0d451",\n      "name": "calculus_-1184-_jpg.rf.2379b174372b903a6191ba7f655a9c92.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1184-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 8,\n      "group": 933,\n      "height": 640,\n      "image_sha256": "ecf549324a89f4f797ece4807fe24bad150fe2a52f9b342d32f3320faf308c9c",\n      "label_sha256": "86ade380d6e04f072aa12ac7770442a8764272dc0d9cc3095759c63292627745",\n      "name": "calculus_-119-_jpg.rf.66c96bbbe9377f99c3876843c53855df.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-119-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 8,\n      "group": 933,\n      "height": 640,\n      "image_sha256": "a47e4b85433c73be164d4146c586e593ff1811be19531aafa3716d696095fe70",\n      "label_sha256": "029480be7c60864efb68249bada4823c2a2abb3119625c0f714d7cb33c9ac75f",\n      "name": "calculus_-119-_jpg.rf.eee14beea5cc8d9038c95a11900c10b2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-119-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1049,\n      "height": 640,\n      "image_sha256": "1e8c21c047f822eb34ecd8541717de2e63d0f819d24963eb52216e8171fe8fe5",\n      "label_sha256": "93c26fea18c507bb899eb98a08ffe566f1a778038759fbb00559f1cf374d4032",\n      "name": "calculus_-1196-_jpg.rf.40db4d332bbc58c326def58ff2821c43.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1196-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1049,\n      "height": 640,\n      "image_sha256": "7947b3ad2e0e98a220b1b4267941f321ab0d998f877edefb0c3b99a43ebb7472",\n      "label_sha256": "d54e438b34ff7e9f95390c43d893825e823e2a82c6a054d358c44e63475ae1d7",\n      "name": "calculus_-1196-_jpg.rf.b8cce1778b2b38fd7f3dedd4ed7ddd21.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1196-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 937,\n      "height": 640,\n      "image_sha256": "117b2afa0253d1f2415e824ca5140823526cbabf6c314d5d3055e9799925340b",\n      "label_sha256": "2192f09c07b5f0170461fda9b560ffe449a4f5d03acea81c3929cbc1748e9cef",\n      "name": "calculus_-1197-_jpg.rf.20f4193bfb56944dc685800bddc63c46.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1197-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 937,\n      "height": 640,\n      "image_sha256": "e8a642a45a52f91c4aebd6c67c08891d73ef919c37540ad4f7cdff2b36084904",\n      "label_sha256": "5e8b0aea84b469f55a8bc8c64dfffe119de86a1363fdf2ece91e5e54d2562c48",\n      "name": "calculus_-1197-_jpg.rf.40346ec136efbf3ca053bdb85f14602f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1197-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1051,\n      "height": 640,\n      "image_sha256": "d4386a5c200b9a28bcc7c9051e44391f000dfaf5a3987c580c605b3aaeb5c86f",\n      "label_sha256": "4fbaead368b04c62fb09ee508fd5b860763fa74a34440c956565e990c4c81b34",\n      "name": "calculus_-1199-_jpg.rf.8b533303774b901241c0a3335f343f89.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1199-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1051,\n      "height": 640,\n      "image_sha256": "3d3e70ea7bcb5ea7bbee3e1028e575ec62b117bd5e930fda52a3a577841233dd",\n      "label_sha256": "5b718f10bcefc51b5de620b544c9558d2e3b0264427f6f72772363f64e1a15b8",\n      "name": "calculus_-1199-_jpg.rf.c3727396624620bd2cf9e7ed44d431b8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1199-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1023,\n      "height": 640,\n      "image_sha256": "3ee473d1e1cd541ac02dc5dc5dc570b249ab7891c6731a1ece8789601769d03e",\n      "label_sha256": "9026d3cbbf4a24e8e78027defc3adf4138b7446a2a4b527252108c5c8308ece5",\n      "name": "calculus_-1200-_jpg.rf.2dbbc904e2c715454d38849c8e798787.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1200-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 247,\n      "height": 640,\n      "image_sha256": "72cfc3541973029c6e933873bd320fc47eaa6eaa58afaddfe85b49ddb1597cd4",\n      "label_sha256": "7bc02eb26d5b77eadce24ffdacfd3c579f2c6d30428b7a1477770cd779d864a3",\n      "name": "calculus_-1202-_jpg.rf.06f3ce1d95dac61f348a88f7ceb54eba.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1202-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 247,\n      "height": 640,\n      "image_sha256": "d0cdd19230ee3cfcc295a3857f94559d434da2c36a0ef67d8cc850e116a27206",\n      "label_sha256": "3c21ea47e7791e26f6f00024ddb7f0c1458763249839e08051508a8d2a1b8c4f",\n      "name": "calculus_-1202-_jpg.rf.591653fbbd9c8f0f357a899d18c60dad.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1202-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 247,\n      "height": 640,\n      "image_sha256": "cb005051b188eeb433b7c42de7fc2d92e4066d62c9d3caa151eb30aff8bf1bc2",\n      "label_sha256": "38c333e91b316df1eb818dfd03609301e363189368d466e4ffe5241d9ba7c917",\n      "name": "calculus_-1202-_jpg.rf.b99b2e75fdabab734ccf6cd79b692dd2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1202-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 908,\n      "height": 640,\n      "image_sha256": "8b582f8f2350aac762663870c86b6f38fb3d2a2152ae258efdf45ddd71b7d07e",\n      "label_sha256": "61c94a5b0b66433a05979a01266ebf45c7113866eca2dd86292eecb50ffbac82",\n      "name": "calculus_-1208-_jpg.rf.0f6f7fc1542b1709fb513746516fadd5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1208-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 909,\n      "height": 640,\n      "image_sha256": "4ad94ba82979bf4bb3e4aca82231543b7d5e8d71032e23cf6eff1b47086471c7",\n      "label_sha256": "135f172e58a1c6994f9b74ab6c8473b3e09bfae63863fb373f8628a44ffdcdd3",\n      "name": "calculus_-1210-_jpg.rf.8933c5fdd6332a2334b3f701631b7b29.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1210-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 263,\n      "height": 640,\n      "image_sha256": "01ec93a7dae3163e0e916d75293ab655dafb85e8100a5fc57a884d84e99a6a50",\n      "label_sha256": "7af61fe09060db685cb7a63d64de8e3971a37b46a415db6894d46d45ec6e1640",\n      "name": "calculus_-1217-_jpg.rf.5a3c664f3d3eefa049b1fe2de390ce7b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1217-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 940,\n      "height": 640,\n      "image_sha256": "d0d37183b5bd25507e3b79fae237fbcbb5a576318dc69542b2cbd96023419d5e",\n      "label_sha256": "4c98aa8767dd8ccd5876dfd41c91ac0733ef83916fc0799e0e636eb41339c65b",\n      "name": "calculus_-1220-_jpg.rf.1ec848b22cbe3f05aeb2de3fe2765560.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1220-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 940,\n      "height": 640,\n      "image_sha256": "4446284291915341290c30b5a1eaf90b2eedba2391ebdedb52476fd53f9e95d0",\n      "label_sha256": "06c2ca525140faca2e218ccef097b9f4a5c183c06196ef45688e91f12b510baa",\n      "name": "calculus_-1220-_jpg.rf.6e151c1d3a300aa4d13836288c37bb30.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1220-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 175,\n      "height": 640,\n      "image_sha256": "35b993028543de2a582bafead9f6b9c179395c172e66de9d77f6ba68c4d01a79",\n      "label_sha256": "9da4fd011823ad4121e3f9943413ad27e60ee8ca57a38a99ba98f19a1e16751b",\n      "name": "calculus_-1223-_jpg.rf.666813ec2927b02f0cabee3d9cd0492b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1223-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 175,\n      "height": 640,\n      "image_sha256": "aceaf7f80125efe8e0489d86218d4048d6460f0c7d1ad69639d8056834bd7872",\n      "label_sha256": "907598e490bfc1ebce28e9e5dd7cca14631a7a0d9969bb03a3f62fbb582d9809",\n      "name": "calculus_-1223-_jpg.rf.91f04ef4bf1b2b632dcfd1f5c17da9a3.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1223-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 175,\n      "height": 640,\n      "image_sha256": "e13958ade8645d73b6251c407159f91cba9e09d3d1fe130686af188ad022a6a4",\n      "label_sha256": "5d3f368c31d7e8f400ce4999188ffd6ec1ddfa0b82d58b8121fb2a0370c9b8aa",\n      "name": "calculus_-1223-_jpg.rf.cd930c0eed1d10f0c255f92a95d938f9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1223-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 932,\n      "height": 640,\n      "image_sha256": "62af40556b845dae197ff55cc2c40766699c3a1eb660c03822816a9aeeda6062",\n      "label_sha256": "f0633ee49fb3f92aea78cdc58c1373e12d7aee6c18d13f2eb18625148dc59506",\n      "name": "calculus_-1224-_jpg.rf.28e04f6203f0cd0073cd1fc4bcd608b9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1224-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 932,\n      "height": 640,\n      "image_sha256": "8545c640e27efdf27c9a630a84303522d17cab1298748bb6575187c34749040c",\n      "label_sha256": "1672adc12faac2d4caadbef78573afb6e40fe8b2ab3b6f0e645bbdae2c8c927e",\n      "name": "calculus_-1224-_jpg.rf.519d4618e37664b02cd73b7f25d89791.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1224-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 932,\n      "height": 640,\n      "image_sha256": "abf8f3efe08a8d7f0d1d13163ca7f7fcc8d1f563a4380d9f1bcb7b24cf9d0a6e",\n      "label_sha256": "f78ce9f56b7bb055a445c7c28c51ff4546dab48fdb0994acf22b0f1e28e67629",\n      "name": "calculus_-1224-_jpg.rf.54c1ad8e016921ef2f23c2429f6d5d23.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1224-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 762,\n      "height": 640,\n      "image_sha256": "dbc6fe8e02433eb4cc0c72e309dc9d47d12adbb067ec83e0a56a14be689f8929",\n      "label_sha256": "90e785bcebf2e1d71c587c2b8d695347c56723cb59598c3f6bb6e36a58c02081",\n      "name": "calculus_-1225-_jpg.rf.946154911ad2d6678b6488ed8586bd91.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1225-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 762,\n      "height": 640,\n      "image_sha256": "d77bf2b4e6f9dd3537b3fb5f828b08aba97b282861fc6f511c9d4dbd0aa755f4",\n      "label_sha256": "e20d92ec21d919f1fabd450b0ee4ed4980478f38b85d9ac945252991f39ae21a",\n      "name": "calculus_-1225-_jpg.rf.b21ad37efd57b2d80eed3d44b27977a7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1225-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1026,\n      "height": 640,\n      "image_sha256": "5f491c611c2fb26447d6913c22749d94059719af9f29f7232ba62a4765c6c2d5",\n      "label_sha256": "c4361fe0732fac7ae493a883d4e4164bf7bc7aa2c45181d2e8c033542d64178d",\n      "name": "calculus_-1233-_jpg.rf.cadd37ce2869b4be14506466a934bd69.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1233-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1026,\n      "height": 640,\n      "image_sha256": "212b63ab12fda537e038977319f561aebe8c401af653a4b2c03b168cd035ce17",\n      "label_sha256": "6c63e21c9c33f35431dd59fd5307fd0932b0a8e489a6060a6ec800883c8a98cc",\n      "name": "calculus_-1233-_jpg.rf.d2f1add33a49bcc6b46532c2fa68695f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1233-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 911,\n      "height": 640,\n      "image_sha256": "42b43c58c491d68d1f2674105bed236ede5ad8174755655caa0007c46c8e5117",\n      "label_sha256": "d5d1b0ae4f5c32df87d2a74001016cf46da25a5d1185ffcd5c84d7a953c15780",\n      "name": "calculus_-1236-_jpg.rf.14a671ebeb641825df9fd5c8e0900c1d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1236-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 911,\n      "height": 640,\n      "image_sha256": "5a5d5861244a35f12bb3b5d7444baf20490c405dbdc3e2039ea1fefb75e7ba81",\n      "label_sha256": "81cc954df6c2e45665b907b2736c1aac26327737b9f902a34a01294cae963dd3",\n      "name": "calculus_-1236-_jpg.rf.28ae549d33a621a9059fedc8aa32d90a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1236-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 269,\n      "height": 640,\n      "image_sha256": "815aa35de007e6777145656a457312f0fe749418618fc3310676a0cfbf538b27",\n      "label_sha256": "50a0184f0d0525928ba2d26b032bcf51057916fa9c4911c66d784ec403b4c740",\n      "name": "calculus_-1239-_jpg.rf.4c391b62c1e63e1b8789a2aa103254b8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1239-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 269,\n      "height": 640,\n      "image_sha256": "4d876ff46a0c91dee316fe7201d7ee66a9aa95aba793f4a173f175363cb436e0",\n      "label_sha256": "e01c8deff5d372c8f59d3cb76b1534dd2394c0d48199a6ad519e1fb3f6f86462",\n      "name": "calculus_-1239-_jpg.rf.6e08fd3850483216ffc104ae0e35a663.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1239-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 269,\n      "height": 640,\n      "image_sha256": "818c813ca6739ca89838ab177878cd9e647cfbe69b695ad74ecbc70c2af22b54",\n      "label_sha256": "b2a6609ce754559143bb95258635ffc55ee5c72ae45e87f3c8b6870aaafa9877",\n      "name": "calculus_-1239-_jpg.rf.fa577e59981b3bd9096f4d08ca63d857.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1239-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 1038,\n      "height": 640,\n      "image_sha256": "e393d94db562a70560faa85b2e83461e1ceee3adcda88d516aa6fa9ec682b128",\n      "label_sha256": "10ea274c0e650d664f0195b285aed4d2cce5d1fa54f07e2d44255f219744c126",\n      "name": "calculus_-1243-_jpg.rf.82548bd1e54e1458cabf92255cd05d3e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1243-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 1038,\n      "height": 640,\n      "image_sha256": "c6f4c3d23c63c3a56eb7883572371f20dd48415b12beac70ed9948dbc3a13d53",\n      "label_sha256": "a1fead64f8ffff9c3224a5cc2ef0db2e6cc5edcd9fa40825c43517691bfafcae",\n      "name": "calculus_-1243-_jpg.rf.c8c7c3f88782e4335354db0332fc177a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1243-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 1038,\n      "height": 640,\n      "image_sha256": "e5ab5d40a4b2a275bae95e0ca2d3a85843ce1b5316bb177d1aab4381fb036082",\n      "label_sha256": "51cb01a0f1702a5df43a9871788ff1c68d8d4ce7d83f60b38110b73e94f2c748",\n      "name": "calculus_-1243-_jpg.rf.da00da39487abc621b42cb1578db6264.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1243-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 912,\n      "height": 640,\n      "image_sha256": "21f666be1b96c8f0efb55f67671edbf6643409cc6317cfd86a18742dd5fff2e3",\n      "label_sha256": "2d67239bda601a9e202db9a0690205eadbf8bc1afc8a094d5a2a8796cd6adf38",\n      "name": "calculus_-1249-_jpg.rf.2214c27fd54e904110d94010b1e28506.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1249-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 9,\n      "group": 933,\n      "height": 640,\n      "image_sha256": "a6a7549c1c910fd2dc162ad2d2c56bfcfeba41a8215d0db5c5ee3c2bfd527186",\n      "label_sha256": "b4e5bdee74e51ec806e06825093a8f530895cd31c2a5d1cb45a005a2bbfc8320",\n      "name": "calculus_-1250-_jpg.rf.b6b7d186524b2978a18295c8118a8d15.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1250-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 9,\n      "group": 933,\n      "height": 640,\n      "image_sha256": "12b356b2b0a309e7da434db3fb5a5dbe10570660aedb324dab5cb728579c1571",\n      "label_sha256": "21bfbd11741c91b5189c4e7ef1ee9372090cee2274e004f08a111b74b6f5b035",\n      "name": "calculus_-1250-_jpg.rf.c3b7309c33a874d86c7b02bde1c7ba48.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1250-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 9,\n      "group": 933,\n      "height": 640,\n      "image_sha256": "ccfcf42afa7e7a469f1bea98b411611c07371cf93ca82071fa1e34e33d5f043d",\n      "label_sha256": "5e3adb7c58502b909044f3d59a44509acb0e14bd85d37f01d1aef57def9fd5ab",\n      "name": "calculus_-1250-_jpg.rf.d2afc0816397931e63ce1603513b8258.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1250-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 913,\n      "height": 640,\n      "image_sha256": "4a8a3b0df2b4c757106df5755f9e418e9043d573163a6b9827f6eb5a9fe61811",\n      "label_sha256": "ce18ed5928563d6df2c85c22a475ae233327891bfddc8c70cdab8256ab3ce6e2",\n      "name": "calculus_-1253-_jpg.rf.346670ab338116144621129f408d8e49.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1253-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 913,\n      "height": 640,\n      "image_sha256": "7c0d29efb81260036b3d300f2603506d08bbe25de5ab9be899964c7fd0f30114",\n      "label_sha256": "c4d42dd2cfdf04dec5a83ccbca6aa2db3f02389f4c6d832a3f91eaede40752ca",\n      "name": "calculus_-1253-_jpg.rf.4a6fcdac49c70fac29ae27d4a09aa396.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1253-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 747,\n      "height": 640,\n      "image_sha256": "336c5cfb872d4d5d7ea30cb02d307c05688915f562fe3ae1d7ecb6780b7edc08",\n      "label_sha256": "05c7ba4d9d7f9e2c2ba26b57af14b590dee3acb375f676b270f74ed045910b3f",\n      "name": "calculus_-1254-_jpg.rf.c92fc6ee4b7a797e788a02ae3e774a42.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1254-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 198,\n      "height": 640,\n      "image_sha256": "c55482552483e60988b9b2fa4dc9f8b637e083041ceb1104eeb1cda9cf863c3b",\n      "label_sha256": "9b1a4060fe100ec8114f762f033cbc6f37a1f095bfced5fc7d8f2aed967e0a3e",\n      "name": "calculus_-1256-_jpg.rf.d8a8a75e826c9d5bc9c63608cb61ab74.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1256-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 914,\n      "height": 640,\n      "image_sha256": "8ee460d10fc3a489a2890efa4c30a9920a2bb29e4e2e498640c0d378eb65b0c3",\n      "label_sha256": "7d761446d830aa66b546383ccf99325533581c67bf82c9b0ad6d2dd13da5a551",\n      "name": "calculus_-1259-_jpg.rf.2edf30ad0ef8d67ce5a013a7f80b9256.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1259-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 914,\n      "height": 640,\n      "image_sha256": "e1deebe239ce134192d70241836f5b5bebb103e6ba6f44344e45066e4ad663ce",\n      "label_sha256": "f520a2e4331c4cf99adceeadea8709b19ff9e3b252c94452e057d8d08d7d63ed",\n      "name": "calculus_-1259-_jpg.rf.9eaa24cfa4c8129ab0bdc99a385864f7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1259-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 203,\n      "height": 640,\n      "image_sha256": "a05eb8dc27b3dea0a3ffff848f34055706ae943e86f6e9cfb8a28966493b1c05",\n      "label_sha256": "23a43d1efd8442880c93bd7dbd8f811dc9d13a8fa71b22803777d88d1c3406ef",\n      "name": "calculus_-1264-_jpg.rf.0b8661a5f8d06ff76f3c7df9d2c4dc1b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1264-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 203,\n      "height": 640,\n      "image_sha256": "8f75a76874d06365821dbabd1d78bb017f399de1a81753873310bc6f6969311e",\n      "label_sha256": "eb7471c3c789db334b33cf4c7289b1ecc66f981f0fb90edaed1d814a347f6b3b",\n      "name": "calculus_-1264-_jpg.rf.53992cfce602f39fd77596acab8b49e0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1264-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 203,\n      "height": 640,\n      "image_sha256": "43bd8c8e75c7b2740dc3253fe87bbf296e11086f13b600f9b5fbae45e12165cb",\n      "label_sha256": "9aac63e240d05773679f1dac70ece22fa805fb68cf389d098436b7424b87b321",\n      "name": "calculus_-1264-_jpg.rf.e0efb7a5f1b7f660a35a677e9f98a6fb.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1264-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 915,\n      "height": 640,\n      "image_sha256": "68afabeba2b9fd6e069f2c59198430fe97643bffd8ff4bedb1dc4cdb3bc5f964",\n      "label_sha256": "85e71891ba285294a9b5d9c9396688afb31f2df71e2455626e756ee895c84ff1",\n      "name": "calculus_-1265-_jpg.rf.3504c88710e3c69e1ea5993197bc4743.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1265-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 917,\n      "height": 640,\n      "image_sha256": "1c518505189d964f1dc33d9ca241892b2584bc1b2b43bf48cf956effb8405171",\n      "label_sha256": "6277984e216169184259dd2bd02d4fdf7b33166680f22cd876fde7f7d2df9db7",\n      "name": "calculus_-1267-_jpg.rf.a2ece728fb83ff2bbda79708d20206e5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1267-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 919,\n      "height": 640,\n      "image_sha256": "e53fc79ff6e902c3aac163df3ea7a17e26341a8f97154a02e03b37073a689fdf",\n      "label_sha256": "f1e58e0be07d052c44f0ef860abb1ced2e98f638465ae3faeafc86d1eff7e491",\n      "name": "calculus_-127-_jpg.rf.0f200bfed7bc68adb81e0ce761e2f131.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-127-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 919,\n      "height": 640,\n      "image_sha256": "6653335c94333013caf3cd7bde6d689895c571dc90216f586f85a7e9a9514323",\n      "label_sha256": "96dc855a214b1e3bfd7441fae6c425f9b883be4266daf2e3790b24f5eff6797f",\n      "name": "calculus_-1279-_jpg.rf.55fea76177206bb5996988ac887d6067.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1279-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 712,\n      "height": 640,\n      "image_sha256": "51931b3ae4d736c2444822fbe02424eea1910080a540dd82612e0b6b39ae70ff",\n      "label_sha256": "15689225abf181c11bea46a226bd83625493fa62b4dc466aba18907f4f425a9c",\n      "name": "calculus_-1280-_jpg.rf.08f2e7e9d6da92cc3c944226318bde68.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1280-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 712,\n      "height": 640,\n      "image_sha256": "4e776a3b2e0712e4db17ff5839a55abf381c6d3825239c790cf191bdb7867d72",\n      "label_sha256": "7451ae5e6137566639c6b351742ad3d07abac267c88f486cee0f0a931b06370d",\n      "name": "calculus_-1280-_jpg.rf.f662656a7ce46adb369a40e800967413.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1280-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 712,\n      "height": 640,\n      "image_sha256": "6cace77bf22d62cb326102a07b8e92c897ccb384d67878cd2f032ed685cab844",\n      "label_sha256": "9f7a0ac09c13cce3b4a2e42b9580774eca38bb3666e90a30e316dbb3893705e2",\n      "name": "calculus_-1280-_jpg.rf.f6f79441a0edf62dd79a0e314305d889.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1280-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 672,\n      "height": 640,\n      "image_sha256": "e1a7bbffdc977deaf1229359e55a06a4c7494f375e4da935d15f83dfd1cb5fa8",\n      "label_sha256": "42195ef02938c84f319d87dd95b95bfd89a9e548fe73e881b22967a255074827",\n      "name": "calculus_-1281-_jpg.rf.5a2196ab8458bf47a6849701db13ce85.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1281-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 672,\n      "height": 640,\n      "image_sha256": "2b507f7905ac9a7c668dfbfcd894912197bc5d123793b3871b65d33155b5c317",\n      "label_sha256": "cdbb47e45c45d94949fc162890dc23b020c2b0ca2cb2fd179b91f67dfb5a4a49",\n      "name": "calculus_-1281-_jpg.rf.91f084bf9e4267dda8c0c97ca9ed377d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1281-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 12,\n      "group": 1041,\n      "height": 640,\n      "image_sha256": "32f279a5c3aebde58d1875d586c4415bd2e8ac868b6d606ec8863578867f307a",\n      "label_sha256": "6168509524c4ce56147fce4d110039761b817991c4e2dfe579ea52f671d04711",\n      "name": "calculus_-1292-_jpg.rf.40ce3c60a27529657a6cfd636a4984dc.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1292-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 12,\n      "group": 1041,\n      "height": 640,\n      "image_sha256": "42d15fbe32f35e37eb8e96ccb33787b2f4394df62097691e48eaaf402371a7df",\n      "label_sha256": "d7587feafcea087485ebbe441fd799039d5ca742169f9d9b5a4d68abc14814b2",\n      "name": "calculus_-1292-_jpg.rf.bda32005ef430cbfa5dd0ff3dd53ae63.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-1292-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 920,\n      "height": 640,\n      "image_sha256": "a0191293d5c921d5cca281130281b9746b4b1594bbd015adb2967d9a8e363a79",\n      "label_sha256": "7da81472b281fd13e1a20f3c8dd984c6c06a1ca6628bb66fe318f9659fd05c53",\n      "name": "calculus_-131-_jpg.rf.a032ca56f9beefc4840b219b84cf1238.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-131-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 920,\n      "height": 640,\n      "image_sha256": "7297df109670f3dc212a242eb8d21270518bf83e73cde680fdf14210c2aa50c5",\n      "label_sha256": "c10fd68fe27f8736d4fd80053655db699ad1449a70024f112642c070c2224ea6",\n      "name": "calculus_-131-_jpg.rf.a530bcdb9f95497e802cd81423de5466.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-131-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 701,\n      "height": 640,\n      "image_sha256": "ff72190ca772cbded9c52c7276d47969d7b3542b437d634ed81fb48c40b9f988",\n      "label_sha256": "dfdc5f12183d03ce203cb81c57ab8d8cfa4b70ac0f3720b4f31413ddecf3d79e",\n      "name": "calculus_-14-_jpg.rf.0baf31a243cc5285df9e22a681de98e9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-14-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 701,\n      "height": 640,\n      "image_sha256": "e2b2fdfb449e4de9a2894c69070b701f2b877cda698dfc2af648b14d48ac6ebb",\n      "label_sha256": "2bf2c85b5913317f3615ea58b5e67051afa9bc0300a986bbbb491d44ee08c395",\n      "name": "calculus_-14-_jpg.rf.aa767abf50e130ff27412af0b6c06fe7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-14-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 701,\n      "height": 640,\n      "image_sha256": "c53197c39f8013b43267daa41c5f3e49b508bd0a365e6d7d673473b8d97eb5a1",\n      "label_sha256": "29562a20f899c59ab995a222ec00647ff0dab00661ed6de4d3996b8319c433cf",\n      "name": "calculus_-14-_jpg.rf.e133b48f8c9e02c789d5910644cc1012.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-14-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 8,\n      "group": 1041,\n      "height": 640,\n      "image_sha256": "e6d988522c5e3dcc0f4712130b1033df0c9f602b5468552d911ff8ebe5402aa1",\n      "label_sha256": "7af4a191bbfef8d95b214286f481c52831537bf8bac0f6bc7a729831bb07b6d8",\n      "name": "calculus_-140-_jpg.rf.1d2d3931198efa84529aaf8201ae0fab.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-140-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 8,\n      "group": 1041,\n      "height": 640,\n      "image_sha256": "6910fda067a5495c1e8c3629017ed1c0282e9fd2f0e18f8a00aac4050022057a",\n      "label_sha256": "8af068e7c7de2724468057731d9f976d069c6ce1ee12261d7a3969edbe019f3c",\n      "name": "calculus_-140-_jpg.rf.6cfc22c99b1705b519e6f93f5fdea6e8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-140-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 683,\n      "height": 640,\n      "image_sha256": "196948a14347179a064e0f6f9e94fcc5ba3923c4cadcd18a76f44e221d5331c1",\n      "label_sha256": "42221d75399bc78268a04d587b1addbede6abf7bcee6410ff276813721e0d9e1",\n      "name": "calculus_-141-_jpg.rf.a444d4465fdcd1d0853d27b6ec635fa9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-141-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 683,\n      "height": 640,\n      "image_sha256": "21cffa9e98542a11cf8aa398750b207f3e66de7dbb5ebdd7ed618dd6fe3ca25c",\n      "label_sha256": "23953fbe22825c29bb0467c3d430c62555bf82b6ab57bba7be8a67d98da792dc",\n      "name": "calculus_-141-_jpg.rf.c0caa10c822a3bd7df58f6ceba7e1eb2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-141-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 683,\n      "height": 640,\n      "image_sha256": "113163be6d1674a5e1da814eba2aa2966f11be54932b3e49ead369341c1b6233",\n      "label_sha256": "f585bedf5ca8d611b779d84b36ee9063f94444a945a1291cd924f54fbd7e0b20",\n      "name": "calculus_-141-_jpg.rf.f1efe6e579213401bd053923f009a341.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-141-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 921,\n      "height": 640,\n      "image_sha256": "c4d6efe3fa3884f46ffbb08cc8c14ea36db63703d3dc8d8e5729bb2fcee35fa5",\n      "label_sha256": "d36a90fd80581cf2975c251bf8afd770533f5feefc2342371ba1d609c9a699a1",\n      "name": "calculus_-146-_jpg.rf.6c278675979b24c40f442fbbfb659ee4.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-146-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 922,\n      "height": 640,\n      "image_sha256": "c30bf9386e28494e4b768cdcd65036d34d728296a26022c24024e6c080733462",\n      "label_sha256": "92fb8dc4b287f2ebcef5db4403948dabaf5534e2ca96122aba6c1359b56e4b40",\n      "name": "calculus_-148-_jpg.rf.f3ead8b25e44b4644af0f57f428dffb8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-148-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 228,\n      "height": 640,\n      "image_sha256": "d4fcebb72136fb58cfffbdebeab2fc69f1bb63d3ea3a137fea08c710876d0a3f",\n      "label_sha256": "91599ecc7cce386aa369a796c4dd54a99f9cc63b650a564fba4c30d0f1a899b7",\n      "name": "calculus_-149-_jpg.rf.42ca53dbfa42947e6793339a3fb45085.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-149-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 228,\n      "height": 640,\n      "image_sha256": "f76cecf09f2f2c9b4d0ce4538bf369bdaf1e24eed5bb8c9c32b57b6405ef49b7",\n      "label_sha256": "e535328a6e41e5a83688acc962b7bf863c41ed29477ac770858b71c2e9804fda",\n      "name": "calculus_-149-_jpg.rf.ea863d5b64a1f4875cde9fb2024d4d2f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-149-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 231,\n      "height": 640,\n      "image_sha256": "fe8a6b88bad64bc93f2f7342d0147cecebbf75c560826cc1d63e3c0e54d3b519",\n      "label_sha256": "9c8adb773a91ea9153dce5b3eae128d8a42b32741589f4f5c57c34d86ccfe110",\n      "name": "calculus_-15-_jpg.rf.4abd1a12ad228ec6c1467e0638574d8b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-15-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 231,\n      "height": 640,\n      "image_sha256": "37bd7c5fe1477d65ee6eb784b74a4441014e0dd46ced82d67d184864d9ea5172",\n      "label_sha256": "4c17597783759114e022e434e9ae8d939c90af0d63cf3b9ad03482b1d63451a1",\n      "name": "calculus_-15-_jpg.rf.5791ebb3846d5d917c53183ee169bbd9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-15-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 231,\n      "height": 640,\n      "image_sha256": "e494d47296338fd0a79a10c3f76ad557c619499cc5a8bff087a587f68bf24bd9",\n      "label_sha256": "9fb32c6b8f9b91900df48560a9c446b6a09d7b00e3b8e1d9c0c74a8fc8ad36fa",\n      "name": "calculus_-15-_jpg.rf.d3133c7f7f13b9a52e382f43f9c03951.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-15-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1031,\n      "height": 640,\n      "image_sha256": "4552ed88df730e47fbe486fb5d9cc9ef83ead14db58d77762cc8e8566b3b02ef",\n      "label_sha256": "fa9f42d7bf36e74b1b4723f48297c0d0d8e767f18a168c1dc284e09f16c43d50",\n      "name": "calculus_-153-_jpg.rf.363cc7b715e828ceeef218201fc91e8b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-153-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1031,\n      "height": 640,\n      "image_sha256": "171395066709920b48a6e5d8ca96c4633251f0398ce223c451f161c67fdafb5a",\n      "label_sha256": "4d22571e221a83f0284af69d4c986dc102d192ffd0f6aba4fbfb5d5ad8c7727a",\n      "name": "calculus_-153-_jpg.rf.b14f170bcdf235238860deaad9fa01ea.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-153-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 847,\n      "height": 640,\n      "image_sha256": "5f3afc2b939d7f1b3e8e4a5cc5713e90e03450f31acd9ea2b7aafd24c923f198",\n      "label_sha256": "a1ea099a43076960faa7694ace833bab33511db809619c6b5d4947c43aae2c8e",\n      "name": "calculus_-154-_jpg.rf.22428d7d45f8c41e088fc54ca7c01ea7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-154-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 847,\n      "height": 640,\n      "image_sha256": "4a33275d40808c1da9acddfb98a2c543fb710dcd488afa5a6844907d95d0a8df",\n      "label_sha256": "b1ed3624248aaeae34c10dcaa49453f8b3a8339f07435247d33b9d25eb06063c",\n      "name": "calculus_-154-_jpg.rf.b5f10e38bd2bfe64b6f87d641bbdf2a1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-154-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 701,\n      "height": 640,\n      "image_sha256": "56ad9d0a5a318d10b094b40c049d5445f47335295657f706f2ccf24a778bafdb",\n      "label_sha256": "ddd0bd8a1101299236deb4a9ac3822a77c9bd2900706230eab3ba2728e7f7017",\n      "name": "calculus_-158-_jpg.rf.28e2f06a83c91721a792cbca0b21328c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-158-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 701,\n      "height": 640,\n      "image_sha256": "4a8f9d00b0287ac4928b061bc139189c964b81e9c0b8a4cc2ea61e9689253f42",\n      "label_sha256": "6f1ac2b38b581d19d2322fcbcb2d04a5bb2cf0e0ddd741b39adcedb6c6dbc554",\n      "name": "calculus_-158-_jpg.rf.97c9a502efb30bf567e3ea4e1febcb45.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-158-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 9,\n      "group": 239,\n      "height": 640,\n      "image_sha256": "a9e3ab93c7b7ee22c675818509a16a2de2d86c2bccfffa9d745029b1c33490e4",\n      "label_sha256": "a4990b84466edd024206bf0b025373262dfb491ca0436ab7f4d0a8e8b145a56c",\n      "name": "calculus_-16-_jpg.rf.b979283e8da0b3cab57c4a87d4ff69de.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-16-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 9,\n      "group": 239,\n      "height": 640,\n      "image_sha256": "22de2b34da7428e8b51772d9fe4376fdc7df8dec50aff7ad02371ffd53a2af55",\n      "label_sha256": "f00cb222eccdea648219fb4ee3b7ed783d3c36c7c5fcbc3b7f65e28ea52bf0b1",\n      "name": "calculus_-16-_jpg.rf.cb6ed76e63bf10d0aac41d2124a1c16d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-16-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 766,\n      "height": 640,\n      "image_sha256": "f812461f41b4b381bbe1774a8fca86dc662fc5460342eed344b26e3b6f2a40c0",\n      "label_sha256": "9331a29bd38a70ecbd803ae122bba3561f3e5e0f846782e817af125b18b6317c",\n      "name": "calculus_-17-_jpg.rf.7798f7102d7f4ded35d7daf358a233da.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-17-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 241,\n      "height": 640,\n      "image_sha256": "7373204d6253321f066b4da9de66e27c167ef3ef3bc1c4214435c833385ec48b",\n      "label_sha256": "130a4b8ce768919ecf3878deeda6ba2919bb2fd58a84d27bd5cf0f397c38f704",\n      "name": "calculus_-186-_jpg.rf.288c4666e500769656c265afdff7b9b7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-186-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 937,\n      "height": 640,\n      "image_sha256": "537ab00d14dd9caaf8bb038175de3e8d576e3f61a65c752b24d40f9efdbd43a6",\n      "label_sha256": "a667baaaafabc768861d9c947ff06b6b08f5d459a98c2fcf9d658df8e4f905e0",\n      "name": "calculus_-189-_jpg.rf.13e49bd4f5918fcb35d425a2d544e421.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-189-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 687,\n      "height": 640,\n      "image_sha256": "b9e2fcb0f6a2bfd37506822e54b98c2705809b2dbfc470afc18b1e3693de3c1a",\n      "label_sha256": "1691d0e2b3d086cc6d011490ed395ba39ba3efca041a9e8ca6ffefab97eaedba",\n      "name": "calculus_-190-_jpg.rf.36fb909c2c94e52ec1ccfc704f718a5a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-190-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 687,\n      "height": 640,\n      "image_sha256": "c3ad1b67e42122b4115ee119604e07051398f5da6db9ba5cbc368a57fb693115",\n      "label_sha256": "915a694c9252f657416a3ab308effc8d412bdb133950223cb77924b427aa1a40",\n      "name": "calculus_-190-_jpg.rf.a4c97008d8c02862acdd56a8d344b584.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-190-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 687,\n      "height": 640,\n      "image_sha256": "408cc8e915d1566542858186d2a9c7f9599b012a083a935e84184f392f3f7e7b",\n      "label_sha256": "c8a69cc2c96cc69ca8a3a052c4ee9588e86e3031e19a98205b1bcef0f47bd425",\n      "name": "calculus_-190-_jpg.rf.ee748e9a20b27efd507abadd23d5acc7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-190-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 247,\n      "height": 640,\n      "image_sha256": "f69e65a1955dd9ea78c63f57498d0516055a3f869dd2d3f8f9b8cc225cb41146",\n      "label_sha256": "74f5d04a3fa6c2412b8538cbb278a729323cb89b6d7031546fc484c0d7762d5c",\n      "name": "calculus_-194-_jpg.rf.34b316fab9eed0b613dc8c04b5c67542.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-194-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 247,\n      "height": 640,\n      "image_sha256": "95cfb341a80cfd31891df456fda722d45cb410d1acaed3793a4c2bd57575645c",\n      "label_sha256": "76b66ce8dffb09ffbd30045669fae1fc55cabc4529c8d9d8814725a494606490",\n      "name": "calculus_-194-_jpg.rf.5ceaabf268a54c940a884fc980f86673.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-194-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 247,\n      "height": 640,\n      "image_sha256": "c98712d63f8c73d31b3c70a14bc58db916b9706f7f1d78f184bbfc43e0359f87",\n      "label_sha256": "7d34a75d846a356100ad0b3cbc7b5fb46daca590ecbcd5096344659e4868d501",\n      "name": "calculus_-194-_jpg.rf.d3cd8bd88223ae9e54fbc3a52dc55d71.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-194-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 1052,\n      "height": 640,\n      "image_sha256": "8b1cfbb4ed128ad21454a298c8199645d64fe937e504389ba84f43a03b46dc67",\n      "label_sha256": "07d6a907b29f4974d8cdc0d6490639edd73345b4952e03d21a001d69a37a1c92",\n      "name": "calculus_-195-_jpg.rf.49da7d238952330ed55eac0b258031b3.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-195-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 1052,\n      "height": 640,\n      "image_sha256": "af05f4ec1cef1cb2e0852d12a5ded9dd888a695b11cc5ff928e186a25f4d6c87",\n      "label_sha256": "a6e0c89f6ce487a4e75e8099b27fcbb02d653a2dd418d57462df3ca23e94d067",\n      "name": "calculus_-195-_jpg.rf.96b8d5113109e13734742686fb223df7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-195-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 1052,\n      "height": 640,\n      "image_sha256": "ff43c334dd01b0f4aac7f78b3bbd7c7ebe4a0a79c159d575a0bf6348dd37e73e",\n      "label_sha256": "f8a381c6a143194cd04fe41d6b1f949422f0a249ecd368b273abe5cf9e7c763b",\n      "name": "calculus_-195-_jpg.rf.bd80c4976357ddc01785188ae853256a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-195-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 926,\n      "height": 640,\n      "image_sha256": "f59af7ed203c66857dacbaf612baabaf3f067127f0e333f5c16156103dfeb2e6",\n      "label_sha256": "b8dd79186e1a19c62759bfede620b8efd23cd56b20a61cb540e0e46c537badd6",\n      "name": "calculus_-197-_jpg.rf.228e0cc99a312d462fea82884a5510bc.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-197-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 926,\n      "height": 640,\n      "image_sha256": "0607ec082cdc0a32d07e3461acf6d69eb6181643eba702079e12bfb2667e2389",\n      "label_sha256": "bd37da6c3413dcb7cdb65fe907e348beeb47c1850575efe3b48740714802f66c",\n      "name": "calculus_-197-_jpg.rf.9c9c6546b05ca72eab761164c0ad3f84.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-197-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 255,\n      "height": 640,\n      "image_sha256": "49e9f3ddd3b6ce74b1a7b016756d0df26523e503e9cbaa2714003d60723aa559",\n      "label_sha256": "87ceaddfdd036e70c8ea9fe68a5c4c377367db54c2a1e139ac4b7699c61adc08",\n      "name": "calculus_-200-_jpg.rf.3aff72471f6753fa06e56396b30800b1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-200-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 255,\n      "height": 640,\n      "image_sha256": "5e4d4219c6e20b3da833c21d121e0cff0c3c5e47405ada2430a1b7b712a0a2f3",\n      "label_sha256": "96bcf4155e668eb33b7fa8c5495e583ff4fda1cfd84f0e701b7c303b36777df9",\n      "name": "calculus_-200-_jpg.rf.b0ddbcd985374e752c81446b7d287a05.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-200-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 909,\n      "height": 640,\n      "image_sha256": "aed977b9237bd4b250fd64b119ff982f88885178c155f39338e29e9f4c5d8c50",\n      "label_sha256": "1deadbee07f538edc551578d739fc33eb260f9185de2d23f59e7809b3027ff90",\n      "name": "calculus_-202-_jpg.rf.33fd475796844d3f57082e9305a18d72.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-202-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 909,\n      "height": 640,\n      "image_sha256": "c14e3541218d80fe9a4a3a118db08a9f6bda4023915d9186b4f3c37a892db649",\n      "label_sha256": "f60c75c25b9aac68d920570db54640dbf82611a3b75859ae6037c61e41310a4f",\n      "name": "calculus_-202-_jpg.rf.57a796c26ceea3ec2ab05c79edfcc45a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-202-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 909,\n      "height": 640,\n      "image_sha256": "31324a2e04c4c8c443b54c50baa6fb3a744340e55ac2b49640457d998c388947",\n      "label_sha256": "3312cbdb68e4ce6e8dddf1d61058a6ba054eb01e6ec0a1d6332ed6916c46f19b",\n      "name": "calculus_-202-_jpg.rf.cc6ac34d5ae35e1d0decea1bd05d878b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-202-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 699,\n      "height": 640,\n      "image_sha256": "64c9283328f9d2d5a074c275e22934380fbd9b89857e5701534c42cd81a4e307",\n      "label_sha256": "6d138a6cfced88f9f144c23d76500fbf87a01cec20f20b87dc48a2f082549cd7",\n      "name": "calculus_-205-_jpg.rf.9213bb4cae55156d9619a39cd270af36.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-205-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 699,\n      "height": 640,\n      "image_sha256": "ae05695ff58e4807f6424363cf6bac7c5c043ecd3de456d5fa5eee36ccc95516",\n      "label_sha256": "fe28e6d5618cc588fa63467daefc71764db3140e34b7c8affaae2952c33725a8",\n      "name": "calculus_-205-_jpg.rf.a66f153aac740dd568da3f10f126e9b5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-205-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 263,\n      "height": 640,\n      "image_sha256": "a3b8b221ea7889d89223eed7f32411aee9c3088647c2802bd779f979e8ac6ebe",\n      "label_sha256": "ca35e705a1a2ba81c1264e6c2b3b694510688b568f7abaedc853c205676a0936",\n      "name": "calculus_-209-_jpg.rf.72837aca2d53f233a07ca3856aab9f61.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-209-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 263,\n      "height": 640,\n      "image_sha256": "7369dac37bf049d64875bf938b158af51a9e6f9135f7eba3371513eb0dc9c158",\n      "label_sha256": "39079ad9d2e6fc660a6b6967960abfb173d07f9166a35664b34cd508d1967f39",\n      "name": "calculus_-209-_jpg.rf.87db002ad5c9d088b2274e7cd4949c7f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-209-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 263,\n      "height": 640,\n      "image_sha256": "0ad6715412aefe6404399ac9757dc8ab9877319ac22cf61ed8a3a4093c37fedd",\n      "label_sha256": "7beab75ed1c907f323556d6c75f8d3811eceb2552e34d79ec22315aef4e6ae53",\n      "name": "calculus_-209-_jpg.rf.acd024bcd0dfd9a84430baf4fb921fc2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-209-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 766,\n      "height": 640,\n      "image_sha256": "2949ed267cdb9d092afc33634b2fc9031a0bc82397b23c0f7285d634dbee5999",\n      "label_sha256": "2cdb198b8334b88f12259aea2ebe78190edb19b028f78cc2a409db18ddd6009a",\n      "name": "calculus_-211-_jpg.rf.10f13827fd7cf8f2c59ca986da88aaeb.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-211-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1060,\n      "height": 640,\n      "image_sha256": "25719baf18bbdf8e2996c10bd7c0b185741385dacdfc1b3ccaa8731c3a4c1c10",\n      "label_sha256": "4af58fd49e7518127010d6d2a483740cb19e07e0676efa62efd9d616534b529a",\n      "name": "calculus_-214-_jpg.rf.a9ca86c71f9a4329f3fe9c32b8fbfd0e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-214-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 932,\n      "height": 640,\n      "image_sha256": "98277f1859d64906953cc844e1669b15ab586c256c24a2b2ce1525e8b355f55f",\n      "label_sha256": "81f08e7e470aa3a8963c65b519b45225c7f8181c4c9a8403ff98b758ba3bc46f",\n      "name": "calculus_-216-_jpg.rf.51d7c108c63f39796509d1afb6a487db.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-216-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 932,\n      "height": 640,\n      "image_sha256": "24b38a14ea3d69d7e12f325a5d49e442a6a879f2e07b3316ae04c12301402508",\n      "label_sha256": "634399f6d9f33356d7c5f882fa604c48e94c0bb904d07880a567dd1854e96e08",\n      "name": "calculus_-216-_jpg.rf.77423f0e6de9cf0f5f397b57564e5b06.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-216-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 269,\n      "height": 640,\n      "image_sha256": "efb4e02fd278f12c67a5529c752f9e0ded5e1eb818c33a982fd16f8269c1edac",\n      "label_sha256": "d2b1db42f905ed77f793d82b1466836f5e9ae295379cd6c50e1582abedf9b8a1",\n      "name": "calculus_-231-_jpg.rf.a143f7481d3c1a012ddde751b67ec43c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-231-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 269,\n      "height": 640,\n      "image_sha256": "f024a101fffd5767a3bbec2e84760c1ebb1320fcb6656ed2de30dd59e66ebc3b",\n      "label_sha256": "99dfe4015aa32194a82ac838e991898fa0b5847dde0d14bf61c27f1cfbd6028b",\n      "name": "calculus_-231-_jpg.rf.dd828e0a5b12f8033844b71d00274e36.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-231-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 269,\n      "height": 640,\n      "image_sha256": "14e97c400d309e98bf0eed006c6d0cec538d5f8e791b3250c82dd716b21a7e2b",\n      "label_sha256": "a5abb3c07bc2df6b86c0746de68d731beb9188fa0f1102b5dd0c7ca01edbb982",\n      "name": "calculus_-231-_jpg.rf.fec8ce8a3f3ae983d8ce5af5c0616c86.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-231-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 933,\n      "height": 640,\n      "image_sha256": "2f7ef70d0c95fe4e3542e9e4476bdf7d1cc9cab8389a9d213c373a0a77f1d5c2",\n      "label_sha256": "63592e5c7fe0e4e7c6c2b2285a6069d100c7ce94d542d656f9fbc9cf36e834c6",\n      "name": "calculus_-242-_jpg.rf.0b7abd9d6f9dfddda0acddb52cb041ac.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-242-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 933,\n      "height": 640,\n      "image_sha256": "159d29e0e6140b71570d5173277ae845675bb5d5e4d5af6afca69268387e6e3c",\n      "label_sha256": "8bae7a7704246d52d8370f0575dd2c20b98fd2476c99aa47d1609f01c30cacfb",\n      "name": "calculus_-242-_jpg.rf.a8c34e057ea2189be050b251eb91e69d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-242-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 913,\n      "height": 640,\n      "image_sha256": "ace7afd2de83edf358c28743e6de64338d495db1526d8ed7efdb040e160ba9d8",\n      "label_sha256": "7e2e51ce15b8bd6a8028120a68de08fa04f0f2e89feef7e34854cfaa56da2b04",\n      "name": "calculus_-245-_jpg.rf.53d8e606a8529570fcf4a5d5686d8118.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-245-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 913,\n      "height": 640,\n      "image_sha256": "45a595d876b730f247b00c985b96a9332891555308554de39366630d6630c5f7",\n      "label_sha256": "2b9b3e6a56c0676e217a2c833789842356263d045a80f5588e59c5f5a2461055",\n      "name": "calculus_-245-_jpg.rf.6437f0450c0d21c9fbfb5994fa5019be.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-245-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 275,\n      "height": 640,\n      "image_sha256": "6f8e567f4ece1070ec2481fba0341035ef4e9bf68081050cc91dc84915e795ef",\n      "label_sha256": "6b86ae4d2584873ca67073c91274a1014b9ba3d77aefa977acd0c26dd83849a8",\n      "name": "calculus_-246-_jpg.rf.98c5a80856cb27df80e686b6b3053488.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-246-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 896,\n      "height": 640,\n      "image_sha256": "c52326c1142f1c483ae55cd785bacd0b6aa4f3f486998a1127da06dce053ddb8",\n      "label_sha256": "2286e388705128d89452d1d9eb4fc8fe7e0ede5d5077cd125b26ce1d8572936a",\n      "name": "calculus_-25-_jpg.rf.2aac3b886a41637679ea29e303f504b7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-25-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 896,\n      "height": 640,\n      "image_sha256": "932c938a9a3cc08a5ee39f91ace9cc046968ae6b85704918cf5afdf0d1f56492",\n      "label_sha256": "46148abbaec3214095a7bf4304de7d018acde17df7b91f8a8bfbe7e823a0df2b",\n      "name": "calculus_-25-_jpg.rf.6759ad6637bf5068c9188a48ba089675.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-25-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 896,\n      "height": 640,\n      "image_sha256": "5880fe49a72a9c1747cbcae6a177bfcd15595fd87f90115060ea6ab2584b953f",\n      "label_sha256": "ebc4550811423a3956ea7a920050260a6b7a5326723baaa86a620ea1d322f4eb",\n      "name": "calculus_-25-_jpg.rf.afa2c170d9a944ca87c2a02a0f7f1af5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-25-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 723,\n      "height": 640,\n      "image_sha256": "30632e0bc31ad2a8688fa38d81a994b7fac3edafaff663cc282e46cd7effcda7",\n      "label_sha256": "b39d7fbf536072645a29954b1a63ac6e00a8523b0ef1359dffde273fb0db5bd3",\n      "name": "calculus_-27-_jpg.rf.0cd215c156b3bebe197dcbb53897fe0c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-27-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 723,\n      "height": 640,\n      "image_sha256": "9f98055ea0c70a85ebed7798b4f589e8c31d68aeb6eb208e0012389f53d2f574",\n      "label_sha256": "68e1ff787b3cff6c958ee55dae5defdfca806d40124ab6a2fc33a12fb3d83137",\n      "name": "calculus_-27-_jpg.rf.67e5eedc048a3ad175c7c4466a4cc21e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-27-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 723,\n      "height": 640,\n      "image_sha256": "346bc58f041f410cb81514198d1e07e4ac4586ff59941b3fa4ea575a439ce872",\n      "label_sha256": "ff341050df10da6605fb7dac27c87f01068d0bb1a5e2506bd880e3f4fe4aea07",\n      "name": "calculus_-27-_jpg.rf.de063216171f31f968920e4bdd75a3fb.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-27-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 1045,\n      "height": 640,\n      "image_sha256": "b502f9f94cfe5e3efb9b9241ceb141f10fffc229390114aed26f6454e7bd10e3",\n      "label_sha256": "2b26d453b4b57462a80d350ba395b53b05b8e191e7c3f016665b99ae2361f150",\n      "name": "calculus_-29-_jpg.rf.abe7917ec460d78f79bc6145841f3af4.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-29-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 1045,\n      "height": 640,\n      "image_sha256": "e63867405eb1e1accbfb8d2973fc95cea1001ae191d0182bb1bfedef298f55ee",\n      "label_sha256": "3546d90fd4fe4d3907350834876e2a5241e5c543bb4ad73e7652216487aee281",\n      "name": "calculus_-29-_jpg.rf.d85548bdab30fac9ce654cf48895e8a4.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-29-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 8,\n      "group": 922,\n      "height": 640,\n      "image_sha256": "43811d20f890ac6de9ab2cc1ad5f93576a6d32b5dfc84f173ebbdd624118118d",\n      "label_sha256": "cec4bc1f038e908cfd938042ff41ee2ab69580fbaca99357efb13126e81b706b",\n      "name": "calculus_-292-_jpg.rf.a8a1517c3412c649a9795b157bcaa03e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-292-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 734,\n      "height": 640,\n      "image_sha256": "50858eb85d4a9a34a9fa5c2c657a8579fd25c5ea8ed0ddaf565b41a82e86e3c7",\n      "label_sha256": "6ead52d25634c016cd1ded3cf16c89e0268be8f46bf2d4751beeb725ce54024c",\n      "name": "calculus_-624-_jpg.rf.70478a6ba333ac2e8befad6b0689fae2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-624-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 734,\n      "height": 640,\n      "image_sha256": "9c1fff2d0818a2934bcb96d3c5b5d401b614633c99e1052fbfb9332dfa819bed",\n      "label_sha256": "07bdf84db2d34e0b6aaf68339fff4131763568bb1aa624fbcdd6eae61525c4cd",\n      "name": "calculus_-624-_jpg.rf.7ee376bd887019496c41fcdba2605155.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-624-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 735,\n      "height": 640,\n      "image_sha256": "e3fc98a8748f4710796430d9d1a9060e5f14e55201e015a90caa9e90fecd85f1",\n      "label_sha256": "9358c80b1cfe1e9ce9d1167cd3b74dcadb496cf769063c0fe2ddee0ea17038d2",\n      "name": "calculus_-626-_jpg.rf.1748ac2be0523be2be1e3909a8e4a121.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-626-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 735,\n      "height": 640,\n      "image_sha256": "e41b68bac4c99ca82bc1075fa16a7ce95cfca3153f7955cd46509c20fedb4add",\n      "label_sha256": "d3dd64584c41c076c6518c60edc7c94942e3134ec6a7748f9443f534c80e16d2",\n      "name": "calculus_-626-_jpg.rf.3a8220522ba9a40099498bffe9b3ea53.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-626-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 11,\n      "group": 1080,\n      "height": 640,\n      "image_sha256": "a67810316fcf1fc91a89cd0d077d567dd62993868ea25e5a7a089bc3d6056466",\n      "label_sha256": "f042d5490878189f0bd45c1f0befba187099f81e0e067a7ab12e20ba3e0ecf5a",\n      "name": "calculus_-640-_jpg.rf.5ddfe5b32f716bee104d62f7ba7e323b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-640-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 11,\n      "group": 1080,\n      "height": 640,\n      "image_sha256": "719367aad041c448063d09b152ddff160c6a8b0525f452536b4ef31b482b0b5f",\n      "label_sha256": "99fa240fd8bc70681ddbfa1e0a28fbbff623c584d7652c32c3057324f574d69a",\n      "name": "calculus_-640-_jpg.rf.f52bdfdc7be74941b96fbe0e109e3d49.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-640-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 940,\n      "height": 640,\n      "image_sha256": "cb66a0da9ced06e5c7ae979b84a796a447cedf1d9d7367ce1b131d5c8302d874",\n      "label_sha256": "febbe9d213fbba9cc2374380ea0de22614860225b930af555f0e10cc0c794b4f",\n      "name": "calculus_-644-_jpg.rf.3c86be4a37f423e44deaa6c7fc35ac51.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-644-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 940,\n      "height": 640,\n      "image_sha256": "e3294bd2303118b7ccded799b90b9fcd70fb6d9f890ff0ef790fb8aad5a89d5e",\n      "label_sha256": "15300b3144cd072fd1d545d534507ffbb9cadfdcc868b2cdce42f09ba82cc4cb",\n      "name": "calculus_-644-_jpg.rf.6e4e952cbf5c79c483ddbf491165ff7d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-644-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 940,\n      "height": 640,\n      "image_sha256": "a5e57c5d0ab6f82b08ae147d2570ccfda91d3713e4cb03316b61c72e00552977",\n      "label_sha256": "111282324c6241d72f0715689cc69a757cec3ffea365c6c211131e42788533a3",\n      "name": "calculus_-644-_jpg.rf.a1758d5d5524a912b42ded6792ad3f4f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-644-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 8,\n      "group": 933,\n      "height": 640,\n      "image_sha256": "8926501221c4fe24cf97262d9dd7d247177d1f40d3883b2dbdab288023167fd9",\n      "label_sha256": "2e70cc9e26e3d3c899d117ebbfdddce48023999119d71077dc9a71af039eca97",\n      "name": "calculus_-674-_jpg.rf.202bfb1e9e74171720490560efd4342e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-674-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 8,\n      "group": 933,\n      "height": 640,\n      "image_sha256": "dee3e7398caaaad79bab7d4958e81891dd12357c4e5d71e17e10420e04bdba9a",\n      "label_sha256": "ee526d0fa6590a8c80177f670656480a6619fb64b0a439621def4fd2c106cf2c",\n      "name": "calculus_-674-_jpg.rf.85fb0a615ee05090cc60eac6ac9c2e8d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-674-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 8,\n      "group": 933,\n      "height": 640,\n      "image_sha256": "1e99f4715a37d7c2a838575d0e44fca194e50e524ee467ded8d13774e7ac6ce8",\n      "label_sha256": "62a3a7be452f719d00b14bf530793a092a0fbe6f5d3c3bf2efb1a7cb357ea08b",\n      "name": "calculus_-674-_jpg.rf.f59f5e8f93e45c6138d378d6a4019613.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-674-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 940,\n      "height": 640,\n      "image_sha256": "b4c5c95a470e32ef5ce02cd6455aae8454e89f8ba0f8833d2c4722bdc1fa0b05",\n      "label_sha256": "9db65847d763771aa8043552e36afecaba6724b8d84fb4d055208ee8ebca2462",\n      "name": "calculus_-68-_jpg.rf.06981a869d141bd5500379e83dbb954c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-68-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 940,\n      "height": 640,\n      "image_sha256": "f1dd2f73fc0769e1a83a3746fc04b9d1e86d711c3ef1fb863891e6bfff098f7c",\n      "label_sha256": "6c8da9e171373274962e3adb434a21d07d7bd381894c8c2063c61d330e9a3965",\n      "name": "calculus_-68-_jpg.rf.6ecc4d1e5de50134a274d317cac523ab.jpg",\n      "original_split": "train",\n      "source_key": "calculus_-68-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 942,\n      "height": 640,\n      "image_sha256": "35a7b73a93edbd66d39b018972442f6a5cd4220d0cab65e10e935946f9acd2c2",\n      "label_sha256": "58c20319f3cd248c04dce16e32f2af1b43d6ef91036fce6764b54109fab72c3f",\n      "name": "calculus_171_jpg.rf.0bea29a066a1ce09ca259dc75ee6ad0a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_171_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 12,\n      "group": 1080,\n      "height": 640,\n      "image_sha256": "06299cbfba109c8fb41dfe8c40111accf0d7852266da618fba5ccab5b8c96b83",\n      "label_sha256": "e446b3cefdea3b8d04604a5b6126d6c7d3d7af2f072a2fd9192430e73c0ef8a0",\n      "name": "calculus_174_jpg.rf.4d9029c6f3fbb8d0d44e2704ccd1fee8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_174_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 12,\n      "group": 1080,\n      "height": 640,\n      "image_sha256": "1be7998f0adc664742fcd9c3b6f85992c529a33a11d89f5b35cf2fe86c66bc8a",\n      "label_sha256": "fdd8c71744f2e34c867a665e182f221b1b48a5171be68350086f8d77a612aad7",\n      "name": "calculus_174_jpg.rf.b0b47647e9b58f5c5ead3c54d26cf8d6.jpg",\n      "original_split": "train",\n      "source_key": "calculus_174_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 616,\n      "height": 640,\n      "image_sha256": "17f9705052708c8b86db1799d2bc4ec07d1bc12d52c6c8c2ab1d8ace69756665",\n      "label_sha256": "d283f2831197ff1b8047a72d1cf02943708212e83aa5b909bee86c5c1fa412be",\n      "name": "calculus_177_jpg.rf.0b6106b252bc810834f92bc67d780290.jpg",\n      "original_split": "train",\n      "source_key": "calculus_177_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 616,\n      "height": 640,\n      "image_sha256": "ef262e1e6ab7f6bbc5564e7a608c30078077de830adaf53e315a18812edb1712",\n      "label_sha256": "488014e678d6dbe35130e06eb840146fb4c5508ea1d0edba9a0a5ecfd7712471",\n      "name": "calculus_177_jpg.rf.2526baac14f2a303a611dbf2fac1e7e9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_177_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 616,\n      "height": 640,\n      "image_sha256": "0143af385ffa8d46f49906d5747ef6450d78c5da14c6a994eaaea9a47dec3d2f",\n      "label_sha256": "26da45ff6cc7c8bc3b721942fea6a55722910ac9f289572777b86346380d3051",\n      "name": "calculus_177_jpg.rf.2dd984e9618be1207324d9e65a39ab31.jpg",\n      "original_split": "train",\n      "source_key": "calculus_177_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 945,\n      "height": 640,\n      "image_sha256": "e0f0a7693432e785e36cbecc5a076f4460da5b5156f606f3d35ee4086bd3deaf",\n      "label_sha256": "56032d5edd526cbc14298b353cde13caed0e3e424e51982de48178a3b60970ef",\n      "name": "calculus_20230302_113922_jpg.rf.13927255e4cc99c4122d8c65d9e93d63.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230302_113922_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 945,\n      "height": 640,\n      "image_sha256": "5643afca9c916acba1e7290ab3a99c49169adf3ecf9d1128ec2230dd72e7bd2b",\n      "label_sha256": "c125668842211a10fd06a009c12844510e57117073128f112782ae549c898907",\n      "name": "calculus_20230302_113922_jpg.rf.1a7fe1d4c33ff598b14d3f9e363e12b9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230302_113922_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 945,\n      "height": 640,\n      "image_sha256": "48431441301e8bdf4ae6ea91650c9a7f8272e3e5c9749a2919f84bd1fc154486",\n      "label_sha256": "d622f7c64d251cf10283116343e4e3f9b8d26dcb9c98cfb5d2fffb56a30cf2d4",\n      "name": "calculus_20230302_113922_jpg.rf.2ae03a72ef074ce4de3e89628b0e6015.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230302_113922_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 945,\n      "height": 640,\n      "image_sha256": "4c9f77232b927cfe4edcdaabc23425fc5de7c6ab106d16e632a391e593ccc166",\n      "label_sha256": "d622f7c64d251cf10283116343e4e3f9b8d26dcb9c98cfb5d2fffb56a30cf2d4",\n      "name": "calculus_20230302_113922_jpg.rf.4a3443005e9811839e8d06aca473cd58.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230302_113922_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 945,\n      "height": 640,\n      "image_sha256": "c7aad2667952b2f91e8a677b5af8306e90f1c9c4841624197fc6c3fc00aae3cd",\n      "label_sha256": "ea7d518c74b70c800da55ff9b1c7e0047a2807a22f21ae0fe81b7d875f51dc75",\n      "name": "calculus_20230302_113922_jpg.rf.675f54ba5dd0b489fa8a6d2626c5104f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230302_113922_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 945,\n      "height": 640,\n      "image_sha256": "55426b45a4d24b1b7b6a36926d780b9a52c5459a04c690901ecc785da38ed3c2",\n      "label_sha256": "c84be8d9937fbd7fc92d763450f725685e10d5de1d12ab962333aecce92bf64d",\n      "name": "calculus_20230302_113922_jpg.rf.8e762f39612047c5b1171fe155f99fba.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230302_113922_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 945,\n      "height": 640,\n      "image_sha256": "b2e43e21d934357fcdac8f88975454b61395720804cf7ed099780e6e3befeee3",\n      "label_sha256": "c6be6a034709ce33c305541a15000f1f5e12d095a29598eee5ff0aa7dd34071d",\n      "name": "calculus_20230302_113922_jpg.rf.aa7179c4218ed5ea7d65da7ca223c0cd.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230302_113922_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 945,\n      "height": 640,\n      "image_sha256": "0dbd7dd8beb61f194094af250c92bd0b4fcae7cc3e26e4e8d8193549bf25761c",\n      "label_sha256": "e00d06c73e01e0622f0f4c502ecdfd3c78f4cacf9de899db004350d42e2ce40c",\n      "name": "calculus_20230302_113922_jpg.rf.b1a745881296cead8ef755d9479baaaf.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230302_113922_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 945,\n      "height": 640,\n      "image_sha256": "91034a42aa8e29bd607e5fc85429635a05b091a6559f20548ab6adc8512f12f5",\n      "label_sha256": "bd29d5c4cc1ca1064168f4d0b210f3ae99ef9c3f19ae3ed4666cbba8737dff3f",\n      "name": "calculus_20230302_113922_jpg.rf.c6fe6be6fc408558b07d9c385ff688db.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230302_113922_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 945,\n      "height": 640,\n      "image_sha256": "21d9644a808efe90bae5f3895e6a3d0b1c337e128ca7a4329ed3e62ef790307b",\n      "label_sha256": "992b552fdaaedf4f3c26111434f87af941387b95ccb59ea7caeee1bc8322d475",\n      "name": "calculus_20230302_113922_jpg.rf.e5c0173e8a70c4f97d97732e9f1d3a03.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230302_113922_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 945,\n      "height": 640,\n      "image_sha256": "1a00f96b1833d5c5ecfaaf936ec75e9afb966527828e92a2342292db8143c8d1",\n      "label_sha256": "351ca629e63774d8535557384db324d3b04de2dde003bb32eb793554970f71ed",\n      "name": "calculus_20230302_113922_jpg.rf.ff4ef644326bc47fa0584bf3504f395a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230302_113922_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 947,\n      "height": 640,\n      "image_sha256": "1acfafbda95a148a5b600cd3827fba9e6249a37b2359f3637b701c66d59f67c9",\n      "label_sha256": "6fe20005545e4bce2223510d277adacbfba8e9f4604fad8685bc73426f921a2c",\n      "name": "calculus_20230315_150546_jpg.rf.0cac260e5ad1254eb3184e715f09232f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230315_150546_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 947,\n      "height": 640,\n      "image_sha256": "8c9cbeba4ba81565834882c0f858fe3d5167224234adf4e2889e2051709def8e",\n      "label_sha256": "3276f0fb1514830b711cb0579b17c39ed0a3819d795f79c9e573ca03b8857d34",\n      "name": "calculus_20230315_150546_jpg.rf.2695f34d66e5ed0bc5442117e1cb7b92.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230315_150546_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 947,\n      "height": 640,\n      "image_sha256": "55accdd9aa4617f0a2911e1c882c92c5d742cb14bef14f7c045b6f92b032253f",\n      "label_sha256": "4c0a85e938829681fbbf5806f4a949b23e89413b3f18129be8e44ef0ea66d607",\n      "name": "calculus_20230315_150546_jpg.rf.2bd64c0bbf7233d8cf455f2e6789399d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230315_150546_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 947,\n      "height": 640,\n      "image_sha256": "32deef38a8c18b540570699896cc33aa7e0815374c9f597f444b92d254f82b50",\n      "label_sha256": "a5de5ac2bcbffb61c5b816405e33103c38cc8528d57512a044d012735856764f",\n      "name": "calculus_20230315_150546_jpg.rf.3274082a9679234e170540598cd8fc03.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230315_150546_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 947,\n      "height": 640,\n      "image_sha256": "5b8ba6c56f35bba913aaf17d0de7350264c86f1e08a4709737416eb62782b476",\n      "label_sha256": "066e9a688a93b8ab00af237815f49598a9b2dda982eddc99b5e35e343c35266f",\n      "name": "calculus_20230315_150546_jpg.rf.b6e9d37c715522e5cd4b3557a6c84003.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230315_150546_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 947,\n      "height": 640,\n      "image_sha256": "1253aa886618e291c826b304c62ef17401d6db9035406b9b79bce0e24e13e4c4",\n      "label_sha256": "78bee2851d0a0290dbd6daf73055b3443f8a3af176c918819e3463973cf2badf",\n      "name": "calculus_20230315_150546_jpg.rf.befd95ec6607c8ef1b5d891b943ec4f5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230315_150546_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 947,\n      "height": 640,\n      "image_sha256": "acb137f64734b76c7134771f83255e0e34e7f329b2d635b2ad78c3f9006defcc",\n      "label_sha256": "428309f73a8fa78243d2b43f98cf1fdc2180e990a59a4ede52e8510169d6894d",\n      "name": "calculus_20230315_150546_jpg.rf.c17e5986251c01b76e20fc68b2be6904.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230315_150546_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 947,\n      "height": 640,\n      "image_sha256": "d4713c339ef3ade3572aa78c6bf32a983ceae5a565c7903ef8a55149d780a511",\n      "label_sha256": "6fe20005545e4bce2223510d277adacbfba8e9f4604fad8685bc73426f921a2c",\n      "name": "calculus_20230315_150546_jpg.rf.cc3a926d412875c4b2eda7cb8172ae0d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230315_150546_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 947,\n      "height": 640,\n      "image_sha256": "e9c619c15be28589fad0851edf7d070ac7c930490440c32e379c878dac02cd6e",\n      "label_sha256": "f74040ba38197088cd83308923d2f2c3023516f93562092b6e88df7f9c73839f",\n      "name": "calculus_20230315_150546_jpg.rf.d37bd708470b4cd1bdb385331c55b12b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230315_150546_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 947,\n      "height": 640,\n      "image_sha256": "6ab13f2c94f281bcfc325c3dd077867e4995b8edab1f0445435df267110f3f1f",\n      "label_sha256": "3f306064535132cd9178ca233b38cb742dcc7f62ff952550f888fa25e2132d51",\n      "name": "calculus_20230315_150546_jpg.rf.e21fd11bf7d15fbee4da05c3db731bc9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230315_150546_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 947,\n      "height": 640,\n      "image_sha256": "f7c742193ab48bf93b09bc47dee2a3c6a92f37172b87ee9ee42e201e1ad49dd7",\n      "label_sha256": "50af146b071063758a99e2d178fb783c3dc8cc21426453e107e57a6846b5e139",\n      "name": "calculus_20230315_150546_jpg.rf.e3a5deb727ab4d7c34520519346d8303.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230315_150546_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 1084,\n      "height": 640,\n      "image_sha256": "3ecf71947840d7752ca475c2e480830f626fff6f80bd592ef259ce0d9d771f96",\n      "label_sha256": "44cfc131fed7821891e415cb680bfad6cf33e965837d18718050b8983956c127",\n      "name": "calculus_20230328_133742_jpg.rf.0f16245df4cc9fd5b663c87af91387a9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133742_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 1084,\n      "height": 640,\n      "image_sha256": "aef91de6e7ce5bd725cceb7051ed3ed194af0e9ee86c42bcc4a77fb2ca91f79c",\n      "label_sha256": "facfbb1f9187ba24622e9747ea9a91f844fcf9377baf56cd39d21a3b2beae2a1",\n      "name": "calculus_20230328_133742_jpg.rf.10dcf2ec24014112ebc66226a13038d7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133742_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 1084,\n      "height": 640,\n      "image_sha256": "8f170cc8b677b82933b40381b17af2e6e842364d7b21fff4a2f0d3cb9bfbbac0",\n      "label_sha256": "d6bdd98cd3006d8d264451acb6705e789d2261a75c6a2b62939f6d8e241e68df",\n      "name": "calculus_20230328_133742_jpg.rf.17097f019fcec41eade2f5955dcad7a9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133742_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 1084,\n      "height": 640,\n      "image_sha256": "a5a910dca268f07202fc506846ee669f1aefd3b7f6bf95e81699f2208dec4974",\n      "label_sha256": "31ce45f695a9f0b2a73797eb3e8eb1043d9cea0114dc6fa0eb059894183160f4",\n      "name": "calculus_20230328_133742_jpg.rf.28b46cb68bd1eb6d6623edfdb404642b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133742_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 1084,\n      "height": 640,\n      "image_sha256": "addacc57331e9f5f7fa3949eaff083e6437eb315b403cefca2c9823cf625bbf8",\n      "label_sha256": "411e68fafbecbdf1a213eeb4c1683f8268568fc4eed07e4a85dd5c5caa3b3fb5",\n      "name": "calculus_20230328_133742_jpg.rf.4b16676f1ea594eb29ffd1ebeb13cd35.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133742_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 1084,\n      "height": 640,\n      "image_sha256": "70a29c4c8394ad1edc004f3fde28bc2e76bc09990b9fa90154923791122d8f0a",\n      "label_sha256": "44cfc131fed7821891e415cb680bfad6cf33e965837d18718050b8983956c127",\n      "name": "calculus_20230328_133742_jpg.rf.6829a9d0a6588e6da702538e4918d33c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133742_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 1084,\n      "height": 640,\n      "image_sha256": "6b1ae3d9744009c772f0f90225053865315c1a82038d0e7ad3d670d0385fc7a2",\n      "label_sha256": "6c5e1824f31cd9abbf995f24b98337c7cec15bf64e0b54266ef27f15d523e6c6",\n      "name": "calculus_20230328_133742_jpg.rf.8783498ec2b83f7362a88856bbae2924.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133742_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 1084,\n      "height": 640,\n      "image_sha256": "04633a96a081d5f3ead049b95c92d9b690242978e6d2bfc4a0c76011cd918a02",\n      "label_sha256": "c023c6da971ee3e341d8b7fd0d2348c13152ae1861e8ee97680ad5e0040487e7",\n      "name": "calculus_20230328_133742_jpg.rf.b566eb259367faed5e168e3483523a79.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133742_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 1084,\n      "height": 640,\n      "image_sha256": "4041209e422d4770812c156263c5b4d21787b6ba0e2e6998b466ff1e0a358271",\n      "label_sha256": "fae47eba7ad74da823240beda59a53f0ab09c44fbc0a390116de22d7dbc77b89",\n      "name": "calculus_20230328_133742_jpg.rf.f128991388fa3e79540ef2f7dcf65702.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133742_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 967,\n      "height": 640,\n      "image_sha256": "aba6f29e912da01738d1439e4b94f58e52e05a5bcb7ef17787a95ef355a0241b",\n      "label_sha256": "5a58f1d96d7de5f96dd9479dad506a22bfa71847870d36b69a7e8b6c9b96a8eb",\n      "name": "calculus_20230328_133746_jpg.rf.099d96cbd040ca485e3f1852c572b8cb.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133746_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 967,\n      "height": 640,\n      "image_sha256": "37769a018b588b64fef0d795b3474ae2b42dcb963fbc83cd36866aaffbf1b293",\n      "label_sha256": "051ad5c25e1ac2e672546b2985474efbadf349766cf89b86467f6d715704c818",\n      "name": "calculus_20230328_133746_jpg.rf.176acc7f39a09031198e378aa1fffcea.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133746_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 967,\n      "height": 640,\n      "image_sha256": "7abda9578b53a3ede8c20eaecdc8ff7cca505733273a1bbbf530b69d1a0999f1",\n      "label_sha256": "051ad5c25e1ac2e672546b2985474efbadf349766cf89b86467f6d715704c818",\n      "name": "calculus_20230328_133746_jpg.rf.2002e3b0887e9275396d9c4be385e027.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133746_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 967,\n      "height": 640,\n      "image_sha256": "43d73f38b08183ef30c53e242a960f108f7131a1f2417aa5cc9d8bb7a9b5122b",\n      "label_sha256": "b9eca0ae693fd67f246a16bc0b06b147d1707a098b04b7eef279c15cbabfe4a3",\n      "name": "calculus_20230328_133746_jpg.rf.27ec2bbbe06dd6f6045ab524bc6b7266.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133746_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 967,\n      "height": 640,\n      "image_sha256": "17cce1d9055149df347deb8d653bf93d79265ef182cc159ccce11f3d7fd57ff3",\n      "label_sha256": "107d047b610fd3a6602158e59c0ae84d212cade755d4ab78f8806bc8b7e5cd19",\n      "name": "calculus_20230328_133746_jpg.rf.31aefba19997a0d2c393bd35bece9782.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133746_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 967,\n      "height": 640,\n      "image_sha256": "967d7d076c8105aa9e013e8b29c4690289d37a766020b93333e0cd3f0f0b90df",\n      "label_sha256": "5681059a04f47ab395b84bfb5d29568fd43ee9c76b1e318ed1734d1fe4b1efd1",\n      "name": "calculus_20230328_133746_jpg.rf.494bf224005862e293a7018ba45c4ac3.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133746_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 967,\n      "height": 640,\n      "image_sha256": "77181ff0cb102183a5f1f9b091cdc7e5dccfeaa08795e36c9820a5644a5efedf",\n      "label_sha256": "d974b0b7d8ee58fbc537b57c89d66ca031809a04a44b5f7ba9f648df2ef64e7e",\n      "name": "calculus_20230328_133746_jpg.rf.620495e9b36bc2da9dd90df1ca647cd3.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133746_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 967,\n      "height": 640,\n      "image_sha256": "1bd347ce4a4a8a472098e90bc8c94253159e5cf624c85472a438ddd0c0712d16",\n      "label_sha256": "7972632c16035c51d45877faf035b2ab3508d45a70593fdec3e9971572de8865",\n      "name": "calculus_20230328_133746_jpg.rf.7aea2c2b11abc44b1ac2af33892d421b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133746_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 967,\n      "height": 640,\n      "image_sha256": "15979a01d81e3333b579a71181844f2aa74eebab84be001e02e26fa79e7139f8",\n      "label_sha256": "b9eca0ae693fd67f246a16bc0b06b147d1707a098b04b7eef279c15cbabfe4a3",\n      "name": "calculus_20230328_133746_jpg.rf.a3c3611d6c199e83d6ca03d8094345f7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133746_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 967,\n      "height": 640,\n      "image_sha256": "6a9a0800e45f76257547199b79e6e495d9c52b88ce619dcb4e3a70e8513839b6",\n      "label_sha256": "3c6fd80608063fc8444da7e6ba676367a8cca7f77f7d6d7e8686878a16d320ca",\n      "name": "calculus_20230328_133746_jpg.rf.bcd12a1b75c22df24f67888b444e9812.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133746_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 967,\n      "height": 640,\n      "image_sha256": "97136f99bd5edfc75797f1a387053367c0e90702ebd899ea0819d396092558f5",\n      "label_sha256": "af46f29159ef8c6003d742c4dd77d847320ca2e1d5342aaf44f6bee402bca859",\n      "name": "calculus_20230328_133746_jpg.rf.d796d63a9176d396129e00197affafb8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133746_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 960,\n      "height": 640,\n      "image_sha256": "07abfc7ef453139823274597ae10ac4e4d1916fd4c902e172cb54d83fe68f60d",\n      "label_sha256": "7f1dbaf21fa7212de2f65edbe280d300375d187a9f7815fc2ed38bfbb013796c",\n      "name": "calculus_20230328_133751_jpg.rf.21fe293ad6353e2c240213647aa7f609.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133751_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 960,\n      "height": 640,\n      "image_sha256": "4706b8bd99c2126f5e5278706aba0c41964a3f2776b4644d3b7ff2790a8e808a",\n      "label_sha256": "ebfc07cf131707aa07632c7cfc201a37f8edb425e356244d8e51adb33b2c7bde",\n      "name": "calculus_20230328_133751_jpg.rf.2feef6f344225a18f34d982b20e41db4.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133751_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 960,\n      "height": 640,\n      "image_sha256": "747ad05650ee18b951b4fcf21dd74d42237fdd633b3273c70728a843b6302a30",\n      "label_sha256": "df9f997c1777ab7655f6ccfbc89f08f8889ac5d81871065541f9fde3947ef51f",\n      "name": "calculus_20230328_133751_jpg.rf.65ca8bf4f9aa25bd74c8cb0de6833948.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133751_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 960,\n      "height": 640,\n      "image_sha256": "623a94928557bd5431d786d88559d9c5b7b3b3719154ecc3881ac5a15c549fe2",\n      "label_sha256": "df057f22d49303939e1d1492ae4d6dfa4514d63ef165fe7400d613a16a4c3075",\n      "name": "calculus_20230328_133751_jpg.rf.6ba9f8add8e8303eccaaee3b00ea4c8a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133751_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 960,\n      "height": 640,\n      "image_sha256": "d41b22c1e7d8009b6e14ec4639aeaadb31c265d516d6f2fc61b1061e9bc802a0",\n      "label_sha256": "3d2f9694e4ccee9e7960e5b4f6bc98d2a6dcd9f342f8e920e6ef64ae18cd3362",\n      "name": "calculus_20230328_133751_jpg.rf.86341b58957af6161979849035f665c7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133751_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 960,\n      "height": 640,\n      "image_sha256": "ac65d811700b881e72b89797417d8a00f2ea44766ea0dd701b4b7a66bbad7938",\n      "label_sha256": "41968f2c6d5e38e4b11282a7e60fdfdad88d8eb9fb89606fc4749b2086cc86a0",\n      "name": "calculus_20230328_133751_jpg.rf.90ac1cec7fed073d5ba9847b7d090c13.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133751_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 960,\n      "height": 640,\n      "image_sha256": "104b9884c9755ede3a6402a13f22caf4f112efb72f4b7eac27a9886ebd0c24ec",\n      "label_sha256": "77eba2fab87116dc9bd635ba53a419d0151096902e22ef3beced32beed578545",\n      "name": "calculus_20230328_133751_jpg.rf.a913a983cde4b26782e02cd134fcd83c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133751_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 960,\n      "height": 640,\n      "image_sha256": "565f5a4fd49f9f350299ee71db795bea939be703ebe1225ae3c27917a7fa6221",\n      "label_sha256": "fe6ff3b08b74208cb186c02cfe7c2dc4b7c4fa25c36f5e920cce4a3ffc9bb1e3",\n      "name": "calculus_20230328_133751_jpg.rf.ccfa14f207430a7992d297d7a4833647.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133751_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 960,\n      "height": 640,\n      "image_sha256": "c3ea8c3d88389f09e82d4c9e22db0f7abaf87419f77f7ed03ceb4f4882aa953e",\n      "label_sha256": "62a02ba7ee6159f18cbcaf61761d3496108b03a257bd3232ca550a9b1ef58a08",\n      "name": "calculus_20230328_133751_jpg.rf.d0f21a216ff7d29feadedfad9fb9f380.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133751_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 960,\n      "height": 640,\n      "image_sha256": "8b2690d127685e05c42c1ff0c310ac77f8889ee671b604952e6fac42d3ed2882",\n      "label_sha256": "df9f997c1777ab7655f6ccfbc89f08f8889ac5d81871065541f9fde3947ef51f",\n      "name": "calculus_20230328_133751_jpg.rf.e0ec74e5a30c610392906286ecdbc529.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133751_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 960,\n      "height": 640,\n      "image_sha256": "dc6aa4d84138a64b07d4edd9ce9460f7cc95dd2a5dfc851dfa09359ebf7b1e9a",\n      "label_sha256": "c742195290f4db26d75c4bfd1147d795deae291b82f5fc86b0bab2c2716a6b2c",\n      "name": "calculus_20230328_133751_jpg.rf.e1cc9ce4bd567c695c03e2ef0cbf2b15.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133751_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 960,\n      "height": 640,\n      "image_sha256": "c68a09fcaee1dda5d60ffe5cf7b96b38a0756c5c7306502174739e25096f7392",\n      "label_sha256": "2941c6d63045b55d72ffba183d770aa561b7a851c1366d9ce92ffdff9450270c",\n      "name": "calculus_20230328_133751_jpg.rf.e844fa569891aa6d411a7ea1f39fc6f7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133751_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 960,\n      "height": 640,\n      "image_sha256": "5b878cd3c4bfb0ffb0496881768157ed1c67338416c9a82f6fa5c4d7896ad064",\n      "label_sha256": "bd132a8c1687934c5472b7bc2b926521c225f4a9837df149911d010e54475bf8",\n      "name": "calculus_20230328_133751_jpg.rf.f26211b3525230785bd6637f3031ab5b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133751_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 966,\n      "height": 640,\n      "image_sha256": "7623c40479dba9d24b70d44587f7aae66383c0fe713b2921d00627d01a22e877",\n      "label_sha256": "15274caea91ffe7d16fee4cb1341f55c33e02627cd02d7708eed4871b30091f9",\n      "name": "calculus_20230328_133757_jpg.rf.2368839b09d70f3c23e3a75a5a6afa78.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133757_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 966,\n      "height": 640,\n      "image_sha256": "c75c0a828a280b11135deacba8c9d67a75d7f4e1d566ab71fbdf29c5a8964c0a",\n      "label_sha256": "2dcec3d06bab50e62368358424dc3646622a5b9ba72c4851fd91f14bdee6d31a",\n      "name": "calculus_20230328_133757_jpg.rf.32aa09844bf696971baacba88bd35459.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133757_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 966,\n      "height": 640,\n      "image_sha256": "cf67328ca95e6c1b17395d6645a62f8b513f5b57ec7efc37eddbaa45b4ef18e1",\n      "label_sha256": "8979599c0fabad4fb30e4242d86d982a926fd6daca2bd9466bd3d7bebeaacf91",\n      "name": "calculus_20230328_133757_jpg.rf.3add964175385274ed175809add75ca0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133757_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 966,\n      "height": 640,\n      "image_sha256": "266742a76b2a7b2b71d9a27c6d89c186a28b09389a37eae9b7bc97fb639f0a7a",\n      "label_sha256": "15274caea91ffe7d16fee4cb1341f55c33e02627cd02d7708eed4871b30091f9",\n      "name": "calculus_20230328_133757_jpg.rf.435782300fb7441e505daba1523d1834.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133757_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 966,\n      "height": 640,\n      "image_sha256": "283f3bbd0109dc4e19c979e0108778f69ad1c3e8f43cd205d1b77535c6977dfa",\n      "label_sha256": "ff4288f975b054d08c7942b4061915195c0227ce27df916eb99217f09715147b",\n      "name": "calculus_20230328_133757_jpg.rf.5b179c957d03b60856a9039a971eb2f0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133757_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 966,\n      "height": 640,\n      "image_sha256": "ad355e89e6313c4b093e0b880dd7d6c8fa7148a07ac88f2b19cf774d5ca81a07",\n      "label_sha256": "1f2616a7c0ab9d6bba2d7a54a88f73504fb85bc616163f0c44e960030131ad7e",\n      "name": "calculus_20230328_133757_jpg.rf.5ec1b69ba08c48180686b79f5b49c240.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133757_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 966,\n      "height": 640,\n      "image_sha256": "28cfbf8ecd3cda78f1ac52aa11f494b887283a174e624e5518e94fdc1a72a78a",\n      "label_sha256": "e035a644a32172ff94fd69c64a57416179e567420f06b8c05a4b9aab396a4065",\n      "name": "calculus_20230328_133757_jpg.rf.ad1362bbeea2c423b2d11648dabc1b8b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133757_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 966,\n      "height": 640,\n      "image_sha256": "0a11e823df9406371905161d0d4d2556fabacd469dbafbf3d377515a6bbb7d4d",\n      "label_sha256": "b06f2fdf5173a4a7c1a52d21ec6d9beb6eaadf25eb84921979bf49c17c0da1b7",\n      "name": "calculus_20230328_133757_jpg.rf.de701bf6862eee6e3207756d0c7e694e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133757_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 966,\n      "height": 640,\n      "image_sha256": "8e42dd288d99ae1b1efc64700797fd002284c9bc92bd6b23f6d660e7c6575ebd",\n      "label_sha256": "c5ccf1ce09709217735462c45d1fc524a447fd871d16a60f072875f4dc3a7a13",\n      "name": "calculus_20230328_133757_jpg.rf.e31a7810ef6554350c7b2bfc638f0b64.jpg",\n      "original_split": "train",\n      "source_key": "calculus_20230328_133757_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 370,\n      "height": 640,\n      "image_sha256": "83799042b017307a9f54e6736f959f619f8db64b05de7cab3bc09097806d98bb",\n      "label_sha256": "a079265b2c2900c7e519fe120b0a81db4db982761b485576ec552d584ff31697",\n      "name": "calculus_calculua34_jpg.rf.6e4da535ae9291725d3a29e45fa5d68a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_calculua34_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 370,\n      "height": 640,\n      "image_sha256": "b0f062b8fa88163159ef32db1141b4e090c765c632d3d5741fd4b0df3130c8c2",\n      "label_sha256": "1ffa05793c1ba945bc6bb87653e036dcb284d01a5b4857dcd2c62cf5e1009564",\n      "name": "calculus_calculua34_jpg.rf.b36e2b929f0cf0f924ec5d699158901b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_calculua34_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 967,\n      "height": 640,\n      "image_sha256": "30e41a41bf875385485f1ccd83e96c065222aa9009a34fd9b46931627f8e02af",\n      "label_sha256": "ae2f759a7519e0d48ce9b18e2b20c54f4cf3a903822b248e6707d4bdea08bea2",\n      "name": "calculus_calculus-3-_jpg.rf.774881cf72a51a93bbda803e1abb07f5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_calculus-3-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 967,\n      "height": 640,\n      "image_sha256": "c7d2203d49d3fba935746d2bf30b3abedfb48f0adecf7fa4b2c3b573d1652c03",\n      "label_sha256": "d43f5be84da81754110b7799bc37614008238bfd7228665327f0170a29dc5b5a",\n      "name": "calculus_calculus-3-_jpg.rf.df4451c5de08859bcfeb3b17ffcb0c9e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_calculus-3-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 375,\n      "height": 640,\n      "image_sha256": "66453345663f4ded697c5855c1952eb263419795abbf3f7c97d2b72d1501acf8",\n      "label_sha256": "d8684e8b7b755103d33b765700a4c5a4946ec588b70f632923aaa34c13b097f5",\n      "name": "calculus_calculus-4-_jpg.rf.2a9d9be343925f0713b63cbd5ace5790.jpg",\n      "original_split": "train",\n      "source_key": "calculus_calculus-4-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 375,\n      "height": 640,\n      "image_sha256": "aeb7d8da9fdf48ddbb371b153b8bc0e6ef502be05bd6658c4b0e910057e31b64",\n      "label_sha256": "18b5408e760283e25fba07f02a42ae20775d2898228486edfb2c0c62ef412695",\n      "name": "calculus_calculus-4-_jpg.rf.a4675d6bc85423a5af2f1bf008df64d5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_calculus-4-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 375,\n      "height": 640,\n      "image_sha256": "13a32c090a2503a7309a74baaae429a04c62c6a89a71f4b2c5b51d5e6c5fdb5c",\n      "label_sha256": "c5cddcfeeffea8156efd1edeb39bcc846195fbc142887bf297a04772a20673c6",\n      "name": "calculus_calculus-4-_jpg.rf.e9b1ecbc07e07bef89e7537be018b8ab.jpg",\n      "original_split": "train",\n      "source_key": "calculus_calculus-4-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 968,\n      "height": 640,\n      "image_sha256": "1838f62a1ed06bac3de71917948d25a90c54634386b21a665d92705bf051ec28",\n      "label_sha256": "e0e7e402005e94a5d7cff63f626facf6a75a23b9cb254e24971efea9f4d4567b",\n      "name": "calculus_calculus_jpg.rf.216cb985c2b9d674719c91fe32ae9903.jpg",\n      "original_split": "train",\n      "source_key": "calculus_calculus_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 968,\n      "height": 640,\n      "image_sha256": "63e3ea9c7836ead6f4c0daebb80f2cb2a947e68d119b2d72663fd899c6ac7fb8",\n      "label_sha256": "a91e543ece0b0b5192c52f86879603ac770d645398ae46891e66574d88b69360",\n      "name": "calculus_calculus_jpg.rf.62d6e3ad3864ce48ea710aa04d882088.jpg",\n      "original_split": "train",\n      "source_key": "calculus_calculus_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 379,\n      "height": 640,\n      "image_sha256": "5fd7f514ee8f8b2e0173b242aa7e8e6406e53c5b9f1964bc08146f82bfee4e01",\n      "label_sha256": "5843972674ae3bd3b447e0454369eee3ae6f5c6e439bf5aa501bd6bbf6db8951",\n      "name": "calculus_calcuslu78_jpg.rf.0687a8f38f5461816131b567e1723cd2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_calcuslu78_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 379,\n      "height": 640,\n      "image_sha256": "cd429f6cec263bf81b962f753134617b70c4859a3da8a6ea4be850adcf4241f4",\n      "label_sha256": "a787f0366f4b74b760688a844de5d295e04d12ab8b53b9a41071a13fa1789046",\n      "name": "calculus_calcuslu78_jpg.rf.41bc3307bf0e5cf997fa65f849c8b18c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_calcuslu78_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 382,\n      "height": 640,\n      "image_sha256": "cd9db506b5d0ba18a0008ce9020e0c2240fd7c03c6cac910c1822c37bc46b329",\n      "label_sha256": "2f33a626f33e34088a3d82bfd97259ce2f3a627bc94e6109cb73c7859d323a16",\n      "name": "calculus_download-4-_jpeg_jpg.rf.29747c90601bbc3f19c375eefb587b90.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-4-_jpeg_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 382,\n      "height": 640,\n      "image_sha256": "be2f8ebd7827e6d5d2eea82b34ec420424aa60adfeb5b45d93b227bf6bf17c33",\n      "label_sha256": "bb9871d45db72a1b599df46600f603f474d57afe33f6acab252683a7647261e2",\n      "name": "calculus_download-4-_jpeg_jpg.rf.8508016886b086435d7ad0e4c3004c44.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-4-_jpeg_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 382,\n      "height": 640,\n      "image_sha256": "51ea179a9d6d98aec1425f3abcc94ce44f4765d0afbaf4cc335a33b61a3b3342",\n      "label_sha256": "a6875ad1b95ee639ba52c7337c9d98fc2d6ddcd227600063f55afecf465049c3",\n      "name": "calculus_download-4-_jpeg_jpg.rf.ce2ec6a6f19cd5edd5635084cfa58ee4.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-4-_jpeg_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 395,\n      "height": 640,\n      "image_sha256": "fe75d7cbb3f60b5e75f80d0cc6ba9d09b47fd09dddc4561437b68a5c72703e19",\n      "label_sha256": "157d2b9dae0caf80d2080791bb3b81aac417b5919b527ba0c6b09f68bd121cb2",\n      "name": "calculus_download-5-_jpeg_jpg.rf.15bc14a4207da3501dfc2aed2f7a725b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-5-_jpeg_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 395,\n      "height": 640,\n      "image_sha256": "96bfa84ebee5cd19fea66239599f223b1792053dc85d17600ffc96839bd64d52",\n      "label_sha256": "3e5f505eacce7a61039d7a10131054774d1ce237818c51f848afc7a975468d7f",\n      "name": "calculus_download-5-_jpeg_jpg.rf.20479731e298302adcce77dbf648603d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-5-_jpeg_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 395,\n      "height": 640,\n      "image_sha256": "1ffd2abe8c2a8f75965836f751fb36a0aed6233a096dcfe9fcd1a5f40a7fc170",\n      "label_sha256": "dafd79be0ad441702dd69a9bafb3bdb3b4a399db3503d784aca5c2d8a253dc72",\n      "name": "calculus_download-5-_jpeg_jpg.rf.229771a02800d06fcc4d0dafbfc28e39.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-5-_jpeg_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 395,\n      "height": 640,\n      "image_sha256": "10af21959dc58d535000113cad3640b87a784fed1682d0b22b8d2c07feca95fd",\n      "label_sha256": "195f2bb64b272ad91c9ec097d986b6fb347841cd84f4abad42356ac93cb4b44b",\n      "name": "calculus_download-5-_jpeg_jpg.rf.442f815ca7b13d26bf1488dc4513179c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-5-_jpeg_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 395,\n      "height": 640,\n      "image_sha256": "a6fe4a9b6df7897032be738ecabe57232e870bc609c45f035b44417f0a09a3af",\n      "label_sha256": "7e1f10631ed6f7877810ab8e47f74318753dc6f426703f4a38db67243f6fdb04",\n      "name": "calculus_download-5-_jpeg_jpg.rf.579e543340cf7210339dccafbbfd0956.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-5-_jpeg_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 395,\n      "height": 640,\n      "image_sha256": "a6fe4a9b6df7897032be738ecabe57232e870bc609c45f035b44417f0a09a3af",\n      "label_sha256": "7e1f10631ed6f7877810ab8e47f74318753dc6f426703f4a38db67243f6fdb04",\n      "name": "calculus_download-5-_jpeg_jpg.rf.6b0df2dfb393deae852a2554a67a80ef.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-5-_jpeg_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 395,\n      "height": 640,\n      "image_sha256": "fe013162b54815025580222b451fc90d0ffd1e56dbd0740d55a9c6c9a9c3bf53",\n      "label_sha256": "d1c8632837be229e7318cac4baf2b255df449c4409b08b64d727b1008fc8273f",\n      "name": "calculus_download-5-_jpeg_jpg.rf.6cb4a9aae6032f15ece3e54a510206c8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-5-_jpeg_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 395,\n      "height": 640,\n      "image_sha256": "8d5a582a165f4f0aaae18c0c929c029a1bc25cc0483330f98910f02d41b88426",\n      "label_sha256": "1ecfabb097e6ccc8c81a537c8275529715289a47a7e670a6d7b492182ffc2475",\n      "name": "calculus_download-5-_jpeg_jpg.rf.82ceaaea122a4e6aa2bf7662830f2fdc.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-5-_jpeg_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 395,\n      "height": 640,\n      "image_sha256": "886b9953e7f058d1d511a65c63c21ec23fbfd0a382707ec3fa6f5a0793cc5256",\n      "label_sha256": "b9f18cccdfa63c5e38239732a27ef77990bed49f007325a4edf8bb330975d080",\n      "name": "calculus_download-5-_jpeg_jpg.rf.977bc77db2973276ead104c948d8ff4a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-5-_jpeg_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 395,\n      "height": 640,\n      "image_sha256": "1ffd2abe8c2a8f75965836f751fb36a0aed6233a096dcfe9fcd1a5f40a7fc170",\n      "label_sha256": "dafd79be0ad441702dd69a9bafb3bdb3b4a399db3503d784aca5c2d8a253dc72",\n      "name": "calculus_download-5-_jpeg_jpg.rf.9fb9911c3095dd67f3da7694a61572c5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-5-_jpeg_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 395,\n      "height": 640,\n      "image_sha256": "917cad6623a6d63246186bc1aa512c5500ede64830261117e5e8229058a60251",\n      "label_sha256": "07ad75be3ac8537999ef18f53c89ae6a141a110f1b5660193387374cd059626c",\n      "name": "calculus_download-5-_jpeg_jpg.rf.bd47cdfbb6f352cd0afc4385074b2663.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-5-_jpeg_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 395,\n      "height": 640,\n      "image_sha256": "8de2a3cad04c0aaad6fa32a39e8709e29c3509207e428068240bb92bef1d8167",\n      "label_sha256": "c680a4a533a00768b050b5fd0d8420325089320735b5e7fd81e47163f9bf157c",\n      "name": "calculus_download-5-_jpeg_jpg.rf.cf22e0836632253bb961f6c213562408.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-5-_jpeg_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 395,\n      "height": 640,\n      "image_sha256": "8a2170e2127a3f2d0791b099086a934f32dace4952890fe0e107abec6d0fd84a",\n      "label_sha256": "61aceaf5aa3f660aa7a42aa95c0193a2a9fcc9171cd164ea5d9a011bea051f0c",\n      "name": "calculus_download-5-_jpeg_jpg.rf.d25353a1c076d9113ba8fb29e1b222b0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-5-_jpeg_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 970,\n      "height": 640,\n      "image_sha256": "90027d459e5de00dcbb429d608011bd2e29faee1c6d46ab3c259366b7d07754a",\n      "label_sha256": "2c20cd090b2c4f7b2f3946f2fecdcd60a60af6b893920ac51d7f4c4cf4da1ec4",\n      "name": "calculus_download-6-_jpeg_jpg.rf.07501304666bc9158a0f58c5e8eed10a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-6-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 970,\n      "height": 640,\n      "image_sha256": "28563900a1b52477dc5f19cd3b0a13d56c5fa844166c30b1c228947fdfab3214",\n      "label_sha256": "db54b1034c7af408c82e8a029401fbd452adc9c72baa49a2cef38a364702ef39",\n      "name": "calculus_download-6-_jpeg_jpg.rf.1fed31c08872aa5925d4e4b0ee670efc.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-6-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 970,\n      "height": 640,\n      "image_sha256": "778d1c5633e4598c100e2a74d37580a719cebc50162a0e955e34cfa2a5c66c25",\n      "label_sha256": "f2f38fdfd33a4e8e1dd1f8428d5cb133faf744e35177444ea857afdcd67678d0",\n      "name": "calculus_download-6-_jpeg_jpg.rf.23474aefc254f1209f16dff5f5bf4a72.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-6-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 970,\n      "height": 640,\n      "image_sha256": "71b5f65b57d9bce70519396d3397e7cdd17b9a8bb4411f8396ced244f3d79900",\n      "label_sha256": "f2f38fdfd33a4e8e1dd1f8428d5cb133faf744e35177444ea857afdcd67678d0",\n      "name": "calculus_download-6-_jpeg_jpg.rf.32064f7d4d9713ef9a365fe05651a42a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-6-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 970,\n      "height": 640,\n      "image_sha256": "3265368cea89bb234b2b97f44146d73a1b9bb0857b0b9ec44e95968e25d2cbf9",\n      "label_sha256": "2bfc3d62921b2f176bcd9d2885c3464b9ca9b4bab8ea294c85f8dbe408a109df",\n      "name": "calculus_download-6-_jpeg_jpg.rf.58457894e9b3677eaa2dd3c8926a2d2d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-6-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 970,\n      "height": 640,\n      "image_sha256": "8d97c71c90c599afd272bf62ecb0fb8a7d06909ad79445459201a6ffbdfba8e8",\n      "label_sha256": "ce1cceb2b1656f868249b18026325f0091ca6be73327d20541578ba8cf882346",\n      "name": "calculus_download-6-_jpeg_jpg.rf.7238be0706731768807b567ff0e4fe84.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-6-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 970,\n      "height": 640,\n      "image_sha256": "ee1238c0b04f318dd684782c4552d08bbe8150db54cc1e2307122d622b0855e7",\n      "label_sha256": "03fdc475007dc55d5638e0ac193c3ac724b72689a46c0293456c7e79887708e4",\n      "name": "calculus_download-6-_jpeg_jpg.rf.a706064663896dc5cc9d9826856e754d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-6-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 970,\n      "height": 640,\n      "image_sha256": "848774926561d6cbc1298e322a6ba523e5fb218cf61fb5f12da043378620af93",\n      "label_sha256": "23cf41e2eeee6e06978c0511714b73f46c9aca5c14d2c900fe18d389dc87d8c3",\n      "name": "calculus_download-6-_jpeg_jpg.rf.ab563e9d603fb3592d263465a203e0f9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-6-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 970,\n      "height": 640,\n      "image_sha256": "39acfaf1010d965b4c4a274fcbe0cfa8ab5266b95304fd71a6b8bd41447bb5e0",\n      "label_sha256": "bbab64afc80255ae9ca6e9a904fe9e583e40db4c105a4f137f7c7d4f5bd17af2",\n      "name": "calculus_download-6-_jpeg_jpg.rf.e1d1d9e23ea9fdb76bd4bc70f565499b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-6-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 970,\n      "height": 640,\n      "image_sha256": "3cc145cd42a870ce7bf2523435ae702013b773771eb2d734020103ba9ae8e194",\n      "label_sha256": "a5f601cb8b587dfc690ebda6ade98b24f6e50112fbf550aa8792faa83b4c74df",\n      "name": "calculus_download-6-_jpeg_jpg.rf.ea411958e3f0d114d4f0fed2bf5f4bbd.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download-6-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 976,\n      "height": 640,\n      "image_sha256": "69abd2e072b483bd9fa7cb755bd676470f18cb7278126944c3baa154c9de3f46",\n      "label_sha256": "029f89a405178016c8a69b83569fb8bd29191326b43ef48a4ae65aa8e5e46291",\n      "name": "calculus_download_jpeg_jpg.rf.38decce75d8e116ee6f0a358938af96b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 976,\n      "height": 640,\n      "image_sha256": "477e9863a91b43bd8db5a1f464a4c4baef38f6d55c089bb88ebb3d05270f0921",\n      "label_sha256": "ccdaec834cd42b79a9e6bc6d6e396f784d50a3512fc1b6f870cf6aae97cf8484",\n      "name": "calculus_download_jpeg_jpg.rf.4ad9b277c2dd2903da23db5df2920860.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 976,\n      "height": 640,\n      "image_sha256": "e52f3200a3812c93a040333ca39db5d96c70acdfe71396c4ee7ccb417fcce933",\n      "label_sha256": "c670af0fcb95aa7ed495719064c80b9859b9ca3b54744ba32d234407c7dc3ee8",\n      "name": "calculus_download_jpeg_jpg.rf.4c7597350a3974d70dffaadbce367231.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 976,\n      "height": 640,\n      "image_sha256": "bcde21f8106474cbd29f9f8fc67433744b30d603cdd83871db814a25aee8a256",\n      "label_sha256": "aa1e1f0aa686ea82d4cfa0ae61cd34ecd2ae5d16f23c6a04704af84a2caaf035",\n      "name": "calculus_download_jpeg_jpg.rf.62da289230903889b8c85db80affed8e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 976,\n      "height": 640,\n      "image_sha256": "6159e2daf5c351e1850d7e03b9cc8bfbc3cf2cd53f87feac77a92d9392ca85a2",\n      "label_sha256": "0c29bbe2040ef7e38234e7d7670432963d132206f67b0f7a6e6410812c47fd57",\n      "name": "calculus_download_jpeg_jpg.rf.63100f87a8eefcb7a8aec7e59603dc88.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 976,\n      "height": 640,\n      "image_sha256": "8d44e41e2097077d6c646046e2db544e2d59b699d2dd368e474c41cd2b0c4231",\n      "label_sha256": "029f89a405178016c8a69b83569fb8bd29191326b43ef48a4ae65aa8e5e46291",\n      "name": "calculus_download_jpeg_jpg.rf.8afd4dfbdc0fdbabe2be318da514f718.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 976,\n      "height": 640,\n      "image_sha256": "2cff384750deed740c7cfd8d0d25f2c0bcefbdd8ddc88288af2d20c4779b5a8f",\n      "label_sha256": "7fc4bca2da07b0ea09b9beb09a9977ef3a438746b97f59aca35b11dbd85317a9",\n      "name": "calculus_download_jpeg_jpg.rf.95b100943136829ace1994a3aaa2bef9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 976,\n      "height": 640,\n      "image_sha256": "556b70632a1cea57135454bb26fbd5be0a134d01eb14978810430ea49f4c291e",\n      "label_sha256": "2c4dfd6e1969e08d3eb00bee2c33e59ad10028641d564062c5efc4c5f024b637",\n      "name": "calculus_download_jpeg_jpg.rf.da30365e7b90d6a67ec48df640412a56.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 976,\n      "height": 640,\n      "image_sha256": "3f58e3ddade2f3bee1133f36c0096c5db507d774386a6afe69e77172d7d117f6",\n      "label_sha256": "4b25c5b0f724ae190274acdc621da2e33c2437753066a2bb4f692fdab227d5df",\n      "name": "calculus_download_jpeg_jpg.rf.eb05dbea3746b5498bc9c9c6b00a54ed.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 976,\n      "height": 640,\n      "image_sha256": "d5d9f18b36eb3bd379e9e9a6604ae7a22b5ce299a6b3062fa89d688f07bb6562",\n      "label_sha256": "ccdaec834cd42b79a9e6bc6d6e396f784d50a3512fc1b6f870cf6aae97cf8484",\n      "name": "calculus_download_jpeg_jpg.rf.f084cce60c455784fea845d3df4f6fa7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_download_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 978,\n      "height": 640,\n      "image_sha256": "a57a55f441d37c74989d027918474b3fb47da0a06a43eaebd1274b6fee8108fa",\n      "label_sha256": "a42944c9d4b7c8ad48a78b37913d4505f1013162b1ccd9703c64fa7ab9f8c7a5",\n      "name": "calculus_images-1-_jpeg_jpg.rf.1b8fdd801233938268ff2ddabf97b7c5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-1-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 978,\n      "height": 640,\n      "image_sha256": "9622c46ca5453b57974988c38aa271f321ec0b2b3de1ec61638f4264967cf5b2",\n      "label_sha256": "c8efb466f529e0877ef7ad3727f2b1bcd1269564e96098de44b89e212ed51460",\n      "name": "calculus_images-1-_jpeg_jpg.rf.2d4770079e1f0e1e84dc1fa385141cff.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-1-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 978,\n      "height": 640,\n      "image_sha256": "dc9f12cc95a1730722dd04f0fffca77ef70481ff8fa8eb768036215100f77c30",\n      "label_sha256": "87ec39cdfcb8b690f52d3756a2061f5f560610296a3dbc07da6a1693336a4407",\n      "name": "calculus_images-1-_jpeg_jpg.rf.4b7c9872f1d6abc2b7a14cfcc17705ce.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-1-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 978,\n      "height": 640,\n      "image_sha256": "b38e11eb7455040e9105ba42b8b49161eae843be6b693d839743483db725edc1",\n      "label_sha256": "0722bac3042e3110a54ad8a220a4a966936ec3724b1c4f457f9fa32e1db130e1",\n      "name": "calculus_images-1-_jpeg_jpg.rf.6ebccf2cc7096a8a4033f6f3eec2f345.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-1-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 978,\n      "height": 640,\n      "image_sha256": "e119de0d9d415db7d4a0ecf265cada71c56c3cee63400989a9d142da65d1662d",\n      "label_sha256": "d68ddbfc0a7383ffcbeef599eae2b4a302b7f204dd062daab692a040b1d876c7",\n      "name": "calculus_images-1-_jpeg_jpg.rf.7bcbe5ebbde2dd36fe0684f93586a569.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-1-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 978,\n      "height": 640,\n      "image_sha256": "18fc3099f0e5d497c3cbfe106047424bd08425f276d12499b11aa93d8e926742",\n      "label_sha256": "c8efb466f529e0877ef7ad3727f2b1bcd1269564e96098de44b89e212ed51460",\n      "name": "calculus_images-1-_jpeg_jpg.rf.9d4d054f3b93c8f42c30c14a5e9e6076.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-1-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 978,\n      "height": 640,\n      "image_sha256": "f5407429130f5187b20a1797dc867aeef2ee2475533aa752b5a39ec73909973b",\n      "label_sha256": "b6d4d8d6ad43ef418aadfdcf2e1bf51e002ca4e9121b3070e70102ca37768f7a",\n      "name": "calculus_images-1-_jpeg_jpg.rf.d91c4b11f2b0c1057a8e4ef7781ef7b1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-1-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 978,\n      "height": 640,\n      "image_sha256": "736a1ed5dab470da4cc888debbe76249cd60c177d163f1b253986a3bfe5f348d",\n      "label_sha256": "1bb2c4f135aed12e2d8ce31312ba6c92e96d192f65d09da4028d4f566ee83cc7",\n      "name": "calculus_images-1-_jpeg_jpg.rf.f3c3b0ad286d125e56d8ad7e5b77929c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-1-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 981,\n      "height": 640,\n      "image_sha256": "6f12bc7b58db2912e7b384fb5bce160b8978697b9d1e40e6b73759f6ac65375c",\n      "label_sha256": "a865ac01c43be3990596250cb3dc36ebb3428c94d5f654fed273344963708cab",\n      "name": "calculus_images-10-_jpeg_jpg.rf.0320822ce211db87137e57640bd54cf2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-10-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 981,\n      "height": 640,\n      "image_sha256": "314cb5e5c22fcbd493d1256649a8a8d72bf5cecfcfed2b714c7c28591e56f71d",\n      "label_sha256": "eb1152746b8bed05844c4aca3f625b7eeb118f9b8e314c8924b816a38970b127",\n      "name": "calculus_images-10-_jpeg_jpg.rf.2032c191e3506e0d3f960b272e3f1a87.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-10-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 981,\n      "height": 640,\n      "image_sha256": "e582d6cf591d431b480d361a2e58d8c4d3c73294cdd83255351497ee212bc8be",\n      "label_sha256": "2b1a5007f6617dbd25fe7e8f35ed6d6c40e4d02072e800de8ac5fc1b0f968ef5",\n      "name": "calculus_images-10-_jpeg_jpg.rf.203f9f4a6be939be73407e4371c3fe0f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-10-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 981,\n      "height": 640,\n      "image_sha256": "af7de009a8486620131ea376d38b4361b5db3667c1e8d0339066bab0d568bdb2",\n      "label_sha256": "ed5061c4ed5b12d78b84be7c6f6b616123c3decc70f86f30e54c5412fa89b669",\n      "name": "calculus_images-10-_jpeg_jpg.rf.3b8d58e6a32d634149d447b96f8a9788.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-10-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 981,\n      "height": 640,\n      "image_sha256": "de3ed613cd1de0a09a88fdbd86fe25be877ad8cd25af1c4cd24ca4b264565f7a",\n      "label_sha256": "a865ac01c43be3990596250cb3dc36ebb3428c94d5f654fed273344963708cab",\n      "name": "calculus_images-10-_jpeg_jpg.rf.5a060dc64ea6d15e1120a38ae105bfcf.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-10-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 981,\n      "height": 640,\n      "image_sha256": "f6a5667b466bba00e5493a0c51d3aa89d1dea8b6f047f4c91f6aded6aa6569d8",\n      "label_sha256": "5f694250c1ebc0289e8373fc68ca6399c263fa20ee2952516da5ce1a0c4b32d3",\n      "name": "calculus_images-10-_jpeg_jpg.rf.6cf92567bff35cf3083e93c9cb0ae5a0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-10-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 981,\n      "height": 640,\n      "image_sha256": "46e1b727a94bcd8cb4964300af17cfd5d817275d61e13b1ca370849f78d9e5a0",\n      "label_sha256": "89416011e9af8db78c285c22edf435448886825b582b65e6348c84e5d8a84714",\n      "name": "calculus_images-10-_jpeg_jpg.rf.84981a973f7210fc6c79301566589045.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-10-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 981,\n      "height": 640,\n      "image_sha256": "763c791934a71253315355722e6dfaeca4e8a0a6bc0bbae72dfc14b0dcf91092",\n      "label_sha256": "ae68dff4d8ffbbdba5492deea1e11648dbbdb16f0dcad67c43c492c1de432b6b",\n      "name": "calculus_images-10-_jpeg_jpg.rf.93a2b2da385a898f13cc57110862150a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-10-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 981,\n      "height": 640,\n      "image_sha256": "01c0ffdd2e2d11d6fab66d7d032c3e1164e43978ae53d8581c4dc8d70664d1c6",\n      "label_sha256": "8bab9752086e192e73809f96637bf9107c05fd1094a882507162dcf23bb9147b",\n      "name": "calculus_images-10-_jpeg_jpg.rf.96652192f67bbc39f6c072671e356695.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-10-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 981,\n      "height": 640,\n      "image_sha256": "c269611299a25e0d32090bda22982ff83ff2bf2462fa8411c72d28fdfe036799",\n      "label_sha256": "f58d6619f0ce977b3886b8f1749521447aabcac381306fbcebd1a956d6c54962",\n      "name": "calculus_images-10-_jpeg_jpg.rf.c2f42a81eb83c6f02bae17f11c4e578d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-10-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 983,\n      "height": 640,\n      "image_sha256": "925cf1d9509f15dbe12fe41ba6f20a14de375a84d8f3169f649a28eb90eb9987",\n      "label_sha256": "9f93abd6ff31191c02eb1edaa6c96a00c1157a3582f6f321315d96122fa03ce6",\n      "name": "calculus_images-11-_jpeg_jpg.rf.3b57f5b4381b909485a334dc0222ad67.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-11-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 983,\n      "height": 640,\n      "image_sha256": "a357b20eef8cfe00ed9c71b9078e43473c92232ae43c0210a33a84ea1d4eee8e",\n      "label_sha256": "611ad0d2402f0701367cf8ac15b3e0992e6f190d1d8455e778a3670c646d8bf6",\n      "name": "calculus_images-11-_jpeg_jpg.rf.47420abb212f18b91f6112a2c609fa6c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-11-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 983,\n      "height": 640,\n      "image_sha256": "d73184fbde19e3e24969ca98a0336a6a3855c968f8ab8d8c17d6112eefe5d070",\n      "label_sha256": "dc60b084e60f8c870fe7e6597877f08bf4bb1bcce7c7ce446f04e11b66c9a71b",\n      "name": "calculus_images-11-_jpeg_jpg.rf.51d2ee74c4d6836f659811618621a356.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-11-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 983,\n      "height": 640,\n      "image_sha256": "657ca90af14173fd39113a747ee6ffb4cd35380facfdd81f7df2e9f9eb416e49",\n      "label_sha256": "dfa3b8f21deeba6edef04ba39e2b3974afda8198fca07467031431a920310345",\n      "name": "calculus_images-11-_jpeg_jpg.rf.6df5f28798e5394e89d89bbc6be5fce7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-11-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 983,\n      "height": 640,\n      "image_sha256": "6fa3b7ad1091a3f810083c7898cc092dd9669e456722a0bf81680e3516fbdea3",\n      "label_sha256": "2399d94895b2f8f42b132e0e89616d3a8258cd9993d02261499fbd7fb5d28e7e",\n      "name": "calculus_images-11-_jpeg_jpg.rf.74aecbf2f038fa430182fee4d25d8a1e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-11-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 983,\n      "height": 640,\n      "image_sha256": "bb74662319393dbd21054c42b3193d3a794369b7f98aeab629c12904f14b87fe",\n      "label_sha256": "57fa04918e026f38a5560defc678f3667b73efcdbac2a8ae08cd390471e18324",\n      "name": "calculus_images-11-_jpeg_jpg.rf.9c279388b09267a684a91534e4e89849.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-11-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 983,\n      "height": 640,\n      "image_sha256": "bbf03f2455f5a502592b63cd03924391b448e8fc9fa3b32d244fe3b4bf78f673",\n      "label_sha256": "ffa0805d47806b90284dca7fdb8f645d5510f03e32a2b9ea0f9aa8dfd9d18581",\n      "name": "calculus_images-11-_jpeg_jpg.rf.cfd5c053320e8bcc3d6bbafd47776a09.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-11-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 983,\n      "height": 640,\n      "image_sha256": "96628a9d1f1a2152849497976731e35ac793b897408d5bc3917db82aac7ea38d",\n      "label_sha256": "57fa04918e026f38a5560defc678f3667b73efcdbac2a8ae08cd390471e18324",\n      "name": "calculus_images-11-_jpeg_jpg.rf.e50f562ff3c7b3975bc0e16285d0cc51.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-11-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 983,\n      "height": 640,\n      "image_sha256": "357c03db974db694847bc5cf623a79c4f5ebfc78c201b01cac0b05fc4dc8f1ac",\n      "label_sha256": "142e71daf13d3531e12e5deeddc67a4a047dbe9448a4b32b18282eb276e23593",\n      "name": "calculus_images-11-_jpeg_jpg.rf.f59aa485994d6a27e2d8c638a3a509ba.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-11-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 983,\n      "height": 640,\n      "image_sha256": "fd2114a273780d8c526023c3fbb2c1abf8fc114fbaa66d2231f7b3b167e67323",\n      "label_sha256": "a8895d1529d99b4965cf94233d155715d86b0fac73e749e103d396e86b770f2d",\n      "name": "calculus_images-11-_jpeg_jpg.rf.f7aa6773d18137004aca68a5f5d6d04b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-11-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 986,\n      "height": 640,\n      "image_sha256": "baffaf408fa2c09a23ab0cf8f0d18840f6daca6304efbdd70867b03a5cdfa52c",\n      "label_sha256": "0b46b058a12e18383cdfee128f35b9665d5f18baca767328588b296479a88d28",\n      "name": "calculus_images-12-_jpeg_jpg.rf.5c5f3b565658753ff44b2fae253cd0c8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-12-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 986,\n      "height": 640,\n      "image_sha256": "32dee8d4ee71dc589e36eb35efac3b80913e42fb90e24c079aa1e895bece83b4",\n      "label_sha256": "0b46b058a12e18383cdfee128f35b9665d5f18baca767328588b296479a88d28",\n      "name": "calculus_images-12-_jpeg_jpg.rf.621b5807cc71f5f2ae472bf0e2c808d7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-12-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 986,\n      "height": 640,\n      "image_sha256": "c81bf40cb003e735daa62357d37d438bdc0522adffaf430b15ce217ec0658f31",\n      "label_sha256": "616b340ae5309ca5e994cc8f744c8af94e89e60ee5a777fe628a090ea693279b",\n      "name": "calculus_images-12-_jpeg_jpg.rf.7415b78cc1d61661b278246a20bb25b3.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-12-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 986,\n      "height": 640,\n      "image_sha256": "473939324463c63bdd077da56f9e81851d6f0aec92c4b188c4e2c984dd50848e",\n      "label_sha256": "4975710296af4ce4df734498674d28c4b8812a5049c51bece296333b83581980",\n      "name": "calculus_images-12-_jpeg_jpg.rf.88b67d527f5a717fe10136f5a9dec6b6.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-12-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 986,\n      "height": 640,\n      "image_sha256": "0a19f1b86ec24bb1defa058f138676e6335b08d114cbe4c8fa9be6be374b01ab",\n      "label_sha256": "99f08fff93b81022f1c1d9ca26f65994028bf01bd214e8b421b6284c582ecf2b",\n      "name": "calculus_images-12-_jpeg_jpg.rf.93db961051e6c5d3542de721b110e809.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-12-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 986,\n      "height": 640,\n      "image_sha256": "da02ff31fd944c60dad0cd85071b34e596cd82246f2afffa4137997e1d4f2177",\n      "label_sha256": "c40492f3f6b5b6394dcedafe80acc4721774a178ef4c471e3f564b2cc1453e01",\n      "name": "calculus_images-12-_jpeg_jpg.rf.aae20b08184bc5dbbeb806735c640090.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-12-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 986,\n      "height": 640,\n      "image_sha256": "3b3200c6dfb2d8ff196657d5c332d771bddd725d546cf3d721e11c25339537c1",\n      "label_sha256": "0b46b058a12e18383cdfee128f35b9665d5f18baca767328588b296479a88d28",\n      "name": "calculus_images-12-_jpeg_jpg.rf.bbcd42f6db12d117beb5d747bc25d561.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-12-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 986,\n      "height": 640,\n      "image_sha256": "9d65c564584264440020794a269ed00dcd1dd4e1a6294ec79e13b65b88b610fb",\n      "label_sha256": "43ef446e19ef2b8ea927e074dbfee3466ddd4d8619f01d2d8986cc1bd1cb684b",\n      "name": "calculus_images-12-_jpeg_jpg.rf.c3ecb51402ff1f4bcd87975f64a9b609.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-12-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 986,\n      "height": 640,\n      "image_sha256": "2682ef50499625e08e0ce40d2ce259df26807b4e6b1fe4d9ba4eeaf2acd2cfbb",\n      "label_sha256": "535ff71112adf0f650b114d1bccb96291d252cc1d54880e6edeb2b1d0179dc9f",\n      "name": "calculus_images-12-_jpeg_jpg.rf.d88ffede69b093e3fb336ee94eb39048.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-12-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 986,\n      "height": 640,\n      "image_sha256": "ac4c558758c90efa2bfc57220de464c0835b6a96ad24830ff89635c143b38a40",\n      "label_sha256": "cbd274e2339c220fdd79d1bfa6aff1986326740709f4c0f566e5f0e86d52e1f5",\n      "name": "calculus_images-12-_jpeg_jpg.rf.dae7a4713651f1114f80ff7277d465fc.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-12-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 986,\n      "height": 640,\n      "image_sha256": "4b00e4022d2257fd33e40cb95b956e00c8e64809b5e7a598cdc31fba35fea587",\n      "label_sha256": "47b9afd28e7f39d9778d3f434069e6b5378e7363ed5b880518d6eee68622df1f",\n      "name": "calculus_images-12-_jpeg_jpg.rf.e3e837e935c13e4405e79dd6932243e6.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-12-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 992,\n      "height": 640,\n      "image_sha256": "773c0e9fd95ecbece1e7d96f0426f4db80d5fa6a34c2e38994862a2ac7ab7a3e",\n      "label_sha256": "e6f004cb334b214a9038076c093ebd5627c22c74b095016694eb79eb54b5ee19",\n      "name": "calculus_images-2-_jpeg_jpg.rf.0cbfa6458240d013017f9bceae7abead.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-2-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 992,\n      "height": 640,\n      "image_sha256": "44d8115f7404bf815c397b88fa7ca7dd23f01cf7dd63a4e59b08425f068dcc30",\n      "label_sha256": "415d3893b0368b8f0ba4edd39b4440fac297b33e84a4b4e1bda2e6813d57aed6",\n      "name": "calculus_images-2-_jpeg_jpg.rf.1235ae4e8ccfa8925c34ee4ea987b121.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-2-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 992,\n      "height": 640,\n      "image_sha256": "872fdac303dfbc19d917ca9069f0130d3af0e9d2b59259c3397d9c70249a7e15",\n      "label_sha256": "e6f004cb334b214a9038076c093ebd5627c22c74b095016694eb79eb54b5ee19",\n      "name": "calculus_images-2-_jpeg_jpg.rf.293a5b2ca17a89d950d3019936705aaa.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-2-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 992,\n      "height": 640,\n      "image_sha256": "1575034df52925c46487354fcb7bf7ba78dae89cc29d6aacde6f3f48ab59c4a3",\n      "label_sha256": "16253eeae497a87bba29e5d615ad08db1ef97a309aca95d4831a82f874f0047a",\n      "name": "calculus_images-2-_jpeg_jpg.rf.800628ced1f6f50359f73857f311e359.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-2-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 992,\n      "height": 640,\n      "image_sha256": "a150eb11c0a925d0f04487a4914023c91e8c9f54d704f8172326b52233900df5",\n      "label_sha256": "e3d7881b4bd0b54846299933d9befba89e9a1fe81e643c57f0ac39e474ef8c25",\n      "name": "calculus_images-2-_jpeg_jpg.rf.83012dd27f1f0b8b9186f79be997a453.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-2-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 992,\n      "height": 640,\n      "image_sha256": "c3dfd54056afe9656f0dc69e444abe4d270fb2c24826d097dfef3879c860c1ba",\n      "label_sha256": "c2ebf9061460a709079c67a41503fb7b3a5868e902ea7e8cb9201346f5fedf1e",\n      "name": "calculus_images-2-_jpeg_jpg.rf.de0ec9fb639db7130ff69cf0b28ba953.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-2-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 992,\n      "height": 640,\n      "image_sha256": "43f9e8e66e0064d28c5aada2c857731c8f032e584162c9d05e2b4596b546d167",\n      "label_sha256": "e3d7881b4bd0b54846299933d9befba89e9a1fe81e643c57f0ac39e474ef8c25",\n      "name": "calculus_images-2-_jpeg_jpg.rf.e4ab44559bbc34d339e220e971cbf716.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-2-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 992,\n      "height": 640,\n      "image_sha256": "6f78dc2c9d0622f983fe431f68d786d217bf3ab626b2d909a58effeca566ef29",\n      "label_sha256": "76b6f88e50bdc6d0db553ffeb56fcb490f51ab9b21be6345937fd282cec60b11",\n      "name": "calculus_images-2-_jpeg_jpg.rf.e6f40170c00655da09f5af0ece1b7ed4.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-2-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 992,\n      "height": 640,\n      "image_sha256": "d9e1888cfec2f325c25382d64e7fe08dd7bcea8f64f48ec55e1710ba4cf9e067",\n      "label_sha256": "125ac76b3c21745d45c5b883546e4aa32a65db199f15cf94b91802441e52550a",\n      "name": "calculus_images-2-_jpeg_jpg.rf.f47bdcdd7c70fb4d4795f43decd2ef93.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-2-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 993,\n      "height": 640,\n      "image_sha256": "df4f8a208db26552d9dc64a1ee862fb996c3b3c1772b896d981912b9ea8b10b5",\n      "label_sha256": "d643bd10a2a3980165be4713506845210737be033beb74afa2c8e56c087daca9",\n      "name": "calculus_images-3-_jpeg_jpg.rf.4f24ad08b1c540f40bbefd38fd778068.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-3-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 993,\n      "height": 640,\n      "image_sha256": "78bccba85c0ac7a59bf19d7a83c68e9d7a3e16049a038c6a83801f53193cd06b",\n      "label_sha256": "72c7be71f338710b2d1805284861410af716022da61df43d732c019317f48aed",\n      "name": "calculus_images-3-_jpeg_jpg.rf.5df47cdeb34c3d4b321e7d80f1ad4fc1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-3-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 993,\n      "height": 640,\n      "image_sha256": "de6df948ce9793ad63551f78a0d589034647f6c9bfcc312c60ffa13feaf62848",\n      "label_sha256": "028059e76f8ecd266ffafe111022e20dc4e05122f47d5d9b759c7111249115cc",\n      "name": "calculus_images-3-_jpeg_jpg.rf.60f592460d7014c377604bbd9d1c0d0b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-3-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 993,\n      "height": 640,\n      "image_sha256": "28e959d38ca7b3dd125504a88ba4cd6dc3b650252213805783127f6e9ae1c7de",\n      "label_sha256": "da824bcf4d82bd81ed9a27bdd2ba09d93e0ad1df64aa1756e79bf2acabc70626",\n      "name": "calculus_images-3-_jpeg_jpg.rf.620d620de25510d8b8faefc04b406907.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-3-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 993,\n      "height": 640,\n      "image_sha256": "ca43ddc0d69ee8114d2c6fef2f696d29c9ff133b038090520447da4b6cb5fe78",\n      "label_sha256": "70415df721ec8f5fba14dd5eacf5b94a02e0ea0c9ec356a8285f7b95238f2845",\n      "name": "calculus_images-3-_jpeg_jpg.rf.8259c6219e02a984b98407aacd6510f7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-3-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 993,\n      "height": 640,\n      "image_sha256": "a22cfe5dbb2ca0b956acba3da81fe92693316e707ac8f0efd6c63030fd01691a",\n      "label_sha256": "a38ad4eefb0597c03b69fa72800b57e0fa60d79fd4e3ff2513499cdc8f923c75",\n      "name": "calculus_images-3-_jpeg_jpg.rf.94f7b9a8954ef13d84a1ec566b806aa1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-3-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 993,\n      "height": 640,\n      "image_sha256": "3fe641661e96e0ed4c8dd030f2a23ed01935048d79f235c728fe35d122bacb01",\n      "label_sha256": "384905648551b83f838f0ac4449ac9dbcd7a602a00a2dc4aa4b856ba86b62f70",\n      "name": "calculus_images-3-_jpeg_jpg.rf.b2a555152ee70e394b4aa61e2215600f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-3-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 993,\n      "height": 640,\n      "image_sha256": "c668205eccf755bc19d55cb71c7a41c1a35813a9368b480c26c1388157bd3ae4",\n      "label_sha256": "fec0262eaf0821a513e114fdfc909909f8d90544e1587bc0d712343eb4866bf0",\n      "name": "calculus_images-3-_jpeg_jpg.rf.cb9fd64f2c2dbc1b488bbb7efcb2dfe5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-3-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 993,\n      "height": 640,\n      "image_sha256": "4bcb1b4e5af1df982d961d31f009fd8596cc46ff5316cb2b94f2150e08b61a44",\n      "label_sha256": "07ef13c204b71de0f8cc2ecc6a861ea22dec46e5cb05bfbdfefc87a9049258e9",\n      "name": "calculus_images-3-_jpeg_jpg.rf.d622d42c51f29b3e87880b50d6be7c34.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-3-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 993,\n      "height": 640,\n      "image_sha256": "6e3d9ec5ca0f4f7382d6e493bd14c74b81e5b47dd84245ae52d2103107744283",\n      "label_sha256": "75f124083bd4eadbeba3ef11351561bf96821649250ed72fd4ba84043a291117",\n      "name": "calculus_images-3-_jpeg_jpg.rf.d64984b8bcadbcb67e14e391c6c86940.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-3-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 993,\n      "height": 640,\n      "image_sha256": "ce3d628d126228d57d6a453b8a35b463fcbcfbbab7d5a6cd79275723a2831311",\n      "label_sha256": "b54f4c86e5033935125435f407dec0e71ad5d149e449e51c5fc8cd143da9109d",\n      "name": "calculus_images-3-_jpeg_jpg.rf.d8470fecabc6b0588c8efdf19396c18a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-3-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 993,\n      "height": 640,\n      "image_sha256": "792ce06312f7a490a44e65469e8270ea465729cfbeaac5b3a897d0f2d0bab3fb",\n      "label_sha256": "d45f28f136635b6c9078a0f8a00ef41efdaa804128a711fa910943ef8cb4112e",\n      "name": "calculus_images-3-_jpeg_jpg.rf.de72c63af58d5fa27b5c73bcadfebbf8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-3-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 993,\n      "height": 640,\n      "image_sha256": "440d496a5212f008e1ab88ae6a4fd60d7526a56c11d1f27e7a0285d2cb90b5a2",\n      "label_sha256": "924372e9adc7c80b748841a076bb17709519e6a299adfc9580889bc73e214275",\n      "name": "calculus_images-3-_jpeg_jpg.rf.e0371063e467dd6c32f301595b8bd087.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-3-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 993,\n      "height": 640,\n      "image_sha256": "c9d02a1e9b3a85233f2a3b63d9d7b69bdee2646d567bbd401645dc307b45b0be",\n      "label_sha256": "d643bd10a2a3980165be4713506845210737be033beb74afa2c8e56c087daca9",\n      "name": "calculus_images-3-_jpeg_jpg.rf.e2c84721449f06c2100cd6f62d1cb28f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-3-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 993,\n      "height": 640,\n      "image_sha256": "b02fa3fe3331bd77346eec56ba0854c9af4dc53e513face923d261d5f0f177ca",\n      "label_sha256": "23533ec9729462c6c306c94a41a01588ec0a97ff9953fde0e243362ead8efb21",\n      "name": "calculus_images-3-_jpeg_jpg.rf.e70ead01498545ad3d95b01bdc0a8916.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-3-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 998,\n      "height": 640,\n      "image_sha256": "ed9d4a3586465d1087022b7d46831012d35f9690552de56f22d869f25e9a0f0a",\n      "label_sha256": "a17cddf6031927c770d4fd7ad6cfc69b72c6412aa8eb3c161b5a2dc230994de0",\n      "name": "calculus_images-4-_jpeg_jpg.rf.034c2ba26676bf31c0c5d30ba6354558.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-4-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 998,\n      "height": 640,\n      "image_sha256": "158a2af7a6542149543bb0a8b6a06907b3ba4e8c8263a7fb8070e877f02c375a",\n      "label_sha256": "c5bfeb6b474f19274b67adab61083c3730ff38f8a77653519ba14719d8850197",\n      "name": "calculus_images-4-_jpeg_jpg.rf.3be068edc6fa2c4752e414e57efb79cb.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-4-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 998,\n      "height": 640,\n      "image_sha256": "80c8ea7e31b8518b9ac22651c42008fc94b568191b7ad433e6fc2725808c476f",\n      "label_sha256": "a7b2550a952cfd193622fa11d89450333d4d4a99b538e6d2c9b7c2a832623b45",\n      "name": "calculus_images-4-_jpeg_jpg.rf.7d2614e8b3d66b5f89179727d5b55f98.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-4-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1000,\n      "height": 640,\n      "image_sha256": "5bb49ba73e4cf064f6cc8fd79a24ffb0c4028c2be4fb9283a368dc07929971e6",\n      "label_sha256": "478f66b6c3dee3033b748941393ea3c7515db9adce1c7a4055f4b7f5bc91312a",\n      "name": "calculus_images-5-_jpeg_jpg.rf.2663ceed9d59855846c50db9a20f54d2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-5-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1000,\n      "height": 640,\n      "image_sha256": "edaeb8f98be0a64f07ed5730126cb229550c71cf0d9a18b3cda082ec9bff9fb7",\n      "label_sha256": "a9305e61da3651c2e2222c4ff5a7530dfaaf255b69cb0d3be2c8c4ba09b3914c",\n      "name": "calculus_images-5-_jpeg_jpg.rf.313beabeb62b619ecd755b90b7b13227.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-5-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1000,\n      "height": 640,\n      "image_sha256": "e7fe3c6086da454264a4a8cd10f6b00eccd1dc61195db02bfb2cb3ce04f59f0d",\n      "label_sha256": "9f381d943aa682f9e16cb5ba01cbd51d8a01c9251094e9ecc1d8d56924c8d34f",\n      "name": "calculus_images-5-_jpeg_jpg.rf.3534b691af77eb0215ea6dd25202ed6a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-5-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1000,\n      "height": 640,\n      "image_sha256": "e0caacb2033dbe9fae13ae97337a63155d275bbeb5de119c803d702f4e97bd99",\n      "label_sha256": "e1c673c4e576ef65bb9b7e6a4f69f9ebe2e04a7a923138231539b14a22f03aa3",\n      "name": "calculus_images-5-_jpeg_jpg.rf.46afa1bebd1a8e58e727612c48282b45.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-5-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1000,\n      "height": 640,\n      "image_sha256": "51bd0550b975feeca56c6d0e47fe89d0171b830d08236be7cf480086ef65ea0d",\n      "label_sha256": "5c4059bfbc372b206a0bc3cc4bd127069f10264c40b64f3cc180e456faa283cb",\n      "name": "calculus_images-5-_jpeg_jpg.rf.55fa9a434334611f80a437b4e07fde7d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-5-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1000,\n      "height": 640,\n      "image_sha256": "e6991c734c243e3ceb9121abfa794a8890b081bf9ba6e467a31040b5049f406a",\n      "label_sha256": "d25e6dd6fc967e15428e10b09ce83271b09c69b3372430782d893ada1f7f6c12",\n      "name": "calculus_images-5-_jpeg_jpg.rf.62551ce64b13c19b6f9b1f47c462e4dd.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-5-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1000,\n      "height": 640,\n      "image_sha256": "a51619ef718039f45072762a1c17162b27fa817574933e51f40ffcdb54eca162",\n      "label_sha256": "a46149467dace8d7aa6085293839b74fa3768164121ccd28904ca0b832dc8b79",\n      "name": "calculus_images-5-_jpeg_jpg.rf.b2562a134579be61d62ad714bc393429.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-5-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1000,\n      "height": 640,\n      "image_sha256": "842a0cb0bfda74e192b6d6741431fc13be4b63ce923676572213243b1feee43d",\n      "label_sha256": "ea576fe55550a4c047db4d572123d4052dbb3163d1ed4d82f122ac9ae4af644f",\n      "name": "calculus_images-5-_jpeg_jpg.rf.fc822e09e728500ac6258aef02e5b13a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-5-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1002,\n      "height": 640,\n      "image_sha256": "429a7d2c75115255c1885cefd374cc131e115cb03754850e4ea7df38c6be90d3",\n      "label_sha256": "3dca2b8fedf969e0c76e093abf01ae1047f497b08b1e5e76d02a3f7d8d319127",\n      "name": "calculus_images-6-_jpeg_jpg.rf.2c4f0d97b8d5493843b0bcdfb93f4ef8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-6-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1002,\n      "height": 640,\n      "image_sha256": "c98dd00f93e45f7c46468b1e54dcdae0fb150e80ed9dbf93f18a71b1a6ad2613",\n      "label_sha256": "b4e367570b12272349b4180c973a8f833013c33f6132850d8ab321c19340916c",\n      "name": "calculus_images-6-_jpeg_jpg.rf.68edfac2762894c73ec98c6c236b39fa.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-6-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1002,\n      "height": 640,\n      "image_sha256": "62e869123124a8e24dc006447451d0a3127377d0d3810e5137e62f3d1cdebab1",\n      "label_sha256": "f0c57098e26ed0a25a68360575fc3207e16c3421cc090c7dde5b6ba489ab8328",\n      "name": "calculus_images-6-_jpeg_jpg.rf.873eee076ce7d7a87710703b97477f71.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-6-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1002,\n      "height": 640,\n      "image_sha256": "cf7cfc56b43e0a38b00e0b382147dcaecc7e33c5cf0b9bf0632faef3f7b7fbe3",\n      "label_sha256": "2404aa1d0ac3888e060d63d25863b934a3d880124075b77b9855e943ba899a5b",\n      "name": "calculus_images-6-_jpeg_jpg.rf.9c25d8e51f8f55bc046e627a438c1b1b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-6-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1002,\n      "height": 640,\n      "image_sha256": "8c60490c9f0c55d887caa46674a5762714363ad0d152c0c32fbe9223d1014378",\n      "label_sha256": "dc83e9881eef1136f323ceb1eaf3908bca7ceadc75e29382364090022e8c935d",\n      "name": "calculus_images-6-_jpeg_jpg.rf.b6bf909f281c6d9e35d7c01b76b28459.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-6-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1002,\n      "height": 640,\n      "image_sha256": "33fd56ef6237ace53f1f75af06ba8056c24bc2a1621df37277767ce5cbd3ca6d",\n      "label_sha256": "a908da7f461bfec82e1b569861ed193b82a1a6025f6d1ffe29a3d2b5d6d9a461",\n      "name": "calculus_images-6-_jpeg_jpg.rf.bb7e9f0be7a8dad3ebe35e2be2ed5de5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-6-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1002,\n      "height": 640,\n      "image_sha256": "1169e9f70dcee6e7f351d1b0fc06241f6211d1e42b6ba33a829ad62e495eb244",\n      "label_sha256": "37b40df1837f50efed32ff1322419afeed988eb868664a4507903c24a4288442",\n      "name": "calculus_images-6-_jpeg_jpg.rf.bebc3aa7bfc8c7984f7ed744386a6b12.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-6-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1002,\n      "height": 640,\n      "image_sha256": "53de33a43da155d0e31c16c306a55d24a3b3e8951854af7922be9c99ae0c7db2",\n      "label_sha256": "af0ab1d3f7973529797eb7d978475c56b22b89dee7d9650364090f3e0eede23e",\n      "name": "calculus_images-6-_jpeg_jpg.rf.ec8be01cb3df801e02bea132d7218c31.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-6-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 1003,\n      "height": 640,\n      "image_sha256": "42f8f2b1d74a7d913111dd7c1f7bd96a57a035023618107af2d7da92d65ae5b4",\n      "label_sha256": "b050335def30711517a242dc459a25a0a94006d159be7382f49d95ce24673557",\n      "name": "calculus_images-8-_jpeg_jpg.rf.174a5ec59020895329a40ec36a994e19.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-8-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 1003,\n      "height": 640,\n      "image_sha256": "5bc0be5ab818e10705f5ac63e2b6c56b5c7fafd68811a730ee7df41cc975f901",\n      "label_sha256": "47e4689bf77246e9159c1b6e23a2e4b941b431fa6c8f61b8785b9d7cc0bff88a",\n      "name": "calculus_images-8-_jpeg_jpg.rf.4c7082f94e1ff45832f024a39dcd1032.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-8-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 1003,\n      "height": 640,\n      "image_sha256": "e5bd185404940a70931f8976b03b07b5bfcf3e521fbadb1a4eee2013e17e1dd6",\n      "label_sha256": "3be8c3275c34ddd5b3fdcb8c25922945f8b76447e9d3c30911276887a67259b9",\n      "name": "calculus_images-8-_jpeg_jpg.rf.77c8247d2255d0b8027a0e9ad29c4434.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-8-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 1003,\n      "height": 640,\n      "image_sha256": "619f1dee857ddd2f4e4c287e1f203e950f9f5b9e145bf1ab168877066be6555e",\n      "label_sha256": "d2207b5483c5948145cd9e41b6021424d096e9fe2ba0fc97082d2067debf6ea6",\n      "name": "calculus_images-8-_jpeg_jpg.rf.96d283f96e24a3092f098cb73c7f299c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-8-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 1003,\n      "height": 640,\n      "image_sha256": "a5d2dbde7bc66a0c174f772195682b839ab5f62add266aefc200e9a1527e8b36",\n      "label_sha256": "517f3560cb8dcbf615b559ae80ff86bde764edb435fcc25b46bebadbc1d50e14",\n      "name": "calculus_images-8-_jpeg_jpg.rf.bc22bdb6de8bc8b4752186ab4a811f2f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-8-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 1003,\n      "height": 640,\n      "image_sha256": "daf6bea8d3d015bea16a0fac142d7ed24cf2146c6d5dace09131527cbfdd1a3d",\n      "label_sha256": "69b4af7dde863eb814d05ed304b98ceedd21d65ba13a234444ad968dab369e7d",\n      "name": "calculus_images-8-_jpeg_jpg.rf.d138263844f08bf2ea15d9141e7c4b8a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-8-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 1003,\n      "height": 640,\n      "image_sha256": "a15b40ca900e3b1d7f4380d6de15c3d85587bc69d73bf16fe88e4a6a5de524ef",\n      "label_sha256": "55832457825da6a80d59e77ec56776076685a5d1d993dc24615133557603e626",\n      "name": "calculus_images-8-_jpeg_jpg.rf.d31f3b96d3273ba8bfffb66680e07c90.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-8-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 1003,\n      "height": 640,\n      "image_sha256": "724980847d67ce2c8867f655ac52ccfc2350d098272079363c424362fa8a94ed",\n      "label_sha256": "91b35d369830213b27f40db0a64f152c5c4e6768edc4015f1fd05fc11b394548",\n      "name": "calculus_images-8-_jpeg_jpg.rf.f10b2be9d7fdfbd046b851094bd12e56.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-8-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1007,\n      "height": 640,\n      "image_sha256": "46339b9c4edee464a66c63547a9ca2fbe2af5a584732ea50e6300fde716f8607",\n      "label_sha256": "48dc873475dc54875f2ffb1ab617d73a9cda16a3afd717af911ed9669a820014",\n      "name": "calculus_images-9-_jpeg_jpg.rf.0ea983c228e769ec61f868faa97a64c4.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-9-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1007,\n      "height": 640,\n      "image_sha256": "84f54fb925a1e6d7afd83a1cf2bc19d146562ba039f9bdae9a04a12f9adffde3",\n      "label_sha256": "b9f0e86799e597c0420fbeac93ff96673e490af0e3a33f1c54f2a50c7a552252",\n      "name": "calculus_images-9-_jpeg_jpg.rf.396520f425456295333b2090de150611.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-9-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1007,\n      "height": 640,\n      "image_sha256": "5a19c6db84266fe2f56b4fe198366009985da2fcf3de841d3cfa4547b30794d5",\n      "label_sha256": "a8d2eb78617e87286d7b754dbc6ab66d6024f23eab0fad5dd31a422d0ff4a83b",\n      "name": "calculus_images-9-_jpeg_jpg.rf.3fcc2d71fc9f392bb5fc0ae42b6a9cf2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-9-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1007,\n      "height": 640,\n      "image_sha256": "2883ac49d6bbd10f37a1132200665be0cfcdadfc4b08c900e518d083bf434575",\n      "label_sha256": "e0e6559bfb90f9ed90796a0fdec81b3e1c025fcd875276b79daa40e2205032a8",\n      "name": "calculus_images-9-_jpeg_jpg.rf.661af5380df6b2ccef9a13fb078eb2d5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-9-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1007,\n      "height": 640,\n      "image_sha256": "85f09932f293f6fb10e167d50a889e19b5cffe9c805c83c178f45336af29f655",\n      "label_sha256": "baf6e79c0d79f180c48d12f37a50e5e24921bc32417113d3a4eff44327960953",\n      "name": "calculus_images-9-_jpeg_jpg.rf.699afd8b8332747281cb364607303ab1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-9-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1007,\n      "height": 640,\n      "image_sha256": "af5f9256806f6d61d37db47ca7e8b927cc2329ef20c23ce18e6d114951eedb35",\n      "label_sha256": "38a37c653a71f060f2157be50a444ea56a3fc661694e706a396ae6f2d7af1cc6",\n      "name": "calculus_images-9-_jpeg_jpg.rf.6a08432242ac3d1150f38626080e4d26.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-9-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1007,\n      "height": 640,\n      "image_sha256": "17c003c9f24d1e3d472dba0801a29209707354a9644d7ace84ba64db4783a5f5",\n      "label_sha256": "fc0f4311bc96216247f7c9e3160d6f22b1e46da96396390a8e8a61862ae06618",\n      "name": "calculus_images-9-_jpeg_jpg.rf.7b3904496755518d6c31546af1db48ab.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-9-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1007,\n      "height": 640,\n      "image_sha256": "6f4c3028d56c58773843d978767158f8d097eeaf7a6e8d3221303556b8c5ae03",\n      "label_sha256": "dbf81a5ab76639318341fb506570884ba331d617066162eb062782f9668fc95f",\n      "name": "calculus_images-9-_jpeg_jpg.rf.ae541b8cd7698c2bdb818db62786862b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-9-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1007,\n      "height": 640,\n      "image_sha256": "c5ee2bcfff32e78929ee7e87db1346b3b1ef4fb2e1e848fd1e358f9d0a163787",\n      "label_sha256": "e0e6559bfb90f9ed90796a0fdec81b3e1c025fcd875276b79daa40e2205032a8",\n      "name": "calculus_images-9-_jpeg_jpg.rf.d12b4528cf2ca3332ce2a7af0ba77aa7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-9-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1007,\n      "height": 640,\n      "image_sha256": "f958598d605b244ff77b9ad475670896cc631d501d8deaea255e433b94d22539",\n      "label_sha256": "fc0f4311bc96216247f7c9e3160d6f22b1e46da96396390a8e8a61862ae06618",\n      "name": "calculus_images-9-_jpeg_jpg.rf.f76d888f2820cccf0fd99d5ac4647337.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images-9-_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 1008,\n      "height": 640,\n      "image_sha256": "71226dd0d04d3462652b5d16d8f7e8f09e399b4343db5300836354208f78db61",\n      "label_sha256": "57401331ababed81953bc91ace43d7d1e27ff3cea68745ee79acf0fd365e9b11",\n      "name": "calculus_images_jpeg_jpg.rf.190fc39c12f9623b48d3d56d6d8178e8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 1008,\n      "height": 640,\n      "image_sha256": "3ca708f8ce3e8e2dc864456d43259add1d8f0236b9dcf10de1ce5b081f99987f",\n      "label_sha256": "d81e9cb7dbb95c81f0cef0222d9b2b56767c10e9748b638b8a87a8944845eca2",\n      "name": "calculus_images_jpeg_jpg.rf.2a4e47d5dcf503b8bb7bccb8682cf165.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 1008,\n      "height": 640,\n      "image_sha256": "30f39c004c6360c4becda7d13a7b72dcd862a27b224b3167c5eb711c9c33efab",\n      "label_sha256": "9513c70c600e890e93bca01c80a1af82bcb1164c22b052296345c640b87e783a",\n      "name": "calculus_images_jpeg_jpg.rf.2c242458592c0f7688bbc95e19c4655e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1008,\n      "height": 640,\n      "image_sha256": "51b2c0536a38aa3e8804c5c76e02b8a1581053491a340becc633d3e472f6e589",\n      "label_sha256": "a79b104277ffcfe1f438ad76874b97217c383ab0da885f4b5ea3a1f085bdfdc0",\n      "name": "calculus_images_jpeg_jpg.rf.5c0dcf12116919a6c7042faa46af17e9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1008,\n      "height": 640,\n      "image_sha256": "153a32f310aa7133b4db8098ceac0cd9f873d7106e9233e5f53f70bba84b7c42",\n      "label_sha256": "e2867cb905dda94851f176823088204c311edd057ac46f91ce62f37b32b9c003",\n      "name": "calculus_images_jpeg_jpg.rf.6898ee79b59db61e0817e669e9ff9450.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 1008,\n      "height": 640,\n      "image_sha256": "1d94cd468a83d4d5bb413eb9154f397f57d8a7a2e1bd4555c558a4df4dea002b",\n      "label_sha256": "b55a7be722e165f85bc90eb81d9c29560211c45974a3a110e7cefca043d3865a",\n      "name": "calculus_images_jpeg_jpg.rf.9db603df13879e9d98961e52b7a03815.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 1008,\n      "height": 640,\n      "image_sha256": "11ee6c5e1319a7f396e22c68693d9cb3a27062e1878e459c9c8ca38e19f75cee",\n      "label_sha256": "551b289596e94a2e01d8840476cea97ce5f020dc216f011f9e3218058e0e7145",\n      "name": "calculus_images_jpeg_jpg.rf.a31e77e574e41cc711ff3562c207d83e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1008,\n      "height": 640,\n      "image_sha256": "8cf93a03fbacb8bc0568a536f652205c0e82f8757e58d0792264c2191783684a",\n      "label_sha256": "60ed5e30e39604ec82557e53502fd6299a364d975309c000e17db8f4f45aeb9b",\n      "name": "calculus_images_jpeg_jpg.rf.af95121ed6b17c0238f73d2c10f3ddc0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 1008,\n      "height": 640,\n      "image_sha256": "97995b8f61049bb9b2016856614535a222f136d6e5bba817bdfcfee6de2c8d8e",\n      "label_sha256": "d81e9cb7dbb95c81f0cef0222d9b2b56767c10e9748b638b8a87a8944845eca2",\n      "name": "calculus_images_jpeg_jpg.rf.b74268fca98bbe45adb201842228b5a2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 1008,\n      "height": 640,\n      "image_sha256": "6d7fd7965e438a33f52acdb7312f0682fb5758ef26160757d7019917e4dadd24",\n      "label_sha256": "1d8499e0f38491dd41f200b9d159351ca4f91b9bcbb430d680af6f60a3ce9d11",\n      "name": "calculus_images_jpeg_jpg.rf.d6fbe431f68cd51d1ec2652d5934d4ec.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 1008,\n      "height": 640,\n      "image_sha256": "9e2d23e0025d23e2f68a79128c7ea98fed2fe622ace1ba31e62f7e32bde03643",\n      "label_sha256": "9482d5c6a690175dbf1fef00011b038ef5eab3e476847590f76bd16814cfd237",\n      "name": "calculus_images_jpeg_jpg.rf.d8dc42bc0cac28a000cc7c9e598dc480.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 1008,\n      "height": 640,\n      "image_sha256": "2502c706b24ff1131501ee07ee41cccf4a5f70a0cc7cdcba4e9709ffc86cf708",\n      "label_sha256": "1f6f3c757dda07b07e77c3cfa9fb7ea79aad3a8e9d5eb77068803556dcffcf45",\n      "name": "calculus_images_jpeg_jpg.rf.dfabbf37731e4c8b908a6a8208c4929e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 1008,\n      "height": 640,\n      "image_sha256": "b8b4032474a46dd96fb99eb1712defae6e2fd7ca9f0cbdf13e376f5cba658eac",\n      "label_sha256": "29c75925817cfe892922e5c0b4f21adcba5167304c5d95419ae6ffb57972506f",\n      "name": "calculus_images_jpeg_jpg.rf.fdd31c4e9b9287d83ad75ae554ba9beb.jpg",\n      "original_split": "train",\n      "source_key": "calculus_images_jpeg_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 529,\n      "height": 640,\n      "image_sha256": "cbc64ec93461fd3b1e3b44bead022022fc1c77351d3bbd6747e4290c576fd2c8",\n      "label_sha256": "4e5a503585e1a7e9181c10998c09353661b1f2183fb483a0e397beee148b3305",\n      "name": "calculus_lingual-surfaces-2-_jpg.rf.d1b9ae8dc3d0d513d27e597d2c983d80.jpg",\n      "original_split": "train",\n      "source_key": "calculus_lingual-surfaces-2-_jpg",\n      "split": "train",\n      "width": 640\n    },\n    {\n      "boxes": 14,\n      "group": 530,\n      "height": 325,\n      "image_sha256": "22284759b74fc663fa139ea745f78400bc83788ebf622b502940ace492523742",\n      "label_sha256": "83e2b0c18fbab5ae7cb8b0e68e7397e6be03dca49add66a844ab526a4ff484e9",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1004-_jpg.rf.4a30b2c2e52bcc6e7c46a0df50fbfb9f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1004-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 531,\n      "height": 290,\n      "image_sha256": "b910ae0f9f5273b3f14c75461d24075b70e8abc734770bde23041cb13b0c47b2",\n      "label_sha256": "1adb20b08b0454cc0927bd64f418b8eff66b829b835df66126f0aaf18ee696b6",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1005-_jpg.rf.fa621249c2bc74c4066f0d4ccb72c336.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1005-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 532,\n      "height": 405,\n      "image_sha256": "6bad344b6352086acadd2b6b2b309d095ad8e1b21922edf3e0c133a67cd57d95",\n      "label_sha256": "02a7607687eb9be6900a78a69db6d7a214871a5eba848cbce7b8c0bbd29aa718",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1007-_jpg.rf.a7adcac2046b4e3ab2f9216d9b908317.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1007-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 3,\n      "group": 1012,\n      "height": 192,\n      "image_sha256": "149739c517dc766a4443d32e8510e4b4e6882763f35085b5b03d6686889cfce9",\n      "label_sha256": "3507d14813f2f34cfcd4c7100f7a2c3fdfac15afc3043605c5f5331d9e3d81f4",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1013-_jpg.rf.45f93cbc21b0c034590e3e7c66857bcb.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1013-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 852,\n      "height": 408,\n      "image_sha256": "13202d5ad6901e8f3099d4d7ffba72881762ca9540106ea218b62fd1ecd14676",\n      "label_sha256": "8afb8c558993be8bfb2697fa64acc5517908f3597f72099d7496e141161ff8ac",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1015-_jpg.rf.ead08c525bc2a5c4df9f26d28f9b5c77.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1015-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 535,\n      "height": 459,\n      "image_sha256": "45c65ed97fa2f5549a760652699717d56865a7c82a500cc32dfa1a01b01d821b",\n      "label_sha256": "7dd0eb111362230077d2d1c9f23b05364aa9afff66e31a6009c0bd272de8a9fb",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1016-_jpg.rf.a6ab5828afadd6c076f546fef07fb9c2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1016-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 536,\n      "height": 408,\n      "image_sha256": "af704ea884af96384d52407f900dd0e810a0329e018af13f6c1795d176e4e7d9",\n      "label_sha256": "2bdde7724281bc804a645eb8da7879d3048dd33dae2115eee1a7cc43743d186b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1018-_jpg.rf.cc9444dd657226f97c1cbcf5310f696e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1018-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 537,\n      "height": 408,\n      "image_sha256": "399c6c4806b8f8df026a38932b061f57eca9d06d308680c819f7d181b7a31ed3",\n      "label_sha256": "ab3bd8e247dca9d5872df7a409d7a45da7c4dc935a9ce73060e190afb91334c6",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1021-_jpg.rf.63318e759c07f744bd6ddb99417c666b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1021-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 856,\n      "height": 375,\n      "image_sha256": "f6bc2d16922500932227f562061ef1051ebbe41d5641cccde251307efee0d54f",\n      "label_sha256": "f6666142c68377f18d1c350240c921d5e4ba519b099efab3e14266217047c356",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1022-_jpg.rf.68ee16bc9687c635ea5c32d0b6226353.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1022-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 17,\n      "group": 539,\n      "height": 408,\n      "image_sha256": "6bc3df0a4e72651e372258504872fae7863b45a23efda3968542196d78bdedf4",\n      "label_sha256": "322c28a19aeade7914d3b57266ca1b22903487962a7e3766de7510e1697a51bc",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1024-_jpg.rf.c918ecf3ceaaceafc9d99198a1835326.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1024-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 540,\n      "height": 405,\n      "image_sha256": "39b34f13eafb0b519efd981ce269da9bcb20853f1ee6033acb844c0119b7eed7",\n      "label_sha256": "f470bdf50ac9c4c9d0f10fce48a24319d1d02802159fbe0aa5101af9bc03331f",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1027-_jpg.rf.059335caf1bbea9dec875ddb05ecea3f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1027-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 541,\n      "height": 255,\n      "image_sha256": "d7c8a6e67097854e1d9e14d0e9c126fe212cfd92176a0e49936170851a840c6c",\n      "label_sha256": "bbc8079ab8aa57d8285b5c09c15fa23b26b6c58dc956195436f7d5878109721c",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1028-_jpg.rf.4bab156755883fdab40321105329a03e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1028-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 1039,\n      "height": 204,\n      "image_sha256": "49fea89dec3dbc0832eb528953dd2e4558eb07be3684481723a81496a05a705c",\n      "label_sha256": "3a5c38a05a33515896753bae24495a4c33e54d91ef33d5c0e31f675b2ad4fea8",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-103-_jpg.rf.d9ea1634c034a2bdc917cf2ef87ea266.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-103-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 3,\n      "group": 860,\n      "height": 408,\n      "image_sha256": "88d5125b5519674bc62465828105fdf4d81857d48a8d35f93407c6472b90dd90",\n      "label_sha256": "079b234b7caf67df4ba6661fac5a9eb21a12918ebcc853b7578288a56b27bc19",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1030-_jpg.rf.8adee2cd8ae34aaf4809e849dff9fcad.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1030-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 544,\n      "height": 193,\n      "image_sha256": "3cea1534b70eb82d8b4e128dc436a4f3a09e083cfe3c150a29d976d6b6bb64fe",\n      "label_sha256": "94b4bc6d07c34117ef51c447df5749e67abb5b51dae4c36efc5f3b7abfbe6f6d",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1033-_jpg.rf.82a48bb3762cc34312ff75fc2d6aa5e0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1033-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 863,\n      "height": 246,\n      "image_sha256": "5eeee617213dd85c88ad88682283981612759ec4eca41e42c272a6945cb77590",\n      "label_sha256": "fa0ca2bcec8621d91fe0ed0aab0607c192243b9d78c76ce4f9f626c84b3b7748",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1036-_jpg.rf.6da2b30b91d1db2ae24f70bcc2f2b4ee.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1036-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 21,\n      "group": 546,\n      "height": 609,\n      "image_sha256": "829520330ec2ebae2049182a91a74ffe710a3527129f2151aa9b2c5e7d33b13c",\n      "label_sha256": "b4a03b78e7915494dfccabaf949e05e6c94e0fdc7ae0e7450de7b63a97035abf",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1039-_jpg.rf.5f39383d1049a9a99cae97cb775a53ec.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1039-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 881,\n      "height": 297,\n      "image_sha256": "c60db940b304c1a65b66c9f2a25ba1c598fc81299a65319e0461d4ddab4ded8c",\n      "label_sha256": "dad10563d9047a74f151fadc8ad8a656f25e74ea8aa0e375a8b3a6aa50c69eb2",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1041-_jpg.rf.5d2f35abff35dcc8f840641db46cee47.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1041-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 548,\n      "height": 408,\n      "image_sha256": "ac1f4778cd048f65f8f5ec0f95da0b326275ec2111a135a5f82cb859f45c5529",\n      "label_sha256": "86b86022486e6901843e3ae8c3918247f5c68c4554fed65e9abe0796d719b0ef",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1043-_jpg.rf.2bd6202d59b14bc099503ab5115ee86e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1043-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 549,\n      "height": 263,\n      "image_sha256": "39fc217ef7b57f4aaafdb744058207f6dba450a875133f858df40f2cf6b337f9",\n      "label_sha256": "7acd7d7ebe5df10f630fbde45b5a0f3cb233aed2c2a7ddf1d940b62f95816e48",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1046-_jpg.rf.d4622727915e0666589979603ad5cbbf.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1046-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 1012,\n      "height": 194,\n      "image_sha256": "d752d372cb4c3e782ca4da6b1837f8ae23a8bc004f1ef1d509c9642cc79850ca",\n      "label_sha256": "16f9eee75adffce906db723fa36a049f2edfe40f2725caca53081fc0a83ebc15",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1047-_jpg.rf.1fdcc3bff05ecc3372ecc8f0cc08df36.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1047-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 1018,\n      "height": 408,\n      "image_sha256": "989f5b8d4fde484628510f7665d58745cc459ff133628bc672c6f628132bd318",\n      "label_sha256": "aad2e45bc967a5327520b9efd923c9d3fac8136d87a535f5209ec265028189ba",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-105-_jpg.rf.f501395234d29cb419e07c7bd34f3b94.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-105-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 552,\n      "height": 398,\n      "image_sha256": "c44d7d4a7456fa1e4f783a18ceb92f7ef4cec693543c54818565a9c103916555",\n      "label_sha256": "f51228e557d6906adcce2d0512a8fa738924a5b156111f9fc23a33e178bb295c",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1056-_jpg.rf.c08bbc37ac762aa46767adda841c8aab.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1056-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 1012,\n      "height": 169,\n      "image_sha256": "9ef451e8c7e934b34f84a9b5892e472fd186850e31390edb2f069a6e19108a18",\n      "label_sha256": "55bf100fe6942c6b837828a6fd3b6944581ae80ae60f09d0c09f76fca6ae3529",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1057-_jpg.rf.28451cd1e1f32ec9c4c1cbe46ef22e2c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1057-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 554,\n      "height": 408,\n      "image_sha256": "d8d415423ce71206a40bbd588bb0ff1830d35f51f2d7ba1d876bb0fc74d5a0e8",\n      "label_sha256": "d3f851dd0057ed6d1c35b5c874fffe770b648ebcb4c7a748dd04680acdec6edc",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1059-_jpg.rf.965ee15ae1ef41572c25ad742a8dae34.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1059-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 555,\n      "height": 459,\n      "image_sha256": "c93d1067c262cdbfa124ef6f432c0bdb6363e79a52dcd948f5e39a337d27f59c",\n      "label_sha256": "a72a629e99489a2060991765147f2ee07e50f32428e69c2291141e34f844659b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1060-_jpg.rf.9105c32893d735acd7606e519ae7c19c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1060-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 1012,\n      "height": 151,\n      "image_sha256": "2ff0e0e2e5ab452140563e495ed54885d2649e928dff7d5426c1cc821184baca",\n      "label_sha256": "6e2c3e3e1a8fd85811ce68bd7a46353f9f06751a30d2cf1a04ceeede134af34b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1061-_jpg.rf.cc2500a45f6370b3f529ea2200cd505c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1061-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 557,\n      "height": 612,\n      "image_sha256": "df17813e83e3935c258e26835e583f18196f0fcd706521f75cf0d9c9a1d4d40d",\n      "label_sha256": "091444c01e8afd2f65132f0d12894529ce314180d62c9326cccab378c09f743c",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1069-_jpg.rf.13f998b9cd0d8ac2ef084411b6f1447b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1069-_jpg",\n      "split": "train",\n      "width": 601\n    },\n    {\n      "boxes": 11,\n      "group": 558,\n      "height": 317,\n      "image_sha256": "11e88e5a47e15353eb3f8629dc1254f0c65c4d79d072b9f03335277a4ea00d3c",\n      "label_sha256": "76dba5379365681e40a2e51b027a5e413fa8e507a02c87dc56d7c3afb2c6e6b2",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1075-_jpg.rf.7977b606900a1e50c96ba36b3ee59956.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1075-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 875,\n      "height": 406,\n      "image_sha256": "5b648cf405315612d7a79bf24acf0be2d8546106e9548fc382b3f5ad0770a692",\n      "label_sha256": "5f71fa99ecdf16aad3a0c6fafc3da98ccc81f78bca96b7422dcab6a82f6592cf",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1076-_jpg.rf.328dc761d412e1e1978fe29be8de0dc4.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1076-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 560,\n      "height": 347,\n      "image_sha256": "be76526b128fd294a00409da089791633de3962847151ce136d736874d3ed736",\n      "label_sha256": "6fef4772cbf70f5a90b004d4e0607b947e42b71d69f8eb14f39d5a19a1f42a29",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1082-_jpg.rf.d3aa18d28ff373ca9e63443be81b254c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1082-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 1012,\n      "height": 195,\n      "image_sha256": "43290be2a1333a4c728abd72e4f3d6498f55b76b69bc543d3fe90310ffb7f9e3",\n      "label_sha256": "ca7cb9e7a508adb53dddcae443e66983e9acf200a9d96d762636a9c0b14c7bf6",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1083-_jpg.rf.6cd1176fc22669672eab59e3d4545485.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1083-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 562,\n      "height": 408,\n      "image_sha256": "568a8174c329ab803c5027b594ca34e42f52dc90fb24765c6e2a10603c424d4d",\n      "label_sha256": "3c77ae844da5d71d4355bf82cd67b747f62cb2691f9ade55e52363f1ff1232d9",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1086-_jpg.rf.f280071531aa9e46fe8bc1ba706192d5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1086-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 858,\n      "height": 390,\n      "image_sha256": "014e29f24155589ed680bd950ab9b6e17b2bfb3c36ab0fcb4408599b48f36183",\n      "label_sha256": "69f7600477fba7d5fc5937bfe6fca2bf3e13256eacb8bb870e7d53bfc3174c58",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1088-_jpg.rf.5ad3c5e1dc5e02d20c9f71c5f5041e90.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1088-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 564,\n      "height": 424,\n      "image_sha256": "95348d894a5cf6958450e1908cc80817a551105c2909b5e5e55260a7bbdfa666",\n      "label_sha256": "012ebc5ce3e7aae6666bd30caffc76e78a1179ff817d582f6d2f586ea4deca7f",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1090-_jpg.rf.32a33301705ba97a75beaa636d828099.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1090-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 565,\n      "height": 420,\n      "image_sha256": "936ba40272baa16bc9ab977b18d30e86b8591512f190f8dd61037eb975f5d8eb",\n      "label_sha256": "d1a6fa582bc87c86a549c6ac682884f7188f87c17eb578863daf21010b42b2f6",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1091-_jpg.rf.e4e8f103b7c9889aa57f9a81622915ae.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1091-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 566,\n      "height": 408,\n      "image_sha256": "a41053b4777d536c0c51989bda0d2d3e5783a4875ef0d6201190996981c54257",\n      "label_sha256": "2d6ca5efd2c8f050e23c19bf03171809810a9822499aa11f2f828f4e15902e0d",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1099-_jpg.rf.95e8d8fa47b345a7aa2280177f04eea6.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1099-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 16,\n      "group": 942,\n      "height": 293,\n      "image_sha256": "b5d2efd29bba66787df754ca98f2f424d335a88a61c4188189dd7d12f2c49179",\n      "label_sha256": "bfea2852bc42c4a52c847120145dcb5db22ef136bc98f2657ccdeaf293b59463",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-110-_jpg.rf.a122b34121067c8d39a58bbd9a71916c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-110-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 568,\n      "height": 408,\n      "image_sha256": "4a75f106ab820c46f847e911f237f8d8a85892ca9604acf40716e8385d028bad",\n      "label_sha256": "3378b22bd33e257d2a6960f74cd71bdec1aacee4f4b6046f31e28adc2244b5af",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1104-_jpg.rf.313f6918c5e2506bca1039b2ee3d5e87.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1104-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 881,\n      "height": 408,\n      "image_sha256": "fdabe3227c4689d0e4318ed604a01686ef386808761149cd9bc9a8ecd0a4c833",\n      "label_sha256": "e493ddda0d75fd9a40d057f84bbd98ca1242a2e844dcd09d0827df831ca238aa",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1106-_jpg.rf.9fbff8c51353287af6f2b6796f2af758.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1106-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 570,\n      "height": 407,\n      "image_sha256": "d4add747e40ad555a59186ae07dc066e4168e670feb15a094b6853b93e33d6af",\n      "label_sha256": "b6bd52682afed54f14edc39ec2f568ca5cda176176865d7b51191fbc97179159",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1107-_jpg.rf.3a395c8d3ae96c53e1a3404d511ad172.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1107-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 571,\n      "height": 408,\n      "image_sha256": "dd4d4832f3eae26d59ec0f73fa90761c20129132de84d101e415e65677dfc253",\n      "label_sha256": "e88415dff6436fd794ca33f779bcf21d2eae6243b6add841f6fa8d0196a25349",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1108-_jpg.rf.f0ac364effee5e1bc5b535d16d8f558e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1108-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 572,\n      "height": 408,\n      "image_sha256": "0e89b164e1e414ea048914ea302c1c3b83ecedac4c7f919221149e2520fc70fe",\n      "label_sha256": "1e6be1e44877960ee6419b58c5bf3332f17afb47ff97036bfa871e0c05474895",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1110-_jpg.rf.3af34ac74749e0fbf30edeeb4c63967d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1110-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 573,\n      "height": 408,\n      "image_sha256": "5fe13ba37174fc35ac7c9aac819babfc2fa9ef1bce8c17a7755678096a63bce6",\n      "label_sha256": "0c6fe970dfe37d1be92f8dec1c1d79de091e5cba5577cb2ba50fe66aa714ea13",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1115-_jpg.rf.75039ae925977fd4a2d02a59d7d10378.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1115-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 875,\n      "height": 408,\n      "image_sha256": "9f6c00b3ff593b2fd03e67b16473c45f617113cd3dc23ecf3c9bc7d5ddd06f59",\n      "label_sha256": "ba94fc121cdb13a88f76e36dc81a1371f86a1ee6eb5ea39aa571b061b59a9559",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1123-_jpg.rf.2148fa3bcdb14ce1d7c8033bbe54cbbd.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1123-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 575,\n      "height": 408,\n      "image_sha256": "04d6a237893d3a15c023b95baa96520ff116e626874bc1230fa1fdabbaadbc29",\n      "label_sha256": "7979a2c53a2889a440ab47076bd3417c5a5ebabb896e160c1763e907568490eb",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1131-_jpg.rf.12d83c09d4cfb4f7dcbfc55e08f4bdf5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1131-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 576,\n      "height": 397,\n      "image_sha256": "56b317b7efcace83f006b83714ccd378a90b9ab8e9bc465f8f10e56fb171dd93",\n      "label_sha256": "448728b0070d22042bc056d7d8f095ba7a7fbdc86e49aa6fc4a727119305f5a7",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1133-_jpg.rf.85de26b0088233fd841cf2666a51617c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1133-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 577,\n      "height": 495,\n      "image_sha256": "0f36929f993fe65c9ed9290be4e83847d19a7d0b9a2c7c44741cc4a58608b09c",\n      "label_sha256": "05f31b27eca277cb75a78b7aea380d0020a9c84c6fc36c5d86dabd0902edc94a",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1136-_jpg.rf.55c5e4b47031f16f68af0d0134143ad7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1136-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 646,\n      "height": 408,\n      "image_sha256": "81fb881aa64478068f13cf4ebecd28ca56532e623fcf79ae43af2dd591285c6a",\n      "label_sha256": "b8dd1351880fc9c49d9acb2c43fcb34b84e1aa81b30257060c766e63a5c8e930",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-114-_jpg.rf.5d8d495aa4f7605fa5e0592e2568dc07.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-114-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 16,\n      "group": 579,\n      "height": 325,\n      "image_sha256": "2fe723e4713b530bc2f6c4a2030b15924a258eeb4812588e439081da51956f35",\n      "label_sha256": "f731aea534cf2b208f70af6df30faef328bd5dd9ed6a69083ee6903bab17fee8",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1148-_jpg.rf.041229d1339ba124434db85e67fbb6f2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1148-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 1031,\n      "height": 195,\n      "image_sha256": "8c66ceddae801f4b669a58f8ac381530b819154c03f9753cb6c639c5fc3ce75e",\n      "label_sha256": "abd791609f5b74e3dc7cd17f671d4a3f0db1e1a86104c6364b18e23aab9a13f8",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1161-_jpg.rf.2428fed8c61b2a8563d2e773273d61b9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1161-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 847,\n      "height": 408,\n      "image_sha256": "f086b2faa55d152481f5daf4d5005583c24eb836d63a7a20a1ccf3a538c271fd",\n      "label_sha256": "7a22d66b54538768beb9a68e9ac4dcb827399c0edee3cd4a5f2f58dcac2a3bc9",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1162-_jpg.rf.d5bfd41c51d288068f8557bd759fb7df.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1162-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 701,\n      "height": 375,\n      "image_sha256": "f08ed62bb91bded9b98b5bf30ba8febb64a6072f71edba44b0e10ea3f46366de",\n      "label_sha256": "e7a41886a22e829c3dc803e0964e948a8e14df416b7613885af48768cf2570aa",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1166-_jpg.rf.1429d3106924d7032c28027d05957f28.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1166-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 16,\n      "group": 942,\n      "height": 293,\n      "image_sha256": "49f3620e84855b67a6bcf39e8280fb49b2e967c0df46afe913b5b298d5244254",\n      "label_sha256": "bc466ae33521c67fb50cc0644f09b2ada381f3516780aa1d1b3e1db8557861bd",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1170-_jpg.rf.fc8505c4fd8d6be77e81b142ea1bb21f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1170-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 19,\n      "group": 1043,\n      "height": 255,\n      "image_sha256": "1dbf1942d325f5fa4980586c0b97aa1c1cc2fe677d3f6cc926b11fb00ba69324",\n      "label_sha256": "d5383468304f3ff1cc5575d6ed965064421d860159020901a385e9781780c772",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1172-_jpg.rf.ac90c725b5b971b306911ac81e8526ca.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1172-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 896,\n      "height": 193,\n      "image_sha256": "e98a6695d245187be557786ee046e0d5437c2698966a790d6529a0046b950ef9",\n      "label_sha256": "605cdfa05f0ebfeb1b7d5810bb016d745185ba88c707e7556aac6da24d3536da",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1177-_jpg.rf.972e5385cb5860f956a254495e989567.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1177-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 770,\n      "height": 246,\n      "image_sha256": "d8a0bf0d9125b5b9c9c477645b9ad3a0359f7de0a6cf0f868b187973cd295762",\n      "label_sha256": "208362d1bbb867c6d326b95186f50cbb22952c0ca32026b97916d0303badfc7b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1180-_jpg.rf.bf453bad5b5027e8103544822685a190.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1180-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 900,\n      "height": 319,\n      "image_sha256": "e91fcc0fff8970ccbd2034d33c8122b22a9f002f8d0fd88c9cdf099e88e7aeae",\n      "label_sha256": "305a25882d5951421ae47899813ade4e96c8428a85a98cf4d9d08e6974ad9ff3",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1182-_jpg.rf.281d2a64983611d0ccaa6635c4543116.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1182-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 23,\n      "group": 1046,\n      "height": 609,\n      "image_sha256": "d2565c5f1fcd04a9fd0f4d2b1f592ac7e60dbda55341626b36d095705f4737bd",\n      "label_sha256": "a1e2e3156349f8c87a847bf55ec7bd974ac24dfcde59e8f0508b16454949712a",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1183-_jpg.rf.5c8c2b19f78de6217d0c791bceaead0d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1183-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 16,\n      "group": 1035,\n      "height": 407,\n      "image_sha256": "c80a4b27f72882cf3fb1badadad31bf463188b0055e258874c52c9c9aa8335d6",\n      "label_sha256": "488f3d9be0dad1ffcf12725f253446df5f3726cdd5256d9bed0befe69e649b01",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1188-_jpg.rf.a0d7ef16b6f03637cd53b288da00d7ba.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1188-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 44,\n      "group": 695,\n      "height": 194,\n      "image_sha256": "d7b5a240cc4a933f173b8ce52e584cf0ce2c525b01c9535e2eca6472e47b1818",\n      "label_sha256": "0473f528c422e8dee7fdc3d89491d3900a95bad89f92dffa265eb240ef7140e7",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1191-_jpg.rf.2834593a99e55e49112782add6740df3.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1191-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 19,\n      "group": 696,\n      "height": 408,\n      "image_sha256": "7f705400b6ba3e08650c988c9ee3061c9df6d41a1c4553449ddbb0d792c2870e",\n      "label_sha256": "e72be0184f114e10e20f851163d42a5401394c6f9df7335176fa2a3d5f0d9f33",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1193-_jpg.rf.95224301226b80e517ccf57827524cdf.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1193-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 1055,\n      "height": 455,\n      "image_sha256": "e586bb96c6cd000032dc901ab082e62dccb38284bbde31175b054c8beeadd1f2",\n      "label_sha256": "2ec88b055fffbddb0fbe674e037820457e885f7057dbeb07fe7b53874133099a",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1195-_jpg.rf.52aee2ebd3ec383b957230c5eba0dd3c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1195-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 15,\n      "group": 247,\n      "height": 408,\n      "image_sha256": "d4a00adc6cc3f669ff07a91d189ae29f63f09cbbf65e22d052d695e34c03286d",\n      "label_sha256": "5be05cf206db59cfabf095cfcdd530c669e0e13fee6132fff163bd42eee06297",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1202-_jpg.rf.e04b467982cde1d3e1767cfd639b8046.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1202-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 1052,\n      "height": 408,\n      "image_sha256": "cb9dd5f22c79d0fb85b1b3a36ef7136763a0bca5f674689186b91147d768b039",\n      "label_sha256": "aba782df73f539448851eba28032e5c7ba0e417380fab5d5045a5e2883eebe79",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1203-_jpg.rf.fc48e522b9e164881147a776e80ac247.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1203-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 42,\n      "group": 1059,\n      "height": 151,\n      "image_sha256": "f2283c0f46b22ea6c2bdbeabecb0887e7fde2c34a6b84d01bcf47f2457b91087",\n      "label_sha256": "8947b2aeb69b867036c1c706f1e3c265a670a2e9bc57b3225970290d1500c5d9",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1205-_jpg.rf.efd67671b9898e9cabceee48db9aaf9f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1205-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 699,\n      "height": 612,\n      "image_sha256": "530e7c1412c0f3b2355886e440c17bc3964b6b67c829d6ebe664d583d0b12f7c",\n      "label_sha256": "c6ebd596c52a8eb375a575556c9e37ffd397502fec00b1fb8e7c2034e3642c8b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1213-_jpg.rf.426e503d9f881f3efea6ec697eb0190e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1213-_jpg",\n      "split": "excluded",\n      "width": 601\n    },\n    {\n      "boxes": 14,\n      "group": 1080,\n      "height": 372,\n      "image_sha256": "0ce1109313bc2bfbe7a30292451241040c0375dc21ccbd01ecc488dabc6845ea",\n      "label_sha256": "dc9c2f25d822f1b2150d3e8e3436d06965ff1c6a72cf627389cc5643a82249c7",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1216-_jpg.rf.c4b4e656b11adfcb312762db96e5d63b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1216-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 671,\n      "height": 408,\n      "image_sha256": "155631d8e50f9d10918478e72a37b456d36ae2d3b65cd08037eb7299cc7fc7b9",\n      "label_sha256": "fce4ac58ae7b5620de82757200376faac3537d1af4c6dd1dbc64fe8b9e70ab30",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-123-_jpg.rf.7cd5fd15c449f41bf80ba9b18007bcb4.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-123-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 702,\n      "height": 424,\n      "image_sha256": "b2b441379582c42f1c678ad91e3b58f2dcf7800d810bf811d4b0e7c53e47ae51",\n      "label_sha256": "70d6b8ec4d5908701740c9fcce0c24c2d750d31d80b3335b125dc61128c4618a",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1234-_jpg.rf.7caf17525ad145906e8838c19c09573b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1234-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 671,\n      "height": 408,\n      "image_sha256": "718d583e5dd8d98ef0f92a6c776784256d5b810d3dad9276ca525b88cf4102fc",\n      "label_sha256": "a26eada8b449a5b3ddf580ee67e67b326d2238e6e28dd438fb658f6c68ef05e5",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1238-_jpg.rf.a6d911cece0a665088c713f09d28f9c1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1238-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 269,\n      "height": 283,\n      "image_sha256": "d3c4ee33fb95d5e909df9b748de93b79d2d660bd801762344d637fdd1d4f89c8",\n      "label_sha256": "18ae5a8307e2eed904b44efcc9e37892d11e00148a6f50e9d6982d21f7998928",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1239-_jpg.rf.08297d68a011644b679ded610c36c53d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1239-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 699,\n      "height": 612,\n      "image_sha256": "8c97726fde162ea112276ed4cde46eab93d6fcbabf1eb9132411a1de36b043c0",\n      "label_sha256": "d435591c56bfce1197601f5d4dede340f802ca1ea985457a1e87b690aa3a7120",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-124-_jpg.rf.19dbd54aa9b63613d3385798d1aa5592.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-124-_jpg",\n      "split": "excluded",\n      "width": 601\n    },\n    {\n      "boxes": 11,\n      "group": 1039,\n      "height": 204,\n      "image_sha256": "60a96a0ed7a73bc855cf0f89a90259725cb78a11e35df8d7cface0e3ecd17b7e",\n      "label_sha256": "e4abaae3ef8f4cfaaa54f62b96373071760e28b0f271b9f75f017237fa274338",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1255-_jpg.rf.df8d63b3a26e9e8967ad6c8a1a88e796.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1255-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 203,\n      "height": 408,\n      "image_sha256": "ee2b6d70f3fc7f911f2d18f9efa82f92cf40911cd752bedbe0e2c6355325827c",\n      "label_sha256": "703ada698ccba0407684d4cf5058e2c18d57e605c275bdcb2091f7e10befd86c",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1264-_jpg.rf.f57f0205b265712f62a87056e2d8c2bd.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1264-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 915,\n      "height": 471,\n      "image_sha256": "ee2b5ed2c01ae2796b1b1a86b33d44e5d0c7886d130b63ff23f8a7e3994f50b0",\n      "label_sha256": "f956bbf4dc847bfb8ca0b2cc40fe13050a582797e1f4e8128c7dc0767ccd3db0",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1265-_jpg.rf.f7ad2f4ffb6b55ca6376974cdc9e0bf9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1265-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 646,\n      "height": 408,\n      "image_sha256": "84c45cce9ae104dc843c5e27c9d7c72ba9b493f0f804df8ef099f384b75b6266",\n      "label_sha256": "bb6821d7a27622a3859f5d02924424a8281d703ec28745fce86e9edd4f303cad",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1266-_jpg.rf.f02ec3733ed3511f87b1bf71a763d656.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1266-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 16,\n      "group": 638,\n      "height": 408,\n      "image_sha256": "3fe40838df77b306db82bee53fd20296bfdf9831b1c7f35614fcd0e0b60612e6",\n      "label_sha256": "ad30fdd339b272e3db7a12e67ec18ebd012b26d3eb7937ffe170431fb811cecc",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1268-_jpg.rf.426d9918e42f0c18baffe02e0bb7357a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1268-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 27,\n      "group": 712,\n      "height": 495,\n      "image_sha256": "b524a019ad87ffe36fc58bf15dd3f1cdbc6b2b0f06bc6d4b74a57ebbbeca10a4",\n      "label_sha256": "3fdea02565e7c631f5b64101e8899e6f74c478aa88947b8052af8fbc8c35d55d",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-128-_jpg.rf.877a5a9a01a40acf78c93652dcaf87b1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-128-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 23,\n      "group": 712,\n      "height": 495,\n      "image_sha256": "6126c43e7e3c444dc201e85381c3b35911c2db68d6310d7b9ae323ba18a8b9d9",\n      "label_sha256": "49ab4cfd38bc233b360c718a93606a74e3e8a5e847938370dd40c708a62f2410",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1280-_jpg.rf.96ff605dd6a57983ed9b7c3939b6766f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1280-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 909,\n      "height": 407,\n      "image_sha256": "8e7b0d30785b4b770932c428892a2e42f3470a26b9999feca00805083cee4e9e",\n      "label_sha256": "6cc097b0a9554611af4d1fa389969059dc53c2d6f11f804e2c3fc158bdf2c044",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1289-_jpg.rf.de2146223a5e51b5a01be7600021be1f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1289-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 672,\n      "height": 408,\n      "image_sha256": "030787109660e19c0ffdb6dd762b21581fb6efbfccd455b6a76b6624562fc337",\n      "label_sha256": "e14cc1db279dda539e3484481018a8f70f70b234973a472b5f99d51741cf1378",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-129-_jpg.rf.dd4e27de0255e1550f8f461862de50b4.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-129-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 913,\n      "height": 409,\n      "image_sha256": "093b49dc5d3e2f2948ce5da552274487d54aa43c3d09a1983f19ee630f826c76",\n      "label_sha256": "7239eb5ec550b0b7cde6c55012d94917539a188dcefeb5e5456a573ba34372ea",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1291-_jpg.rf.52b2f8abdb6cd5bf4d92d0ee2e6c835a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1291-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 683,\n      "height": 290,\n      "image_sha256": "5bd6a24b1821af62fc40f35a986383a3bbce762f05d52326375f008a8d172c32",\n      "label_sha256": "deafdec05f262430d8f4f7403e859e189674f83c50b41fe0d770d15c1d7416b0",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1293-_jpg.rf.7466d4c11156302a4c5b37e505c81fd9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1293-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 16,\n      "group": 696,\n      "height": 408,\n      "image_sha256": "15b71989461cf82e9b3e54b288af8e9faf5211c2ea8d97bac2e4d217e9440a57",\n      "label_sha256": "62124c1323c0322a5793982fbcb5e476ba5568ccd7b8734f1b508654421e6a23",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-132-_jpg.rf.1975385d5995867217cdb58a23489e5b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-132-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 701,\n      "height": 375,\n      "image_sha256": "fe9409839c5840bb399dc7110fe8cc0f30e991911de9abfe05e4300edf96efa6",\n      "label_sha256": "36a6e6a6dc8f81eaf5973324cf312ac5a15db8e98c6ec5c9b0d448db75cb2a1c",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-14-_jpg.rf.67930655207381e53c3029e4bb8d7697.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-14-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 616,\n      "height": 405,\n      "image_sha256": "607e4472fd27bf9592d3599b564a4abfbc26fb7ae861b7555205415b62c687fe",\n      "label_sha256": "e32e9e1134b9ad2dde1c24e73e621421b67cd14fcee483e6995e2c12622c19ce",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-147-_jpg.rf.ea79d77c77ebf1d832af92376c9b4bc8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-147-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 922,\n      "height": 408,\n      "image_sha256": "3642de3dff25370476590632dca249a78c44edeef5ec3cfee1d20388df6c5a2a",\n      "label_sha256": "8e1b729a5303a2752075026d433f46b9f0a995be6e5fbe551081953ec2c29367",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-148-_jpg.rf.e99f6292f2a9f24bd4870d9e4eed6b00.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-148-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 3,\n      "group": 618,\n      "height": 408,\n      "image_sha256": "6d296a943e2b24110f03e1af4a6ca0382f87287f96d6ea963c87fafd474c4f39",\n      "label_sha256": "a7f074c4e09fda19060e71534e40ab9ec2cb152760cd99a82146a0ed349036b9",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-155-_jpg.rf.c795be85d5d4656227f143166a61b424.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-155-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 619,\n      "height": 408,\n      "image_sha256": "0d6096ef6e322503b8f818e39faa756144be3822543704d727a65967a949d81f",\n      "label_sha256": "611c5b2e2f7f59410d4d2a67289cc45f290916ee5c7de1389946eedaa82993f6",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-157-_jpg.rf.82f802da14f0d540c32f1a21bbc8043e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-157-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 701,\n      "height": 375,\n      "image_sha256": "2dd684aca4a6f112d6a5a192ace9e44c22bfb65bfb0173472a143b9676054fb1",\n      "label_sha256": "d2899b3f515a52ace24a0809c5567acf9fe53a59ee792fb3969a3758d7faacbc",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-158-_jpg.rf.a2823a7e78316281b34afb8e78006b87.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-158-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 766,\n      "height": 408,\n      "image_sha256": "578b61721011bbd3660fa20b1273e8cdc08c08ca6b6160bae818705a2253f00d",\n      "label_sha256": "7483337fd91d673d427be891a2dc7957c6a6fa714dd6bc1f0bbe965e5da22f9f",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-161-_jpg.rf.27cea5076032f3ebb1fd78ec097d1065.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-161-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 1043,\n      "height": 255,\n      "image_sha256": "bc01f9c7ed8674b0b41ebf877cf25b4765a3c57607710acd618e8b569c2326e5",\n      "label_sha256": "28693d9a8a511476553b7c3ba91333f13e240b4f50694f85354ab688cee64843",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-164-_jpg.rf.b188ed0aa64cdbee5e0f7ad47447a4d0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-164-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 653,\n      "height": 408,\n      "image_sha256": "63c6de9157c99ab23dc88aba909ee344a0cb9b467571357b81e35e7ecb395360",\n      "label_sha256": "79927acde4cd96d2a959c44fdfbac513c547bf8dcfd74e775033c9ab9b602236",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-167-_jpg.rf.9ed90db13f3c620c98ce92dc8b9ff99e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-167-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 1044,\n      "height": 402,\n      "image_sha256": "f5ac9eae8f494f65f05ed58f4173cbebc54bb118831477cf4b0eb872ce625c52",\n      "label_sha256": "ed6579b81b35be37e86d3da75703c9b41ae8d5c4ab94bf4e81a78cf759bc0770",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-170-_jpg.rf.85ed4af19cd7378b22f7cdd14cdf1754.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-170-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 723,\n      "height": 355,\n      "image_sha256": "bed14488373021d267d16ee8b4253af1ab03d42a127eb7fd3910e3aacf5cbdd9",\n      "label_sha256": "fa35b307a596cb80f20ddf2c6af78a8e49bc2556f010df0e42329337d311e622",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-171-_jpg.rf.c6ca4a25cd8dbabce54b7593e43e3498.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-171-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 770,\n      "height": 246,\n      "image_sha256": "75c2baee87eb1162e62edd70965a7ece748e237217acdba87225cf06bb9b123d",\n      "label_sha256": "84a50b9a0cebce9568345b633299b2106f30858070e54b386dc50768da031acb",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-172-_jpg.rf.8e6a42b3812cfc1dd9e3775077272784.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-172-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 1,\n      "group": 627,\n      "height": 340,\n      "image_sha256": "b8edf7eaf5cd68184058cef773798e78be3ebda9c32cbd9039e02eb571ab95a4",\n      "label_sha256": "97946b11828eef0683d7fc76c276856f430b4930cee72e13d24a371749e0d74a",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-173-_jpg.rf.387dfd20eff033f3f4f8cb71d0f7c8f8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-173-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 900,\n      "height": 319,\n      "image_sha256": "de8a187caf4fdaf4d33d888e1c417c314e0e579dbb0d6fc44610cc0768150919",\n      "label_sha256": "4632211e013f124f91d4fe9d0bd104058c419e8d741baa2112d859d0211494b3",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-174-_jpg.rf.59ec469abd36c1d98864094e2afd698d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-174-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 629,\n      "height": 297,\n      "image_sha256": "665579c8164f5d429ea5b9cd417cc1cfab0285d76163696db7b6a9b9e980246d",\n      "label_sha256": "b11fa3595525838ffac8dc089dd225c77f9c61e901d15eefb45a608b0d6220f7",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-177-_jpg.rf.450a43ef296f54f97999146e9c5cd798.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-177-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 1048,\n      "height": 263,\n      "image_sha256": "964d46d5a9da8df924bbf91963c94e2aab5ae3b0d68dde10fe79b929c776d39a",\n      "label_sha256": "6c2dcaa711a849b5fed7ad43ae20939ef6f0eb8790163c99dd1dba0729b45a47",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-182-_jpg.rf.6491fe39e0c583700136bd100b4275a9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-182-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 37,\n      "group": 695,\n      "height": 194,\n      "image_sha256": "3350cc89b2bd8ecc5d502bec694857ba6c9e494425a8f00f99ec6cece5d11e7e",\n      "label_sha256": "942ebaa7c286a15f5eb0614f3ba4aaca6026dcb1cc8853c369636d3096f00f1f",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-183-_jpg.rf.6bbce72392a795a315750191532d1aeb.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-183-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 632,\n      "height": 455,\n      "image_sha256": "c6826ced00c59ebb0d9fb3879c9c4d958996a3b1d7a3dd7ca5d1e2967901f797",\n      "label_sha256": "7df0aeda06a4d9bf56ee0d473a617cfb9dc121edcd1112c707ea338f93c6a59f",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-187-_jpg.rf.5a037e1a273a97437e6ad0b2e0d65d5a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-187-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 18,\n      "group": 1049,\n      "height": 125,\n      "image_sha256": "953902ea5b13634222b2bfe77399c7519a0587a926631b13887236cfcaa59bd4",\n      "label_sha256": "8ed4b55c55f7278d7a897414a086031b9171ccb3860503f637b08e427963978c",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-188-_jpg.rf.f0f5f82628f4665ff6677e90308adbe6.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-188-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 687,\n      "height": 408,\n      "image_sha256": "50024210afa33936bbba8628dce9ba15d391e089c5a6169f3a47a7fe608dcfb8",\n      "label_sha256": "8a2bc483cc961f6e2a5fb03f918d90ae37b398a7fda9a4b382cf4978a28582b6",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-190-_jpg.rf.b46ff9fbafa717735282bd8104680459.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-190-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 16,\n      "group": 1051,\n      "height": 204,\n      "image_sha256": "84bb8a29911defaf85b6c76cba4ec7c1633551450c1436eded9e8ddcfaf7bdf2",\n      "label_sha256": "20ad8c7141d1a0f099f11b12e0568988c69b2aaedaeb364220e3de9c2d395c21",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-191-_jpg.rf.2059c383b1d71df364989d2d318ba72b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-191-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 247,\n      "height": 408,\n      "image_sha256": "422455d41cc69f76ae39807771a11dfcb2dacd76c9e3ff255c948ed3fd905a5e",\n      "label_sha256": "8bb1ae34b4db6a0bb4f33f4bc4b1c3327631a4060787f60cf0199c8bbad45323",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-194-_jpg.rf.81730d70fb5b646c2fbaa3a332e4a77f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-194-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 27,\n      "group": 926,\n      "height": 151,\n      "image_sha256": "9244937f349e440e174e05f4b958f74e2f058008f76eb063e133299c1b8a281a",\n      "label_sha256": "8f8edaf2ddeeac04e3a1dc8925d16ec0625777b369175b0560b0e5f15b1d869d",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-197-_jpg.rf.9dde94fd1ab4af965055dc3b65439af2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-197-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 15,\n      "group": 638,\n      "height": 408,\n      "image_sha256": "2a78dcd8e4bb4c55ac676424d1fe0a60d724886be486cc9a6457c21c11a8fcf2",\n      "label_sha256": "ab0363eab708ee37351f85dd89c0e425493805123a97709ed66749ff158bbb3a",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-204-_jpg.rf.fb69b4bcf1a293d0cef9b70779f19176.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-204-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 699,\n      "height": 612,\n      "image_sha256": "91e0db485fd6e1d5e00250f826d248c494ac7a4f8219b46efb68ee31321e2134",\n      "label_sha256": "e028dc8cfbcd05b7a01bbfb7d857aba771cdae7049947837a67ef2f89f88e0ec",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-205-_jpg.rf.6ccda58d377cfe859233d6427ae98b2c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-205-_jpg",\n      "split": "excluded",\n      "width": 601\n    },\n    {\n      "boxes": 13,\n      "group": 674,\n      "height": 405,\n      "image_sha256": "027bd2d069a68d043245a398a66b7a0ff0604ca63595a2e2c1cd83fe7eac3db5",\n      "label_sha256": "32e5ab4a8c824cdbde9e19e2015598f49e984612a681a00d9d147595c86f7b24",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-207-_jpg.rf.73b43df98ae1dee223cd8afcfa7ef8ea.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-207-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 263,\n      "height": 408,\n      "image_sha256": "10f1472ae66297399c9a61e4780582d7d00585d9c6165e8ae326a8d1eb588243",\n      "label_sha256": "4bee2b22f1ab92fcc294e5fed7eaa749bb2b2ae456ffaa11bf893a3590507ace",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-209-_jpg.rf.64f70c3630e196db65e24f5f80c51c52.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-209-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 642,\n      "height": 408,\n      "image_sha256": "a9b34949c1eb8005fbfd43e4911c3032c97a74b6cae30a56d1f46c2051701436",\n      "label_sha256": "136eef02b87eb759e0883dcb4a4c3a070822f36950f1056147a1675b84d5a709",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-210-_jpg.rf.986dcda2c9276c976f4373b1ebb09c20.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-210-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 940,\n      "height": 406,\n      "image_sha256": "f0c0d85ed5ac3843396b58c118debbc9f7e4f24b7ab3bca09aed45a8d439250e",\n      "label_sha256": "4cfc51293032223584505bf5879d0f3f4912c3ad2549bd44ce6987d156a76d2c",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-212-_jpg.rf.3ce803201a314418a8cc663ec73a14d3.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-212-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 747,\n      "height": 408,\n      "image_sha256": "f5b45fad747a505337e6977ab50ed94da100353700738e33bbcc6282e2df14a6",\n      "label_sha256": "ddc4371c0a052c249fe563fb50906c602e061a162a90cdb895bdaf2d41e24c07",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-246-_jpg.rf.1c979e9b36ff69b4e91c3cd566c85a37.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-246-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 896,\n      "height": 193,\n      "image_sha256": "5b4e9327ba39f3b0ea5052f5b2dead9e206d6bff8866fd3a410a38bea158965f",\n      "label_sha256": "ff333104fa771f48d8a0033e36c7bd28aab86c054152c4cd72739fba6a64fdc6",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-25-_jpg.rf.4ba37fe0988e0f3380d8e629efcdb0c9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-25-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 646,\n      "height": 408,\n      "image_sha256": "8244284e163985c64073d48b2f72b04891fd1d567ebb453a569bfd094cef6288",\n      "label_sha256": "1663ad0aaeb47ebe4905e255d00f518859d1aba4ef2563f22ef8cef7378aa2cf",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-258-_jpg.rf.040f4ebce284479f5ec883e8bf12b7af.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-258-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 19,\n      "group": 712,\n      "height": 495,\n      "image_sha256": "af9d1e73dafbcf595b55cb77232ac7039b25af0f19e87b56b5ef6e6f66a467c0",\n      "label_sha256": "f9be20889cde45636e569794578c867e67306a90220eb60e8b442a108b81ae35",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-272-_jpg.rf.8a36afba27b12ea4c8546b54a43cf2a9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-272-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 766,\n      "height": 317,\n      "image_sha256": "c935e3fee7d50f4a2fefe4dd5d01ac708ed01e4029b0a795678f815d4a2932d0",\n      "label_sha256": "0974e1104311afd3573662572a8ff8387bcab53c978fcac035b5a276e1d5fb86",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-278-_jpg.rf.bb18ccac03e4a4875880122652dd97f6.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-278-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 683,\n      "height": 290,\n      "image_sha256": "b5eef4bb3507f5e651b85f00725b082d96afc4ca482730216fe41d1c50ee6197",\n      "label_sha256": "3f615f4e5f72e36803cf4ffd34834245838f5958d275a2d0ca3864d9325f0911",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-285-_jpg.rf.9e353079f8afceb5f6599e59426ac288.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-285-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 1031,\n      "height": 195,\n      "image_sha256": "2bd6999275dc72cb3cc9428a249a116b397e8a8f9c605c4205d891f83342af5b",\n      "label_sha256": "563c08cf6d064a72f651e15d518636847b5285dd66da6edeb34d4910c5e2af9c",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-297-_jpg.rf.8ffc15049d61980455d2e1791e059a30.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-297-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 701,\n      "height": 375,\n      "image_sha256": "f4573ed85a66f0ae0e2700e94c2548f1ffa20e71584b2fb0dda83b12d00d3b1f",\n      "label_sha256": "09275d18d2e0240da07f21608108392af9a766c76194007b441c23f40e3070ac",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-302-_jpg.rf.9d9e92808f43342596d0c2f9c4a8d13c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-302-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 3,\n      "group": 1037,\n      "height": 408,\n      "image_sha256": "baa588711fe51c4053b8875d028987fe282dca5c8522000aa11ab3dcd20ceaca",\n      "label_sha256": "47a7c3ab04cfb833663ad07ca06a8714d82a28230b6b53404ba6a215f5b61de7",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-310-_jpg.rf.46ab084042ca3df86fba115b530ae834.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-310-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 653,\n      "height": 408,\n      "image_sha256": "59baa5e6c39df3730bb8a39cce88deecbf1df23a8775bc9778be964486cc4231",\n      "label_sha256": "ed45a92b957351989335dde60453c672c149e6a0fee21286b354b7a946b1ffe2",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-311-_jpg.rf.a1e071d3ab1fbd5116f0f0a70bd798fa.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-311-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 896,\n      "height": 193,\n      "image_sha256": "f1ff2046b99341bbdfe9104476f3f12402dcca7127496e10a08028a261c3d908",\n      "label_sha256": "ddb6f92bb940d59e31f4e5e403ab44d26e39fe4f3c50ff328137b613bbcaad01",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-313-_jpg.rf.983d53e8aed9feafa429e8b490e51b15.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-313-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 655,\n      "height": 408,\n      "image_sha256": "54a55e2b8d317b2b970449ef1fc340b383c70fc383b43077702497a5f0916642",\n      "label_sha256": "72ffc880bd788d57a6a54be187b374ba5740149d0b5f9f665df3a27a58c9182c",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-323-_jpg.rf.2717e7e70bec9b3d9e8c0355cfcfcdc8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-323-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 33,\n      "group": 695,\n      "height": 194,\n      "image_sha256": "f1f0fa63787d3de2c3d33d552547144513437c4a50d503d49203e6344d4d2ec7",\n      "label_sha256": "78f0862b778ff36f5bbc0463505ff9e9bab7a3acfc5bd39506d9aa8c6c34c40a",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-327-_jpg.rf.28cb6c4bfb4ac67ae90a1cfadb1a8495.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-327-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 18,\n      "group": 696,\n      "height": 408,\n      "image_sha256": "8f28bf796a111951ad4500f8d4040a554ec5e6c58beba5730c4c8d9fe4b14cb4",\n      "label_sha256": "f47dd886bc881879c434c9ed996138238b59072250840c534b12d811156702fc",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-329-_jpg.rf.698395b0f580b0d3c04495f7e7e562a7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-329-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 1055,\n      "height": 455,\n      "image_sha256": "5c18744f3d02624605cebe7e1745b8fbc4334ffa06a98d6dbf457cba0f85d273",\n      "label_sha256": "295b926fae93c248c9fcd24723dcf2b8c3f0cd5b61da82885e96f9911fbbdfc5",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-331-_jpg.rf.514410a2a63efbdbe39a639e11e399ec.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-331-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 1023,\n      "height": 398,\n      "image_sha256": "19eca6b433785e240b620d8c6f8289766540e1d5068487c4f0827eb27a191f73",\n      "label_sha256": "e2fbef2ea491a097306e314d43240e369cc69ff3428f11a1f7a3f8e25a337ead",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-336-_jpg.rf.714638b7f17e3c8d783219695e7122e1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-336-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 247,\n      "height": 408,\n      "image_sha256": "a86835e2b2b04623069df169ecda253803ab12c60f6b977be2a1050002928e73",\n      "label_sha256": "40034959f0ddcd647a2444ba1d0f5b32adf14cd62e846dcacf5b79223dbf25f4",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-338-_jpg.rf.e43c1603e30ec5ce82c11b2ed010aae9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-338-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 699,\n      "height": 612,\n      "image_sha256": "c6a32588fcf9bfdfb084a0a923de82c649ea23382329210b589b34b757be48fe",\n      "label_sha256": "6a6c15e08adb0a04703bc55b3c1637e9c88c6d851888a9cf0a8e4da1514e102b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-349-_jpg.rf.b89923a52435328045a6469a18a2d487.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-349-_jpg",\n      "split": "excluded",\n      "width": 601\n    },\n    {\n      "boxes": 15,\n      "group": 1034,\n      "height": 408,\n      "image_sha256": "690e8f2f3cbd70cf0f23e3a5a327c87f559094af527603fe10cde2591b1301fc",\n      "label_sha256": "4ebe3cc3abc3e04faa46ca11fc9d17ca68b3ef6cf3c062f9c60a8a54582d7618",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-35-_jpg.rf.c356ea2f1eeeff6b20f7f7c84a2a8ee1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-35-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 15,\n      "group": 1035,\n      "height": 407,\n      "image_sha256": "c32df111503d21b9615c3f038a4c6e531ee127cdbf5e887f48a7de76ba4f078c",\n      "label_sha256": "39113a6f07b927a675f610f3bccf88e99d11ba1630db78ab7ce70b51bae63bfb",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-36-_jpg.rf.5baa2be35f12529af0786ab39b3df98a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-36-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 664,\n      "height": 420,\n      "image_sha256": "8b0bd2c8993c040fe9cf63ad60bc686539402ddfebf6356d137428513e9fa6e9",\n      "label_sha256": "dbbb65a56b6689fc8c651daf9f8528cb50d77d8148df772fee7fb67a78f313a8",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-371-_jpg.rf.e15b77320e9ea14646a8d03254f581a8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-371-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 665,\n      "height": 283,\n      "image_sha256": "da8d08ff652fc4dd553526d9f08e8d9c581b6d4f4767ecf35c96ccf185215911",\n      "label_sha256": "fc9d8b60adbf2f874788e597884eb97bcd5b9959020477057df784116bb19b6b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-375-_jpg.rf.4c3386ea40c6ab6d8727736a60cee905.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-375-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 1038,\n      "height": 408,\n      "image_sha256": "f8eeacdf2e93bc59a0f1e35acb4755405cff0d1f3de1babac10b2fc888c52b33",\n      "label_sha256": "14e2173d8344f281c9ccc7d3d0df0bde0dc166506c945dd414c8deaeff9045ff",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-379-_jpg.rf.2fdde1cd5c4e2b39fda162a0be5ba973.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-379-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 41,\n      "group": 695,\n      "height": 194,\n      "image_sha256": "8a68992c218e613f7c50249f746c870a73d4a08a9c5e5130e9a5e9c08e32d542",\n      "label_sha256": "a961821e95fe4f5ab39554f644da6d188dfc8742865ecf9441092b55c8ca7877",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-39-_jpg.rf.a30d05b8715bb73b5a196768971b4917.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-39-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 15,\n      "group": 915,\n      "height": 471,\n      "image_sha256": "6bdce738a7dc459bd40eecfdf0d4bdfda90688f6704dd1ddc20fbbec41d36bac",\n      "label_sha256": "ad0f722c0cc27706312fcc3350dd806d649baaabc030133bd7980864479e4d9b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-401-_jpg.rf.56db730f6579b85738334ebda4c8d8b1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-401-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 26,\n      "group": 933,\n      "height": 408,\n      "image_sha256": "d836e2cc7a09ca1f741c065450fe61329432db5f6778f457bc0e5f247662f33d",\n      "label_sha256": "05d5cf54f736cb7a083884badbb803f792e4029dbd0179e9833f0498ae6f8325",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-407-_jpg.rf.488f8c887612a2495d118e0c56ff6fa9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-407-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 1080,\n      "height": 372,\n      "image_sha256": "fb36211e94475c75198059f0c229619238d76e33210050872b01c956b484ba69",\n      "label_sha256": "08845531cf88ae5bf6a8c44a417fdb59d47c328ac5f90dbffe951977ac636837",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-410-_jpg.rf.a9a52f67dd9f320567a1c5ee004b1aea.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-410-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 671,\n      "height": 408,\n      "image_sha256": "6c76ab764b3661e82394760f157c7efc88c2ffe4baec51bcc17a3ca81151682b",\n      "label_sha256": "b78091a6f33b19dd8a78c7f6ea56579bb230e4a9e6c9c3b786a07f58e8141cc8",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-411-_jpg.rf.bc94f5322c77c49a7bfab20d3e42333e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-411-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 672,\n      "height": 408,\n      "image_sha256": "f4471b12a55947bd6b5fa5bfd87c688a208a3c6f94a49888425c2f8d84c811fc",\n      "label_sha256": "aadd16000f4c87b8c0b8d6711a72e1a344902d20379b8d2252855c22b035bec4",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-417-_jpg.rf.aaab16e4f24f8ff9b7dc80c78f9e2c99.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-417-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 16,\n      "group": 1040,\n      "height": 459,\n      "image_sha256": "fd09ed4c662f7442fc48d8e857989824554b4f602a34597c708f59df2bba681c",\n      "label_sha256": "2f9cda0e4f124939f6abbaf1bc2ae527821c2a9ef3f05226e71583f145fd9e97",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-418-_jpg.rf.b156490b11501747f26cf05b68bff773.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-418-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 674,\n      "height": 405,\n      "image_sha256": "6e641ab46f4a6f88d6e21303dfb8eaad054886f57fd91736696543b62703e94c",\n      "label_sha256": "1f2245d2bc364c780e45a37f13e220e50c30ca99aa32659b2eb987c16a792cc3",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-431-_jpg.rf.164f4b017cffe632aac79f1d22eebc69.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-431-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 718,\n      "height": 408,\n      "image_sha256": "c999d425e63ce3e0e2004381fceeaebf580efdafb095420e751a0081d29c216d",\n      "label_sha256": "048218a2b92bee9a4ed4c07210053137df959e1f1bbaa004b5d6106f7773fcda",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-433-_jpg.rf.4fe4e6519b609af80e6bd5b0abf0c04f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-433-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 922,\n      "height": 408,\n      "image_sha256": "132d85e203a8b7dba5b9d5d357e34f02eee09c92739c6b85ec81fa389fee00cf",\n      "label_sha256": "b817932430937e0b1df88fcedf72cd6a9cdfef6e3b42b5794b41f6b803afbe03",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-436-_jpg.rf.b26cf1720d1787ffc83892f4aad19bd5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-436-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 1017,\n      "height": 192,\n      "image_sha256": "dbabd6b8281721411eaf3393ec2919f52c51f7069bc06300b890f2b4371e0812",\n      "label_sha256": "16c08cb3782551db4c5aacddaf354b69d08136e2088ed1c88b4b829b03990465",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-437-_jpg.rf.a034d6bf4a5a636b99a19d50d7a49fdf.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-437-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 25,\n      "group": 1049,\n      "height": 125,\n      "image_sha256": "94f8987e65d728e5d802098766d089de3bbad52a9e4f98c32e248e5876c52566",\n      "label_sha256": "5f07201742b4a1353474bc6c28efa0a7a258bfdc568f7fef264d8d68216106b0",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-44-_jpg.rf.c29b97f15e757e3d72d1f67cb25ba059.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-44-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 679,\n      "height": 459,\n      "image_sha256": "e85ecec0621f7ab6bd3d85718bbafa05dd56f6f4716ab17b2b070cd3b9caa574",\n      "label_sha256": "b79a9300f60dfe02aebef5ca8f4c3046da4c0ebfe8e736ad7c60bdec03244136",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-440-_jpg.rf.31f9e172e1775efef195a64ea1c1153b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-440-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 17,\n      "group": 1042,\n      "height": 408,\n      "image_sha256": "ac484304a1575a2f9aca98da3d463d0415a7a2b9b355898d7b2fe9d7b85fa31c",\n      "label_sha256": "d77a34779efcfc4e364bcf9897343c036a3522993658cec290959ac1c78f0b0f",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-444-_jpg.rf.80b4b352bbd6a51bacf7ff2e011bb691.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-444-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 937,\n      "height": 408,\n      "image_sha256": "765d91b234540e3028e183a99f03f8824fc13db9eaaf64e4087d7f78be1d32df",\n      "label_sha256": "98c215d13cabebfb4a5ed60741f89b59c402db840353aded67d5c15f507793aa",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-45-_jpg.rf.dcee8ce3208a20be78f932b9ececc10f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-45-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 1043,\n      "height": 255,\n      "image_sha256": "510edad7dffac328beeca08340a57f3358062a357e5d41ab90dc9890b03baeb9",\n      "label_sha256": "12005c6de7f90608bf7427be6f164ec637d65da4d2b11226fa4588c66533e3db",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-452-_jpg.rf.267edfcd22dac9d2efc536f59a6ed964.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-452-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 683,\n      "height": 290,\n      "image_sha256": "35dd9aadf2ffdc4f8acd7d766de8ce7819813c592e7361aefa743b3e24cda373",\n      "label_sha256": "e2ca67d12f9f2f4833d792bc79f2fcee423319e18cde6b6ea9f870d38cce5866",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-453-_jpg.rf.b79abeeaf694d6c6ade8a48503f8466c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-453-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 1037,\n      "height": 408,\n      "image_sha256": "a27fc088a996a76527f69a154e362ca4b4604d3bf9c568ed6946bfc9f18bdf84",\n      "label_sha256": "54091767339daeb5a7c57da3ec7c36e5b4220de3dc37d526d35dcd694bab9d69",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-454-_jpg.rf.afed1028813d02dcc01f8efbbca0843f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-454-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 896,\n      "height": 193,\n      "image_sha256": "20124b8ae2b02d80eb38bb6dc7675bd9f1346fb19fa622dd36164d1f76908b1a",\n      "label_sha256": "d2ba260873bf67aba3d1d5f87f55a7132e98ec0fbdd31816aa6183f41ebee4ff",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-457-_jpg.rf.5a94834be11a0ea8ac24d59aba5ee2fa.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-457-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 1044,\n      "height": 402,\n      "image_sha256": "91b61d53009bdf172b9fada6cc63a04fa2c8c897ed20ff2b685d95cb868f6a37",\n      "label_sha256": "c5ec64ccb2f4c12df7dbdf10f9962d2dbd4f5c47339526514107a87ea2319414",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-458-_jpg.rf.e31c0f747a1072af8f0f19bda905cd33.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-458-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 687,\n      "height": 408,\n      "image_sha256": "d0ab13ba7fc2f2f6b0a56260846c9aaf956f1c8b3575a149079130d26ec3d6a2",\n      "label_sha256": "1de2f280e3cb4f4d1630ec531da4d2b53f346b0f839fe3190cf42f25f2e0c6f7",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-46-_jpg.rf.7d90364b33613c11a0d1fc33a6f5929e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-46-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 1,\n      "group": 766,\n      "height": 246,\n      "image_sha256": "3b70b22861aa9cf94c605a5d7db961c39c28e1f59532348ccc011029ad303766",\n      "label_sha256": "4e857d6c9b2f4369517b5c21b4abc751b0ebee2b776470aeacfa2f02426d70a0",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-460-_jpg.rf.abee16ae224b769d0e2ce02ff2881d98.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-460-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 900,\n      "height": 319,\n      "image_sha256": "1508334cc432f6b00bd411930076c7691b9e1c6a9e1ac0b70798367706d11f89",\n      "label_sha256": "eeeb4a58b3926cb277fe7a3d81441113c5ddc0d31e2942ced615115c60b55859",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-462-_jpg.rf.fac33f99791b1dd3827dcfebbc5bead9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-462-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 31,\n      "group": 1046,\n      "height": 609,\n      "image_sha256": "2994fcb4f74450665339d13d38a1d818c8f5302fc67ace25b28ba43864d41281",\n      "label_sha256": "a035a7fdd635c2cbc2cbb3bec39abcb34038f3656b635993b0a0a9148ddf4c51",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-463-_jpg.rf.8f5afda1b21641f4ea9d322f32193715.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-463-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 691,\n      "height": 297,\n      "image_sha256": "6e7f367e255d158801189405732d92613f6240ad0b68db3b275d85c90b0cbdc7",\n      "label_sha256": "3c6a10a0897ff996a4f23a26f7728a6e42cc51793433ec203709524f259ca700",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-465-_jpg.rf.cdc6b29adcd2d862ba4dd9612f30d83a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-465-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 692,\n      "height": 408,\n      "image_sha256": "86c61b4727a47f4b03a1a33a51b329eb2438200d562b50440d4cebfeb5c7ea43",\n      "label_sha256": "302706162681a86d13a1118545f71a993e1f7957f48634ca201596dc6f5d1b82",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-466-_jpg.rf.2d65d0c124db6bdc9c0b975eedbece70.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-466-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 17,\n      "group": 1035,\n      "height": 407,\n      "image_sha256": "35bc25935d8bd7f82cb03e58647c094c4527bde674cac360d7ea3100af5ce857",\n      "label_sha256": "c9823818d22bb897682767f7a515d8bc8f96ed8c8cd03eb79920265b60214927",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-468-_jpg.rf.a38b115924b6253d8d3b570648be6417.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-468-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 19,\n      "group": 1051,\n      "height": 204,\n      "image_sha256": "ec33b362adca5bcf98730a4df7d2eab816ed7b83afbe36197d6cf53b57945ad5",\n      "label_sha256": "6483812a02372f964ac4f4f71e09b65d86acf36ae97977540fa2c52d94eea4bd",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-47-_jpg.rf.0f8a33b7930dac1eead21988a35105ab.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-47-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 16,\n      "group": 695,\n      "height": 194,\n      "image_sha256": "31edb9419e05ae6d1ca182473a58d2059e88cc22691be0ee0043d9c7b853b526",\n      "label_sha256": "467fdea1591d49c65c3df7b458ff4f31e621ce39de54dfe4b03216fddd70dd34",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-471-_jpg.rf.0d3488d41ded50dc35805134e7b15fa7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-471-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 696,\n      "height": 408,\n      "image_sha256": "11607189e8492118e071103d9ab2adb7e3030abf91f89eb506758dd4f65d916e",\n      "label_sha256": "330a227a8c85193a447a1b0e4f8949172290f63620cf6fc02087a5e8cdfde85d",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-473-_jpg.rf.96281b5d1d1ecabd96c72f297499421c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-473-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 937,\n      "height": 408,\n      "image_sha256": "e26c0eca6590d83d22876dbded407c9c718e77442090989152e1a9170a5f9e56",\n      "label_sha256": "18599a3edae1d2e83c45d9bccf92d0f3461b15b16a9c99f95d9df119a5271239",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-477-_jpg.rf.43ec7b02e698f071dcc718028f71fccc.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-477-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 1052,\n      "height": 408,\n      "image_sha256": "fde66c4b40120614ad7ea08488b521467e6c5752c3fff6e5c7963e4c92740f9d",\n      "label_sha256": "10ec7b6dc28dc599206cfa99cece16e6a8d78b4ffebf3f818286f6304d13609a",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-483-_jpg.rf.836a553da0a15766838f089bed00bdca.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-483-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 3,\n      "group": 699,\n      "height": 612,\n      "image_sha256": "b311471348168e159db8761ab297887917a276c1b59b882679f155829e4b319a",\n      "label_sha256": "83331037b4939a0ba0e7bc2574db6a4dffd5592a2f506803b3c1cc8639b7b5ae",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-493-_jpg.rf.2ce3651f1b99454e165bd0b4422624df.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-493-_jpg",\n      "split": "excluded",\n      "width": 601\n    },\n    {\n      "boxes": 12,\n      "group": 247,\n      "height": 408,\n      "image_sha256": "64ec0f9ec442a4d360506aa7d949f7f25360331f9d6c6c36be6cc2c52902a17e",\n      "label_sha256": "aae1fe589a3d9d5dd1df9d8f5a0f5c84a7a353483db6ea7da601a23169f728df",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-50-_jpg.rf.8552d9493229351197731ea7f6f13c43.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-50-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 701,\n      "height": 375,\n      "image_sha256": "074612e7bac3aadc2bdc398c4b419a9f35a933f457403b409cc43fa0e72f806f",\n      "label_sha256": "092be9769444004a7ca8ac0ad29ea6f3f22840e19864536ea0c32fb999fbe383",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-509-_jpg.rf.96d3f7ba37c096205d809f2ff75ba18c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-509-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 702,\n      "height": 424,\n      "image_sha256": "30428b7bfb80fe7bebccbfc3934f0578d298884dab4b2e694de9f8305e8531ab",\n      "label_sha256": "00b36bfb03bb46d22821c843e36f31f196b428ac91a70ce8a8cd925a59f59bae",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-514-_jpg.rf.bd2fe623c1a532d33e19b7f3396a060c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-514-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 703,\n      "height": 408,\n      "image_sha256": "e4e9dede3691fdc0e3234ad8a6b952e5875aecc39426f33fe6d563ad0ce5c1cc",\n      "label_sha256": "0aac15d606b32fe331e2cd75382832b234932b34f5cbc67f06bacd43abbad24b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-523-_jpg.rf.444bdcad2d42c459bd1adb6711d12a03.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-523-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 2,\n      "group": 704,\n      "height": 408,\n      "image_sha256": "854bfbe1fe224ca5f233d632ec09fa2dad6578d8728064f8f0aad362f18fd4db",\n      "label_sha256": "f8e2b8d81ebb1e0d48215407408e0074570564ba5b0662db94bbf53550077a46",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-537-_jpg.rf.b2aaaca859490a742483783b0367edbf.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-537-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 705,\n      "height": 408,\n      "image_sha256": "fa335cc19ec44d2e682f009250e93d92bd73626b701ea71a02381ace80d75df6",\n      "label_sha256": "713a2a486cd5a2a5738eaaae764be54b864b7473d041ae530881ae5a8e9e6b0e",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-539-_jpg.rf.45edaaeebfaa9e5d34cda1f6d7ded36e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-539-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 766,\n      "height": 317,\n      "image_sha256": "a6960cfcda853873f8a54e6dc8406ff091783199aa836047e1dabf7960075e1e",\n      "label_sha256": "60432879e8c886c783bbc6029e385e6ef6c8c3ca8ac7f435bf360a7765ced2e0",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-541-_jpg.rf.caac4334eb0401c8410bd218a1858a98.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-541-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 707,\n      "height": 293,\n      "image_sha256": "29422577980e09e1dd526224036171e905d7188c433bf95c89bf36a603a00078",\n      "label_sha256": "26070810ebae65fa040a69271c8e8e94527330507fb47ea896822fe1dfabef82",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-543-_jpg.rf.1a8085a8cc7b3f12fca05a844eb92aeb.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-543-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 915,\n      "height": 471,\n      "image_sha256": "2c2c1259b2d6909b67035513ffbfe643728a8d514742e0c4335803e5cb7ec184",\n      "label_sha256": "ddb5592fbdd56e500b1acb706802e2d4a6d2cfcb6b46ef694b351b067c66bf79",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-545-_jpg.rf.21bd835d323ba810604d489454b6d841.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-545-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 709,\n      "height": 408,\n      "image_sha256": "3834851ba0c13a8829a94377276e53467e33fb1e0a90d64196b9471054c60360",\n      "label_sha256": "aa3df6b3fdc0ba3aef9853c1fc1e124f43de7eb644fefcda65c824c69fda15a5",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-547-_jpg.rf.1515ca4888cd94737ee90ac6702ce93e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-547-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 710,\n      "height": 372,\n      "image_sha256": "cc319c300e94eae80ff2ab9a90540c01a6a600760fe0baa0823398d22bb33cb5",\n      "label_sha256": "3eac2654cf70b97d9c99b84cc8e6ac2f7e7671d5ebd6f2a4b2f934455e26dbd4",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-554-_jpg.rf.ea211091bad7d706f7ded9d4caec5c4f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-554-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 711,\n      "height": 398,\n      "image_sha256": "7c54f040a98fdbd0e6cba95d4a663ce690d7724f7282393b364dd4004c97aa1f",\n      "label_sha256": "86d78260191bf5322f6dcf4db668c3cd78fae70c575caba6815950c41414d5b2",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-559-_jpg.rf.f8fe6c6282d40f68ee4212ee9765e643.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-559-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 16,\n      "group": 712,\n      "height": 495,\n      "image_sha256": "0a18f6874fe76c820158c31075cb3e60d1d7ad00cc8cbef07d89199875dc490e",\n      "label_sha256": "c2e65b80510ca303d3d80af0d903765700e7608c14fa18e3a8ce8f463f1b5546",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-560-_jpg.rf.f86362e5d5f58fabe41c57fde0e6c51b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-560-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 713,\n      "height": 408,\n      "image_sha256": "4761c1ff84312881f31cf1399059c0d4227128b1e994703f0b036cd8c1f98382",\n      "label_sha256": "7d465614334a062cdb224af5317e5f966484c3f5c774882bbb332bb2586b8404",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-561-_jpg.rf.f93fdf129973752e551e4f6ece0ed582.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-561-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 714,\n      "height": 408,\n      "image_sha256": "f5577bcaa96a36e39fdcfe13015176269bc28c7d7e943fa4409759b02f48ae5d",\n      "label_sha256": "66188377a3e06ce781443e32b5ecc90a3d571c00282201a51b569297aaf9d113",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-568-_jpg.rf.03a1b8f5f6bf7965bb927ad2fd1c93e0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-568-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 909,\n      "height": 407,\n      "image_sha256": "763ea9e147b9fb5347d618aa01ee20fd8e0cdc856f3e745fc495e0e4641c5be5",\n      "label_sha256": "26194c9c72ac68cf631ea52ad9a9a8e882142036d55c13c3d3c0d86b54f7ade8",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-569-_jpg.rf.ad31c492c07a6f3d5d07683af7cc8d79.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-569-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 1041,\n      "height": 325,\n      "image_sha256": "862e6a43391087dd9d9b104c9e3fe399796a3f6e0b8c1550eb34e7b961aa4330",\n      "label_sha256": "6f9397d6b3c4c486d10e4f754c7bb68b7b1490765da5ceaf9bf8e8b3d4fa14f6",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-572-_jpg.rf.33b16a9e994700ced7dfcee2420af217.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-572-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 920,\n      "height": 408,\n      "image_sha256": "3198d855aa8fc944523bc882a895ce4b60d2dd9049b0cd3211179c62a76a3e86",\n      "label_sha256": "ae2cdc3dccc02d8a71712fc0dd3809bc9970c4bef6afc70a71c024a322c3b545",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-576-_jpg.rf.0a7193e30816efb1cba86e72c2e133ea.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-576-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 718,\n      "height": 408,\n      "image_sha256": "65c1989c3a0c894a64082d58a3ec8637cf4f980031cc1b34dd4d65bdf5cc75d1",\n      "label_sha256": "41f53809b111779f7cf3551fe373e4531342205149b18eb3fcaf8ce81f93031d",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-577-_jpg.rf.19804ad3f867e82030a99a46f7bc120e.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-577-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 719,\n      "height": 405,\n      "image_sha256": "ab08db9fecb5ed0671006d9da56043a8b2faad12629ce5b5bd8edaf39168adf2",\n      "label_sha256": "05ad22ca090a709964fe2db65d749297da7f0d6afb64ea9179269d0dd0f4800b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-579-_jpg.rf.7bedbaa8d1a4fd094c17247ee632d408.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-579-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 19,\n      "group": 720,\n      "height": 255,\n      "image_sha256": "25deabfa5eb112b0514fbb78451b643ecb7ba5dfb6ca016c124dc366a0622de1",\n      "label_sha256": "53fc8c0e032dc9e8a36cc162986206289e636860bc559b1a01be9d8a22ea40c0",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-596-_jpg.rf.b125eae4add5d561e31a9f29821df7e7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-596-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 721,\n      "height": 408,\n      "image_sha256": "5afe27d490aa0444b6f73e4ca1a4d3d64fb521a6aa35e621237e17f2f2685f90",\n      "label_sha256": "c768f5341db06e96cab2e2e88736ad3dc8803623fc72f6a470625a33e03b7a4b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-598-_jpg.rf.83558c6a36ff95a6f0d7f2d70f82ede1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-598-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 3,\n      "group": 722,\n      "height": 402,\n      "image_sha256": "83863220a652354a6e6760c3586d4719b2d3ea7d8941fc45461ed1d984a41b06",\n      "label_sha256": "a3b64e14c2baa87ba8b1cd784a204c68b591119beb2c0b5b595a2f2a91ab0d63",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-602-_jpg.rf.f0f3de1b0057033c9cd57b8d5ca86540.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-602-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 723,\n      "height": 355,\n      "image_sha256": "87487383fa66d611f121cd8d3b7c63d37fe7a0fc5d1aa4ae0b6475c98bdeeac1",\n      "label_sha256": "5f2c6a210d8f5d96b281c8f8960ca0b9b8db400be0dfc98dbf722f0f4e870218",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-603-_jpg.rf.4c47b1b2a3805daf67a3aa7c7e48a3e1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-603-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 724,\n      "height": 319,\n      "image_sha256": "d970172914a0f1b4753bf62639ef19e6266ba9cd40eaed4b5317c34334fe8179",\n      "label_sha256": "a3cdddad5a21e4c110c8a0fe92014c05b4daf13a39abaa0d27673728284ecdab",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-606-_jpg.rf.0d5fd65003e8402a20f5ed2947423490.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-606-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 26,\n      "group": 725,\n      "height": 609,\n      "image_sha256": "50e46f360b3611cebfc8f7226539f72a853dee4ed5e8e41470876cff68a53278",\n      "label_sha256": "7ba1bc30d85de440a0a1abb779cd62886ac03335fb525d8df5d94603673c116a",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-607-_jpg.rf.d48336e9469357a53db0bfe78d340419.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-607-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 3,\n      "group": 726,\n      "height": 408,\n      "image_sha256": "33b670643c1c7ea3da5c5e48f5163a68804eac4280732b06917f5909e74ec53c",\n      "label_sha256": "603e0cb8579a25a0ca164fa0b3a1e042524fedf1cc301768635d9e2cb1fca9e1",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-610-_jpg.rf.d9efaa695d928ff7dc69143d3979a1a4.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-610-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 727,\n      "height": 263,\n      "image_sha256": "8401b7aebbd90ded2af0f2d5968f4c1005d7bd93441fa611f777c1b9ed37add3",\n      "label_sha256": "4e00d8aa70149bbc9165fd38f932c2a6baf5db58f77ae974b5eba23a9d647b82",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-614-_jpg.rf.3f9ea8d521fe4b1399bd2bd48e64bac4.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-614-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 16,\n      "group": 728,\n      "height": 408,\n      "image_sha256": "0474ab395923e7f45cacc107a7124d7b96ffa744e95223c0b7a0b37f5aef9154",\n      "label_sha256": "cb571211ac2603e5f5a70a28831974d67a342d4c8dd0017fe0a0b782e92e5be1",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-617-_jpg.rf.5dfef652083a15dc286ed6a09530ada7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-617-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 729,\n      "height": 408,\n      "image_sha256": "e159a36fc9a8adc7108b1370fcbdea7d286943e6c5113836ecda86e0dec7433b",\n      "label_sha256": "ed01b1e14bf3338954528681a8be633a1403f6d71e3631c0907b2f5f79255b42",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-618-_jpg.rf.39f023227052e3ff47608b94acccf027.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-618-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 3,\n      "group": 730,\n      "height": 455,\n      "image_sha256": "db6278628dfdfd31a141acd619535e01b6e08e2191d31fe16aafefee70dacbaa",\n      "label_sha256": "92bede59d940481c46fb82b70350f85c450a2035e48e92144248dc4e8e087d13",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-619-_jpg.rf.52272421f8afad91232f96c747e0a763.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-619-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 937,\n      "height": 408,\n      "image_sha256": "539c63bc0a93af18226c36ce1074ae7101d0ea68ca59f49342f59d0f7ef76741",\n      "label_sha256": "ea1cc0fa5e266e51e0c81cbb718c99da21b88b870ecbe7cf3072ea5d56809b19",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-621-_jpg.rf.c0c566d734518f821abd905ac5a55a45.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-621-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 3,\n      "group": 732,\n      "height": 408,\n      "image_sha256": "ed60068b70c9b3311074c880305f672938a66e1dae169afca09efd7d44d350fc",\n      "label_sha256": "8abfc262de55bffc487a71f10db8e2b8108fda09042e5226fd904ac12267512c",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-622-_jpg.rf.8844181248740ee1321a455c968b13f0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-622-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 35,\n      "group": 733,\n      "height": 204,\n      "image_sha256": "2f7d9e0e430c3ee04f77749eaa87906c950d80ffa8446d828fd49823391dcf70",\n      "label_sha256": "deefe192f5cefc9224fe64f73325896f0354dc236c0e75f162b446ed8598da89",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-623-_jpg.rf.d4ae3e2f20ce10bac05f705a9ab47be0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-623-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 16,\n      "group": 734,\n      "height": 398,\n      "image_sha256": "e51e12f154ce2f37c5a6c6bd0852a37b19d596aefe1795af40fb76bb0a1d263d",\n      "label_sha256": "b7061e8dcfff8aed00b3e701ac2ee4cd19040fad6fc2ba9aac8ca65454cac1d4",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-624-_jpg.rf.62be2628cc909aa9ab65bd3a541169e0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-624-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 735,\n      "height": 408,\n      "image_sha256": "cf6911f53e29d92bfdc91124fc8d0da2aa2d4434505ef4e3216da277f370f2d6",\n      "label_sha256": "135b13088f8709c244a05a4e82def7303b9a1f2a7a7b703121bb9ee85aec097e",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-626-_jpg.rf.c2baa9f8ac2983f7e68ad8601b74c2f3.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-626-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 38,\n      "group": 736,\n      "height": 151,\n      "image_sha256": "e882a75af5e2f5247a1bf68f2be96be9b3826f2af6b4cd2b964236f563ce7ff7",\n      "label_sha256": "4146d78c6ec1b33ae18e7ff6250773bde17562ec6b5ff009642839e8fb356d5a",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-629-_jpg.rf.7901b755ffc158c8d2bcf57c26a32e06.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-629-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 737,\n      "height": 612,\n      "image_sha256": "fa66a0171ecadb72618d2cc5a6f5dc8a4ce0d0d87f994a0e84e1a54447409256",\n      "label_sha256": "b12b5a3ee01b16a4fff6ca5229f616ecf666924480f9fd84619df7bf2a69650f",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-637-_jpg.rf.a7ec2109c262a24cae07dd43b4bb1589.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-637-_jpg",\n      "split": "train",\n      "width": 601\n    },\n    {\n      "boxes": 19,\n      "group": 738,\n      "height": 406,\n      "image_sha256": "c70b0ac224d9d44d7ef148d52e7afd49b2d5aa2d2bb30599a44de8140c45d7a4",\n      "label_sha256": "76bb9eeb1b568a6b532e8f963083476e36c76540b638e647e96490ee2593cec5",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-646-_jpg.rf.f89ce4144db3ad51070a0d1093f02ece.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-646-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 739,\n      "height": 408,\n      "image_sha256": "c628b5bee1aabd7150020bc650a73b2b096d8365d345412ff50cb586c9fbcccc",\n      "label_sha256": "58efc58339f46bb176c655ec31b548c3d7be222ac2ea11460aab62ee9e27f650",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-649-_jpg.rf.07f331ff76f9c3f615d095ec9f2c17d6.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-649-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 740,\n      "height": 347,\n      "image_sha256": "12e8c6a9f0d22864ea54a37e84490ee34d11fa297dff4ebf408f5d61f1da5324",\n      "label_sha256": "5ee931bcdad3758a1c61547f56981c450850da31f57f18a01506e871fe767639",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-650-_jpg.rf.cc1d21530ab0e5ea9d2aac1065ba6c2f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-650-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 3,\n      "group": 741,\n      "height": 360,\n      "image_sha256": "8d15b8a2fa09dbf8cb62724596a91390c0a7dac9ffbb50f43399347f269ca74a",\n      "label_sha256": "881427c56eb6e40fa4fde96265dadb8b941ba3faadc5da2c0a552089b6b810cc",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-657-_jpg.rf.cc27cc476eee5b53c92b9d3eb797f833.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-657-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 742,\n      "height": 420,\n      "image_sha256": "86789e35994fb0dc2e8895c12228ae422f0c11c3247bde46a3611ee4413b9f3f",\n      "label_sha256": "cf7a48fc260031684656c36562549e67b688bfe150f7279c4217b7a6508b9c61",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-659-_jpg.rf.449758dfc3e74a145d50c705e463c274.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-659-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 743,\n      "height": 408,\n      "image_sha256": "30dac7e9d00260bdee527f1ba5f6b1d7504d3b3f0a2c1a93972629252149f4e4",\n      "label_sha256": "c5edd17b40e95cabad2f690cc65c2db0672365de4386c96352d5b397356edf06",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-662-_jpg.rf.4e5869f72691dba838519d5c15e421fb.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-662-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 744,\n      "height": 408,\n      "image_sha256": "ed764c3fe3b8aaa1693597f331229a9a1dc335662891eb4b5200aff1f898ecae",\n      "label_sha256": "13de34e746bd1444a25473a1449ceae7c62093a3d2d8f2051fea78f203060ec1",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-667-_jpg.rf.bba4cb64b7a39324e716e4c11de87f1c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-667-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 745,\n      "height": 408,\n      "image_sha256": "caad0882d5b8d0f2b67633fa2e41514f1a7c1e69268c6659897b6eb19892df0b",\n      "label_sha256": "0e7694a128115f7c54e39af9ec36703bf5b229f70a53f42281505eddec2e14b6",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-669-_jpg.rf.c4f20bd45748780d45d1a299cf93576a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-669-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 746,\n      "height": 408,\n      "image_sha256": "c692060ccc768ba868d09752b3bff6001ddcfd856977c465a2c11fb248769e93",\n      "label_sha256": "f2ccb62be74e485c94ae09e8742fa9ae72547d3341359bb15e239a5c1acd2da6",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-676-_jpg.rf.b8cc18d9e97bc7bb738aa5fa861d3406.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-676-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 747,\n      "height": 408,\n      "image_sha256": "eb6abaa978081b891e0385877f39135f758c4f3d5c57f7f61b03bd8e40f70f3f",\n      "label_sha256": "99f286e9391e6ee3a3c6134a87a170c13d606bbe51d45617ff74a8b6d22b63d1",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-678-_jpg.rf.50a43ca3f91f0401659645bf84a6cae6.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-678-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 3,\n      "group": 748,\n      "height": 204,\n      "image_sha256": "0f3d6284e0a3a5b3f9b816702bc9758022c1583d0fdb03227335025cafacc325",\n      "label_sha256": "c7cfe37eeef3b8acfd3711b10454cc673640a62ff7048a232751d61336a99580",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-679-_jpg.rf.599a65554d202dae48272ab4ba60ae26.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-679-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 749,\n      "height": 408,\n      "image_sha256": "bee319fa47d4af5f1c1ed53e18c46ace1c5f9257769f16316985b0be9b75e7a2",\n      "label_sha256": "098ac017a81e84693290519460bc9d806a75d1b2be7f2f66e18a89387316c4bb",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-681-_jpg.rf.41b60e589667b0609385489dadde4201.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-681-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 16,\n      "group": 1042,\n      "height": 408,\n      "image_sha256": "aba760cce4ab8199f822ea81e835eb7a1ebe35b777c92d0cb445c8b6ccf006f0",\n      "label_sha256": "60c04e1371ef227e6cec447f85b906e1d52a4b06c286ef0ce134f0b51526696d",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-684-_jpg.rf.6e60de3a00b01a04d015c3513bab89f0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-684-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 751,\n      "height": 471,\n      "image_sha256": "dde14133f7fca47fb2a6761b481208d96c3a78658bc592a0b646e03e670396f8",\n      "label_sha256": "383f22b8cbf4c6981a91c5a28b6f639ba8b5f1ed28d06cdd9fb55ab65a11c6f5",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-689-_jpg.rf.03ce70cb1a6bc1c782ad0dd8b129c5da.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-689-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 752,\n      "height": 407,\n      "image_sha256": "a04d571c76c5378a660df785ee6b1045748565f6e16b48c232d807c15b5b5542",\n      "label_sha256": "30fcaae358513f601209428a6853fd5eea385b29bb913408d2b67502cd70a2c1",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-694-_jpg.rf.4cd3393b019b3135bddf09e478af31c5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-694-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 29,\n      "group": 1060,\n      "height": 406,\n      "image_sha256": "672fd8cc20b0f46b8a8a594eeadbfc41e38e45b8f91fc1e89308d3d1e2f87aaf",\n      "label_sha256": "5f36abc5387450be529858319e816254043878eb28760e209dc4763e67719fe6",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-70-_jpg.rf.5e5c4efdd02c88a0f379d0138d6558bf.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-70-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 23,\n      "group": 754,\n      "height": 495,\n      "image_sha256": "6b4814d1b49774933b4f301f02e51643049d53e27722174708f7e5b986fcb315",\n      "label_sha256": "3dea1df764893a5690a4b533e4a13aeead21dba7ef9d0b0b6be0cc8b343d1b76",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-704-_jpg.rf.4273b8dfc0353405a53aadd1bd62245c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-704-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 755,\n      "height": 408,\n      "image_sha256": "c8172200112874f5cf485f27c1dc9fc9f67c54695cca018dbbe71d2ea4a9c6e9",\n      "label_sha256": "fd2133149154c044844c0ff8880656ff0c9d2d56ec660608f6fd9401842f039d",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-707-_jpg.rf.ac6b75ef3edb72feb6a9d874af527ede.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-707-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 756,\n      "height": 409,\n      "image_sha256": "a5d58b2e7478f7c6b4d0a39fe9635e66ad1d4a872568a256611778464c3064ff",\n      "label_sha256": "d18e04b750356db770e01750ee7da5f3a8c2cca8ed2b749fd3b74a1a382f0169",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-715-_jpg.rf.86e9b08d672d9608d569653a535e8710.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-715-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 757,\n      "height": 325,\n      "image_sha256": "24d28ad17da08a2b5b6cd1cc2ce8ee140663be3163b3e631558aa1b9b1be9f38",\n      "label_sha256": "e6c5b4487a164ef828a668a8c2406b72ae6ad809b88f2101528987238fad9e73",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-716-_jpg.rf.5b956c9b64508e70f30fd49a0e5c8ea0.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-716-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 758,\n      "height": 408,\n      "image_sha256": "1560e146a5e23732b82f5fd1912f5270bf5c6d7e6b558b7ee9529b72b1992e9e",\n      "label_sha256": "eafc6f4ac58bb9891ee49853739847f4235c1340a5053d182e5779a701674e42",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-718-_jpg.rf.641445b6c84aa562fa27a9382e92063f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-718-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 759,\n      "height": 408,\n      "image_sha256": "7b877e627610e8b38175cddf04b3b35d1030853779fe010e7c104f75df010105",\n      "label_sha256": "ed13f5b6735ca00ce654cde88118b73f7b64ddcb98eb7526a9f7a83b886c6aa9",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-722-_jpg.rf.f4840fbbd43a93ceba731c55ba6de766.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-722-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 760,\n      "height": 459,\n      "image_sha256": "00da8e19f7a3c5aac92a4bd118b2d4ad5555c1dc2c8fa124738086e9b915e7cb",\n      "label_sha256": "cb9c220b11ee7e8ede4b8938ded90eafd55244607964ff5158d90c20939f42c2",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-728-_jpg.rf.0acbe50aa3ebd28f7b709396fc17f8d8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-728-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 17,\n      "group": 761,\n      "height": 195,\n      "image_sha256": "75e6b264408b720b996e66cafb33c7cd6729a01e72911b8b6bbf7d77035c253c",\n      "label_sha256": "542e3129c3afb491a757fec76a8f4f3d6ac675334cd3600702cff4b23e9963c2",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-729-_jpg.rf.0fc873bd5c55b28b3fc14f60fbc8ccbc.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-729-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 25,\n      "group": 762,\n      "height": 408,\n      "image_sha256": "e2e5e4f6abeac23eee06dcc5e390407dd3455d19ef8a7f74876c79bfc3a36e52",\n      "label_sha256": "f38e559afaf625e947e6715322d6babd2cc5f8ff5e86c1a92376720c0e9ad2d9",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-73-_jpg.rf.46eb226b008250f4b50e28f84b7f7573.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-73-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 763,\n      "height": 408,\n      "image_sha256": "77a7887c8ecf53e54c505757f2098629bfa15573141977dd4604683a6c6a2bb4",\n      "label_sha256": "f8d56a4a5f5b6c04cd3bf113144b167e537ff0b45f20a4f89b0ba7907ec4d6c8",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-732-_jpg.rf.f5be66295b89a5ac5ebc623c6baff1c3.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-732-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 764,\n      "height": 390,\n      "image_sha256": "6b162f2abd1c65e967725a18ee504183d4b094f8da3bef14e15542a8abbde46f",\n      "label_sha256": "d12300c64287b072239ecc01da7e4ecaad7f790806e4ade339485ed16359e68a",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-735-_jpg.rf.6d416ba7277fe363d5789e1c5761c414.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-735-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 765,\n      "height": 408,\n      "image_sha256": "6be618de13566e3e741e0f2d7b43490c0a73a8491484f247d7f1da37f92b8059",\n      "label_sha256": "1bf13365b138354a52fea5d9d4f27b165157f8ac0709aa0a8d8e8f880e6e9859",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-736-_jpg.rf.581d3496dc3f617e239fd1e996af8a69.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-736-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 766,\n      "height": 408,\n      "image_sha256": "b656d74fbb039313f73d6fc7d46616aad41dbf8480687978ad99f9a8d97bdb46",\n      "label_sha256": "62fa9fb1c7fa201cead9105e3ea4c29f9461a66f5bd54b66685a8ba965b1396a",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-737-_jpg.rf.0283310c251f72cd4dc2fb031835ba24.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-737-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 19,\n      "group": 767,\n      "height": 255,\n      "image_sha256": "a21908ceb2de130618a47c1f519a837fca5184db2c1a16cac414adabd38b1b9d",\n      "label_sha256": "00f760242f1c5ec750021a9c21092212f23d9f142f561f1dc24d519039ac53ce",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-740-_jpg.rf.0adbfd01aff057e38189ebbb30276993.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-740-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 768,\n      "height": 290,\n      "image_sha256": "b19981efdc8fec30ceab5f166b405280b313b83ea4a3a296e98122b24f5fd2c6",\n      "label_sha256": "f8528a1a4b9de33791a54c0c30141fec9d525f8c0d684b2bcb9120de221dedbd",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-741-_jpg.rf.ad3c4c87181807887cef20ec7214bc0f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-741-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 769,\n      "height": 408,\n      "image_sha256": "35653f57fe00eb518c7c022d13647e32c26f393d268d2c0236c6d9f4ba929671",\n      "label_sha256": "144e7e66f07f6bbd12e477546778abf670e9ab80478012cec2d99b950c331aac",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-742-_jpg.rf.9792d2fc35fed5e7fc5fc7c38b21ac5d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-742-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 770,\n      "height": 246,\n      "image_sha256": "ebe7213853fd479b999d1e0033efa8872729bcd4030d6cd6d75df5c92f44e9f2",\n      "label_sha256": "043fab833d71d5359a95aa85c35b25688984e2fc18b42e4800f3ee7def5288cd",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-748-_jpg.rf.dae33fa6c614c447f3d229f4c5a30de8.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-748-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 1031,\n      "height": 195,\n      "image_sha256": "6c5a1e3af98c292640a6f329dccd304dd20a2b619d5b67f47dbb94647c27e5aa",\n      "label_sha256": "2289b20ff0e3b1b93ae82b7715656fb626be3cb6e9752e26c2892b2d48767bbd",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-75-_jpg.rf.fa1f6b1bf88de355d1c1dc868dad5b83.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-75-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 23,\n      "group": 772,\n      "height": 609,\n      "image_sha256": "3555ced6bf8eadf32545c6e404daee53767a57cbf5798511a3df3b0783b80963",\n      "label_sha256": "a096b502ac6dfd9365907df653ddf3057d757c38b9b19871d7a23a3f7bb330b2",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-751-_jpg.rf.2f50175b622492a0ce7ffc04f2a13d02.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-751-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 773,\n      "height": 530,\n      "image_sha256": "e2f18e770132e0e6a94cbf879c2a56fe902236260a58617c4286e006c6da0ae8",\n      "label_sha256": "1d98616a0926089a63e5a92c950540f0361cd0067516d8000e4391214e35b790",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-760-_jpg.rf.6936e6f7f2f0d79f5e96f731a09fa6f9.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-760-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 15,\n      "group": 774,\n      "height": 408,\n      "image_sha256": "bb548d454d706752b112bea5c46ff03c69ed9287f4b9672ac72658a32f5b1124",\n      "label_sha256": "bd468ff7803d4a036b3b3ae0b0399a2ae0385a7b10d42b6ed805e172c541cdab",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-761-_jpg.rf.906a3eb80fca36694a35fe4c26653854.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-761-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 775,\n      "height": 408,\n      "image_sha256": "02fb8849ecd3e4de33bf78ffc78fe2ba157df124958e24f89835a926e1a86861",\n      "label_sha256": "41c11cb3084afd1861cacbbd50f33fdc43449111c1d71fc9e789551a987741a4",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-765-_jpg.rf.27ec5caabd147a8318901256b3b6e095.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-765-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 31,\n      "group": 776,\n      "height": 204,\n      "image_sha256": "e90aa5e03ddfbe7d6b0e1656b938133b76f62e5b5a692e440e336aca90a88a77",\n      "label_sha256": "d6ee1c719c5da4c681c9819ce9810377c083f7b97dbc9264947ee65f4c0f0741",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-767-_jpg.rf.493fe0d87e68d76c37e6e3e69c82eb0f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-767-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 777,\n      "height": 398,\n      "image_sha256": "8196dc1b8b5d5882b14a2452f3710c580c1a1a4735d3613e1937860659c46a86",\n      "label_sha256": "b5b00208d4eb808de8935a5f00d355b3b4d94adb60172cfc31cd417ad78308a0",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-768-_jpg.rf.45bb565a237f1f2f36ecfc141eed3cdf.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-768-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 778,\n      "height": 408,\n      "image_sha256": "d2ac3aa16627b06316610ea29b7bf485dee0c53de3a66cd91f23198590274028",\n      "label_sha256": "01a994562b78a55b9972f46fa76a000ee3d1811687842b7040e5d88ee48b8bd9",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-770-_jpg.rf.906057e41cc70da562d16db05722a2b4.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-770-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 779,\n      "height": 408,\n      "image_sha256": "cc704ef3ca52f01a6e57d56e05e0b2fd1e6f364f9850e1fbd9ca06e57c3fad0f",\n      "label_sha256": "785875cd35b82034e3eaffe0c92385aa1d33ada81b53d01190dcad5e998fa536",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-771-_jpg.rf.32ad1306bacd655d0a543b2797740500.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-771-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 43,\n      "group": 780,\n      "height": 151,\n      "image_sha256": "502009c38cee5b9c726d85d5672a930aa3afa157755fb0efec74e193cfbbd0f3",\n      "label_sha256": "b27bfa464885a34174a5acb94cb5b7dfc77455e8eb2ae1cf35f92c41535e3d85",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-773-_jpg.rf.476f0aff6ff2a9b8689189cb26fd131a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-773-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 16,\n      "group": 781,\n      "height": 355,\n      "image_sha256": "a2da046e36218e189eede94b3cfd29f48119f8dddd9c2144c88398bcd49d7d4a",\n      "label_sha256": "cabe0e356aa01ccae359d4f577ee411611b0ac92117b0f527dcdef865e7c6241",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-777-_jpg.rf.3a0d99750b7e4f96afd7a3714fadcc9d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-777-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 782,\n      "height": 405,\n      "image_sha256": "cb46683d032b9ac2c46531748449e7a77c6bf501ae2551fd31757897d544fdc8",\n      "label_sha256": "8f62d63b15d86d22269dc4bf6b4b2ade81eae649db2ecc40b099d10d935e3a72",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-782-_jpg.rf.1b4bd1e68624d119dbe328b8f718925b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-782-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 16,\n      "group": 783,\n      "height": 408,\n      "image_sha256": "53acd9cfda17a7d26e9a64787c1184d6dab033e91e14c586486fa23d5dd1d722",\n      "label_sha256": "f01a68b55688769cf1ce382a7bb17ef88b5434d2d25d2a4d0404747597b0e08f",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-785-_jpg.rf.25b7e76ca346eac258499bcff9b23b69.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-785-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 784,\n      "height": 408,\n      "image_sha256": "ab9d8abec7317becfab183995582df94765b6a267e57337549ffb773ce2c92a8",\n      "label_sha256": "b0a4840825d89e8e4aff8fdbaee02819c1c2dc2177f0b1c053f908cca06b0e17",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-786-_jpg.rf.dec48c1ee18d499261b6b615e815d2d1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-786-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 785,\n      "height": 317,\n      "image_sha256": "5874d3977cf868a1b705a29286623a9205eee15d4053e14440dcc3167e2210ec",\n      "label_sha256": "0ddf983f811f4b55d685729facf5c3d8908ee8746e744447b9ec888c892e55d7",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-787-_jpg.rf.906ce1133dd53e1a11f10e4c7a48ef75.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-787-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 22,\n      "group": 786,\n      "height": 406,\n      "image_sha256": "2ce0b06c248644b2d6e578a8fc59f8a54c48caa86ff47a87c241e09eb5f2fc05",\n      "label_sha256": "59d5a587bfeafa8f7f99af85aa09da1de08cb3688b39baed564b0dc2e282dead",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-790-_jpg.rf.6ae1877efdcb20df4e45ca39d9545331.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-790-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 787,\n      "height": 408,\n      "image_sha256": "57820f4e7dbc1b20c53047bee08041fa98a25fc9c976cfe271574dfc7d077bee",\n      "label_sha256": "bad4e94b1d2185418bc9e01f24e9da6e3d680eb09cac09e88a044314eebe6fc1",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-792-_jpg.rf.db151d0c3ac0ba4217c1634f6744b12a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-792-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 15,\n      "group": 788,\n      "height": 408,\n      "image_sha256": "95de33e8b0ada8928946ae42e4e2911674db54f48e7532a59554aa9c2525c9a1",\n      "label_sha256": "428dab512fc9fb98efc1a47f0a632a70fb661748ceefe3dc8d0c443899205201",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-793-_jpg.rf.8e5eb48009418990249520a3f7cbf2de.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-793-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 789,\n      "height": 347,\n      "image_sha256": "a271e07e1c3500917306e4634963cc59285c476cd73ed8eeafb0e7ccef707bc1",\n      "label_sha256": "40b6eb89304d6b2d749f28c3785456303ad81981bfba70d67b2d07e0ade83503",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-794-_jpg.rf.20f810d3449bd25c0b5c149319601652.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-794-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 790,\n      "height": 360,\n      "image_sha256": "6e0473a1742b494f82bab3121996b61f232413eb4c41c26c744da2313dcf434c",\n      "label_sha256": "99938f755b6e6d5858d5187a29cf5b79d8a7a4b047aa8293b196fa0ff6a57c64",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-801-_jpg.rf.fae48633f1947e1fdabb77ed25543be6.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-801-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 791,\n      "height": 283,\n      "image_sha256": "a1b734e21cc72cd6460673b5cad5d4b5715389ea346aaad115b4f4b86ad11ac8",\n      "label_sha256": "98666024b31cecbb29da515f16b9d7129ad00ec931d9bb9fdfa815cf3ce98d08",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-807-_jpg.rf.5bb1660ffdab0e15851b1fd89dee5cc5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-807-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 1026,\n      "height": 360,\n      "image_sha256": "7d8942c90fd2c42c4e1eba6fd403c9a7b2f1ec627cb25a73161e7fe27b4ef145",\n      "label_sha256": "318babd0ee7e40ea45cc62699c8f9e217da6faf0d98db2cad40572160696d19b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-81-_jpg.rf.5aa6bd4beea8549d14e834e00a439157.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-81-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 793,\n      "height": 408,\n      "image_sha256": "2832743d9cd68b324f2bdd12dfd14081e0b0dd58844b62de5d7bc4283417ad38",\n      "label_sha256": "b7089eba2a431351d33b84bae50a045dd5bee2a95eeaf09126c45e451aa217c0",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-811-_jpg.rf.62f3b5cb5b7bae6b1aff57331f086a3c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-811-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 794,\n      "height": 408,\n      "image_sha256": "0789643d5c31cd39ef52773f77b804affe04af1d124ca8b455ba4f083e3cde38",\n      "label_sha256": "1ef6562a017561b5b443005ddf94a9ee27d52ad6f5dbf87f5f3db1bc7f74a41c",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-822-_jpg.rf.6e532c66c82f216b4a97162b5f3a2801.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-822-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 795,\n      "height": 204,\n      "image_sha256": "8aaf1203a5bfde13fdb623aedf1d37a95bbfca4ad5c62812ded8ebe135923466",\n      "label_sha256": "4fab255c54d08b91b283100f97f56edd4dad326ae0528cb9b55cdf4e3cf19943",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-823-_jpg.rf.92a7240657d03d86c2890413393c5b04.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-823-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 16,\n      "group": 796,\n      "height": 293,\n      "image_sha256": "50365b4f01fc75d4be8b9efd2862da8be59ae42008651e779b3ce0a0bc34dfb3",\n      "label_sha256": "33742436b32f069b638af94c0f44ef60a31209e5de6e8cd41b61053df644823b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-830-_jpg.rf.17bbf579a1b0ae0698fa1db648b68dba.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-830-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 797,\n      "height": 471,\n      "image_sha256": "a78224b899ef59a5f3187568434aa8df71564a421f915259604cea8f9c8b9cf2",\n      "label_sha256": "631bb4e54f2cad65ab645e8e40bf5e21ed35a3ec9b5af2a8b519f3c801ded8b5",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-833-_jpg.rf.22b3cae38cd8d4d8584d777aaf929e2c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-833-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 798,\n      "height": 408,\n      "image_sha256": "f0a380cf151ae8c3b56480380256ac0127689c5eb6d3629c29f6305762cbe699",\n      "label_sha256": "8d4e98c2bec5ca815092668233904b29e6ca0144ef559fc294b713beac8d8661",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-834-_jpg.rf.a7eab473e3c430be1fbd469ff49c33a2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-834-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 799,\n      "height": 408,\n      "image_sha256": "0b2e7b910a382b7ba55773994ca1ae89c9f8a6b9586a9f343e9daad95688d987",\n      "label_sha256": "f97f6a04b416daf231d98acd2aab03003d5ff277bfeeeb2131fdf3ed244273c6",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-839-_jpg.rf.515f0d7c075be6f146ace7f694134ecd.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-839-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 911,\n      "height": 375,\n      "image_sha256": "eb212c576503105b2aeafc3a6dc010aedf23760b0fb3d84457d8cced583694cf",\n      "label_sha256": "b7b726c7a9c0247907417ee762554bc3cf8e341de3942bc02fac5a0ed4c9e1c0",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-84-_jpg.rf.f84141e0fe84e4ce0b949360749aeb79.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-84-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 801,\n      "height": 408,\n      "image_sha256": "49639b485b6e69dffe2e90ca19b1d0f43727e13c13c39cd52a3ba617c503c0fd",\n      "label_sha256": "d9bbde17b866b3b7055fe8142aa4a9a5fef15756ee19f63334ff1d502648e13d",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-843-_jpg.rf.cf8cb2ebfead5644e5362807715aaa09.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-843-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 802,\n      "height": 398,\n      "image_sha256": "b4e6bb029e0308223856ab02f4409f330d2e631c078ebcd6f3959561361bc15e",\n      "label_sha256": "fe1aa885b26108bf3a4078788d2ad3b4a86b67fd8552928fe4f00ac1514e2095",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-847-_jpg.rf.ba724b2908438bfdc7be644fa715a15c.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-847-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 23,\n      "group": 803,\n      "height": 495,\n      "image_sha256": "a8be9a654ac19da86e3f61f866f76d85e76ffd1aa1a6e23c9054c4a5677c5e60",\n      "label_sha256": "d6c2f15491bb48985785e212c5707ce0aeb40df366711dc004a5dbd66ee05e7e",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-848-_jpg.rf.b8bd755e70c965e4d8e39e514a9d3a0b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-848-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 804,\n      "height": 408,\n      "image_sha256": "0f55fcee4738b8b8434ee65550840b36031466cc36e3567b03a77b5598928ff4",\n      "label_sha256": "8f45db62ca73ad44736304479835d57381f6d034b215ddff0b0ab5ef3d88ad1b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-849-_jpg.rf.d5bc3268226f882326e563f21b2b5e18.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-849-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 805,\n      "height": 325,\n      "image_sha256": "8de9eb7802e8a0cdab81755a72775e7b4345f6d0e08bbe1cf1d615dc297bee3a",\n      "label_sha256": "897486d391c758b954f344b52aa559d09e4202034f0211b8a1fc16aba5564088",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-860-_jpg.rf.3539b20e10361e1e97fddcf7c9d6f12f.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-860-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 806,\n      "height": 408,\n      "image_sha256": "aba06f8de787cab4abaec9018b24a8ea4d6e81504ef8ffb638f27a01351665ed",\n      "label_sha256": "fd68acd03787611ddaa179fabbbf5e6adff55c3ad251733b51f9d9d7fa0e88c1",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-865-_jpg.rf.254e927393c53512bee0d5dfa1aac728.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-865-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 807,\n      "height": 408,\n      "image_sha256": "8dfea4abf3a5d50e6536a7c7caf9a1a504a3a2901a4a7334ba2aec43b81b2309",\n      "label_sha256": "4f517a71998e28bb2951064e7b6ef945d77ae75829ccb4444186fdb3e9750507",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-870-_jpg.rf.1b1ae32fe8af8a08453ca60eee517e4b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-870-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 808,\n      "height": 459,\n      "image_sha256": "665ef4cfa1403d9583179e107c24f719e8013b1c9f43532b98e9b94f92a90dd1",\n      "label_sha256": "ecf65d486f856a1969fc52c3a5747e5e9b45a502b49aaf0bb3696d45e66ae43c",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-872-_jpg.rf.7534329f1828854f253648376b43bee4.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-872-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 809,\n      "height": 408,\n      "image_sha256": "906318b3adc9d65b7a631ed4f384a4c7b3f625363b876f2fd4b569d12c3150a9",\n      "label_sha256": "23255ee17f3ffa332e056cd3a231805293d54df336bb4c3677fe9429987a409e",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-877-_jpg.rf.5c01d82452410f348b547f197d6a5d8a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-877-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 766,\n      "height": 408,\n      "image_sha256": "490f6fe476b5ff740e7db762bfd140eac943e45fe52bdb1cd90b72a8f1ade23e",\n      "label_sha256": "ca71389d18cee7c5b7de90fc49d0dfb56a936628a7724e4d35d3a442d82a1aa0",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-881-_jpg.rf.0ff3c6ab5e47278be46f50a51418fa60.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-881-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 811,\n      "height": 408,\n      "image_sha256": "697ae497c9de4df0104a8e8e41fd6b299bed5d7732bfaf1eefcf576ac3e0eea4",\n      "label_sha256": "f9d39c65f3f185b63e2990aaac7be660ee7c117d673e708723dead5cbeeb8a90",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-886-_jpg.rf.2cad15e87c004807417da59716f1b50b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-886-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 812,\n      "height": 193,\n      "image_sha256": "3a1a4ecf4e81608383b442b382a7f0fd1a2f5e7738e33c10d3a4c070fc849df8",\n      "label_sha256": "24f575471008cae3a9db61621a3d06a94d078912e60fdc085591a951848f19d4",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-889-_jpg.rf.94f8f2d4492faa9ddfa2d6ad211add9b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-889-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 813,\n      "height": 402,\n      "image_sha256": "fa098e1470a53bb347f4a2a708913e965721a2c2ca9e700319d0c804fcb3dd98",\n      "label_sha256": "ba11600fa81e62e88b2186e2513da72e1c4eb5c282d772c2e987e5ad4a265118",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-890-_jpg.rf.80b0c0f5561309d3a43b6ca15c75eecd.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-890-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 814,\n      "height": 246,\n      "image_sha256": "044c4573a0aef0cb8ad8a6d648a964b8bc6f39d3acebeffed1374b0a65a41eac",\n      "label_sha256": "9914955762568d8bb6ddd837ed45a0c6be5efdebe6a5e975e57e27cae18e02b5",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-892-_jpg.rf.5cd22f227d55d52c220ff635ca4e37e7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-892-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 3,\n      "group": 815,\n      "height": 340,\n      "image_sha256": "1cbde1f89e3b8af1ddd7e050db4b74589d7fc5bf073fe5e8836b27139c849e65",\n      "label_sha256": "291b14e9e88315ff9a949249a72eaa60ce29c6d425660abb587cf4e84ac338ca",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-893-_jpg.rf.1574b2d481f876fc192df0bbc52ef4a2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-893-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 816,\n      "height": 319,\n      "image_sha256": "5c2ac25cc7ba53f832412095291fd40a9c9d616a3763fd14e9e4f3db0b803c2e",\n      "label_sha256": "cce07676fac5d35edcccbc9587084283ed461b74024f1c99d762f77a80270fb7",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-894-_jpg.rf.c7c90e8850643aa9d4d7d2ba35c191cd.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-894-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 817,\n      "height": 297,\n      "image_sha256": "654787b32eb311905c9612b73997c0264243b38e9dcc35470ec3f5db0cb2d873",\n      "label_sha256": "7baa027f3928852339bac62131a4ffbe0ed1b35db77e78f8990af90419bee2cd",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-897-_jpg.rf.84a1049af18bcd1cba0152f00fd0dddf.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-897-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 818,\n      "height": 408,\n      "image_sha256": "0250d4b0cb053bf52ab1ef2c8567ca67407e123426f25a7520afd0d9a27e6909",\n      "label_sha256": "b2d2b5c20405372f856928511954578940b96e5f979c2d658bb202c64f1e0089",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-899-_jpg.rf.1bdf619adca65edccc7d30bd4885f2a5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-899-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 819,\n      "height": 263,\n      "image_sha256": "ac3ac272d9a072f45f0bbf0f8de3d9058655f457ad17608bfe171ec2d2179087",\n      "label_sha256": "e6f5b13cafcd599afac31e71fe350ec039bff345ce151a971bcd50615f09a079",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-902-_jpg.rf.766493b9ab1023c995e5dedca5ae2227.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-902-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 25,\n      "group": 820,\n      "height": 125,\n      "image_sha256": "11895f8527819fe2a329dce28fd535ae6beac148b7f07ee80c155373f788c200",\n      "label_sha256": "006db22f0ac540eb3766f8b8661162d84fc9ddb5ec34808007897ca81131214c",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-908-_jpg.rf.60fa80a2bb90aa6462220b3c49ec20b1.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-908-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 18,\n      "group": 1038,\n      "height": 408,\n      "image_sha256": "0c0b8b8fbe700992a3f28df7882339a07f87b18c604e9340987f2f134b3bd3c8",\n      "label_sha256": "a5c8c2135e97ff38d438fde56588bcde581527c070a5bb44b1f754ea901d97f4",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-91-_jpg.rf.f4b68ef82e195b72d55145ea90656dd6.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-91-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 822,\n      "height": 398,\n      "image_sha256": "8eb179149f3dd6dc54ee287313863cc494187dcf6b6f5ec0c4dba0ac2e6a5edc",\n      "label_sha256": "f711ca7bcfbc1f382e0e16ed522ce4d044cccbb73563a4bc15bac34d8fc2de6e",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-912-_jpg.rf.a50cbf764180e055072fa10aba076bb7.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-912-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 823,\n      "height": 408,\n      "image_sha256": "1c63313bfc617b5836653b80f6338fae46da371d011aa390cf9ba08602410142",\n      "label_sha256": "6f90c7ec6476aa6de3f46ef43581144fc37d30e68b0cb71b38fc385507184ade",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-914-_jpg.rf.659e6eff447f83a5a62398468c4b9fca.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-914-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 824,\n      "height": 408,\n      "image_sha256": "37d5e4eec7d8ffc77814a0ff1d5ef89cba65c88b7877777aaff106725f008fbc",\n      "label_sha256": "527e8522692a59a44b6641e5ac8cb5ca8dbd92b4009ea631d8065caa4b5ce0d7",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-920-_jpg.rf.7568a195910ef1eb09d3de004ba40206.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-920-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 766,\n      "height": 407,\n      "image_sha256": "4776aa8922aef64d5980d917d7bb4dad8d22972367adbab5c3b3bd51b57d1ad2",\n      "label_sha256": "14f2eee0a311a76136cb92518de90127083431eec7861eea9d6f8b1a85771ab1",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-923-_jpg.rf.927d805a3276b9cc0e01328bcd2de4da.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-923-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 826,\n      "height": 408,\n      "image_sha256": "625f63b18b65befab86182e1e74c1b488aedaa329a0d0090e715368ca88b092c",\n      "label_sha256": "d15538fd7c6911b8153b5cc61785a59fbd0ec4f3715fb6ac4ecd1c8a00dc36ef",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-929-_jpg.rf.430a08af3a0a15ca4f0b7aee6809460a.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-929-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 17,\n      "group": 1060,\n      "height": 406,\n      "image_sha256": "7aa71ce70afa7a4e911f2fff856eec1818c895de0ca51c6ec945edb1d4c28c98",\n      "label_sha256": "84d904920217f13256e974c3971e23908af897e366fa8f59112949084d4e6e8b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-934-_jpg.rf.bfaf0971e8412e70442120dd78294d1b.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-934-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 828,\n      "height": 408,\n      "image_sha256": "78a41a47eb9137b5a6c99ab3cb1b9fc45fac5b9d7e00292998a5ebb70abb0c82",\n      "label_sha256": "d205dc146771f5dde04a7d220fd8408210b9ea4463425f0687c37b17489b48f6",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-937-_jpg.rf.081f177adc25ba1c1cc2fac81336e0f2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-937-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 829,\n      "height": 347,\n      "image_sha256": "f85a3ae366ae97b32fbd60f97e02b8f4c3ea4f3546b0750a89683dc7786136f5",\n      "label_sha256": "1cd36515994d7f2355a24d7f17651506373102df8a31a63055a54d8e1ca75d9b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-938-_jpg.rf.7208a34ac37012bab3059097673f2336.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-938-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 830,\n      "height": 424,\n      "image_sha256": "1d2a5ea64a67b55ff26d8710ad9ede61817f4f0ea72fb35b3b55c6e38a4f8df4",\n      "label_sha256": "a5ba45dc995f39a1a33d6c3e4d644b9851ce319c8291d07ece2fe761bac8b009",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-946-_jpg.rf.c368f713ffa68be241ff09adc67e9bf4.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-946-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 831,\n      "height": 420,\n      "image_sha256": "385479844e6de3c2ee7b81c5c7ddad7e36352d5446952790730e6dceeeb9ae06",\n      "label_sha256": "c96392a95ba678934ddaac60d5837fa832173cb72c1762e849680ea9abbaab83",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-947-_jpg.rf.07b57e35442a43fffd71e7166d5f4d20.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-947-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 832,\n      "height": 283,\n      "image_sha256": "ac039579ee732090e5093867ff35e371bf816bfdfb60cea072b1ea78aa7298cc",\n      "label_sha256": "5393a8566f4f463b0dce127080e76d416b6a6dd962b6c5ada8a99f390071a466",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-951-_jpg.rf.466b2068ade07dc55133b6dda2d24529.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-951-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 833,\n      "height": 408,\n      "image_sha256": "b35ae1ec8a7d8c313fd3d0611e3f6db7278a1ec517409817eef0219f86c8e00a",\n      "label_sha256": "3a66b25401d67737a0db3bbc12d3ee31eba67b90237a6f518a5435908b3dd3dd",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-957-_jpg.rf.468b820e96b5e199a415595ac61a70ae.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-957-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 1020,\n      "height": 408,\n      "image_sha256": "3b88be0990645fd0f41c45eca00d11f5c5fb00e10d327e9b3a33fcf6e01a0a0d",\n      "label_sha256": "e93f04d50f4d36100cff19122a91ab3cfa648cf2138d86e13afe310c574d91e3",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-96-_jpg.rf.a68e6b093ff9ad51c11a0e28a156e6e2.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-96-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 912,\n      "height": 406,\n      "image_sha256": "e407db35b1f6cd7a3c1d51162d5946dbecfeeb2f54f7c77aa8295c02fa274bda",\n      "label_sha256": "fc5f1b49b93629828c0f648dd0147f9e86656d0b24b106a25ab7954c8d45e45e",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-97-_jpg.rf.e094bc49ae41ea79d7f6373d94075355.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-97-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 914,\n      "height": 408,\n      "image_sha256": "c8f74bba8db474a3cd56e2814d0276b5b64a1fce777ad14c488f958299b29acc",\n      "label_sha256": "d4f2b9a4dfd6296307ad1ecb3d1a35abac3bbe2fda49ef6fe247083971b7c91e",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-971-_jpg.rf.1fc2e015e0cf977bc1945eadd737b572.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-971-_jpg",\n      "split": "excluded",\n      "width": 612\n    },\n    {\n      "boxes": 17,\n      "group": 837,\n      "height": 293,\n      "image_sha256": "9edcc7dd4b5062cbd0d55fd7aeee5a3f1c408efe73a46a174b2148287c8298fc",\n      "label_sha256": "e33cea09a20f831180e78a1ec69387898d6d572c2221ffbeffccff6e09385d5a",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-974-_jpg.rf.b460649ff94942f37a98b6cde400320d.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-974-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 838,\n      "height": 408,\n      "image_sha256": "48ac5523fb64d2f12660b176c59aa8270b3a84d26f73dbd594833a6232ba426a",\n      "label_sha256": "636d8cb903cbc40ee392dff27ff7ed4150537e98b11816a2b3cf953659a13a2c",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-976-_jpg.rf.b09c2eb10a00d7c83ad59fdfda620269.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-976-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 839,\n      "height": 409,\n      "image_sha256": "e68e0851bebbdfcc4d0638c6c7a2831bde5597268a8106ef75ae6ce196e9bf7e",\n      "label_sha256": "7164be9f18b0859391fc5864c98353d475faa75d8821f404ccc85d198e7cf517",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-985-_jpg.rf.c1e91085d21ff720f7e7c80d15064032.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-985-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 840,\n      "height": 612,\n      "image_sha256": "91be459bb43e277b951b7e971ce041b134d811491b4c3a735b91b947c57a8f8c",\n      "label_sha256": "a24709c311ffd137121cbf97d96aa5191f8ed3964b59d7370c4764a82633b3be",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-988-_jpg.rf.94e29407f0d1655db05991cc01327216.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-988-_jpg",\n      "split": "train",\n      "width": 601\n    },\n    {\n      "boxes": 24,\n      "group": 841,\n      "height": 495,\n      "image_sha256": "1da0afe9c5a9e1cc9db00907a32873049cff4441f953b8fb6002bae2a6b3000c",\n      "label_sha256": "bc0286cba5a6958d4035b76ebdafe7f0fbb8d8428d3eea0e5f57ff9628336d32",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-992-_jpg.rf.ae22c713aca5d8c4fb0d063d29ce9323.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-992-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 842,\n      "height": 408,\n      "image_sha256": "90adc1243fbfd99a8840063adcff1ab99f7f5c872cdfe4a897de6281c00a458d",\n      "label_sha256": "50454c15bd2663ad84a20fc4598719eaf1413c0f1ab4aa7faa2a06af6fa9a12f",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-993-_jpg.rf.92321f7446e8b311879f887f8821e771.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-993-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 843,\n      "height": 408,\n      "image_sha256": "2cd63045279a2ff10239b387c72f646bfc6e40aaad92b928746258e8c8095d92",\n      "label_sha256": "fdd87dc0d13752213158393b89f5d1322aad3c306064a0954dd09c21adbdb6a4",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-996-_jpg.rf.145e11f68488ef70f2d2f4300a4a29a6.jpg",\n      "original_split": "train",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-996-_jpg",\n      "split": "train",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 1084,\n      "height": 640,\n      "image_sha256": "0dbdd78541d7d3d486a0f586e56a01ae2e3003be79bc4e8fc62ef9f312468c46",\n      "label_sha256": "fa9876aafc7feb5dd5bfe6e59f96eadef5dc5214ea42dc7802e9bcaea8c5b577",\n      "name": "calculus_poor-peiro-2-_jpg.rf.17f0d8babcf85d6278ff8a604bfe4526.jpg",\n      "original_split": "train",\n      "source_key": "calculus_poor-peiro-2-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 9,\n      "group": 1084,\n      "height": 640,\n      "image_sha256": "8d40d198526383325edca385e194c694b13545f67e1b05c0cea37cbc75f4e212",\n      "label_sha256": "b822314ed358b5f00a34aea3836b3ec101263f825e0e4d3ae4fb3f3fd2bb47a2",\n      "name": "calculus_poor-peiro-2-_jpg.rf.fa4c745a4ac71653d21f1708640f0dc5.jpg",\n      "original_split": "train",\n      "source_key": "calculus_poor-peiro-2-_jpg",\n      "split": "excluded",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 718,\n      "height": 640,\n      "image_sha256": "7a1de6e0aedfafe8118a70aa8ee499e5f51d6158324e9fe18b3d5534512615f7",\n      "label_sha256": "0b668310cecaa8c4cbc485a861203e6c1f434185703a97f1fc44a16f6b86adee",\n      "name": "calculus_-1-_jpg.rf.cfd174cea309634b22d04c521bbcce8a.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 847,\n      "height": 640,\n      "image_sha256": "8cfff4a34e50dc44e9504ed0452b6959215d03d72734e7b9b72841f5cc30e9c5",\n      "label_sha256": "748cdf76290c5e6ea78431ede9f8b7564c8a69cb869d8ac377f47acc862cd7aa",\n      "name": "calculus_-10-_jpg.rf.83ccac4b935b9d0f5e243e5832b27d76.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-10-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 848,\n      "height": 640,\n      "image_sha256": "c0e76bc0c38f8b4ae7856a6f9944c1d974f3aff6ee3ae2432c3bd36c37b2c808",\n      "label_sha256": "4ba28b625fcca8aad640a0472b4a173802393e537e720a907e2797907eff4a71",\n      "name": "calculus_-1000-_jpg.rf.d93f9728decff65af77b67f35f85154e.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1000-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 849,\n      "height": 640,\n      "image_sha256": "1eeb2fee75324521fcd5737e288e756d62359fe84002597c7cef434f14141110",\n      "label_sha256": "a1f94455ad8daaf4661ae7387935263ac6b0ba7863fdfef6b6f81ce7d1ebc60b",\n      "name": "calculus_-1002-_jpg.rf.5e5fdf4d35e9c450233dd5bfa8d45a8f.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1002-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 913,\n      "height": 640,\n      "image_sha256": "3ba5f989b3abb2db9ddf402362e580ff0fd196591f9224938fcab9627cb3755d",\n      "label_sha256": "a2e69a663d35c741022b9f8b5804079514671ef18a6ce56b00b2cd627067a481",\n      "name": "calculus_-101-_jpg.rf.6001e177fd5cbce6edc2d94d774f9759.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-101-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 851,\n      "height": 640,\n      "image_sha256": "2706313ba0f00de3f98626e294435f27bde28bf7846315e9131bd27b6a9cb8a5",\n      "label_sha256": "6c821b1b1713d57de1592f8b712fd436a8a856abd276bb6980dcfb25c81f87c6",\n      "name": "calculus_-1010-_jpg.rf.67a056acfc8fe1c99b94156fc6f6b8f6.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1010-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 852,\n      "height": 640,\n      "image_sha256": "162b6d5316f784481e5830888befaf6fec4bbf697bf570f463cf1a4ea6db5a59",\n      "label_sha256": "5555ef306bd6612cd650cdea9a5756d40d2e8447e0183cf34d620b2f46d4b29a",\n      "name": "calculus_-1015-_jpg.rf.a02d81f76f47496cd7a88142ddb8dd76.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1015-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 853,\n      "height": 640,\n      "image_sha256": "ef322193d45551786746e0beb9367c9d8c9f78c95217994b46f0ca778dafbccf",\n      "label_sha256": "a460c4e39d1d29236e0576c22b5c4f1b88815612720f483ace97d31b125191be",\n      "name": "calculus_-1018-_jpg.rf.82f233bb7225d6eca104dc2ea1799c28.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1018-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 854,\n      "height": 640,\n      "image_sha256": "9392e88305122b44bb42690e082d6694f1d9bd967a38254f8ab8c919e2942f70",\n      "label_sha256": "fd4c82ed2498d4a953e2b1e1200a9ec3537601906b16575151d5f9c8784fc155",\n      "name": "calculus_-1019-_jpg.rf.1f55861b19a0088c5cf860508c9d3d4a.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1019-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 562,\n      "height": 640,\n      "image_sha256": "8d6c23e6ac36383522272908f22e35a444240e0e7ae8741abfce8c27d86c5341",\n      "label_sha256": "4251205ce20f0d06e0a28bc06c0316d2b47ab99f7339875d4b2fa841054f9b0b",\n      "name": "calculus_-1020-_jpg.rf.ce8aaabd9d58e71c70e2b3a3cc999ecf.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1020-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 856,\n      "height": 640,\n      "image_sha256": "8956ca29a9796da9ac303ccec43fc7a58976c2494e7ee79a356ab7a7f386a69e",\n      "label_sha256": "2934bac7e862fe734656a3103410b3cccffb49d65c4e32784e94924851110117",\n      "name": "calculus_-1022-_jpg.rf.f2f8bbe14d59979d766015d2295467b8.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1022-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 858,\n      "height": 640,\n      "image_sha256": "68be86bfe6eb476f468d3be7892c7ed1e40d41a43d9496f4f4c5facc17d26820",\n      "label_sha256": "4937d74b0a5442f3ab056b6e675de3b631cb048e1f000a81326e17298ff07b99",\n      "name": "calculus_-1023-_jpg.rf.3cf20c9227c53a77d73de09940fabbd2.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1023-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 858,\n      "height": 640,\n      "image_sha256": "36966d6f3d8323ef34a33ee972ebb168293dc1bb4e688594f6ccf080c7e4d4df",\n      "label_sha256": "34d777af37bc3b712f041dd818db0c3239de0c44cdccb0f06bd70b16d47bce0a",\n      "name": "calculus_-1023-_jpg.rf.5bcffd4e093240813bb2bdce4879e44a.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1023-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 859,\n      "height": 640,\n      "image_sha256": "32c9b9ac0603e89c24b08170f6128cc974566a3bafef9d3176398f664cf75572",\n      "label_sha256": "42357613352f238b86c2268a6ff2671c3a37d7ce5a81fb5b978edb8160fe600b",\n      "name": "calculus_-1025-_jpg.rf.dbd052da60462bac77fb5195efb6b0a4.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1025-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 860,\n      "height": 640,\n      "image_sha256": "364e16704952f0e86ec82afaae4bec5ed65182850ab62610be6886585ec262c5",\n      "label_sha256": "766f19a7250df2226b97f8c58572ffd5d02083c955c606d7546fa176a9fdc105",\n      "name": "calculus_-1030-_jpg.rf.f4cdfc6ff1562b2c0aeab4d2462f2f96.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1030-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 861,\n      "height": 640,\n      "image_sha256": "33b7c66eab270e93e2dc0f6171b61e3fb43e552084fae832460d73575175a1a2",\n      "label_sha256": "e7b59e447e17cb9e3701dce2425e3e0a38115cd43842a62988074b4081a36079",\n      "name": "calculus_-1031-_jpg.rf.d1605ec0c11cdc356cf326f38996bf33.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1031-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 863,\n      "height": 640,\n      "image_sha256": "fd657cff926ed1c07c2e35f1a43cb64b2245ccc1dd92397ecf5096684f7440a3",\n      "label_sha256": "54cf53aac1efc130ad410f3ba59d16cd1dbb2e0e06ad5d2dd18ea8d018886a53",\n      "name": "calculus_-1036-_jpg.rf.3ada19d10911efeca9826f8013289355.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1036-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 863,\n      "height": 640,\n      "image_sha256": "0c3a6794ca7701d7cc3bb37467d183a22b7c4730771c9e71b68b375185de889c",\n      "label_sha256": "d5d7da82433b8b262771e5b74e0459f31535f8b6e5a71b34f46d1d344df0a72c",\n      "name": "calculus_-1036-_jpg.rf.fb5c4fd61a8d8dd0c9327913de6e820a.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1036-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 864,\n      "height": 640,\n      "image_sha256": "ab8243973af957545bc3f449a7a5be8fee10d0ae52ef3a6b28fffff6b2af78fd",\n      "label_sha256": "9e44c136fe16e0247558670b9fdc16e0ec79be9ddbe9659423e32d80e81725e9",\n      "name": "calculus_-1037-_jpg.rf.c8f5104f8d240028c646b084355dd5df.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1037-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1011,\n      "height": 640,\n      "image_sha256": "be887b0cd56ab981e3e71e6e4fbd7e7bae8ce6bc45721f15ceaef80836634e89",\n      "label_sha256": "2a3e8f33b4aade211105822ba2fa0e8a687179125722406f4a2d1b8eca2a3f5b",\n      "name": "calculus_-1038-_jpg.rf.a6b18d1034483f002fa2061ed3cbc22b.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1038-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 867,\n      "height": 640,\n      "image_sha256": "ac6a94b2022c17b281e6720ed3131e3e582490005b43b8d2a351ef75c5c19d1e",\n      "label_sha256": "d6a17877f51a48c2f03fde9bb91c2d7954141e2a99d206b461dc0dd5ea660a98",\n      "name": "calculus_-1040-_jpg.rf.b760ce4e825fbda447c85b6699ab0865.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1040-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 867,\n      "height": 640,\n      "image_sha256": "9a9766588681b4090d6df09e7cbf1c4527142725cbbcbe80f06b2262cbe70ffa",\n      "label_sha256": "d8bca1c9dbce8250364ad1badd7f26c32720b2d84e38f9c505c68293d8245777",\n      "name": "calculus_-1040-_jpg.rf.f76ea89b1be5aac87cda8d0b71fb529a.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1040-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 868,\n      "height": 640,\n      "image_sha256": "51cb97b5077879b053dabf4105c7b040a10c87366e48ec4136018f13a472861f",\n      "label_sha256": "6637af0e4b14b98f0ca1b5623cf31c42313a2810b1867a6eb7d507b4cf92f7cc",\n      "name": "calculus_-1042-_jpg.rf.96a8dae6a23357476417a8c4e248b25a.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1042-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 869,\n      "height": 640,\n      "image_sha256": "9dace6a959e94d034f9c535fadbb9a2ce92a448c1f35c873952c4e9925f3905a",\n      "label_sha256": "89c509f029edcf2af224e3ec30094502cc97b5e6d16a312134f73ee727721d91",\n      "name": "calculus_-1054-_jpg.rf.bf56f5a9fe10e31f0f6b5718ca6ad2a4.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1054-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 871,\n      "height": 640,\n      "image_sha256": "0c1cd65d7f5e6486d46627cd18d1a244abf7389da5f074783ded3de6654abadf",\n      "label_sha256": "ffd322b7823271c77bfbb019ed546e8cd7d135ee6d9788b7eb7f68c2ca864bcf",\n      "name": "calculus_-1058-_jpg.rf.54adec3619cccd35e652b2489e5a3fec.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1058-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 871,\n      "height": 640,\n      "image_sha256": "9845afa07ff3cec93865850a3128b1c7512be609bbd3f46b5d1553619753db83",\n      "label_sha256": "dbfe827cfb0da84fc28ef431551f4e25fc93b69f35ae226e6e828e8a7313f409",\n      "name": "calculus_-1058-_jpg.rf.fdaf03db4db69ffc0a0c1a0d269c7314.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1058-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 873,\n      "height": 640,\n      "image_sha256": "4fbf7161e5d399b8f87118ff4c22304f45019382451bb01dda38854cecf107f8",\n      "label_sha256": "32812ed146698ee295a1197b6c6a79b9a2d91692d6e5f3cb8b8428c52b2759c7",\n      "name": "calculus_-1059-_jpg.rf.a279a0e99671cd1c06aee4084b6d971a.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1059-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 873,\n      "height": 640,\n      "image_sha256": "31a1f49fc2e824a936ae6a2b01f7b1cd23aeb7e8e77faf24fa7146e83fa5f4aa",\n      "label_sha256": "71bca98ffa416cffca57791061be53079a68d94928858f4501e4614437feba84",\n      "name": "calculus_-1059-_jpg.rf.d656e2c608e15c970ac284beb1902b93.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1059-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 874,\n      "height": 640,\n      "image_sha256": "1a9b8d724a65754ac3b7f0c54af09f104e8ef236210d4dde7ed1991b347fca0a",\n      "label_sha256": "976de3d77f81cbe094567d53c194b1d46579a30a93c42b9f39d096f3cf10d0b8",\n      "name": "calculus_-1074-_jpg.rf.e7b747c810d72bc51fe57780eab7240f.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1074-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 875,\n      "height": 640,\n      "image_sha256": "f4bcf2991e22be518da1f89dc8cb70992498f446673c6cff7d50c7aa1ea41666",\n      "label_sha256": "f8e7af1e0dee039b087c13021f77cf747a38f47b677151875e3ca017fec646c8",\n      "name": "calculus_-1076-_jpg.rf.6148266e8abbc6ac8e4c6006ea13dc16.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1076-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 876,\n      "height": 640,\n      "image_sha256": "cfc35437525558a0b50616e92dc462daf7a1bd9fd42fec398a63bb5b5b7a130c",\n      "label_sha256": "bf843dceb9e7ae79bef57996324b225eca71b2a36c504f64871525405dc5ae1b",\n      "name": "calculus_-1077-_jpg.rf.7fe6952151e8f7c4131d71b479e7a661.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1077-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 878,\n      "height": 640,\n      "image_sha256": "316e148d6ce100f127b3c3862f209442195c31b7b9fd7e407cc9d91b8a1bc0e0",\n      "label_sha256": "c96b120568d22c89302c0d67838483eec6ea4ec01b1cf30fba75c28d4c1ea0c8",\n      "name": "calculus_-1092-_jpg.rf.0b1da02d059665f16995768faec7a479.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1092-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 878,\n      "height": 640,\n      "image_sha256": "01dc84cb53b79c97132b1a4ede221081105b3d998e20b4c8956325db845e29f2",\n      "label_sha256": "7fa01711763756c0b4aba69691a3056497991f9737b28acd8e304ea829fff641",\n      "name": "calculus_-1092-_jpg.rf.db3c52d4db67947538c8710d363a3679.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1092-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 880,\n      "height": 640,\n      "image_sha256": "27ee9b138b57d64b5835bd3b463f062e3cf0ea9c5ac15c6929fe16e03984770d",\n      "label_sha256": "be49f26f2302cde46d0bc55ae85cc54c2cc358844353bba3eb251be7ab470e6f",\n      "name": "calculus_-1095-_jpg.rf.1208d4a56490e3867ffc402fd13e2e7e.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1095-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 880,\n      "height": 640,\n      "image_sha256": "633a9d69f7142655ed8eef96c04700f5b51644fe72704b127f52214e5905c237",\n      "label_sha256": "d4ff90a0a0fe1479d17b9d770e4f3d1739e2bbe5233c7dac75e3e2ea619e22d7",\n      "name": "calculus_-1095-_jpg.rf.c7a62572eb808c4e901cce530d692181.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1095-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 881,\n      "height": 640,\n      "image_sha256": "866d404e054c6d7afc0c6349800702657cfc1257917f53e6fff6ff7dd45fc930",\n      "label_sha256": "1666669d5394bc53471d3ba9fb6b74e8027818586bde238977926601acbe16ec",\n      "name": "calculus_-1106-_jpg.rf.f072ac8df79db17f4ba178b48cc760d7.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1106-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1015,\n      "height": 640,\n      "image_sha256": "30176d9a3921ee36e34cbcfe4b7975a9372531d726051f05d4b08d13d2364e6d",\n      "label_sha256": "5bfd633a7dca3b86e0f4e91faa87eaab93572bc3f91b0c28ca5050dcfdc4b021",\n      "name": "calculus_-1109-_jpg.rf.24effdcb336be5d2b684993d6be933b3.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1109-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1015,\n      "height": 640,\n      "image_sha256": "4ded15d4a0f83f3822058210c45690dc36b1ce23042bc53a9c2d870fbc7e4b19",\n      "label_sha256": "9a7e0d04a4501fb8f264d9dd30facf0e21c0fc960ada34bd03a807ad37aeddf9",\n      "name": "calculus_-1109-_jpg.rf.66bdd2c205cb60317c0e099d74b99a12.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1109-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 884,\n      "height": 640,\n      "image_sha256": "706ceaf36b3e7a472d4cdc514b6325dfb711bc1414a19437bc2b1c7c574c621d",\n      "label_sha256": "1c7d7fcf1f0718942839388c76269f8380bc25d6276b3ce6752218a51843ba0e",\n      "name": "calculus_-1120-_jpg.rf.658136f3eced9afd2febb1263793666c.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1120-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 885,\n      "height": 640,\n      "image_sha256": "b1e8cc58d895d8c765c46e921484a739fea923bf659abb7d826d9cc71ef25a4f",\n      "label_sha256": "45f9c91573dabffd0652587d662331784fb379d1ebf37f76edf50c50447fa2fa",\n      "name": "calculus_-1122-_jpg.rf.bb7f4f6b27cb130460bbf413cd2ab815.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1122-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 886,\n      "height": 640,\n      "image_sha256": "3f52ca93119bf0316b592d3ae7a8f5182870567dc87c07ed565497f00501628b",\n      "label_sha256": "0e03c440ee3d1e2507eda8162eea530341da7b20b5584055a3c05a08989598cc",\n      "name": "calculus_-1137-_jpg.rf.53777d5e232bf050a63d83c565894cfc.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1137-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 887,\n      "height": 640,\n      "image_sha256": "fef50761c41f187e183beaeced5f7ef2fa44ab4250eb6ed4f2b1ff55d9f54c1d",\n      "label_sha256": "b9f66a970c5e02aa404b31a664d5a0adbd41e0a2cb4127d64a51dbdfcaeddcec",\n      "name": "calculus_-1150-_jpg.rf.1a4ec54d1461649cff06183b6cd35490.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1150-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 718,\n      "height": 640,\n      "image_sha256": "c57b2a6d6cec407ca7738cfa9747fe9766557fc3d1b3655f52725ba43793c861",\n      "label_sha256": "52639ad1c757c94ad27cdcee41993d707ac7187f8f441fb7f03d71ca63773de4",\n      "name": "calculus_-1153-_jpg.rf.c9bef2f4db286b8a8464352fc538a7e5.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1153-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1017,\n      "height": 640,\n      "image_sha256": "b5ae0f9abcae371063b04e24ebe2533f996b2ac7201d4c5e2b6f57f1f34311d7",\n      "label_sha256": "f55d825e1a9a7fd757df34134689bfdcb9428df8d553068a70093aa8640f511d",\n      "name": "calculus_-1157-_jpg.rf.9bac6e15ec7bff951b048b2a077fcf0b.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1157-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 9,\n      "group": 638,\n      "height": 640,\n      "image_sha256": "43af1cd3a5c5f25d2f2039efdf965686c18f5a859b0cd0d1d7fe48638f488b26",\n      "label_sha256": "16d992cc3cde9e5b5caae4af007881b67bef3a6526eec6c2408f41d7c973c1b9",\n      "name": "calculus_-1159-_jpg.rf.e2addd0fc819e0ce26fef3ac73287777.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1159-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1031,\n      "height": 640,\n      "image_sha256": "14ce5508000733771ffdc6979cd3b8dcc31e557307772340bdec81e8b50e2e29",\n      "label_sha256": "8fe9a0aed6b83cbefe6b306eb93eb1628b93fbc4ae2b1e4a7088aa89c74764c0",\n      "name": "calculus_-1161-_jpg.rf.5df7943affba94e7d5fadc0591a4bb0d.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1161-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 231,\n      "height": 640,\n      "image_sha256": "5326154fb4934df0d802fa70366016f96bd0835c6ba7b987732190f76e5a3f90",\n      "label_sha256": "f8439ec06983f3408f93056d546c546c1a356bcbba9705118fc0ceba1fd1a4da",\n      "name": "calculus_-1167-_jpg.rf.b274761296d513581cf5c5d8a10cd375.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1167-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 942,\n      "height": 640,\n      "image_sha256": "e349aaf1f3ddb92381f060c85a5a52194d897e566ae552e4c59d7480f2c90551",\n      "label_sha256": "904496cae0a064aac0b7cbb11d3fecb17e13b43ef1b6e85a456b13ac36c2294b",\n      "name": "calculus_-1170-_jpg.rf.28b2178b0abd49411ec90285040a858e.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1170-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 1033,\n      "height": 640,\n      "image_sha256": "3e20c8fd74123b2215d5baa95eb233234735d8959c48006227168174a17dd09c",\n      "label_sha256": "048d98b609e6d5bbffcca68b777ca92baa2e0c3c257a94e24c9687a5838a000b",\n      "name": "calculus_-1176-_jpg.rf.3acc5357928604ae93567cf0f9945846.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1176-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 1033,\n      "height": 640,\n      "image_sha256": "bc80a82ea2b55efc848ed73ebf79982c79b23ae10df2b7788c72404157e7e02b",\n      "label_sha256": "fede06133d50f29197f49fd940ad47628fa96b7007980992bfaa4e1ae0c12f8f",\n      "name": "calculus_-1176-_jpg.rf.a31bfd238c6ddf759ac83d40fbca87f6.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1176-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 896,\n      "height": 640,\n      "image_sha256": "49a784a496fb128c7166963771bc9440aaeef73bb8034e686472935971447f3b",\n      "label_sha256": "5c9d8444e9d1b9993657653dee364eea7c51ad49c187b79028b83d2e767b1742",\n      "name": "calculus_-1177-_jpg.rf.5bc274c972698efb14d276650bbc2b5c.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1177-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 897,\n      "height": 640,\n      "image_sha256": "5b22182c0fa736cdfbd2453a965ee32cc558c9f643bb2e6ff4f9a66453ec1902",\n      "label_sha256": "2f07da42b85cb33b5d61ae5cbb985d8d21d6a68563ce1aa44479e02f9a4f01a4",\n      "name": "calculus_-118-_jpg.rf.c806e0fcd944ebcd58c97ae0e219b72d.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-118-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 1045,\n      "height": 640,\n      "image_sha256": "546e7b261457236fd3478cd27517ec8bbdfe663db6454d56284befd34cec85e5",\n      "label_sha256": "acaa16854d0aa9a975b6eda16a1d22883b1986abee841ff9f50454a59e10e408",\n      "name": "calculus_-1181-_jpg.rf.88dd26e9376f7abfe0391ab4db366cda.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1181-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 1045,\n      "height": 640,\n      "image_sha256": "6cd827e44026adfbfc8254e1a0971729c4179005b9bcb6bb77cf87ec419cb02c",\n      "label_sha256": "cb9821a2aaf6305fd68a9c073973498ede82b42c982e59bd83168e623752197e",\n      "name": "calculus_-1181-_jpg.rf.e411d6e17b974f4280558ee432c7d0b4.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1181-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 900,\n      "height": 640,\n      "image_sha256": "e6c90e2ded18889b4f11168ef96f9865bd465a517a395b28278df0767e30f799",\n      "label_sha256": "7c276c97e6791b9a9f625b873957f9cbaa16c5ff8491aa27f7e01cc2f6185947",\n      "name": "calculus_-1182-_jpg.rf.de8809ae61d752333f095a7df0ca5c61.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1182-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 920,\n      "height": 640,\n      "image_sha256": "e8292f9eef9bcea007c5c4c0048c2621ed22b8f8c76c3a2fb9b4a9dbbb3f4b49",\n      "label_sha256": "e275ede50e3687ac76d5103ba2244afa0e0777a1e1e664d4e6b3427070c83a0e",\n      "name": "calculus_-1184-_jpg.rf.b77df57f94521142c2034285dd1c6dd6.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1184-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 920,\n      "height": 640,\n      "image_sha256": "ebdf1102e4214e74637661bc528b3c4a62f754a130e2fa91038d99ac3de86d9d",\n      "label_sha256": "cd8705109deea7544fca7b67f2933c18e834253c5a12b2f2f1155a759d5fa07c",\n      "name": "calculus_-1184-_jpg.rf.e772fab2f154a2a03d676cff1276269f.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1184-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 904,\n      "height": 640,\n      "image_sha256": "51ba8d18134792a2e607a4c2faff3e88b778f27b2058d3f70a785e23d197c652",\n      "label_sha256": "3ca53e68dccfb9e9c41895683d4fdf4728c2e3e7e73d5a344d950390e290ed4c",\n      "name": "calculus_-1189-_jpg.rf.7b09684537c3e76386359214f70c199f.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1189-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 904,\n      "height": 640,\n      "image_sha256": "fb6938dc83165d17c93d0d29ba43dae5c66a32a852c69fcc0f8edc320ccf8d0f",\n      "label_sha256": "be3cee4dcf20039f9ffc2f751984ec1a9781e8956faf6831aa53ddd4c32780da",\n      "name": "calculus_-1189-_jpg.rf.835f3d1fefdff2ca4a96571c6420faa6.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1189-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 8,\n      "group": 933,\n      "height": 640,\n      "image_sha256": "99dd87253308553b71494163babe208f8a269d5636d17c9bd47535adfdb1de4d",\n      "label_sha256": "9e445029000c017efa546350b5efbffcab8f33a29dd3a0ce59fb4c39a0fd0780",\n      "name": "calculus_-119-_jpg.rf.20b62495c20d7417fbcaf453f3e6ff55.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-119-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1023,\n      "height": 640,\n      "image_sha256": "0f72d2371fead1d68e6e7fcaa38ebc85e7cf4bf778cde2ce4ce1a1a54f005807",\n      "label_sha256": "4f87f64291a01317e8b696bbdba81b79fd1565a5c681794df2088c119c1e44e7",\n      "name": "calculus_-1200-_jpg.rf.4e0a78835870c65f9c0e78b390a615a2.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1200-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1023,\n      "height": 640,\n      "image_sha256": "4836923effb0504dc30ad7e04c2c8ca68216df70df6c5b924a79e077eea15467",\n      "label_sha256": "8050b444e9f8b02b9e6abc0ef82b6e005df9b68dcda8b5eb5adee03889dbf2ca",\n      "name": "calculus_-1200-_jpg.rf.604a04674bb4837026396702aeb5c7bc.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1200-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 908,\n      "height": 640,\n      "image_sha256": "fcddb8175af128e3337ff0480f72472c3bd3db14e0dfeb005c82ee6cc822f00f",\n      "label_sha256": "56cd6fa3ad2239b781e9e41cbe27184de4873c4f5b2e2d0747ccecc1c44e90ed",\n      "name": "calculus_-1208-_jpg.rf.b50ee9eba675e6b7bf0ef4c2fadc72db.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1208-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 909,\n      "height": 640,\n      "image_sha256": "5c804cf396df96c2ce5c9ac6a5c467a133a8042dbb8558a6c31d20ebc73e26e5",\n      "label_sha256": "b240148609040ea46edc1d56195a4dd3c593267dd2b9b540ef3263eaf8f1b78d",\n      "name": "calculus_-1210-_jpg.rf.791fc850a4814f6577b6adaf28f30e7e.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1210-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 263,\n      "height": 640,\n      "image_sha256": "d175dc01a1c5803500760e6da68657baf2d145492678a2325e26d469a4869b4f",\n      "label_sha256": "e44a5e46f60f80004adaf3da7b24ce5c4e4c156f7607ce6f1470fd44c1217d7f",\n      "name": "calculus_-1217-_jpg.rf.cac6eb1573529ff58602c45bc6eb3f05.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1217-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 911,\n      "height": 640,\n      "image_sha256": "27c6828f2dfd192ae7aeb6248db9cf5f1a3a9bb9b580ef6366e6f6d8f884b508",\n      "label_sha256": "5a72290ad3ee81ae6b31c79586689ba0dd50b2ded976c4d123abd357d615330e",\n      "name": "calculus_-1236-_jpg.rf.0b8455bc808f3f9e8ba320a54bde9854.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1236-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 912,\n      "height": 640,\n      "image_sha256": "6b3bf5da4767335bc7cf83057ef8ab6ad2cfbf47ad4459ce9c0c5558f4537b64",\n      "label_sha256": "74fcc7a189796228faeacffb57d6aa8a7c68aa3c4c6c5a545186f86274ee844e",\n      "name": "calculus_-1249-_jpg.rf.212cba1baea84ab9962104e160b801c6.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1249-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 913,\n      "height": 640,\n      "image_sha256": "084bd0c81999efc7272c987c8a6558cdf895093bd188971bf6cf4efd5a3e1d72",\n      "label_sha256": "1bc30fdb0470cca2e60a2dc1d834f0a8eb7d30c1b83902efce118fabb72e7bc3",\n      "name": "calculus_-1253-_jpg.rf.be6865869fd7b5bad1d24086830b176f.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1253-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 914,\n      "height": 640,\n      "image_sha256": "890f25229bce5c0e95c53070ee234a99d177c7584fd7692d3db0be0ba29369c9",\n      "label_sha256": "19d1a7bc343fb4bd14d3f046e2c2138a8e05b36b16f39bd1423448bc330eee6b",\n      "name": "calculus_-1259-_jpg.rf.792701ef2a66e7c1abb99583789d8e31.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1259-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 915,\n      "height": 640,\n      "image_sha256": "4798b617b3fef80ab440f53d9e77970bdba82212c969e6bab40eb3219223b237",\n      "label_sha256": "e554b080176cb760da3bdad4ea5580016d8bc11cbf734dcdb2d2a5f3237da43d",\n      "name": "calculus_-1265-_jpg.rf.a010ab537ca2d1491169859c19d6ddcb.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1265-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 917,\n      "height": 640,\n      "image_sha256": "316183720dfa9f8a3aaf58329f85e0f27d5765e258272c95472e9a2c548c8adf",\n      "label_sha256": "5ecb6f95261c9407dd972a3a8e08c149412fb8f75fae889dd8bd5c19f4440c4e",\n      "name": "calculus_-1267-_jpg.rf.c3852e238bcb5d997cc2df07d1e6f4dc.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1267-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 917,\n      "height": 640,\n      "image_sha256": "5988e2ce9fc58368d2633adddb6a4c615559f2dcd4cdb2db3ef2832b07d6a6bb",\n      "label_sha256": "bdbedfdec3d72a4abf05911c07e01ea434f3cf3757b567874143449d499cc8ae",\n      "name": "calculus_-1267-_jpg.rf.df45f07dca194c7973c95650749dfbc2.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1267-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 919,\n      "height": 640,\n      "image_sha256": "5318b4510fded421cd7e18e66e6962b6ace3aa8b5fa51ea0a1040975f325eff2",\n      "label_sha256": "3fba33cd6704f04fd13d6d1fd483d02878bb3bff46dae3007ceb93f43573b9fe",\n      "name": "calculus_-127-_jpg.rf.4a3f6a397c3b73caef29ca92f66aafb1.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-127-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 919,\n      "height": 640,\n      "image_sha256": "918e2c6cd29f24aec194e966ac0f9fb6f53f6e2bb848f9f1f8d09c3e30db3728",\n      "label_sha256": "fd3a26a9022f61cd9ae167c10d8e36ee408f749e1cdcbc5adefd2dfa06a79314",\n      "name": "calculus_-1279-_jpg.rf.e56193bf64d26f0d7be7ec676d57810e.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-1279-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 920,\n      "height": 640,\n      "image_sha256": "14f21bb3e735738c02934a6d279b6566a90cff9739380c8fe692466185dd9f74",\n      "label_sha256": "4e4401351d4ccc3c20fba71c246af5608aea4c7de191601f10a41ab49b44edcf",\n      "name": "calculus_-131-_jpg.rf.ce6ab0283683c0bc1f4067d291191d80.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-131-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 921,\n      "height": 640,\n      "image_sha256": "75345f65d7cdbfc7cacfb7894cb17a16af8c79d84b8646d617fecb35e52900e3",\n      "label_sha256": "bf48c11e02e1e1df58ee5816d68ddd6e47127c9b8b1f1df99db3deaf860ecaea",\n      "name": "calculus_-146-_jpg.rf.39d5222fc9d610ffda00b62cc2aa611f.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-146-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 922,\n      "height": 640,\n      "image_sha256": "5b90dd85f3803a4713c7552e74d2f96c7731caf691a8401d7074cdb9de68ea5f",\n      "label_sha256": "063d934dececcd184fbad68a8b8fb062824c9a6aab2979ad27258d2fcd72b578",\n      "name": "calculus_-148-_jpg.rf.8c869ad594448ed2aec8187c702b319e.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-148-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 922,\n      "height": 640,\n      "image_sha256": "3029c65e0171f6831990a958ea263ca7e231fd91400c30ad19fe738ff981e3a7",\n      "label_sha256": "26ae949b45caaad99bbbae3b540d47fa38aba87e03d5bd4ac3bbf5d03dd9c110",\n      "name": "calculus_-148-_jpg.rf.caf56df14466b2a75576d0852d9654f8.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-148-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 766,\n      "height": 640,\n      "image_sha256": "7dd1ab0de8af2b9e5f7197978a0ea54a3e107842952f0d4443679167b547f1bf",\n      "label_sha256": "c1d3c2c752e408eb1c6729231e789d106bc06f29cea9d72c544fcdbc220bed1b",\n      "name": "calculus_-17-_jpg.rf.cd1a8768369625ce703fe582557655ab.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-17-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 937,\n      "height": 640,\n      "image_sha256": "33e4a362789d931f6f94e488eac603a428729157222ebacd25ad95c8836c8f51",\n      "label_sha256": "b473660f5a28c42bb954bb821028ca74db97ccc7a98f456c6d3043c2d1acf953",\n      "name": "calculus_-189-_jpg.rf.eb387059cc09fb3ed90487d7f02c97ca.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-189-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 926,\n      "height": 640,\n      "image_sha256": "510d4f43235097c97e09bf4f0e4073d097e040d4f7ebdcfa8b853c7243bc54dc",\n      "label_sha256": "8d10c52c9ea22cc66bb1eff2c4425592c9879800d2ea199fc62b4869cc730858",\n      "name": "calculus_-197-_jpg.rf.0de8a2964c99dd2e781c37b6ba62ad86.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-197-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 699,\n      "height": 640,\n      "image_sha256": "9c5adf003c665672d4b78cc1b7d5afd18096e284553a25acf53652e611afb27b",\n      "label_sha256": "44987a2bee02dd6fd30d1230e74403a1590b0c97ab02a797f34bc57c1bdc8a94",\n      "name": "calculus_-205-_jpg.rf.fd9387dbdb67ec5c63ec3bec90703928.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-205-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 766,\n      "height": 640,\n      "image_sha256": "a045b655171bb5897971a6c5c11556605af8ab3b0e4447850e3898be0e82871d",\n      "label_sha256": "56e8aa209449cd79ccb01a2f4b868f2d17151923f62f5b9ee57a8223f4e42d67",\n      "name": "calculus_-211-_jpg.rf.39900dd93598aa53b6cab1396f9c7426.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-211-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 6,\n      "group": 766,\n      "height": 640,\n      "image_sha256": "7eb07b2c5efbca9fd8f05bbbcc6d2f062b8323aa4763d49c606596ec828fc7ff",\n      "label_sha256": "896f7e7d7f3ef7ad776756eefc804b38b84c6b9198f21d8187b78132a72dd6f2",\n      "name": "calculus_-211-_jpg.rf.52301ecc3a9c6baf86740d05de8e1d3f.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-211-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 930,\n      "height": 640,\n      "image_sha256": "e1ca8c2ab884d014f077bec548a85bdba2b479362425be2a1941a89ec8365d14",\n      "label_sha256": "2932138b5b673252af2a883aa37d6ea9435a26114e6da1a53e1c2a73f40ada25",\n      "name": "calculus_-212-_jpg.rf.0133434b34af64d3797f35ce581fff33.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-212-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1060,\n      "height": 640,\n      "image_sha256": "052895b31918eeb5bf02abcce2a4bcc01f7ec0cdf8bba4731e757ae56523221b",\n      "label_sha256": "3393a11c481b67b00887013d401b56b4f5487524eeed0c2e87c2b7aa1ccff9ec",\n      "name": "calculus_-214-_jpg.rf.cc5806b3ed48d34fb7935051208cfb1b.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-214-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 932,\n      "height": 640,\n      "image_sha256": "83e952373e741a331287cebd59f75bbfa9e443d7d3d680b3dc7b17d6b691e90b",\n      "label_sha256": "20f796e9a1031eb9c676d39e281c9b79d94d3b6552ab11683bb0a59b9b29e01c",\n      "name": "calculus_-216-_jpg.rf.88be07bb49f35d9997789696e34ce96c.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-216-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 933,\n      "height": 640,\n      "image_sha256": "1c748f3cd64514e2793bcc711ff4753b10ce4a5597f24f95d57f9d4336490e9a",\n      "label_sha256": "909cdfc5544b775b75c0167d93649c9b7cf452d8067041bc2f092329952b8b74",\n      "name": "calculus_-242-_jpg.rf.d020dcf5a492d8c89562d29dbf75be1e.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-242-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 1,\n      "group": 1045,\n      "height": 640,\n      "image_sha256": "4cafef64f1740b31e98780427b0a5fc9cdb34b4b07608a9e87a9a1f3609166e5",\n      "label_sha256": "9dc76b0f28e6d0f68e84ef83511d07fb4f38384edc729c7b42f845c99cbb216e",\n      "name": "calculus_-29-_jpg.rf.7d0a3dc46fd00aef9d349b1629281153.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-29-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 8,\n      "group": 922,\n      "height": 640,\n      "image_sha256": "5ab3e20b4517010c350d0d36310002f3585ad2d9ede16ba710078b35d38ef361",\n      "label_sha256": "40116ba5f0e8a95c7f34cbd1dc29b628ed495aff3da709c4e102b3e0b60d3377",\n      "name": "calculus_-292-_jpg.rf.4a68f915c31422c3d34ba9c08a80d550.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-292-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 937,\n      "height": 640,\n      "image_sha256": "75a7f726c9522a3f7dd6a7e945cd2e20bf4e243b841091ecd4dbe81f720a6010",\n      "label_sha256": "04eedf35f84d9363d2570e5f20aebe97c57310cd5237dd8a5b3e8bd6a274fa26",\n      "name": "calculus_-621-_jpg.rf.754ff49481044d509410d53478a51506.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-621-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 937,\n      "height": 640,\n      "image_sha256": "8df69c5af01a4a82dc539fb09f1bc827f9ccc0044edf6cf84f3b3ec3faa7b751",\n      "label_sha256": "43269b7666b2fd35fb0d1f369ad9fa9916903cbe034a564af4b9dfac587f5de0",\n      "name": "calculus_-621-_jpg.rf.9ada93611a2a1a0d146949956fa6a42a.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-621-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 735,\n      "height": 640,\n      "image_sha256": "16014f5dab554e07ebf289b7ad9392123fe75f6a04578f24e85fe3589d308708",\n      "label_sha256": "af9efb3f54e503fe5f3c45138e15565501e4cd476eda5ca8bcb4079813734473",\n      "name": "calculus_-626-_jpg.rf.7f4cb6f828e3d62667b75c1e39954064.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-626-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 11,\n      "group": 1080,\n      "height": 640,\n      "image_sha256": "81c5f7ebf1952e19ef67b5a4eaba6e3ec86bb3f64c8eb34a3c557e509b5ac84e",\n      "label_sha256": "dbe67d847df498d8f7d3d8b2452a9be3ac8e1b5c55148cbde14529fee4cd3354",\n      "name": "calculus_-640-_jpg.rf.86e70d8a1e0614c0eb6bdc86f31811b4.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-640-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 940,\n      "height": 640,\n      "image_sha256": "efde971a0b438f2a4a42d0694045740ddce52cbd44274b5546fe311b088910b9",\n      "label_sha256": "79fd76b510be26d807810bdf8d2a7dc41a06fb0f118b9b7aa3eb42b7c10177bd",\n      "name": "calculus_-68-_jpg.rf.082fa45b61effefeca4533d8a80457ea.jpg",\n      "original_split": "val",\n      "source_key": "calculus_-68-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 942,\n      "height": 640,\n      "image_sha256": "a8cc8a65cdaddeeea0c5a7037ca1421547385b0d4f188e0f83419674bb67e993",\n      "label_sha256": "c5a4b5285dca3a000df003a743807f9d7740a305db38d2a55cb4f629f6555696",\n      "name": "calculus_171_jpg.rf.bec06215ca5d4c22aa5fbcf41298f5f4.jpg",\n      "original_split": "val",\n      "source_key": "calculus_171_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 942,\n      "height": 640,\n      "image_sha256": "bd5dbadd2e93e3dbcb40e0a2356d37c7aafb2c04ff16f105d2c62bb482277781",\n      "label_sha256": "d4ae2eabac4e74a395f0d4a438568c813a7025fe3ada2ad081d141ebf9276d0f",\n      "name": "calculus_171_jpg.rf.c2df6d5d219b86140638073097330571.jpg",\n      "original_split": "val",\n      "source_key": "calculus_171_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 945,\n      "height": 640,\n      "image_sha256": "d729a76d08195db7a030028bdd4a4e690116c310d54d248b75431770dfe55197",\n      "label_sha256": "c84be8d9937fbd7fc92d763450f725685e10d5de1d12ab962333aecce92bf64d",\n      "name": "calculus_20230302_113922_jpg.rf.0a13bcfd99aca6c5b5cf736b659d34f5.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230302_113922_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 945,\n      "height": 640,\n      "image_sha256": "2bf84eaf0a1d60867e49947434d1790fa0ed7916dffc435fc562fe609534cf56",\n      "label_sha256": "c7d5ce37bee63ef83f3ef60e11e52e7a6eec4e302fa5a9574985572b0d6b7b4e",\n      "name": "calculus_20230302_113922_jpg.rf.a1be570267680c3bf723346f4bd4a313.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230302_113922_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 945,\n      "height": 640,\n      "image_sha256": "34dff733501262acfc8e41978e8d33faeaacb03e6cf15f45c94786aee2a3c210",\n      "label_sha256": "0a5ba87d8e7c1671d65fdeb6e38f492a24416ab8083d588293abde4e2d9c2c1d",\n      "name": "calculus_20230302_113922_jpg.rf.f9e19d87a23f362ceabcae9472a9ded0.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230302_113922_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 947,\n      "height": 640,\n      "image_sha256": "2e5c409eb9fbcf9440307343bb5d0d8d2bd3dfac55b7031618c3f9fffeb93907",\n      "label_sha256": "a8039908fd67822566aaf5c1d1fe7adda855023623dd502a5dfa8514142516b9",\n      "name": "calculus_20230315_150546_jpg.rf.1c51021acb43a38db7146c90a70b61c0.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230315_150546_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 947,\n      "height": 640,\n      "image_sha256": "5855c8f071bb08dea6affdc846ac492d51a4df138dbac8f42371619438c08b49",\n      "label_sha256": "e299b65a0c9b59ef2c283e1275ad4174e51d6a3e8598c751d70b6e9003f31f1f",\n      "name": "calculus_20230315_150546_jpg.rf.99dd5cca5a789bfb27adbed8f0a35318.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230315_150546_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 1084,\n      "height": 640,\n      "image_sha256": "b68afdee3cf7aa06cc4fac0a3045dd5ae54ed54d3e9aaa7bd954bd3e85669c63",\n      "label_sha256": "8af7c539c941630be82822dee875309ee79d2a83f32f6c8245a6b71e42a814f8",\n      "name": "calculus_20230328_133742_jpg.rf.026c8bdd30d6eac01093de46b10943f2.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133742_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 1084,\n      "height": 640,\n      "image_sha256": "105402ada178504d7bb4ecf96969778741a26dbce29182a8bf2e6a484fc13b19",\n      "label_sha256": "eb7bbd9b35d1c69202d70b23c38285212168e72423ee04a14d0fc97bf6f4e116",\n      "name": "calculus_20230328_133742_jpg.rf.3926ca6e4e2c772e773ad5e037ccfe0a.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133742_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 1084,\n      "height": 640,\n      "image_sha256": "57ecba7d7400c350fb36134e55077db735304a72f9ed2f7e923219d8c68a698a",\n      "label_sha256": "d6bdd98cd3006d8d264451acb6705e789d2261a75c6a2b62939f6d8e241e68df",\n      "name": "calculus_20230328_133742_jpg.rf.7972f1a7285f7c617e3a5885d0989727.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133742_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 1084,\n      "height": 640,\n      "image_sha256": "43f1c2a196237c76e529991ce48667ebce3115d2aa48840286d8b3cdc0bf21d0",\n      "label_sha256": "41885941ee3380bdf5714563890d160aca5401ec4b34e23a1000ba8fbf4c2ee2",\n      "name": "calculus_20230328_133742_jpg.rf.9a16618d9612e2f250f2c5e24607fd73.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133742_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 1084,\n      "height": 640,\n      "image_sha256": "3336f09e1e1f8bd2c2a77681cab6248e0742b89d2e0a48d5c52798e89678a681",\n      "label_sha256": "1f6d143be18fc29ea9d08ed3c78ee053be254da62064c022eb16ce2b6fbd6cc1",\n      "name": "calculus_20230328_133742_jpg.rf.aa695dbfec9d3b22337b2e80ec7b1e45.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133742_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 1084,\n      "height": 640,\n      "image_sha256": "20faa49a0f1645718615d7f5531e702a52365d9b3c6037b2541e6cb3e9f68f9e",\n      "label_sha256": "790de523b04887f4f24debc366f7bb78fc9999787f96606ddddcfbccce091f2b",\n      "name": "calculus_20230328_133742_jpg.rf.c1117351b85a1eab46894e1adb269108.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133742_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 967,\n      "height": 640,\n      "image_sha256": "064ca2c81d5dd2a43e7ef2f8d9e3149cfc758cee34af61f459047de736ad9b6a",\n      "label_sha256": "3f31a1d5350f62ff85b285ea4c32632396485508ff06ee6779653d4a7be5bcf6",\n      "name": "calculus_20230328_133746_jpg.rf.449bf7855c11599b93ed96e41314a963.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133746_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 967,\n      "height": 640,\n      "image_sha256": "95bc70cae7a2459e5c06bbab3088b81331d286daba11c0319c393670f6798a44",\n      "label_sha256": "cd2ded60015b167e08ea30c1c8bc0b7c60a80f7438ba6a9ed17b321377f6618b",\n      "name": "calculus_20230328_133746_jpg.rf.7bd89c39ea12af3b3f2cc30f608802d4.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133746_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 967,\n      "height": 640,\n      "image_sha256": "b7e91caad45a62dfa95adde8e8f0ce6d8b7f31d2ecf5339cf7d7bdc2fa06fe8c",\n      "label_sha256": "5e59ffe67b451b84ef98858e022a0608707e8ed738fbc78b084fc45cdbf6cb98",\n      "name": "calculus_20230328_133746_jpg.rf.cc7ab044ab2c5645407cb87d5d12edc3.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133746_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 967,\n      "height": 640,\n      "image_sha256": "604bf03dd9b05571256ed537db7d4da5169ff4581504a8bf61ab61a481e1c317",\n      "label_sha256": "107d047b610fd3a6602158e59c0ae84d212cade755d4ab78f8806bc8b7e5cd19",\n      "name": "calculus_20230328_133746_jpg.rf.d347b00eccc338324b653ff4e2716d3e.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133746_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 960,\n      "height": 640,\n      "image_sha256": "26e96b41e9e10f1ff58ba2fa14055fe8c874b3fde543d603ab91bae5b1ce4195",\n      "label_sha256": "c742195290f4db26d75c4bfd1147d795deae291b82f5fc86b0bab2c2716a6b2c",\n      "name": "calculus_20230328_133751_jpg.rf.014926b9229aaf75279490f5f187993d.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133751_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 960,\n      "height": 640,\n      "image_sha256": "b583bd269e08fa9b3b2ef732d97482e801db3e302702046dd4094bfb653929a5",\n      "label_sha256": "7a754d163ad194aa5f40454816a6ee1f8ff8b328e6712364be1f4c2f1523bb31",\n      "name": "calculus_20230328_133751_jpg.rf.0174bab4fba3ec58198c1a24402fc8a1.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133751_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 960,\n      "height": 640,\n      "image_sha256": "9e7140307c8062816dbfaf146b66095e439d9e1c54adaef77d075bd923157755",\n      "label_sha256": "cb2948d4a5ec9bddece3edace170590bb1952187750b6c62c6b4416d226aa1ac",\n      "name": "calculus_20230328_133751_jpg.rf.88f578bccc35940b5f6290b8c58876b6.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133751_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 966,\n      "height": 640,\n      "image_sha256": "116ee94b6534181b906f5ebefb5053b9fbef7650e29d9fd3c94a8abc52c88ba3",\n      "label_sha256": "324c60eaf4774a7d93aafb6d30907b47ead411f8e8228d2b6154ed3e80ea5788",\n      "name": "calculus_20230328_133757_jpg.rf.0bc2ae453b89104ba6b37d2a92dc42a3.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133757_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 966,\n      "height": 640,\n      "image_sha256": "59c2238b9e95d42350fa963b4dc1b00859b10194bd93b8dbad28480a657dda1a",\n      "label_sha256": "631e30d11a0d754a55d9f6609864e84d4cebec767b4ef497613f9913050d8f78",\n      "name": "calculus_20230328_133757_jpg.rf.0ece41b6bf7ead9d1013ee2b12689fa6.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133757_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 966,\n      "height": 640,\n      "image_sha256": "d2b9b6dec6e263623842e758bb7343380e5ab2df6ac168abcd531a50f1852cd9",\n      "label_sha256": "798457a218192116ac40990c3b2808e5c4266bce860fbb60be6918749613879c",\n      "name": "calculus_20230328_133757_jpg.rf.41cb7af29720f6503d7a9f596328c130.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133757_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 966,\n      "height": 640,\n      "image_sha256": "baa31010e7897fc0ce2706091c69a31ca45a411d850b1de97fb05a9b34511dbc",\n      "label_sha256": "0436c2553f1b224742ad20bc9099783197f8eb9b6048286ba2932cb1d9cd4911",\n      "name": "calculus_20230328_133757_jpg.rf.79fbe8c9f3d13a1440b5bc355fb6a95c.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133757_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 966,\n      "height": 640,\n      "image_sha256": "35898f8c95791b235d080343d515f449a7df929bdf62cc4076bce6dc92402a24",\n      "label_sha256": "b91fc54b6307a5f5df8923b7f430a402f0614c7c4e11e8627ae83edf352edf90",\n      "name": "calculus_20230328_133757_jpg.rf.99202c706bcc2e31b1dc67059f8e1fed.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133757_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 966,\n      "height": 640,\n      "image_sha256": "85040c76cb85aad5574620e92dedaa478a56c0b90765a8afd66998de1d37e1fa",\n      "label_sha256": "b91fc54b6307a5f5df8923b7f430a402f0614c7c4e11e8627ae83edf352edf90",\n      "name": "calculus_20230328_133757_jpg.rf.e3cdb680dca212acb8927977b1c29ed5.jpg",\n      "original_split": "val",\n      "source_key": "calculus_20230328_133757_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 967,\n      "height": 640,\n      "image_sha256": "9e1ee16310c762c6d0fd626cf80407d54f42feb61bc7fe4562ed014cb5e493f8",\n      "label_sha256": "dcfa2ee948377d7c6ed59cda4d638d55bb4c6a27165c849f604f2f31f3d6d656",\n      "name": "calculus_calculus-3-_jpg.rf.fd4868de5102701d8a1ad3c172bafafb.jpg",\n      "original_split": "val",\n      "source_key": "calculus_calculus-3-_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 10,\n      "group": 968,\n      "height": 640,\n      "image_sha256": "f51057db6b038091652553bc46e4524b4b5d1fbd3b06563484aab84261e4fee3",\n      "label_sha256": "867c5b14a1b79c491a0ec3877ad11f468de3444c9feaed78c0a22e0de23f9b1b",\n      "name": "calculus_calculus_jpg.rf.dacc6a42f6efeb4db3d24385aa89df12.jpg",\n      "original_split": "val",\n      "source_key": "calculus_calculus_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 970,\n      "height": 640,\n      "image_sha256": "ef0b0443f16645c11ee5e10326b7fdafb0a87d79f8de75f1a58643dae2bf5b06",\n      "label_sha256": "4e3071244b1c58aa95a88c5da6322c031604528aa0d74da595da1fa0d4c6df13",\n      "name": "calculus_download-6-_jpeg_jpg.rf.578bcbd2a120293f2123a83b4e854330.jpg",\n      "original_split": "val",\n      "source_key": "calculus_download-6-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 970,\n      "height": 640,\n      "image_sha256": "4e62f2ea5a84c258dc69f864e6526e4e16c9a0eeae7e594e31d783de9bc9c20f",\n      "label_sha256": "5847a8341a40f9b7898b3e5def24caefbacee807ec1414f8f9b51c3ed5a145b4",\n      "name": "calculus_download-6-_jpeg_jpg.rf.6f8aacd402066401f03396fbd306ed85.jpg",\n      "original_split": "val",\n      "source_key": "calculus_download-6-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 976,\n      "height": 640,\n      "image_sha256": "5cd848c5f52d5149ec32546ffac88216974401e30f6777a3c5e411ae11551071",\n      "label_sha256": "97058c9510a71d7ea0aa4d0b32cfaa03df6a8ee4c68e0d5aec94f50b736fc883",\n      "name": "calculus_download_jpeg_jpg.rf.235c438f5c257ce8106a92209feb2980.jpg",\n      "original_split": "val",\n      "source_key": "calculus_download_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 976,\n      "height": 640,\n      "image_sha256": "b8f09bd61e332b1fc28717f3ba1589b5ab73897e2b7c52bfde09a04ffe4c2d5e",\n      "label_sha256": "10f3d0dd5b67a02e6e592fffcf60a6de5e05e377c55f8baa75f098f4b6066ec5",\n      "name": "calculus_download_jpeg_jpg.rf.8cddccedd9a9f6a947cdd9f012ffdd97.jpg",\n      "original_split": "val",\n      "source_key": "calculus_download_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 976,\n      "height": 640,\n      "image_sha256": "8a9fe8cffe4a6ef75bfe53df9cc8f58708913bfbdfb63d5ea107986c8df2bf55",\n      "label_sha256": "e0b4445fb295bec86722aee5906e9ef0ad15a0a174a37a1470cd93046d3de07d",\n      "name": "calculus_download_jpeg_jpg.rf.ad1e380af58348c02595c82d439c05b8.jpg",\n      "original_split": "val",\n      "source_key": "calculus_download_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 976,\n      "height": 640,\n      "image_sha256": "85fab53fb3fcca6ee7cb2b32f551463eb9e36f68978bda633b62d98e18b56cf4",\n      "label_sha256": "5a2fb9f8b90797e54a0871ed586062cb411001323a0ff0bc9d647c53fe9e8e6d",\n      "name": "calculus_download_jpeg_jpg.rf.af6f2a444057e92fcf72c9325c98207b.jpg",\n      "original_split": "val",\n      "source_key": "calculus_download_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 976,\n      "height": 640,\n      "image_sha256": "ead61b59a551d11d8960961c4bb23f20c662212d77d7fdb37aad5ec855de37a6",\n      "label_sha256": "5458e0072ef81b6b1db39a59a7fa0cbf472096c52b12c23656bf693077bc009e",\n      "name": "calculus_download_jpeg_jpg.rf.d7c6809a098b850d759a16c3cebc2cc9.jpg",\n      "original_split": "val",\n      "source_key": "calculus_download_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 976,\n      "height": 640,\n      "image_sha256": "132a4c949fa073b0f40295ef7158ff8627539c76f732160f84f223f7684d1989",\n      "label_sha256": "75951fba2ed69e335442c69e8f224701497a08485a14ed5ed0c79739221411eb",\n      "name": "calculus_download_jpeg_jpg.rf.fb28eafdf973c905d13f2d10f5ae0610.jpg",\n      "original_split": "val",\n      "source_key": "calculus_download_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 978,\n      "height": 640,\n      "image_sha256": "f715276150d0cf3a607539c3b4539aac884bd4226bb7dd422f4331c144e9b798",\n      "label_sha256": "11c3032702966b8fea43f565675f3e669d06cd7c19d636f30a84d55ac94326be",\n      "name": "calculus_images-1-_jpeg_jpg.rf.ba42ea9ae0b6912ec5a0eedc1fd1b46d.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-1-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 978,\n      "height": 640,\n      "image_sha256": "4936baca43eaff79b3af02864cf170e19e3a16cb10f8197d8ba1210d2c551df6",\n      "label_sha256": "c0d13dddbf3ddf987e0ac1dc25c2e9e7c81234f12642dae711bf9c7804c7be42",\n      "name": "calculus_images-1-_jpeg_jpg.rf.dc590b459e482037da2c04c5230f21e8.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-1-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 981,\n      "height": 640,\n      "image_sha256": "3e4dd4e02eeeb790a964c1bcc26cf65251d652cf109a66faf4dd1b6146f6d13f",\n      "label_sha256": "df2a3be51e0999f91e88988f09c1c7c9204bf85f0843b6f573b3618a4132929a",\n      "name": "calculus_images-10-_jpeg_jpg.rf.89a8ec67c047d136e2736046d8376686.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-10-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 981,\n      "height": 640,\n      "image_sha256": "86a870f3ec93ea117791c59b506e27f06e7b02d03be05de8624607537b574f22",\n      "label_sha256": "145a8559d9c085fb883688c9872d58f76554b82d9fb6a30c8e3b5d427bd08a08",\n      "name": "calculus_images-10-_jpeg_jpg.rf.9b7c1ed22c8e6d68eea27b8d2937c156.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-10-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 981,\n      "height": 640,\n      "image_sha256": "2e0038d9e8492a036ae0394c4781bed2b7033369e8fbbb1c347be05857250000",\n      "label_sha256": "2b1a5007f6617dbd25fe7e8f35ed6d6c40e4d02072e800de8ac5fc1b0f968ef5",\n      "name": "calculus_images-10-_jpeg_jpg.rf.cb4f06560b773daa3828590584e20235.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-10-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 983,\n      "height": 640,\n      "image_sha256": "927c91a1fe2b0bea794c717c968bf2ee02101e0539c5db01575c8947f85f18ec",\n      "label_sha256": "44ba95be5462276b49b83448c81f1876e1911da2d58238248a3f4243be784736",\n      "name": "calculus_images-11-_jpeg_jpg.rf.67edd4d3a3cee56930672b63108ff691.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-11-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 983,\n      "height": 640,\n      "image_sha256": "b2f0be8a5b36100c9f7a39136e9a75e71c8fe9fe9f3809bc9f91d4bbeb572dc4",\n      "label_sha256": "2d8a4fedf38bb15c208148204fbab25c7fc09b02a843579c83ea58008df61d94",\n      "name": "calculus_images-11-_jpeg_jpg.rf.be0340a291a697f958baaaa983feabfa.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-11-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 986,\n      "height": 640,\n      "image_sha256": "98479de328cf3ac50b6ba3ed5e48e7dc903fa69765f85ab1fa11c0e882ce5054",\n      "label_sha256": "1f9d50b98c435cfe13d6b242e7ecd40294e2d0ef191b4258512f1a5019026751",\n      "name": "calculus_images-12-_jpeg_jpg.rf.22bcc42f6a29bbb98398cc083bba1eb4.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-12-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 986,\n      "height": 640,\n      "image_sha256": "98429854b00a922f97421698a02a988b7166bd40ccc9677b751d05453f34ee2e",\n      "label_sha256": "cbb386ac2a04f38c63b68244ae50714975e1a864e545a40e854e2150cfe7e307",\n      "name": "calculus_images-12-_jpeg_jpg.rf.8390d8615bae59e7b73cc103b32b10ef.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-12-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 4,\n      "group": 986,\n      "height": 640,\n      "image_sha256": "bb5eb69940de8ae65e81a3e594f95007272bf341af6da21aa4945c86f039bce7",\n      "label_sha256": "285e6589080be07a866ccb32502e00fa4408d9caee94b78351e7c7d650dc93e3",\n      "name": "calculus_images-12-_jpeg_jpg.rf.83c490911502980384221b058731ff7a.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-12-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 992,\n      "height": 640,\n      "image_sha256": "7ff935d1c38b8e72299bcf31457109ea1bdce9373d56cd02a6b08e4eb41cae9f",\n      "label_sha256": "1fb2fc95a714c27ec5757be209622f3dd54c045e99c1367fea95348b1009d26c",\n      "name": "calculus_images-2-_jpeg_jpg.rf.1b499816635d289f1701b7822a786536.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-2-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 992,\n      "height": 640,\n      "image_sha256": "441ed03128d98603c18eb409d3e094d727ec288e310732003b0bda598f79ed61",\n      "label_sha256": "c576991a4244511fa806410d56130ae976542d8dc0ac7d1295eee55592261eea",\n      "name": "calculus_images-2-_jpeg_jpg.rf.617ea4286a4de301ca496c2144ef3db0.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-2-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 992,\n      "height": 640,\n      "image_sha256": "93b0cad0c998641a066e820718c006e53828f2f739424e7b2d7a6b39a0e50380",\n      "label_sha256": "ea2289344efde67596aae7d86f34b661403ce8e78931372316c991af944d83d3",\n      "name": "calculus_images-2-_jpeg_jpg.rf.6aa311c127d4f30aa579a57ee895656a.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-2-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 992,\n      "height": 640,\n      "image_sha256": "682c78a76f58769fb00f5e3a02a69513c763dd301838a3e9603875bbe0de7824",\n      "label_sha256": "c2ebf9061460a709079c67a41503fb7b3a5868e902ea7e8cb9201346f5fedf1e",\n      "name": "calculus_images-2-_jpeg_jpg.rf.768be793300bc5819c6da7103c2ca00b.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-2-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 992,\n      "height": 640,\n      "image_sha256": "3b24b91c90119c125dc7a7fe66a5de219daefd87616eba27a6e18fdbd3e8fa2f",\n      "label_sha256": "f2cd3bdac52826a39c4ee93f55f0179fbd20a72f736b101e4115cbb18b50e9a8",\n      "name": "calculus_images-2-_jpeg_jpg.rf.988fe1b66a38a9c15ea600fc826bbd82.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-2-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 992,\n      "height": 640,\n      "image_sha256": "72191bc4e7affa0d919af3303a7f10e1ce729310edc65a3c62e3fbc44df55ed4",\n      "label_sha256": "339cf2989a46925af4d9d68d73d44c17370d36a447a2ff3821f34e961287f378",\n      "name": "calculus_images-2-_jpeg_jpg.rf.e23298dc8ae817f8446de22411a5aa89.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-2-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 993,\n      "height": 640,\n      "image_sha256": "dfd025a8dfee3ccdbd166a4a13058b36a3fb065f7976d064884024e114dc5028",\n      "label_sha256": "384905648551b83f838f0ac4449ac9dbcd7a602a00a2dc4aa4b856ba86b62f70",\n      "name": "calculus_images-3-_jpeg_jpg.rf.a0e3f0640b13b14dfe488b896a2eab05.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-3-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 998,\n      "height": 640,\n      "image_sha256": "f767a2a3fd7a32d43871cd665335b774922d8ae67bd0adfd83ad109cac714157",\n      "label_sha256": "7975a325c516bf9b375913129fa5620fac5defda74e24ff94f9e878736ffea06",\n      "name": "calculus_images-4-_jpeg_jpg.rf.1d6029e5a2468d4eb6353320134b8e9b.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-4-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 998,\n      "height": 640,\n      "image_sha256": "2ec4a16f9f5d2353743b3506213c7da46af52e77ad8941d492ca48e6bc14cd8a",\n      "label_sha256": "ab4dad5414f8050a63d8a90012827064087f6f8daae34e4bf6246fd77c7f76a1",\n      "name": "calculus_images-4-_jpeg_jpg.rf.3a7cd9278d969b1e87ad1d6cb3d2d71e.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-4-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 998,\n      "height": 640,\n      "image_sha256": "63c4910fcacd118fdbeee084394a158c447c2fbae85acd991b020125d8282592",\n      "label_sha256": "33056b110d11b92586bc62ead1796288154fbf43ba81556c31ecbd3274f09035",\n      "name": "calculus_images-4-_jpeg_jpg.rf.6d8eef93ab0f6ef30a798b30b571c013.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-4-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 998,\n      "height": 640,\n      "image_sha256": "4c1fbd91ea874b5acc3c6fae57d404d19673063ea10e0f1aecfaf0ab510fa4f3",\n      "label_sha256": "bbd59e2703439a2413d133a021839ed48d81e39c626fcf615c6522abc23e76ce",\n      "name": "calculus_images-4-_jpeg_jpg.rf.aed9531cd1c9536c51e95d77b40a583f.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-4-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 998,\n      "height": 640,\n      "image_sha256": "bfc70d6dbfc21977e09b537781f07cf9b43e2f04b9af37cb8951fbbfbad3e6c2",\n      "label_sha256": "8f01a4acf02938fc6e3a7e4564f3fc1d31f21c25bcefb719384d8908a2811c91",\n      "name": "calculus_images-4-_jpeg_jpg.rf.d61701789047b86aa60fe3b60434a4af.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-4-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1000,\n      "height": 640,\n      "image_sha256": "86d556b17c4379014a04df9e75d19f0617c64992da6f3af21391e5580ca150e5",\n      "label_sha256": "47cb9078c0bc1a96dc43c1dc435ae6e4121e48a35c263b8ef5b3860971ee24cf",\n      "name": "calculus_images-5-_jpeg_jpg.rf.b6b51c25452016771bd8f2ccd23e829b.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-5-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1000,\n      "height": 640,\n      "image_sha256": "92a81437d2f6244e112f12abf8be086eb6e1fbb36fd1da7990adf14a0ef427d9",\n      "label_sha256": "73db038dd538a414b1a19b42591bbdecd33c13493d4512d94dff4691e02d505b",\n      "name": "calculus_images-5-_jpeg_jpg.rf.fd603f17b2d6698e0bf8a95230f03152.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-5-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1002,\n      "height": 640,\n      "image_sha256": "d0fa090bc71be6ef40ea6a93f42ade51b47ff9fe56ade062e76645f92279d3f7",\n      "label_sha256": "98a0417e90a22091f8f7a31dc4186caaca1b6cc38df1d68d34bfd77e6be5f119",\n      "name": "calculus_images-6-_jpeg_jpg.rf.61cb75353162054d021787fbf65be083.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-6-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 3,\n      "group": 1002,\n      "height": 640,\n      "image_sha256": "aa63c66369fb5e7faba0b76171b67ee35d025a79bd07277738c054a258ddaf5c",\n      "label_sha256": "46a6bf6e37b12c6c2b90e7bc0298c20070882216fc889405b2fa4a609ddde522",\n      "name": "calculus_images-6-_jpeg_jpg.rf.66b4028488bba84026a545335adf06ee.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-6-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 2,\n      "group": 1003,\n      "height": 640,\n      "image_sha256": "678a150cc2f7591e62c1ead608bae00df72d262916eeef688e2d1388996aefd3",\n      "label_sha256": "358f678ceead3400dc275d67c6a11d60580505aaecc9b8a3bcdb0b7223f9f985",\n      "name": "calculus_images-8-_jpeg_jpg.rf.920172ff320f480caa23df54290fe0c8.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-8-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1007,\n      "height": 640,\n      "image_sha256": "efd2e3943ce4f87b9f99056434456c5b977a645456f687c69a0bc3d244c9ff3f",\n      "label_sha256": "b32b8fb63d03d4df76de78034db3b1db05126fe5e600077287952e949245ae3d",\n      "name": "calculus_images-9-_jpeg_jpg.rf.1e94bcb1aece3c6b3f128905516a687a.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-9-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1007,\n      "height": 640,\n      "image_sha256": "38e5f687ca1d902ee7b493f74b7c7503c50f2a6a2ff3ab4c3671d82a2872525c",\n      "label_sha256": "905c22ed3215433b83bc7ef674ad8a3b36a684f738f0b31123eccc5bdbd866a7",\n      "name": "calculus_images-9-_jpeg_jpg.rf.6e1743e9e6dd53399296794ae9f35861.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-9-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1007,\n      "height": 640,\n      "image_sha256": "83f3769d833b297efd0deda16215a3b3b070ba08128bbf8f5f0d4c5348816a33",\n      "label_sha256": "a138fe0981f5a28b15c8c4907da1d3297d414a4bfcee230146c05cbe5acacb95",\n      "name": "calculus_images-9-_jpeg_jpg.rf.92a5c9ec378e1a26949e91b576fdeeeb.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-9-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 5,\n      "group": 1007,\n      "height": 640,\n      "image_sha256": "7f80f9e4122488403021e94ca9c9056eb9477dbcbdd707735d3e617b6810b70a",\n      "label_sha256": "985569540317852528024f62b4df5884513ed120d472a684e81f02b9ff492263",\n      "name": "calculus_images-9-_jpeg_jpg.rf.b2cf0a50ae8fd9ccd80dea0855295495.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images-9-_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 7,\n      "group": 1008,\n      "height": 640,\n      "image_sha256": "dbc21570f919764cf078d1dee18bd79b9706aeecd63d9383004a29f3224c1a1d",\n      "label_sha256": "1f6f3c757dda07b07e77c3cfa9fb7ea79aad3a8e9d5eb77068803556dcffcf45",\n      "name": "calculus_images_jpeg_jpg.rf.e2850fdba854bbec2fde90f49dd6cc3c.jpg",\n      "original_split": "val",\n      "source_key": "calculus_images_jpeg_jpg",\n      "split": "val",\n      "width": 640\n    },\n    {\n      "boxes": 9,\n      "group": 1009,\n      "height": 405,\n      "image_sha256": "5e239cf5a4657e25e5c088d03c87b0de3050657a4bde0e4b47a9b3ffcf8f00e0",\n      "label_sha256": "562a7852553f43ece080de75821b489ad6874ca0b01148df1c20aa4f1bd231fb",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1011-_jpg.rf.7969d23dda9490f449fa12d72f8fa2a6.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1011-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 1010,\n      "height": 402,\n      "image_sha256": "69b9e3127c08ab96e4b04bf41007f227a3884f8fa73e7c72b97ee4ff359e4ccb",\n      "label_sha256": "b833d39fc761578e0fd1c1f8d69a33b2d8ff6d255c0239bbdcf7137b5922f028",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1034-_jpg.rf.952787befa6cc8d8d1063fb950fafb35.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1034-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 1011,\n      "height": 319,\n      "image_sha256": "5962c869c65e1355e7c26016df8b9088d20c34de144a35f2589357e336699821",\n      "label_sha256": "44a1462982fcaa3520ba9bcb5460841aae09edc65201b5907e00de730bd77d3d",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1038-_jpg.rf.e1369349f4493c15f8ac785ca41c6940.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1038-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 1012,\n      "height": 204,\n      "image_sha256": "285327930a80797fb276fcece275cfda70569808e1f395ca582f5c787d7ea8ca",\n      "label_sha256": "70c649fc28b2af496f3f13375626f3a61ff78b93f7ca079942d185e74685b04e",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1055-_jpg.rf.7d25f22195a5fbe8448d58c0760de4b7.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1055-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 1013,\n      "height": 293,\n      "image_sha256": "1b664cfa6f0f96ab6ac3d9baac3fba5b3d4efa45d3b838a4e2a5b8d99849f309",\n      "label_sha256": "72b99b2699551d8bf9f530c04b2984f45fa9ef54d6f077eb972582633f48b9ac",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1118-_jpg.rf.339b508f7034e51b9f3d678fb552de1c.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1118-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 881,\n      "height": 372,\n      "image_sha256": "515b2b3c1181f27ae4112271071c82d7aa8c55bd971fb8c635695df89a73608c",\n      "label_sha256": "e4a36e1fefb70c93ccd090d313081ab7b1c2343285970eabb71af3ea2984bb93",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1130-_jpg.rf.a7e7d82ae379a787dbd1c38b1635d8f8.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1130-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 1015,\n      "height": 409,\n      "image_sha256": "6e91c1d7a053c9480d697b9e00ed610ad7eb8188cbd11e451825abcdb610f556",\n      "label_sha256": "d10ac1f7ddc3335cedbcbdf678bfee39c061c29806dca6d67250f54f52963ed0",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1147-_jpg.rf.331ca2cee783b315e6835034af212ee9.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1147-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 1016,\n      "height": 408,\n      "image_sha256": "9d9432e583147f0e47105f1ab807bd5ba153ddd7a6a845182298b99ed71cefce",\n      "label_sha256": "214abe45c0523a23d5565856c7bf304c119963434fdce6e7fbb9fed9a9297da6",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1154-_jpg.rf.2f3240a7d685ca59f789e5dab88e6042.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1154-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 1017,\n      "height": 192,\n      "image_sha256": "2743f5c1f49e595b41a68bb55eaedcaf19b2ce1e4e88f1fb022c1ab1dd6942f8",\n      "label_sha256": "af00826bf496d8fb25a3b658199e681d9eb2f3d8503f4fc0ffcf81680cd79838",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1157-_jpg.rf.0553142bd6ebe635996419c9db2267ca.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1157-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 1018,\n      "height": 408,\n      "image_sha256": "a6340046e73731be5f3d146f69da02bce1703f32724cc5f0ffcba2aa55ab0c00",\n      "label_sha256": "fc644f0911b2aecc4389905696323c06b0098918319543728e5d6a06914679c3",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1158-_jpg.rf.f9662eea873dfa8f9123b66108b214eb.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1158-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 1044,\n      "height": 402,\n      "image_sha256": "686dd8d3152b0c3c508bc9b3e1adec8524d8070383336a3ec3a58fb9fa586503",\n      "label_sha256": "3e46ba0d6857bc02751f186ea72fb806b3afc804322a43c1f1dc977cb7a57dc1",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1178-_jpg.rf.e83c7d093c602bbca589a4ccd5c39591.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1178-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 18,\n      "group": 1020,\n      "height": 408,\n      "image_sha256": "f83881ac54900100c65910467b652196018ff7d9bdb1663dd7faf6ba67041902",\n      "label_sha256": "a34682ace93f3dc239676f0a5668e03172cc365de7fe96a1829074c8c1e58350",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1189-_jpg.rf.316418b6ab087900e1e7f8eb511618df.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1189-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 1048,\n      "height": 263,\n      "image_sha256": "1f0c4874b7d5bb2776668a9055832aeae2ac1aad813655592b3e7906c43e027e",\n      "label_sha256": "50e3f694469300452c44d7e69f6f0f5d4b5425f9e5441c248bded526deea8f1d",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1190-_jpg.rf.b599f119910977752ef4ad43d91f4ad5.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1190-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 46,\n      "group": 1051,\n      "height": 204,\n      "image_sha256": "cff13048ef69811d56da06a597e3e0f8339ae6fec79e57c5a9200843416558d0",\n      "label_sha256": "c43687570b845c84a1034ba6af1cc5404b678061411ddfa1d86c0e68186eca36",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1199-_jpg.rf.0d33f588f08b0dcb2d4237d6291f7f0f.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1199-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 1023,\n      "height": 398,\n      "image_sha256": "1f4db218074574f2b2b5011e4d61e7ef87a966538303c153acc4b56f01547d2b",\n      "label_sha256": "623655b49be92057ac479afef672a77c8953fbe3cb842903ce0fa4fc31f207d8",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1200-_jpg.rf.26824d60b497a9bf3b8a184f42660898.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1200-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 36,\n      "group": 1024,\n      "height": 169,\n      "image_sha256": "732acf451886f25e4576ba629bd561b59ed7d2a1383447c239765f8ea1e9c948",\n      "label_sha256": "d5e82724e04b45642035eefe3ac74d6ac3a692ae6e1539f59d97a76c5a319364",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1201-_jpg.rf.314bfb6a827f7c233b7a03a0c4816f72.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1201-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 1054,\n      "height": 347,\n      "image_sha256": "ddebe42f257a30c9b69e6adf61ec118ae7c890bc4b750c2d4670204145d3a35d",\n      "label_sha256": "41e85b0464d910e88f68394b25cb2d0de5446373a9cb2cb7336e87ede4a1599b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1226-_jpg.rf.a1035954ef9cf45b58a713312a61ac61.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1226-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 1026,\n      "height": 360,\n      "image_sha256": "f90e8449ce2ea3ba8fdd79bb1013efd1028b352b3e64e94c90e075d7cfb13428",\n      "label_sha256": "5c13b026fd0d339321a6eb071cd85542c90a674cbf1be0112f41c241e3611df4",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1233-_jpg.rf.9635d8a302843e71422c3d41f5c23ddc.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1233-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 17,\n      "group": 747,\n      "height": 408,\n      "image_sha256": "e8e4dd5b87cdc98186578d2f63abd016ab61fb78b81556406abb70be46613c22",\n      "label_sha256": "ff994bd612df4dcd17fe70b23d3397467320356d9b5c5afe5d22321e01e8ebde",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1254-_jpg.rf.ec9e62dfa0c7616fe9ab8c27422cf86d.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1254-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 17,\n      "group": 1028,\n      "height": 397,\n      "image_sha256": "e141cea7af7cfc0fe9c8af22c3f82a85da4c14e2455affd039879b6af6475b0d",\n      "label_sha256": "54e1a6742afaa5f3a46c855ae3fc1e33359a4d00767cc7aae8c23ca8da3030cf",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1277-_jpg.rf.a8d2c2fa10345a1cc178849b8880b039.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1277-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 1041,\n      "height": 325,\n      "image_sha256": "55ac2385b8c041fe1266cbbe9c8bcec4384447e8356e9d98a3324e903ad55ab4",\n      "label_sha256": "a95d63433e18f812c603ab0545143d22e98075592b5e8eedcdf4b064c8e4663b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-1292-_jpg.rf.df2c255cd15190576f05fc96c7c1ae75.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-1292-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 1030,\n      "height": 459,\n      "image_sha256": "cc9f45d3dffa7fe2bbac4ac954b96a9d5c4c002a227a9670be38c8df85ae2376",\n      "label_sha256": "de1228a48d8edcd34817848689d6121df96ac7ea8352158a0795545ed7be5b82",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-152-_jpg.rf.b3321d3453278cb6112a8be485f63c6a.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-152-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 1031,\n      "height": 195,\n      "image_sha256": "69c47dcec8796aff02dbbd63847316333461155487796cbdad3a64da3dbdbbff",\n      "label_sha256": "094c4337f58f070f29ce291a64132795ae1d9e99400178ed9f3c459185c1956e",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-153-_jpg.rf.1dcf819f2ac1be3859505e55de3ce283.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-153-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 3,\n      "group": 1037,\n      "height": 408,\n      "image_sha256": "6c841d80f8b0a82eac463e0cce82474e71dfbebc7b7ca96376b218da4abd0977",\n      "label_sha256": "4c2d0ed1ef02e8413bba5f77b1b6197ee7f031248f874ee2d85169effd9c8efe",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-166-_jpg.rf.aaf374dec797ecd8d404f548582163b0.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-166-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 1033,\n      "height": 408,\n      "image_sha256": "0ab394db83d8eb729359876795a30004408f91938c786d4291f904f90ea467c9",\n      "label_sha256": "e69011d3f3151ec8d01cfab93cbbf1515e1d61e1afeca535f585919b25cf3172",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-168-_jpg.rf.c9a110dc154a863857818241fab964b7.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-168-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 1034,\n      "height": 408,\n      "image_sha256": "7905883adb26ac12c2b35b7bb83b0ac8782ad5fa4de8d67ddbd977bd2e3a4332",\n      "label_sha256": "a064a0585c1185a07dea9371440f218ca522fea57bee2c37a54599bc425e6377",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-179-_jpg.rf.62593cf772f458637b4aaee66743ec97.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-179-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 20,\n      "group": 1035,\n      "height": 407,\n      "image_sha256": "09e2bebc62ad0a9720f35a3b59c72e1e01852dac7dea2fb0f425f1d6c842e96a",\n      "label_sha256": "e9e7685d433cc5f40a44561089e31fb8c272a87b1b9d1314fe020f28c5aa4253",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-180-_jpg.rf.9ed2f3854c5a534734b103828453a5c4.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-180-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 1036,\n      "height": 398,\n      "image_sha256": "12b75df535f07b6148426404d3c23435362f4ea31d88284713b18a4cb8f5ac97",\n      "label_sha256": "1acc904d3973176ef9debfe12ad7018689388cefa353af7f52074d3232f66df9",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-192-_jpg.rf.d1fe83ae6905998915d2bf8c8b2bdf06.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-192-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 1037,\n      "height": 408,\n      "image_sha256": "a29fddf971289c8f6601ca0edc9ce7236af18f078af38998d8797316fbba935a",\n      "label_sha256": "27b1867d835c3089e53b9215f63705ed1e14409b58f9b3836f5ae21351ffbefa",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-22-_jpg.rf.123a6cf0f4b66fe3ad9a71635d7c36e7.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-22-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 1038,\n      "height": 408,\n      "image_sha256": "99ac6f25606b3a83e7ca0672fe1096723193c048519b4a6fa246f5e67bf4b7d2",\n      "label_sha256": "6baadf6f74de3a135ad5f659c8baad957167b8021e575da3e60404e2fe806845",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-235-_jpg.rf.c16b9198cc78306a078bded37fce4670.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-235-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 1039,\n      "height": 204,\n      "image_sha256": "d461991070d7dcd9ba78819b8970596ca62aa318132e32970aab9be08df60dac",\n      "label_sha256": "e050d79a77895275e2a83dc5991b081627a985f969444c0ac2fb5abc2130ed0f",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-247-_jpg.rf.5c43ad7f6d2dd711c73aafd6eda9e1dc.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-247-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 1040,\n      "height": 459,\n      "image_sha256": "683eb6fc3cc38764fcdb0e9317aa4a366246d0ed2c8558836acce2aea1a5c0bd",\n      "label_sha256": "ef680a2299216e3417c72870c5170f44d0017d879cd20de2ea6d61992ebe695e",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-274-_jpg.rf.de35a380efbc3530d438c0fdda0596ef.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-274-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 3,\n      "group": 1041,\n      "height": 325,\n      "image_sha256": "fc4e85d7128cc7e9b4071fb76244a33165fc8c7ede2187cb024bcec41be8656c",\n      "label_sha256": "e264813601fc9fde4f50fe4edc515b5d13ea769a8172b45383a236df6d2b407c",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-284-_jpg.rf.e93b8e706b297a7c1f6107d229ade6bc.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-284-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 1042,\n      "height": 408,\n      "image_sha256": "c83d80bd13fa4b5cedd0d62fbbc481709b80afe0123ad71025571c77bd2ad8d4",\n      "label_sha256": "af93af7489373e99333b4c7e89932412ad1ae149d04426ecf5f51a6f2828581c",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-300-_jpg.rf.5ad5c475a69ce1b1a6f5131f866245be.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-300-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 1043,\n      "height": 255,\n      "image_sha256": "80c8f5b1a31bacc34285a164e47e4558708134842b36effb68e4ac3bc49b88b3",\n      "label_sha256": "d2f3cc998b5aa24da92e29e9c3ef6fcdc4d9734cd0b104ac375e2a35e7ffa40b",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-308-_jpg.rf.d967638b3ef26509b86b27f47bf2c2fc.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-308-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 1044,\n      "height": 402,\n      "image_sha256": "c72c7fa14cd4541809366133aa54a0377ad87e22bf678aa4cfe7fed8232419dc",\n      "label_sha256": "59f0faeddad2e610251efec03da492a6f984e4173bd59bd7acf4b8233c6c7349",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-314-_jpg.rf.0cbfa8e8529f4af926f433922791c637.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-314-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 1045,\n      "height": 340,\n      "image_sha256": "185bd1c2417def70b5efdf718261a511e80126013403d0d4d0b7136b9fb00d13",\n      "label_sha256": "76d34511733d364690b0902aed97eab1e08dbdefc2612c7d820595d541430d50",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-317-_jpg.rf.40e9fcb1636d81ff91696da90b70e5d7.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-317-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 25,\n      "group": 1046,\n      "height": 609,\n      "image_sha256": "761e6a8fe273b874eb6144191b1a9005645ec307c29a1f5d413b189036ba051f",\n      "label_sha256": "fb9bf2becd545b5f09248bb1323b6be8a6eb8bd04ad0d3dca728cbf446d260be",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-319-_jpg.rf.5979ff294a141161fafdb472abb20f2a.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-319-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 16,\n      "group": 1047,\n      "height": 297,\n      "image_sha256": "4255ef2f7958c8727c0257057e16ed168ea4aa2dc8089c372fcc471980a7a153",\n      "label_sha256": "f46bf2a00728759a73e01e578a762995bd3e1448f0a47d07f4b8f0e6c219f4c4",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-321-_jpg.rf.92fa3a9b5118c4457080b9cc8e1b7282.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-321-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 4,\n      "group": 1048,\n      "height": 263,\n      "image_sha256": "8289307e0a0c70214f1e0a5fe6fff73f855b507a1c7c508347b974d6e785b427",\n      "label_sha256": "611450e5a25947334f82dbb25dc668caff94ba2e810e95b705bcc7951b8b0708",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-326-_jpg.rf.34abdb7f84ca8edb914aff946b90b79c.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-326-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 15,\n      "group": 1049,\n      "height": 125,\n      "image_sha256": "1b7c1c068af4d81a1e6ac966587a220fac36de09c27897361cf56b3eb90df602",\n      "label_sha256": "03e98ea169f7c9d185f74b6e2ee01c936d9c031b6122df85450cefbe6a1290ee",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-332-_jpg.rf.d95fe17eea3122bf5789d8621e1fb197.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-332-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 14,\n      "group": 937,\n      "height": 408,\n      "image_sha256": "3e6581b5982924124eaff6ea900c2ef2f9457c8a99d24e05968f63b2aea73184",\n      "label_sha256": "b0b41ac24922ac1f6851fabbe1bbe359bea5bf2c62dcee4d1e3237e830fe1618",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-333-_jpg.rf.9732388c3e18eeacb3604b0b70bdec1f.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-333-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 28,\n      "group": 1051,\n      "height": 204,\n      "image_sha256": "fb263a43fe8fde1a28b371811dc68f2c595e30f7e56a0958fccdfc42d2c6b454",\n      "label_sha256": "607e64577d36c8472f02458dc37777d493fdbc61e3015391991ba04c2470f754",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-335-_jpg.rf.a02cc38b22d78198a6fe0bc6f3f875a6.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-335-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 1052,\n      "height": 408,\n      "image_sha256": "896b250922dae359e7831e552db02d8c972594047d21944a8975f351c12d7a12",\n      "label_sha256": "f324c7cdf109c7dea4e394c3a3a1b768b42295095eaacb4c6bcbfde96f0bbcba",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-339-_jpg.rf.91cce3f621511dc6aac69a130d661755.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-339-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 24,\n      "group": 1060,\n      "height": 406,\n      "image_sha256": "cc955d2afccaeb700dc97f67c5eeeb36b710becec7214fa0fbdd5f5fe1c865ae",\n      "label_sha256": "af9c46f2365603d79f7be86094d74146bc4bc22d78f9bfcde358d7bbe2232c90",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-358-_jpg.rf.2db88d9f4c40cc2eecd58c99648ff297.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-358-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 1054,\n      "height": 347,\n      "image_sha256": "1ad1313878efc27551caa6a8b27d8e60084c6bef0149bb9ff32a1ec1229efc7b",\n      "label_sha256": "e01b26946804c7572422c5417b1131b98b3bea54bae29126273ee0dc014cdf15",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-362-_jpg.rf.1975bbc6de4d711e0dbb12928f60769a.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-362-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 1055,\n      "height": 455,\n      "image_sha256": "efd6f0da5599e74e75491870e72984e5a2dd6feb5b44659f4b13661032d9c5b4",\n      "label_sha256": "e6393f7f34bbffcf62ad7059218ba9a43d1ba9cab6f6c56b82bfdbcc9633dbe5",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-43-_jpg.rf.61fb84e6b925e9f116569c2605d72b61.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-43-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 1056,\n      "height": 408,\n      "image_sha256": "a2b49525f4a4080aa8b1d027fbf4a79261aab9fcbf6483db65de4f7d1896a417",\n      "label_sha256": "cd2685c1ecdb772ad78fc9d1a46a6cd1299e037eb93462eae816f5184dbbb441",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-467-_jpg.rf.70e8a4ec8284f65cb34aa5a3fb7af718.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-467-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 1057,\n      "height": 455,\n      "image_sha256": "9675075859f1d0e219783f28ddad462836f0005029607e2c00f0c30d9da32318",\n      "label_sha256": "427d29ea19ccad896d5138a9a78d14d78b4ddea66206c7ea408fd90c9b900bde",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-475-_jpg.rf.56c2c7c1fd066d24271addc6d5e5ec11.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-475-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 1058,\n      "height": 398,\n      "image_sha256": "76b710e711cc834a33ec670fbc817271c1cd22f2529006996887951d2f2ba9fe",\n      "label_sha256": "bb9a1795f39515f7397e8282c30124f5f39d932a68b187bb90dd8aac6eb8dbfd",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-480-_jpg.rf.a63a4fc8c41cf093f000160cc098bca7.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-480-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 24,\n      "group": 1059,\n      "height": 151,\n      "image_sha256": "ab05df6d1bcc0355881dbe08c826afcade0748bbf56e333dcc6218b47f8e51bb",\n      "label_sha256": "517b713a20ecaf124c153dfc1cc38715ba83acf4fa0a9b097aae337ac3928f07",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-485-_jpg.rf.ccb601192725326a50e3e6d176259860.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-485-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 13,\n      "group": 1060,\n      "height": 406,\n      "image_sha256": "d6c90603a8b1b0dec44b2e9239f9cd3dca05b563d38a9d9ad1567d29bfab3095",\n      "label_sha256": "62c4bdc71e90000bd89934a8ac83f9f160148f05ee2d3835a6f7f4f2a66fc177",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-502-_jpg.rf.970ccf73d58f20c0d0e347332124231c.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-502-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 1061,\n      "height": 408,\n      "image_sha256": "ff39fd80bcd328fb57e379898bbce2635e39d4959475b9c244a32c81e917516b",\n      "label_sha256": "57fca386907894204fc4aedba8a41e482de859954b35d1b68992ede0e177a1f3",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-544-_jpg.rf.e1bc59efe6a7ed7533b9406f99b10a30.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-544-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 17,\n      "group": 933,\n      "height": 408,\n      "image_sha256": "081c38ac2ea89d9d1a6780d3287d74032274234101c28e7e8515ff7ac05cf0c6",\n      "label_sha256": "468a7cf89dde10b23797354541e56e798cc1074402d5926b8d0f3cf7c99f73f5",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-551-_jpg.rf.0dcfb2bd135ccad45401ca3aef00fd88.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-551-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 1063,\n      "height": 375,\n      "image_sha256": "29a8ebeae17c069833f503cda1b53317cb872724d25aea59a015de92890a35df",\n      "label_sha256": "c5b64934fe3cce7933af290f9d2cffe57b030b6e538d79fad32c0851e882d581",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-590-_jpg.rf.26c7d9846db377adee6afb3220f47235.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-590-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 6,\n      "group": 1064,\n      "height": 408,\n      "image_sha256": "dccba5fd3ccb6e913704ab5ce2d09cdbe915024ee51ccd97e9721e97595c02bf",\n      "label_sha256": "9cad2aeda1b5176946c60e95fc2bc0a8649b5189ef5fa4011c25e2d412896f1c",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-613-_jpg.rf.704294c66cdd02f852dc6dde961b701a.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-613-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 1065,\n      "height": 424,\n      "image_sha256": "92f0e6b376b3696d64f8dec63f1022f7fa71079a62f30f431044a4099f6cb6d4",\n      "label_sha256": "5cf42a2f0a19dc1a42ff2255f1f71edf9cd850efd023ccb60925a6c30989b22f",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-658-_jpg.rf.862819f1b8123c6835bc2d583938018c.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-658-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 1066,\n      "height": 283,\n      "image_sha256": "8ce2f6b58e498f759ba16e4018a70c6bda6c9bc7847c3241d7996ca367cf0448",\n      "label_sha256": "349b28ff25121c3918614f4ec7c0c86cc90e7ba36b273f29bc1acc3da4c4d7cd",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-663-_jpg.rf.7158c08882cf086891e9f8f1cb43debf.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-663-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 23,\n      "group": 766,\n      "height": 317,\n      "image_sha256": "fcf3527a3c231c97cdd8e6031c2ad1b287a0f1e2112e516d0f4438cd1ee1d8ff",\n      "label_sha256": "5cd55877af0de647755b1533d1928110a3c5e02d8ce02eac60e73ab3955f4b09",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-67-_jpg.rf.7f4fb90f7a43672755d46ee45328e712.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-67-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 1068,\n      "height": 408,\n      "image_sha256": "60d6bd9a65eb0af0fa94fa9866576c130779943f04bb7084204051cdabc1d963",\n      "label_sha256": "786faad8d857f427e7b5cfb378a47538c94b5442eafdace6effc3a594a9a432d",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-721-_jpg.rf.f664bea3e32700927ff0f60399b41a99.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-721-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 2,\n      "group": 1069,\n      "height": 340,\n      "image_sha256": "24f44c516742ac07a28d1b4e86d1ec0f697428acb1ff890dde6f47966dcfe712",\n      "label_sha256": "83fab9da1f086f77888b58515066082a9547f818f9367cda65e7acd5dd2727ed",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-749-_jpg.rf.b24fe7514d37ac1b9d3a4c04f25c5e75.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-749-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 1070,\n      "height": 420,\n      "image_sha256": "1d6a0c4428105d04ad1d7f1ecc4698ae6784ecf2fe69fef235adcf91c624920f",\n      "label_sha256": "671b3638890037f4ee455c89304047a9b7197e15f855a91048e8164d16f73aa5",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-803-_jpg.rf.b799a8a90d2eb430a3a82189518b0ca4.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-803-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 7,\n      "group": 1071,\n      "height": 409,\n      "image_sha256": "2b92299b91f2c7747061fca761b5455d7b10562d915797669aac7e93bb576e33",\n      "label_sha256": "0c740227df0fe1079b6325aebab56197273ab2ab23e42298bd3f6a995de8317e",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-821-_jpg.rf.6c365c1cce338b4f95ef14268fb210b9.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-821-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 18,\n      "group": 1072,\n      "height": 255,\n      "image_sha256": "093d9514d3e9c934b83a3ec698618a17d96a3bb70e951dd7f6d4bfb16d8de027",\n      "label_sha256": "cdf7569c799b672f6a9c7cc0cc5df24e01c0480071c3a443bb900ff57b5d4d44",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-884-_jpg.rf.0e209e9eab29ba34289cd7b458e273ca.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-884-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 8,\n      "group": 1073,\n      "height": 408,\n      "image_sha256": "157db2c9c0f0ee6645024c65f0f71e5edd7780aa384f77ed806ded30057b6e5d",\n      "label_sha256": "f27eadbe7109f7a626fc011cb32f48e7db617516c56194fb723aa7934aeca3f7",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-887-_jpg.rf.2896b1746e0db6676224496c18cad810.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-887-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 11,\n      "group": 1074,\n      "height": 408,\n      "image_sha256": "ae2240149f85cb66b1254c7d47bba6b50298daf8ba935b60b3e9c340f043f79e",\n      "label_sha256": "7eca4e06d1ad3d54d04759fd0ba9e3e1b451580a40e90c1bb8cd803715610aef",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-901-_jpg.rf.e0ec52c788e571de959869afb5169316.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-901-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 36,\n      "group": 1075,\n      "height": 194,\n      "image_sha256": "7725c46c2bd65a37e6ee851081ed14b0d70b5643b7e0733d2951d4a9a263baf8",\n      "label_sha256": "9491f930188913a04d0902c87f5ebcad91dbbd7368d9ff1ecfa10fb55e44f656",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-903-_jpg.rf.dcaea757854ebd773ef27897a692af7a.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-903-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 3,\n      "group": 1076,\n      "height": 360,\n      "image_sha256": "cfac1864ec3bb94ed1da9742d84598c360afd21b678fcfd8b26c9cc74c0888cf",\n      "label_sha256": "cfff67200d2b56406d28e1df5147c5b372b0bdb781b136551c5e73e81ab95a43",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-945-_jpg.rf.c40d9a0b8380fc96418051cfb5f7c69b.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-945-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 1077,\n      "height": 408,\n      "image_sha256": "0e090007b753730d1230db09d111ea571a38ebe98f83654d06175af71bdbfa0f",\n      "label_sha256": "e75c1471df416bc31d62c6b933e1c05bc9818e3c65fbe00c0a49374703febef1",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-955-_jpg.rf.5fd6fff73ad6f7d038f54d05fe0d542b.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-955-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 3,\n      "group": 1078,\n      "height": 204,\n      "image_sha256": "02d1e49fd150a651dd0271874c2c69cfa916fda0020491adfbd3bf255191f299",\n      "label_sha256": "51e2ce4eb20ea25bc42c89829abae0047d289d0d1960891cffbda6528c01dbdf",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-967-_jpg.rf.c96cb56923d92c29764a1e3366410592.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-967-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 10,\n      "group": 1079,\n      "height": 471,\n      "image_sha256": "488e1bd46e07be5506fa7b5ac78a73fb7fe436eba615918d4bc41b0c87d4a433",\n      "label_sha256": "4d68ea5b43c5bad039d8ddbc7ba73953d73c4e0b58e4cb02220dd53957123d76",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-977-_jpg.rf.238b2bb56f00cb023f7cb401611a98e1.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-977-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 16,\n      "group": 1080,\n      "height": 372,\n      "image_sha256": "333883b744c7277cea7f6a15a909d16f5e9fea462977ccc6c0bc7ecb8bab60c5",\n      "label_sha256": "38453f39aedd549ad21008a267564cd8959cdf444536fb8f93a8ceb28c86b74f",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-986-_jpg.rf.17e00c0a5b34b1453f64e0f105b657a8.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-986-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 12,\n      "group": 1081,\n      "height": 459,\n      "image_sha256": "0b8cc96d9395060148e179d860ac2516a31a51742b397cc2e7cb7e8abc962a26",\n      "label_sha256": "9e19ca3fd436eee26e9463c39e7eee76141b46ec183a54c97b2a873ba78c7084",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-994-_jpg.rf.5000c1a92538809f6be9beb487141d49.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-994-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 5,\n      "group": 1082,\n      "height": 408,\n      "image_sha256": "52fc9076c37e110a1d3ea3e052332038d528a8ec09116100a9b85a514a1c6d14",\n      "label_sha256": "c5122559ba3e8cd3cda01fce867b0a304e3e99cfb8dc0911cc7ee7df3f1e2e39",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-995-_jpg.rf.8896435f639954602a96e5d1a40869f0.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-995-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 15,\n      "group": 1083,\n      "height": 408,\n      "image_sha256": "f44dab35255fbf6295bd6754a8402e9ef4e70715feb7607428bdfa1b38d79d24",\n      "label_sha256": "8c161296e30c813dfbaf8fe4a90755eaef1421eac65b942a5a211a6c1c71e447",\n      "name": "calculus_oral-disease.v1i.yolov8_calculus-997-_jpg.rf.04a905cf0545aece2935598d280f0572.jpg",\n      "original_split": "val",\n      "source_key": "calculus_oral-disease.v1i.yolov8_calculus-997-_jpg",\n      "split": "val",\n      "width": 612\n    },\n    {\n      "boxes": 9,\n      "group": 1084,\n      "height": 640,\n      "image_sha256": "3e7d1281c61409d780a7c07936c0e9d5182283eab82f0557aa30c9921a7e46fc",\n      "label_sha256": "1c419e97d6a9965c33dc2663462e059b0626185aa3df8d8a21688d442d7142e5",\n      "name": "calculus_poor-peiro-2-_jpg.rf.48bbf5851bd3f8824f1bf19525d304c8.jpg",\n      "original_split": "val",\n      "source_key": "calculus_poor-peiro-2-_jpg",\n      "split": "val",\n      "width": 640\n    }\n  ],\n  "source": "singh868/multi-class-oral-disease-detection-dataset/1"\n}'
assert hashlib.sha256(payload.encode('utf-8')).hexdigest() == '831e8768df7ed11d7608b008297380a2014be4a1badeb56c21a1d0f70e828a50'
manifest = json.loads(payload)
Path('/kaggle/working/grouped_manifest.json').write_text(payload)
print('Manifest:', '831e8768df7ed11d7608b008297380a2014be4a1badeb56c21a1d0f70e828a50')
print('Visible GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available()
display(pd.DataFrame(manifest['records']).groupby('split').agg(images=('name','size'), boxes=('boxes','sum')))

## 3. Stage verified images and labels
Resolve each original train/valid file and check both SHA-256 values. Retain calculus rows only. Excluded images are never staged. Source test is never read.

In [ ]:
candidates = list(Path('/kaggle/input').rglob('data.yaml'))
assert len(candidates) == 1, candidates
source = candidates[0].parent
assert '0: calculus' in candidates[0].read_text().lower()
target = Path('/tmp/calculus_grouped_v1')
for row in manifest['records']:
    if row['split'] == 'excluded': continue
    original_split = 'valid' if row['original_split'] == 'val' else 'train'
    image = source / original_split / 'images' / row['name']
    label = source / original_split / 'labels' / (image.stem + '.txt')
    assert hashlib.sha256(image.read_bytes()).hexdigest() == row['image_sha256'], image.name
    rows = [line for line in label.read_text().splitlines() if line.split() and float(line.split()[0]) == 0]
    label_bytes = ('\n'.join(rows) + '\n').encode()
    assert hashlib.sha256(label_bytes).hexdigest() == row['label_sha256'], label.name
    for kind in ['images', 'labels']: (target / kind / row['split']).mkdir(parents=True, exist_ok=True)
    shutil.copyfile(image, target / 'images' / row['split'] / image.name)
    (target / 'labels' / row['split'] / label.name).write_bytes(label_bytes)
for split in ['train', 'val']:
    actual = len(list((target / 'images' / split).iterdir()))
    assert actual == sum(r['split'] == split for r in manifest['records'])
    print(split, actual)
(target / 'data.yaml').write_text(f'path: {target}\ntrain: images/train\nval: images/val\nnames:\n  0: calculus\n')

## 4. Visualize annotations and image transformation
Show a retained image, its boxes, and aspect-preserving letterbox geometry at 640. During training YOLO applies its own matched augmentation and converts pixels to floating point in [0,1].

In [ ]:
from ultralytics.data.augment import LetterBox
import numpy as np
image_path = sorted((target / 'images/train').iterdir())[0]
im = Image.open(image_path).convert('RGB')
draw = ImageDraw.Draw(im)
for line in (target / 'labels/train' / (image_path.stem + '.txt')).read_text().splitlines():
    _, x, y, w, h = map(float, line.split())
    draw.rectangle(((x-w/2)*im.width, (y-h/2)*im.height, (x+w/2)*im.width, (y+h/2)*im.height), outline='red', width=2)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(im); axes[0].set_title('Original with verified boxes')
axes[1].imshow(LetterBox((640,640))(image=np.asarray(im))); axes[1].set_title('Aspect-preserving 640 letterbox')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Train the matched baseline
Only model scale differs between the two notebooks. Fixed 60 epochs, patience disabled, AdamW, batch 16, seed 42, matched augmentation and deterministic request. A five-hour wall-time callback saves last.pt and stops at an epoch boundary. Checkpoints are saved every 10 epochs for manual continuation. Kaggle outputs must be collected before a fresh session.

In [ ]:
model = YOLO('yolov8n.pt')
start = time.monotonic()
def bounded_training(trainer):
    if time.monotonic() - start > 5 * 3600:
        trainer.stop = True
        print('Five-hour training guard reached; retain last.pt for continuation')
model.add_callback('on_train_epoch_end', bounded_training)
results = model.train(data=str(target / 'data.yaml'), epochs=60, patience=0,
    imgsz=640, batch=16, device=0, workers=2, seed=42, deterministic=True,
    optimizer='AdamW', lr0=0.001, weight_decay=0.0005,
    mosaic=1.0, close_mosaic=10, mixup=0.0, degrees=0.0, translate=0.1,
    scale=0.5, fliplr=0.5, flipud=0.0, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    project='/kaggle/working/runs', name='calculus_grouped_v8n', exist_ok=False,
    save=True, save_period=10, plots=True, amp=True)
print(results.results_dict)

## 6. Show learning curves and selected-checkpoint evidence
Best is selected by validation fitness. Save reported metrics and checkpoint hashes. These are source localization results on positive images, with no negative-image specificity estimate.

In [ ]:
run = Path(results.save_dir)
history = pd.read_csv(run / 'results.csv')
history.columns = history.columns.str.strip()
display(history.tail())
history.plot(x='epoch', y=['metrics/mAP50(B)', 'metrics/mAP50-95(B)'], figsize=(10,4), grid=True)
plt.title('Grouped validation localization'); plt.tight_layout(); plt.show()
best = run / 'weights/best.pt'
summary = dict(model='yolov8n', manifest_sha256='831e8768df7ed11d7608b008297380a2014be4a1badeb56c21a1d0f70e828a50',
    completed_epochs=len(history), best_sha256=hashlib.sha256(best.read_bytes()).hexdigest(),
    metrics=results.results_dict)
(Path('/kaggle/working') / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps(summary, indent=2, default=str))
for filename in ['results.png', 'val_batch0_pred.jpg']:
    path = run / filename
    if path.exists():
        display(Image.open(path))